<a href="https://colab.research.google.com/github/Anjlika27/amazon_ml_business_entity_resolution/blob/main/Amazon_ML_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
import os
print(os.listdir("/content"))

In [ ]:
import pandas as pd

train_s1 = pd.read_csv("/content/train_source1.tsv", sep="\t")
train_s2 = pd.read_csv("/content/train_source2.tsv", sep="\t")
train_s3 = pd.read_csv("/content/train_source3.tsv", sep="\t")
ground_truth = pd.read_csv("/content/train_ground_truth.tsv", sep="\t")

print("Source 1:", train_s1.shape)
print("Source 2:", train_s2.shape)
print("Source 3:", train_s3.shape)
print("Ground Truth:", ground_truth.shape)

In [ ]:
# Make sure empty values are treated as no match
gt_matches = (
    ground_truth["matched_entity_ids"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Number of matches per Source 1 entity
num_matches = gt_matches.where(
    gt_matches != "",
    ""
).apply(
    lambda x: 0 if x == "" else len(x.split(","))
)

print("Total Source 1 entities:", len(ground_truth))

print("\nMatch count distribution:")
print(num_matches.value_counts().sort_index())

print("\nSingletons:", (num_matches == 0).sum())
print("One match :", (num_matches == 1).sum())
print("Multiple  :", (num_matches > 1).sum())

print("\nMaximum matches for one Source 1 entity:", num_matches.max())

print("\nPercentage:")
print("Singletons:", round((num_matches == 0).mean() * 100, 2), "%")
print("One match :", round((num_matches == 1).mean() * 100, 2), "%")
print("Multiple  :", round((num_matches > 1).mean() * 100, 2), "%")

In [ ]:
import pandas as pd

def split_matches(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    return [v.strip() for v in str(x).split(",") if v.strip()]


# Count S2 and S3 matches for every S1 entity
def count_sources(x):
    matches = split_matches(x)

    s2_count = sum(m.startswith("S2-") for m in matches)
    s3_count = sum(m.startswith("S3-") for m in matches)

    return pd.Series({
        "s2_count": s2_count,
        "s3_count": s3_count
    })


source_counts = ground_truth["matched_entity_ids"].apply(count_sources)

print("Total matches from Source 2:", source_counts["s2_count"].sum())
print("Total matches from Source 3:", source_counts["s3_count"].sum())

print("\nAverage S2 matches per S1:",
      source_counts["s2_count"].mean())

print("Average S3 matches per S1:",
      source_counts["s3_count"].mean())


# How many S1 entities have S2/S3 matches?
has_s2 = source_counts["s2_count"] > 0
has_s3 = source_counts["s3_count"] > 0

print("\nS1 entities with at least one S2 match:",
      has_s2.sum())

print("S1 entities with at least one S3 match:",
      has_s3.sum())

print("S1 entities with BOTH S2 and S3 matches:",
      (has_s2 & has_s3).sum())

print("S1 entities with ONLY S2 matches:",
      (has_s2 & ~has_s3).sum())

print("S1 entities with ONLY S3 matches:",
      (~has_s2 & has_s3).sum())

print("S1 entities with NO matches:",
      (~has_s2 & ~has_s3).sum())

In [ ]:
import pandas as pd

ground_truth = pd.read_csv(
    "/content/train_ground_truth.tsv",
    sep="\t"
)

print(ground_truth.shape)
ground_truth.head()

In [ ]:
matches = ground_truth["matched_entity_ids"].fillna("")

s2_count = matches.str.count("S2-")
s3_count = matches.str.count("S3-")

has_s2 = s2_count > 0
has_s3 = s3_count > 0

print("Total S2 matches:", s2_count.sum())
print("Total S3 matches:", s3_count.sum())

print("Average S2 matches per S1:", s2_count.mean())
print("Average S3 matches per S1:", s3_count.mean())

print("S1 entities with at least one S2 match:", has_s2.sum())
print("S1 entities with at least one S3 match:", has_s3.sum())

print("S1 entities with BOTH S2 and S3 matches:",
      (has_s2 & has_s3).sum())

print("S1 entities with ONLY S2 matches:",
      (has_s2 & ~has_s3).sum())

print("S1 entities with ONLY S3 matches:",
      (~has_s2 & has_s3).sum())

print("S1 entities with NO matches:",
      (~has_s2 & ~has_s3).sum())

In [ ]:
import pandas as pd

# Select 50 S1 entities that actually have at least one match
matched_gt = ground_truth[
    ground_truth["matched_entity_ids"].fillna("").str.strip() != ""
].sample(n=50, random_state=42)

# Get the S1 IDs
sample_s1_ids = set(matched_gt["source1_entity_id"])

# Extract all S2/S3 IDs associated with these 50 S1 entities
matched_ids = set()

for value in matched_gt["matched_entity_ids"]:
    if pd.notna(value):
        for entity_id in str(value).split(","):
            entity_id = entity_id.strip()
            if entity_id:
                matched_ids.add(entity_id)

s2_ids = {x for x in matched_ids if x.startswith("S2-")}
s3_ids = {x for x in matched_ids if x.startswith("S3-")}

print("Sample S1 entities:", len(sample_s1_ids))
print("S2 IDs to retrieve:", len(s2_ids))
print("S3 IDs to retrieve:", len(s3_ids))

In [ ]:
def get_rows_by_ids(filepath, wanted_ids, chunksize=200_000):
    results = []

    for chunk in pd.read_csv(
        filepath,
        sep="\t",
        chunksize=chunksize,
        dtype=str
    ):
        found = chunk[chunk["entity_id"].isin(wanted_ids)]

        if len(found) > 0:
            results.append(found)

    if results:
        return pd.concat(results, ignore_index=True)

    return pd.DataFrame()


sample_s2 = get_rows_by_ids(
    "/content/train_source2.tsv",
    s2_ids
)

print("Retrieved S2 rows:", len(sample_s2))
display(sample_s2.head())

In [ ]:
sample_s3 = get_rows_by_ids(
    "/content/train_source3.tsv",
    s3_ids
)

print("Retrieved S3 rows:", len(sample_s3))
display(sample_s3.head())

In [ ]:
sample_s1 = get_rows_by_ids(
    "/content/train_source1.tsv",
    sample_s1_ids
)

print("Retrieved S1 rows:", len(sample_s1))
display(sample_s1.head(10))

In [ ]:
# Build S1 -> matched entity relationships
relationships = []

for _, row in matched_gt.iterrows():
    s1_id = row["source1_entity_id"]

    if pd.notna(row["matched_entity_ids"]):
        for entity_id in str(row["matched_entity_ids"]).split(","):
            entity_id = entity_id.strip()

            if entity_id:
                relationships.append({
                    "source1_entity_id": s1_id,
                    "matched_entity_id": entity_id
                })

relationships = pd.DataFrame(relationships)

# Identify whether the match comes from S2 or S3
relationships["source"] = relationships["matched_entity_id"].str[:2]

print("Total sampled relationships:", len(relationships))
print("\nRelationships by source:")
print(relationships["source"].value_counts())

display(relationships.head(20))

In [ ]:
s1_lookup = sample_s1.rename(columns={
    "entity_id": "source1_entity_id",
    "business_name": "s1_name",
    "business_address": "s1_address",
    "country": "s1_country"
})[
    ["source1_entity_id", "s1_name", "s1_address", "s1_country"]
]

s2_lookup = sample_s2.rename(columns={
    "entity_id": "matched_entity_id",
    "business_name": "matched_name",
    "business_address": "matched_address",
    "country": "matched_country"
})[
    ["matched_entity_id", "matched_name", "matched_address", "matched_country"]
]

s3_lookup = sample_s3.rename(columns={
    "entity_id": "matched_entity_id",
    "business_name": "matched_name",
    "business_address": "matched_address",
    "country": "matched_country"
})[
    ["matched_entity_id", "matched_name", "matched_address", "matched_country"]
]

source_lookup = pd.concat(
    [s2_lookup, s3_lookup],
    ignore_index=True
)

matched_examples = (
    relationships
    .merge(s1_lookup, on="source1_entity_id", how="left")
    .merge(source_lookup, on="matched_entity_id", how="left")
)

print("Matched examples:", len(matched_examples))

display(
    matched_examples[
        [
            "source1_entity_id",
            "s1_name",
            "s1_address",
            "s1_country",
            "source",
            "matched_entity_id",
            "matched_name",
            "matched_address",
            "matched_country"
        ]
    ].head(30)
)

In [ ]:
# Basic quality analysis of the 188 genuine matches

df = matched_examples.copy()

# Convert fields to strings for analysis
df["s1_name_str"] = df["s1_name"].fillna("").astype(str)
df["matched_name_str"] = df["matched_name"].fillna("").astype(str)

df["s1_address_str"] = df["s1_address"].fillna("").astype(str)
df["matched_address_str"] = df["matched_address"].fillna("").astype(str)

print("Total genuine pairs:", len(df))

print("\n--- Missing values ---")

print(
    "S1 names missing:",
    (df["s1_name_str"].str.strip() == "").sum()
)

print(
    "Matched names missing:",
    (df["matched_name_str"].str.strip() == "").sum()
)

print(
    "S1 addresses missing:",
    (df["s1_address_str"].str.strip() == "").sum()
)

print(
    "Matched addresses missing:",
    (df["matched_address_str"].str.strip() == "").sum()
)

print("\n--- Country agreement ---")

print(
    "Same country:",
    (df["s1_country"] == df["matched_country"]).sum()
)

print(
    "Different country:",
    (df["s1_country"] != df["matched_country"]).sum()
)

print("\n--- Name exact equality before normalization ---")

print(
    "Exact same name:",
    (
        df["s1_name_str"].str.lower().str.strip()
        ==
        df["matched_name_str"].str.lower().str.strip()
    ).sum()
)

print("\n--- Address exact equality before normalization ---")

print(
    "Exact same address:",
    (
        df["s1_address_str"].str.lower().str.strip()
        ==
        df["matched_address_str"].str.lower().str.strip()
    ).sum()
)

In [ ]:
import re
import unicodedata

def normalize_text(text):
    """
    Conservative normalization:
    - Unicode normalization
    - lowercase/casefold
    - punctuation -> spaces
    - & -> and
    - normalize whitespace
    """
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Case-insensitive normalization
    text = text.casefold()

    # & -> and
    text = text.replace("&", " and ")

    # Replace punctuation/symbols with spaces.
    # Keep letters/numbers from all scripts.
    cleaned = []

    for ch in text:
        category = unicodedata.category(ch)

        if category.startswith(("L", "N")):
            cleaned.append(ch)
        else:
            cleaned.append(" ")

    text = "".join(cleaned)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Test a few examples
examples = [
    "Davis Family Office",
    "DAVIS FAMILY OFFICE",
    "Pvt. Ltd.",
    "Private Limited",
    "A Cure 2 All PLLC",
    "Team Air Pvt. Ltd.",
    "Team Air Pvt. Limited",
    "88 Olive Circle, Lebanon, TN",
    "88 OLIVE CIR, LEBANON, TN"
]

for x in examples:
    print(f"{x:40} -> {normalize_text(x)}")

In [ ]:
df = matched_examples.copy()

df["s1_name_norm"] = df["s1_name"].apply(normalize_text)
df["matched_name_norm"] = df["matched_name"].apply(normalize_text)

df["s1_address_norm"] = df["s1_address"].apply(normalize_text)
df["matched_address_norm"] = df["matched_address"].apply(normalize_text)

name_exact_norm = (
    df["s1_name_norm"] == df["matched_name_norm"]
)

address_exact_norm = (
    df["s1_address_norm"] == df["matched_address_norm"]
)

print("Total genuine pairs:", len(df))

print(
    "Names exactly equal after base normalization:",
    name_exact_norm.sum(),
    f"({name_exact_norm.mean()*100:.1f}%)"
)

print(
    "Addresses exactly equal after base normalization:",
    address_exact_norm.sum(),
    f"({address_exact_norm.mean()*100:.1f}%)"
)

print(
    "Both name AND address exactly equal:",
    (name_exact_norm & address_exact_norm).sum()
)

In [ ]:
def normalize_address(text):
    if pd.isna(text):
        return ""

    text = normalize_text(text)

    # Common street/address abbreviations
    replacements = {
        "street": "st",
        "st": "st",
        "road": "rd",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "highway": "hwy",
        "parkway": "pkwy",
        "place": "pl",
        "court": "ct",
        "circle": "cir",
        "terrace": "ter",
        "way": "way",
        "suite": "ste",
        "floor": "fl",
        "apartment": "apt",
    }

    # Replace whole words only
    for old, new in replacements.items():
        text = re.sub(
            rf"\b{re.escape(old)}\b",
            new,
            text
        )

    # US state name -> abbreviation
    states = {
        "alabama": "al",
        "alaska": "ak",
        "arizona": "az",
        "arkansas": "ar",
        "california": "ca",
        "colorado": "co",
        "connecticut": "ct",
        "delaware": "de",
        "florida": "fl",
        "georgia": "ga",
        "illinois": "il",
        "indiana": "in",
        "maryland": "md",
        "massachusetts": "ma",
        "michigan": "mi",
        "minnesota": "mn",
        "mississippi": "ms",
        "missouri": "mo",
        "nebraska": "ne",
        "nevada": "nv",
        "new jersey": "nj",
        "new mexico": "nm",
        "new york": "ny",
        "north carolina": "nc",
        "ohio": "oh",
        "oklahoma": "ok",
        "oregon": "or",
        "pennsylvania": "pa",
        "south carolina": "sc",
        "tennessee": "tn",
        "texas": "tx",
        "virginia": "va",
        "washington": "wa",
        "wisconsin": "wi",
    }

    for old, new in states.items():
        text = re.sub(
            rf"\b{re.escape(old)}\b",
            new,
            text
        )

    text = re.sub(r"\s+", " ", text).strip()

    return text


# Test
address_examples = [
    "88 Olive Circle, Lebanon, TN",
    "88 OLIVE CIR, LEBANON, TN",
    "Lebanon, Tennessee, 88 Olive Cir",
    "120 Autumn Woods Boulevard, Mount Holly, NC",
    "120. Autumn Woods Blvd, MOUNT HOLLY, NC",
]

for x in address_examples:
    print(f"{x:65} -> {normalize_address(x)}")

In [ ]:
df["s1_address_norm_v2"] = df["s1_address"].apply(normalize_address)
df["matched_address_norm_v2"] = df["matched_address"].apply(normalize_address)

address_exact_v2 = (
    df["s1_address_norm_v2"] ==
    df["matched_address_norm_v2"]
)

print(
    "Addresses exactly equal after address normalization:",
    address_exact_v2.sum(),
    f"({address_exact_v2.mean()*100:.1f}%)"
)

print(
    "Improvement over base normalization:",
    address_exact_v2.sum() - address_exact_norm.sum(),
    "additional exact matches"
)

In [ ]:
!pip -q install rapidfuzz

In [ ]:
from rapidfuzz import fuzz

# Name similarities
df["name_ratio"] = df.apply(
    lambda r: fuzz.ratio(
        r["s1_name_norm"],
        r["matched_name_norm"]
    ),
    axis=1
)

df["name_token_ratio"] = df.apply(
    lambda r: fuzz.token_set_ratio(
        r["s1_name_norm"],
        r["matched_name_norm"]
    ),
    axis=1
)

# Address similarities
df["address_ratio"] = df.apply(
    lambda r: fuzz.ratio(
        r["s1_address_norm_v2"],
        r["matched_address_norm_v2"]
    ),
    axis=1
)

df["address_token_ratio"] = df.apply(
    lambda r: fuzz.token_set_ratio(
        r["s1_address_norm_v2"],
        r["matched_address_norm_v2"]
    ),
    axis=1
)

print("Name ratio:")
print(df["name_ratio"].describe())

print("\nName token-set ratio:")
print(df["name_token_ratio"].describe())

print("\nAddress ratio:")
print(df["address_ratio"].describe())

print("\nAddress token-set ratio:")
print(df["address_token_ratio"].describe())

In [ ]:
display(
    df[
        [
            "s1_name",
            "matched_name",
            "name_ratio",
            "name_token_ratio",
            "s1_address",
            "matched_address",
            "address_ratio",
            "address_token_ratio",
            "source"
        ]
    ]
    .sort_values(
        ["name_ratio", "address_token_ratio"]
    )
    .head(20)
)

In [ ]:
# Create all S1 × sampled S2/S3 candidate pairs

s1_small = sample_s1[
    ["entity_id", "business_name", "business_address", "country"]
].copy()

source_small = pd.concat(
    [
        sample_s2.assign(source="S2"),
        sample_s3.assign(source="S3")
    ],
    ignore_index=True
)

source_small = source_small[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "source"
    ]
].copy()

# Cartesian product
s1_small["_key"] = 1
source_small["_key"] = 1

candidate_sample = s1_small.merge(
    source_small,
    on="_key",
    suffixes=("_s1", "_matched")
).drop(columns="_key")

print("Candidate pairs:", len(candidate_sample))

In [ ]:
# Create set of genuine relationships
positive_pairs = set(
    zip(
        relationships["source1_entity_id"],
        relationships["matched_entity_id"]
    )
)

candidate_sample["is_match"] = [
    (s1_id, matched_id) in positive_pairs
    for s1_id, matched_id in zip(
        candidate_sample["entity_id_s1"],
        candidate_sample["entity_id_matched"]
    )
]

print("Positive pairs:", candidate_sample["is_match"].sum())
print("Negative pairs:", (~candidate_sample["is_match"]).sum())

In [ ]:
candidate_sample["s1_name_norm"] = (
    candidate_sample["business_name_s1"]
    .apply(normalize_text)
)

candidate_sample["matched_name_norm"] = (
    candidate_sample["business_name_matched"]
    .apply(normalize_text)
)

candidate_sample["s1_address_norm"] = (
    candidate_sample["business_address_s1"]
    .apply(normalize_address)
)

candidate_sample["matched_address_norm"] = (
    candidate_sample["business_address_matched"]
    .apply(normalize_address)
)

candidate_sample["name_ratio"] = candidate_sample.apply(
    lambda r: fuzz.ratio(
        r["s1_name_norm"],
        r["matched_name_norm"]
    ),
    axis=1
)

candidate_sample["name_token_ratio"] = candidate_sample.apply(
    lambda r: fuzz.token_set_ratio(
        r["s1_name_norm"],
        r["matched_name_norm"]
    ),
    axis=1
)

candidate_sample["address_ratio"] = candidate_sample.apply(
    lambda r: fuzz.ratio(
        r["s1_address_norm"],
        r["matched_address_norm"]
    ),
    axis=1
)

candidate_sample["address_token_ratio"] = candidate_sample.apply(
    lambda r: fuzz.token_set_ratio(
        r["s1_address_norm"],
        r["matched_address_norm"]
    ),
    axis=1
)

candidate_sample["country_same"] = (
    candidate_sample["country_s1"]
    ==
    candidate_sample["country_matched"]
)

print("Feature calculation complete.")

In [ ]:
features = [
    "name_ratio",
    "name_token_ratio",
    "address_ratio",
    "address_token_ratio"
]

for feature in features:
    print("\n" + "=" * 60)
    print(feature)

    print("\nGENUINE MATCHES:")
    print(
        candidate_sample.loc[
            candidate_sample["is_match"],
            feature
        ].describe()
    )

    print("\nNON-MATCHES:")
    print(
        candidate_sample.loc[
            ~candidate_sample["is_match"],
            feature
        ].describe()
    )

In [ ]:
# Look at the most difficult negative pairs

negative_pairs = candidate_sample[
    ~candidate_sample["is_match"]
].copy()

# Create a simple inspection score.
# This is NOT our final matching score.
negative_pairs["inspection_score"] = (
    0.30 * negative_pairs["name_token_ratio"] +
    0.20 * negative_pairs["name_ratio"] +
    0.30 * negative_pairs["address_token_ratio"] +
    0.20 * negative_pairs["address_ratio"]
)

hard_negatives = negative_pairs.sort_values(
    "inspection_score",
    ascending=False
)

print("Number of negative pairs:", len(hard_negatives))

display(
    hard_negatives[
        [
            "entity_id_s1",
            "business_name_s1",
            "business_address_s1",
            "entity_id_matched",
            "business_name_matched",
            "business_address_matched",
            "country_s1",
            "source",
            "name_ratio",
            "name_token_ratio",
            "address_ratio",
            "address_token_ratio",
            "inspection_score"
        ]
    ].head(30)
)

In [ ]:
!pip -q install scikit-learn

In [ ]:
import sklearn

print("scikit-learn version:", sklearn.__version__)

In [ ]:
print("Available variables:")

for name in [
    "source1",
    "source2",
    "source3",
    "ground_truth",
    "candidate_sample",
    "sample_s1",
    "sample_s2",
    "sample_s3"
]:
    obj = globals().get(name)

    if obj is None:
        print(f"{name} -> NOT FOUND")
    else:
        print(f"{name} -> {type(obj).__name__}", end="")

        if hasattr(obj, "shape"):
            print(f" | shape = {obj.shape}")
        else:
            print()

In [ ]:
import os

files = [
    "/content/train_source1.tsv",
    "/content/train_source2.tsv",
    "/content/train_source3.tsv",
    "/content/train_ground_truth.tsv"
]

for f in files:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"{os.path.basename(f)} -> FOUND ({size_mb:.1f} MB)")
    else:
        print(f"{os.path.basename(f)} -> NOT FOUND")

In [ ]:
import pandas as pd
import numpy as np

# Reproducible sample of 500 S1 entities
np.random.seed(42)

training_s1 = ground_truth.sample(
    n=500,
    random_state=42
).copy()

print("Training S1 entities:", len(training_s1))

# Convert ground-truth match lists into individual IDs
def split_ids(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    return [v.strip() for v in str(x).split(",") if v.strip()]

training_s1["matched_ids"] = training_s1["matched_entity_ids"].apply(split_ids)

# Flatten all positive IDs
positive_ids = [
    match_id
    for matches in training_s1["matched_ids"]
    for match_id in matches
]

s2_positive_ids = {
    x for x in positive_ids if x.startswith("S2-")
}

s3_positive_ids = {
    x for x in positive_ids if x.startswith("S3-")
}

print("S2 positive IDs:", len(s2_positive_ids))
print("S3 positive IDs:", len(s3_positive_ids))
print("Total positive IDs:", len(positive_ids))

print("\nSample ground-truth rows:")
display(training_s1.head())

In [ ]:
# STEP 13 — Retrieve positive S2/S3 records without loading full files

import pandas as pd

def retrieve_ids_chunked(file_path, target_ids, chunksize=100_000):
    target_ids = set(target_ids)
    found_chunks = []

    print(f"\nReading: {file_path}")
    print(f"Target IDs: {len(target_ids)}")

    for chunk_no, chunk in enumerate(
        pd.read_csv(file_path, sep="\t", chunksize=chunksize),
        start=1
    ):
        found = chunk[chunk["entity_id"].isin(target_ids)]

        if len(found) > 0:
            found_chunks.append(found)

        if chunk_no % 10 == 0:
            print(f"  Processed {chunk_no} chunks...")

    if found_chunks:
        result = pd.concat(found_chunks, ignore_index=True)
    else:
        result = pd.DataFrame(
            columns=["entity_id", "business_name", "business_address", "country"]
        )

    return result


# Retrieve S2 positives
positive_s2 = retrieve_ids_chunked(
    "/content/train_source2.tsv",
    s2_positive_ids
)

print("\nRetrieved S2 positive rows:", len(positive_s2))


# Retrieve S3 positives
positive_s3 = retrieve_ids_chunked(
    "/content/train_source3.tsv",
    s3_positive_ids
)

print("Retrieved S3 positive rows:", len(positive_s3))


print("\nS2 positive records:")
display(positive_s2.head())

print("\nS3 positive records:")
display(positive_s3.head())

In [ ]:
# STEP 14 — Retrieve the 500 sampled S1 records

training_s1_ids = set(training_s1["source1_entity_id"])

training_s1_records = retrieve_ids_chunked(
    "/content/train_source1.tsv",
    training_s1_ids
)

print("\nRetrieved S1 rows:", len(training_s1_records))

display(training_s1_records.head())

In [ ]:
# STEP 15 — Build positive training pairs

# Rename S1 columns
s1_pos = training_s1_records.rename(columns={
    "entity_id": "entity_id_s1",
    "business_name": "business_name_s1",
    "business_address": "business_address_s1",
    "country": "country_s1"
}).copy()

# Prepare S2 records
s2_pos = positive_s2.rename(columns={
    "entity_id": "entity_id_matched",
    "business_name": "business_name_matched",
    "business_address": "business_address_matched",
    "country": "country_matched"
}).copy()

# Prepare S3 records
s3_pos = positive_s3.rename(columns={
    "entity_id": "entity_id_matched",
    "business_name": "business_name_matched",
    "business_address": "business_address_matched",
    "country": "country_matched"
}).copy()

# Combine S2 + S3
matched_records = pd.concat([s2_pos, s3_pos], ignore_index=True)

# Join S1 with its known matched IDs
positive_pairs = training_s1_records.merge(
    training_s1[["source1_entity_id", "matched_ids"]],
    left_on="entity_id",
    right_on="source1_entity_id",
    how="inner"
)

# One row per matched ID
positive_pairs = positive_pairs.explode("matched_ids")

# Rename
positive_pairs = positive_pairs.rename(columns={
    "entity_id": "entity_id_s1",
    "business_name": "business_name_s1",
    "business_address": "business_address_s1",
    "country": "country_s1",
    "matched_ids": "entity_id_matched"
})

# Attach matched S2/S3 record
positive_pairs = positive_pairs.merge(
    matched_records,
    on="entity_id_matched",
    how="inner"
)

# Add label
positive_pairs["label"] = 1

print("Positive pairs:", len(positive_pairs))
print("\nSource distribution:")
print(positive_pairs["entity_id_matched"].str[:2].value_counts())

display(positive_pairs.head(10))

In [ ]:
# STEP 16 — Create hard negative pairs

import numpy as np

# ---------------------------------------------------------
# 1. Prepare candidate records from the sampled S2/S3 data
# ---------------------------------------------------------

negative_candidates = pd.concat([
    sample_s2.assign(source="S2"),
    sample_s3.assign(source="S3")
], ignore_index=True)

negative_candidates = negative_candidates.rename(columns={
    "entity_id": "entity_id_matched",
    "business_name": "business_name_matched",
    "business_address": "business_address_matched",
    "country": "country_matched"
})

# ---------------------------------------------------------
# 2. Prepare S1 records
# ---------------------------------------------------------

s1_for_neg = training_s1_records.rename(columns={
    "entity_id": "entity_id_s1",
    "business_name": "business_name_s1",
    "business_address": "business_address_s1",
    "country": "country_s1"
}).copy()

# ---------------------------------------------------------
# 3. Add normalized fields
# ---------------------------------------------------------

def normalize_basic(x):
    if pd.isna(x):
        return ""
    return (
        str(x)
        .lower()
        .replace("&", " and ")
        .replace("-", " ")
        .replace(",", " ")
        .replace(".", " ")
        .replace("/", " ")
    )

s1_for_neg["name_norm"] = s1_for_neg["business_name_s1"].map(normalize_basic)
s1_for_neg["address_norm"] = s1_for_neg["business_address_s1"].map(normalize_basic)

negative_candidates["name_norm"] = (
    negative_candidates["business_name_matched"].map(normalize_basic)
)

negative_candidates["address_norm"] = (
    negative_candidates["business_address_matched"].map(normalize_basic)
)

# ---------------------------------------------------------
# 4. Build lookup indexes
# ---------------------------------------------------------

name_index = {}

for idx, name in enumerate(negative_candidates["name_norm"]):
    if name:
        name_index.setdefault(name, []).append(idx)

# ---------------------------------------------------------
# 5. Generate hard negatives
# ---------------------------------------------------------

hard_negatives = []

rng = np.random.default_rng(42)

for _, s1 in s1_for_neg.iterrows():

    # Candidates from same country
    candidates = negative_candidates[
        negative_candidates["country_matched"] == s1["country_s1"]
    ]

    if len(candidates) == 0:
        continue

    # Random subset so we don't create a huge Cartesian product
    sample_size = min(20, len(candidates))

    sampled = candidates.sample(
        n=sample_size,
        random_state=int(rng.integers(0, 1_000_000))
    )

    # Known true matches for this S1
    true_ids = set(
        positive_pairs.loc[
            positive_pairs["entity_id_s1"] == s1["entity_id_s1"],
            "entity_id_matched"
        ]
    )

    for _, cand in sampled.iterrows():

        # Never use a genuine match as a negative
        if cand["entity_id_matched"] in true_ids:
            continue

        hard_negatives.append({
            "entity_id_s1": s1["entity_id_s1"],
            "business_name_s1": s1["business_name_s1"],
            "business_address_s1": s1["business_address_s1"],
            "country_s1": s1["country_s1"],

            "entity_id_matched": cand["entity_id_matched"],
            "business_name_matched": cand["business_name_matched"],
            "business_address_matched": cand["business_address_matched"],
            "country_matched": cand["country_matched"],

            "label": 0
        })

hard_negatives = pd.DataFrame(hard_negatives)

print("Hard negative pairs:", len(hard_negatives))

print("\nLabel distribution:")
print(pd.Series({
    "positive": len(positive_pairs),
    "negative": len(hard_negatives)
}))

display(hard_negatives.head(10))

In [ ]:
# STEP 17 — Combine positive and negative pairs

# Keep only the columns needed for model training
training_columns = [
    "entity_id_s1",
    "business_name_s1",
    "business_address_s1",
    "country_s1",
    "entity_id_matched",
    "business_name_matched",
    "business_address_matched",
    "country_matched",
    "label"
]

positive_train = positive_pairs[training_columns].copy()
negative_train = hard_negatives[training_columns].copy()

# Combine
training_pairs = pd.concat(
    [positive_train, negative_train],
    ignore_index=True
)

# Shuffle
training_pairs = training_pairs.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Total training pairs:", len(training_pairs))

print("\nLabel distribution:")
print(training_pairs["label"].value_counts())

print("\nPositive ratio:",
      round(training_pairs["label"].mean() * 100, 2), "%")

print("\nColumns:")
print(training_pairs.columns.tolist())

display(training_pairs.head(10))

In [ ]:
!pip install -q rapidfuzz

In [ ]:
# STEP 18A — Feature engineering

from rapidfuzz.fuzz import ratio, token_set_ratio
import re
import unicodedata
import numpy as np
import pandas as pd

print("RapidFuzz imported successfully.")

In [ ]:
# STEP 18B — Create matching features

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Lowercase
    text = text.lower()

    # Common symbol normalization
    text = text.replace("&", " and ")

    # Remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_address(text):
    text = normalize_text(text)

    # Common address abbreviations
    replacements = {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "highway": "hwy",
        "apartment": "apt",
        "floor": "fl",
        "suite": "ste",
        "parkway": "pkwy",
        "place": "pl",
        "court": "ct",
        "circle": "cir",
        "terrace": "ter",
        "north": "n",
        "south": "s",
        "east": "e",
        "west": "w"
    }

    words = text.split()

    words = [
        replacements.get(word, word)
        for word in words
    ]

    return " ".join(words)


def numeric_tokens(text):
    """
    Extract numeric tokens such as:
    101, 1901, 82, 1/2 etc.
    """
    text = str(text)

    return set(
        re.findall(r"\d+(?:/\d+)?", text)
    )


def calculate_features(row):

    # -------------------------
    # Normalize fields
    # -------------------------

    name1 = normalize_text(row["business_name_s1"])
    name2 = normalize_text(row["business_name_matched"])

    addr1 = normalize_address(row["business_address_s1"])
    addr2 = normalize_address(row["business_address_matched"])

    # -------------------------
    # String similarities
    # -------------------------

    name_ratio = ratio(name1, name2)

    name_token_ratio = token_set_ratio(
        name1,
        name2
    )

    address_ratio = ratio(
        addr1,
        addr2
    )

    address_token_ratio = token_set_ratio(
        addr1,
        addr2
    )

    # -------------------------
    # Exact matches
    # -------------------------

    name_exact = int(name1 == name2 and name1 != "")

    address_exact = int(addr1 == addr2 and addr1 != "")

    # -------------------------
    # Country agreement
    # -------------------------

    country_match = int(
        str(row["country_s1"]).strip().lower()
        ==
        str(row["country_matched"]).strip().lower()
    )

    # -------------------------
    # Length features
    # -------------------------

    name_length_diff = abs(
        len(name1) - len(name2)
    )

    address_length_diff = abs(
        len(addr1) - len(addr2)
    )

    # -------------------------
    # Numeric token overlap
    # -------------------------

    nums1 = numeric_tokens(
        row["business_address_s1"]
    )

    nums2 = numeric_tokens(
        row["business_address_matched"]
    )

    if len(nums1) == 0 and len(nums2) == 0:
        numeric_overlap = 1.0

    elif len(nums1) == 0 or len(nums2) == 0:
        numeric_overlap = 0.0

    else:
        numeric_overlap = (
            len(nums1 & nums2)
            /
            len(nums1 | nums2)
        )

    return pd.Series({
        "name_ratio": name_ratio,
        "name_token_ratio": name_token_ratio,
        "address_ratio": address_ratio,
        "address_token_ratio": address_token_ratio,
        "name_exact": name_exact,
        "address_exact": address_exact,
        "country_match": country_match,
        "name_length_diff": name_length_diff,
        "address_length_diff": address_length_diff,
        "numeric_overlap": numeric_overlap
    })


# Apply feature generation
feature_data = training_pairs.apply(
    calculate_features,
    axis=1
)

# Add labels
feature_data["label"] = training_pairs["label"].values

print("Feature matrix shape:", feature_data.shape)

print("\nFeatures:")
print(feature_data.columns.tolist())

display(feature_data.head(10))

In [ ]:
# STEP 19 — Split by Source 1 entity

from sklearn.model_selection import train_test_split

# Unique S1 entities
unique_s1 = training_pairs["entity_id_s1"].unique()

print("Unique S1 entities:", len(unique_s1))

# 80% entities for training, 20% for validation
train_s1_ids, val_s1_ids = train_test_split(
    unique_s1,
    test_size=0.20,
    random_state=42
)

# Create masks
train_mask = training_pairs["entity_id_s1"].isin(train_s1_ids)
val_mask = training_pairs["entity_id_s1"].isin(val_s1_ids)

# Feature matrices
feature_columns = [
    "name_ratio",
    "name_token_ratio",
    "address_ratio",
    "address_token_ratio",
    "name_exact",
    "address_exact",
    "country_match",
    "name_length_diff",
    "address_length_diff",
    "numeric_overlap"
]

X_train = feature_data.loc[train_mask, feature_columns].copy()
y_train = feature_data.loc[train_mask, "label"].copy()

X_val = feature_data.loc[val_mask, feature_columns].copy()
y_val = feature_data.loc[val_mask, "label"].copy()

print("\nTraining:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nValidation:")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTraining labels:")
print(y_train.value_counts())

print("\nValidation labels:")
print(y_val.value_counts())

In [ ]:
# STEP 20 — Train Random Forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")

rf_model.fit(X_train, y_train)

print("Training completed.")

In [ ]:
# STEP 20B — Validation evaluation

y_val_pred = rf_model.predict(X_val)
y_val_prob = rf_model.predict_proba(X_val)[:, 1]

print("Classification Report:")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=["Non-match", "Match"],
    digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nPrecision:", precision_score(y_val, y_val_pred))
print("Recall:", recall_score(y_val, y_val_pred))
print("F1:", f1_score(y_val, y_val_pred))

In [ ]:
# STEP 21 — F0.5 threshold tuning

from sklearn.metrics import precision_score, recall_score, fbeta_score

thresholds = [
    0.30, 0.40, 0.45, 0.50,
    0.55, 0.60, 0.65, 0.70,
    0.75, 0.80, 0.85, 0.90,
    0.92, 0.94, 0.96, 0.98
]

results = []

for threshold in thresholds:
    y_pred_threshold = (y_val_prob >= threshold).astype(int)

    precision = precision_score(
        y_val,
        y_pred_threshold,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_pred_threshold,
        zero_division=0
    )

    f05 = fbeta_score(
        y_val,
        y_pred_threshold,
        beta=0.5,
        zero_division=0
    )

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f0.5": f05
    })

threshold_results = pd.DataFrame(results)

display(
    threshold_results.sort_values(
        "f0.5",
        ascending=False
    )
)

In [ ]:
# Best threshold according to validation F0.5

best_row = threshold_results.loc[
    threshold_results["f0.5"].idxmax()
]

print("Best threshold:")
print(best_row)

In [ ]:
# ============================================================
# STEP 22A — Prepare blocking keys
# ============================================================

import re
import unicodedata
import pandas as pd
import numpy as np

print("Preparing blocking keys...")

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Lowercase
    text = text.lower()

    # Replace & with and
    text = text.replace("&", " and ")

    # Remove punctuation
    text = re.sub(r"[^a-z0-9\u0900-\u097f\u0c00-\u0c7f]+", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_name(text):
    return normalize_text(text)


def normalize_address(text):
    return normalize_text(text)


def get_first_token(text):
    tokens = text.split()
    return tokens[0] if tokens else ""


def get_numeric_tokens(text):
    return " ".join(re.findall(r"\d+", text))


# ------------------------------------------------------------
# Prepare S1 sample
# ------------------------------------------------------------

blocking_s1 = training_s1_records.copy()

blocking_s1["name_norm"] = (
    blocking_s1["business_name"]
    .apply(normalize_name)
)

blocking_s1["address_norm"] = (
    blocking_s1["business_address"]
    .apply(normalize_address)
)

blocking_s1["country_norm"] = (
    blocking_s1["country"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

# Blocking keys
blocking_s1["name_exact_key"] = (
    blocking_s1["country_norm"]
    + "|"
    + blocking_s1["name_norm"]
)

blocking_s1["name_prefix_key"] = (
    blocking_s1["country_norm"]
    + "|"
    + blocking_s1["name_norm"].str[:6]
)

blocking_s1["address_exact_key"] = (
    blocking_s1["country_norm"]
    + "|"
    + blocking_s1["address_norm"]
)

blocking_s1["address_num_key"] = (
    blocking_s1["country_norm"]
    + "|"
    + blocking_s1["address_norm"].apply(get_numeric_tokens)
)

print("Blocking keys prepared.")
print("S1 records:", len(blocking_s1))

display(
    blocking_s1[
        [
            "entity_id",
            "business_name",
            "name_norm",
            "address_norm",
            "country_norm",
            "name_exact_key",
            "name_prefix_key",
            "address_exact_key",
            "address_num_key"
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# STEP 22B — Blocking candidate retrieval
# ============================================================

SOURCE_FILES = {
    "S2": "/content/train_source2.tsv",
    "S3": "/content/train_source3.tsv"
}

CHUNK_SIZE = 100_000


# ------------------------------------------------------------
# Query keys from our 500 S1 records
# ------------------------------------------------------------

query_name_exact = set(
    blocking_s1["name_exact_key"]
)

query_name_prefix = set(
    blocking_s1["name_prefix_key"]
)

query_address_exact = set(
    blocking_s1["address_exact_key"]
)

query_address_num = set(
    blocking_s1["address_num_key"]
)


# Store candidates separately for each blocking rule
candidate_results = {
    "name_exact": set(),
    "name_prefix": set(),
    "address_exact": set(),
    "address_num": set()
}


def process_source_for_blocking(source_name, file_path):

    print("\n========================================")
    print("Processing", source_name)
    print("========================================")

    chunk_number = 0

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        chunksize=CHUNK_SIZE,
        dtype=str
    ):

        chunk_number += 1

        # Normalize fields
        chunk["name_norm"] = (
            chunk["business_name"]
            .apply(normalize_name)
        )

        chunk["address_norm"] = (
            chunk["business_address"]
            .apply(normalize_address)
        )

        chunk["country_norm"] = (
            chunk["country"]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.strip()
        )

        # Keys
        chunk["name_exact_key"] = (
            chunk["country_norm"]
            + "|"
            + chunk["name_norm"]
        )

        chunk["name_prefix_key"] = (
            chunk["country_norm"]
            + "|"
            + chunk["name_norm"].str[:6]
        )

        chunk["address_exact_key"] = (
            chunk["country_norm"]
            + "|"
            + chunk["address_norm"]
        )

        chunk["address_num_key"] = (
            chunk["country_norm"]
            + "|"
            + chunk["address_norm"].apply(get_numeric_tokens)
        )

        # ----------------------------------------------------
        # Block 1: exact normalized name
        # ----------------------------------------------------

        mask = chunk["name_exact_key"].isin(
            query_name_exact
        )

        if mask.any():
            candidate_results["name_exact"].update(
                chunk.loc[mask, "entity_id"].tolist()
            )

        # ----------------------------------------------------
        # Block 2: name prefix
        # ----------------------------------------------------

        mask = chunk["name_prefix_key"].isin(
            query_name_prefix
        )

        if mask.any():
            candidate_results["name_prefix"].update(
                chunk.loc[mask, "entity_id"].tolist()
            )

        # ----------------------------------------------------
        # Block 3: exact normalized address
        # ----------------------------------------------------

        mask = chunk["address_exact_key"].isin(
            query_address_exact
        )

        if mask.any():
            candidate_results["address_exact"].update(
                chunk.loc[mask, "entity_id"].tolist()
            )

        # ----------------------------------------------------
        # Block 4: numeric address key
        # ----------------------------------------------------

        # Avoid empty numeric keys creating huge blocks
        valid_numeric = chunk["address_num_key"].str.split("|").str[-1] != ""

        mask = (
            valid_numeric
            & chunk["address_num_key"].isin(query_address_num)
        )

        if mask.any():
            candidate_results["address_num"].update(
                chunk.loc[mask, "entity_id"].tolist()
            )

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: processed "
                f"{chunk_number} chunks"
            )

    print(f"{source_name}: done")


# Run for S2 and S3
for source_name, file_path in SOURCE_FILES.items():
    process_source_for_blocking(
        source_name,
        file_path
    )


print("\nBlocking retrieval complete.")


Processing S2
S2: processed 10 chunks
S2: processed 20 chunks
S2: processed 30 chunks
S2: processed 40 chunks
S2: processed 50 chunks
S2: done

Processing S3
S3: processed 10 chunks
S3: processed 20 chunks
S3: processed 30 chunks
S3: processed 40 chunks
S3: processed 50 chunks
S3: done

Blocking retrieval complete.


In [ ]:
print("ground_truth:", "ground_truth" in globals())
print("training_s1_records:", "training_s1_records" in globals())
print("candidate_results:", "candidate_results" in globals())
print("blocking_s1:", "blocking_s1" in globals())
print("feature_data:", "feature_data" in globals())
print("model:", "model" in globals())

ground_truth: True
training_s1_records: True
candidate_results: True
blocking_s1: True
feature_data: True
model: False


In [ ]:
# STEP 23 — Recreate the Random Forest model

from sklearn.ensemble import RandomForestClassifier

# Check that training data still exists
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training model...")

model.fit(X_train, y_train)

print("Model training complete.")

X_train: (9429, 10)
y_train: (9429,)
Training model...
Model training complete.


In [ ]:
print("model:", type(model))
print("Model trained:", hasattr(model, "predict_proba"))

model: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Model trained: True


In [ ]:
# STEP 24 — Score candidate pairs with the trained model

print("Candidate rows:", len(candidate_results))

# Use the same feature columns used during training
feature_columns = [
    'name_ratio',
    'name_token_ratio',
    'address_ratio',
    'address_token_ratio',
    'name_exact',
    'address_exact',
    'country_match',
    'name_length_diff',
    'address_length_diff',
    'numeric_overlap'
]

# Make sure all required features exist
missing_features = [
    col for col in feature_columns
    if col not in candidate_results.columns
]

print("Missing features:", missing_features)

if missing_features:
    raise ValueError(f"Missing required features: {missing_features}")

# Extract features
X_candidates = candidate_results[feature_columns].fillna(0)

print("Candidate feature matrix:", X_candidates.shape)

# Predict match probability
candidate_results["match_probability"] = model.predict_proba(
    X_candidates
)[:, 1]

print("\nScoring complete.")
print(candidate_results["match_probability"].describe())

Candidate rows: 4


AttributeError: 'dict' object has no attribute 'columns'

In [ ]:
# STEP 24A — Inspect candidate_results structure

print("Type:", type(candidate_results))

if isinstance(candidate_results, dict):
    print("Dictionary keys:", candidate_results.keys())

    for key, value in candidate_results.items():
        print(
            f"\nKey: {key}"
            f"\nType: {type(value)}"
            f"\nShape: {getattr(value, 'shape', 'N/A')}"
        )

        if hasattr(value, "columns"):
            print("Columns:")
            print(value.columns.tolist())
            print("\nFirst 2 rows:")
            display(value.head(2))
else:
    print("Shape:", candidate_results.shape)
    print("Columns:")
    print(candidate_results.columns.tolist())

Type: <class 'dict'>
Dictionary keys: dict_keys(['name_exact', 'name_prefix', 'address_exact', 'address_num'])

Key: name_exact
Type: <class 'set'>
Shape: N/A

Key: name_prefix
Type: <class 'set'>
Shape: N/A

Key: address_exact
Type: <class 'set'>
Shape: N/A

Key: address_num
Type: <class 'set'>
Shape: N/A


In [ ]:
# STEP 24B — Find candidate/feature DataFrames

print("Candidate-related variables:\n")

for name, obj in globals().items():
    if any(word in name.lower() for word in ["candidate", "feature", "pair", "blocking"]):
        if name.startswith("_"):
            continue

        print(
            f"{name:25} | "
            f"type = {type(obj).__name__:15} | "
            f"shape = {getattr(obj, 'shape', 'N/A')}"
        )

Candidate-related variables:

candidate_sample          | type = DataFrame       | shape = (9400, 19)
positive_pairs            | type = DataFrame       | shape = (1762, 10)
features                  | type = list            | shape = N/A
feature                   | type = str             | shape = N/A
negative_pairs            | type = DataFrame       | shape = (9212, 20)
negative_candidates       | type = DataFrame       | shape = (188, 7)
candidates                | type = DataFrame       | shape = (58, 7)
training_pairs            | type = DataFrame       | shape = (11762, 9)
calculate_features        | type = function        | shape = N/A
feature_data              | type = DataFrame       | shape = (11762, 11)
feature_columns           | type = list            | shape = N/A
blocking_s1               | type = DataFrame       | shape = (500, 11)
candidate_results         | type = dict            | shape = N/A
process_source_for_blocking | type = function        | shape = N/A


In [ ]:
# STEP 24C — Inspect candidate DataFrames

print("=== candidates ===")
print("Shape:", candidates.shape)
print("Columns:", candidates.columns.tolist())
display(candidates.head(10))

print("\n=== candidate_sample ===")
print("Shape:", candidate_sample.shape)
print("Columns:", candidate_sample.columns.tolist())
display(candidate_sample.head(5))

=== candidates ===
Shape: (58, 7)
Columns: ['entity_id_matched', 'business_name_matched', 'business_address_matched', 'country_matched', 'source', 'name_norm', 'address_norm']


,entity_id_matched,business_name_matched,business_address_matched,country_matched,source,name_norm,address_norm
3,S2-686400156,మై కన్‌స్ట్రక్షన్ ప్రైవేట్ లిమిటెడ్,"1-1-742/3, తెలంగాణ, HYDERABAD",India,S2,మై కన్‌స్ట్రక్షన్ ప్రైవేట్ లిమిటెడ్,1 1 742 3 తెలంగాణ hyderabad
5,S2-176314669,AGRO PRODUCTS,"#634 PLOT NO. X-0037, ANSAL SUSHANT CITY-II BL...",India,S2,agro products,#634 plot no x 0037 ansal sushant city ii bl...
6,S2-942337168,Clairvoyant Récord Private Ltd,"7-1ST FLOOR, TAMRAKARMALL, NEW BUS STAND, SEHO...",India,S2,clairvoyant récord private ltd,7 1st floor tamrakarmall new bus stand seho...
8,S2-901286769,Aarvva Cthit Private Limited,"DIAMOND MOTORS, N.H -8 SUKHER, GIRWA, Rajasthan",India,S2,aarvva cthit private limited,diamond motors n h 8 sukher girwa rajasthan
11,S2-742819060,TEAMAIR.COM,"FLAT NO:101, ANUSKA TOWERS, OPP. MERCEDES BENZ...",India,S2,teamair com,flat no:101 anuska towers opp mercedes benz...
12,S2-959907396,तिरुपति टेक्नोलॉजी प्राइवेट लिमिटेड,"1901, DELPHI TOWERS, 19TH FLOOR, RAHEJA GARDEN...",India,S2,तिरुपति टेक्नोलॉजी प्राइवेट लिमिटेड,1901 delphi towers 19th floor raheja garden...
13,S2-896700091,Sauhard Welfare Society-Ltd,"H.NO 26 & 27, GROUND FLOOR, PARSAVNATH METRO M...",India,S2,sauhard welfare society ltd,h no 26 and 27 ground floor parsavnath met...
16,S2-621477890,Unique & Sons Private [Limited],"PLOT NO.118, 119 SYNO:190, IP CHANDANVELLY, RA...",India,S2,unique and sons private [limited],plot no 118 119 syno:190 ip chandanvelly ra...
22,S2-517483574,Trip Weblfare Society,"1/254, BARABAN KHURD THANA KAKORI, Uttar Pradesh",India,S2,trip weblfare society,1 254 baraban khurd thana kakori uttar pradesh
28,S2-945105765,Trip Welfare Socaieety,"1/254, BARABAN KHURD THANA KAKORI, Uttar Pradesh",India,S2,trip welfare socaieety,1 254 baraban khurd thana kakori uttar pradesh



=== candidate_sample ===
Shape: (9400, 19)
Columns: ['entity_id_s1', 'business_name_s1', 'business_address_s1', 'country_s1', 'entity_id_matched', 'business_name_matched', 'business_address_matched', 'country_matched', 'source', 'is_match', 's1_name_norm', 'matched_name_norm', 's1_address_norm', 'matched_address_norm', 'name_ratio', 'name_token_ratio', 'address_ratio', 'address_token_ratio', 'country_same']


,entity_id_s1,business_name_s1,business_address_s1,country_s1,entity_id_matched,business_name_matched,business_address_matched,country_matched,source,is_match,s1_name_norm,matched_name_norm,s1_address_norm,matched_address_norm,name_ratio,name_token_ratio,address_ratio,address_token_ratio,country_same
0,S1-793615008,Clairvoyant Record Private Limited,"1St Floor, Tamrakarmall, New Bus Stand, Sehore...",India,S2-614824581,"Gonzalez, Rosa Poreis and","3236 -300-3240 SILVER SANDS CIR, VIRGINIA BEAC...",US,S2,False,clairvoyant record private limited,gonzalez rosa poreis and,1st fl tamrakarmall new bus stand sehore madhy...,3236 300 3240 silver sands cir va beach city va,37.931034,41.379310,33.333333,32.323232,False
1,S1-793615008,Clairvoyant Record Private Limited,"1St Floor, Tamrakarmall, New Bus Stand, Sehore...",India,S2-18897662,familyempirepartners.com,"2716 VALDEZ DRIVE, TEMPLE, TX",US,S2,False,clairvoyant record private limited,familyempirepartners com,1st fl tamrakarmall new bus stand sehore madhy...,2716 valdez dr temple tx,37.931034,41.379310,30.379747,30.379747,False
2,S1-793615008,Clairvoyant Record Private Limited,"1St Floor, Tamrakarmall, New Bus Stand, Sehore...",India,S2-405677992,rabunboral.com,"0226 87 STREET, NEW YORK, NY",US,S2,False,clairvoyant record private limited,rabunboral com,1st fl tamrakarmall new bus stand sehore madhy...,0226 87 st ny ny,33.333333,33.333333,22.535211,17.647059,False
3,S1-793615008,Clairvoyant Record Private Limited,"1St Floor, Tamrakarmall, New Bus Stand, Sehore...",India,S2-686400156,మై కన్‌స్ట్రక్షన్ ప్రైవేట్ లిమిటెడ్,"1-1-742/3, తెలంగాణ, HYDERABAD",India,S2,False,clairvoyant record private limited,మ కన స ట రక షన ప ర వ ట ల మ ట డ,1st fl tamrakarmall new bus stand sehore madhy...,1 1 742 3 త ల గ ణ hyderabad,9.375000,10.344828,31.707317,32.500000,True
4,S1-793615008,Clairvoyant Record Private Limited,"1St Floor, Tamrakarmall, New Bus Stand, Sehore...",India,S2-607008616,A Cure 2 All PLLC PLLC,"14840 60ST AVENUE EAST, COLLINSVILLE, OK",US,S2,False,clairvoyant record private limited,a cure 2 all pllc pllc,1st fl tamrakarmall new bus stand sehore madhy...,14840 60st ave east collinsville ok,28.571429,31.372549,28.888889,31.111111,False


In [ ]:
# STEP 24D — Inspect how blocking candidates are generated

import inspect

print(inspect.getsource(process_source_for_blocking))

print("\n\n=== blocking_s1 ===")
print("Shape:", blocking_s1.shape)
print("Columns:", blocking_s1.columns.tolist())
display(blocking_s1.head())

def process_source_for_blocking(source_name, file_path):

    print("\n========================================")
    print("Processing", source_name)
    print("========================================")

    chunk_number = 0

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        chunksize=CHUNK_SIZE,
        dtype=str
    ):

        chunk_number += 1

        # Normalize fields
        chunk["name_norm"] = (
            chunk["business_name"]
            .apply(normalize_name)
        )

        chunk["address_norm"] = (
            chunk["business_address"]
            .apply(normalize_address)
        )

        chunk["country_norm"] = (
            chunk["country"]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.strip()
        )

        # Keys
        chunk["name_exact_key"] = (
            chunk["country_norm"]
            + "|"
            + chunk["name_norm"]
        )

        chunk["name_prefix_key"] = (
     

,entity_id,business_name,business_address,country,name_norm,address_norm,country_norm,name_exact_key,name_prefix_key,address_exact_key,address_num_key
0,S1-791737059,DK Solutions Limited,"Faridabad, Haryana, H.No-216 Sector-9",India,dk solutions limited,faridabad haryana h no 216 sector 9,india,india|dk solutions limited,india|dk sol,india|faridabad haryana h no 216 sector 9,india|216 9
1,S1-587681015,Cardiology Medicine,"5270 Mckendree Church Road, Tarboro, NC",US,cardiology medicine,5270 mckendree church road tarboro nc,us,us|cardiology medicine,us|cardio,us|5270 mckendree church road tarboro nc,us|5270
2,S1-71656643,Dynamic Producer LLP,"Bihar, Patna, C/O-Rajeshwar Singh, Dinapur-Cum...",India,dynamic producer llp,bihar patna c o rajeshwar singh dinapur cum kh...,india,india|dynamic producer llp,india|dynami,india|bihar patna c o rajeshwar singh dinapur ...,india|
3,S1-532502121,Continental Veterinary Clinic,"1667 85th Street, Cleveland, OH",US,continental veterinary clinic,1667 85th street cleveland oh,us,us|continental veterinary clinic,us|contin,us|1667 85th street cleveland oh,us|1667 85
4,S1-516946648,Kochi Research Care,"No Js-1, First Floor, Heavenly Plaza Apartment...",India,kochi research care,no js 1 first floor heavenly plaza apartment p...,india,india|kochi research care,india|kochi,india|no js 1 first floor heavenly plaza apart...,india|1


In [ ]:
# STEP 25A — Correct per-S1 candidate blocking

from collections import defaultdict
import pandas as pd

def build_query_maps(blocking_df):
    """
    Build:
        blocking key -> set/list of S1 entity IDs

    This preserves the S1 -> blocking-key relationship.
    """

    maps = {
        "name_exact": defaultdict(set),
        "name_prefix": defaultdict(set),
        "address_exact": defaultdict(set),
        "address_num": defaultdict(set)
    }

    for _, row in blocking_df.iterrows():

        s1_id = row["entity_id"]

        if row["name_exact_key"]:
            maps["name_exact"][row["name_exact_key"]].add(s1_id)

        if row["name_prefix_key"]:
            maps["name_prefix"][row["name_prefix_key"]].add(s1_id)

        if row["address_exact_key"]:
            maps["address_exact"][row["address_exact_key"]].add(s1_id)

        if row["address_num_key"]:
            maps["address_num"][row["address_num_key"]].add(s1_id)

    return maps


query_maps = build_query_maps(blocking_s1)

print("Query maps created.")

for key, value in query_maps.items():
    print(
        key,
        "unique blocking keys =", len(value)
    )

Query maps created.
name_exact unique blocking keys = 500
name_prefix unique blocking keys = 429
address_exact unique blocking keys = 500
address_num unique blocking keys = 462


In [ ]:
# STEP 25B — Generate per-S1 candidate pairs

def generate_candidate_pairs(source_name, file_path, query_maps):

    candidate_pairs = []

    chunk_number = 0

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        chunksize=CHUNK_SIZE,
        dtype=str
    ):

        chunk_number += 1

        # Normalize
        chunk["name_norm"] = (
            chunk["business_name"]
            .apply(normalize_name)
        )

        chunk["address_norm"] = (
            chunk["business_address"]
            .apply(normalize_address)
        )

        chunk["country_norm"] = (
            chunk["country"]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.strip()
        )

        # Blocking keys
        chunk["name_exact_key"] = (
            chunk["country_norm"]
            + "|"
            + chunk["name_norm"]
        )

        chunk["name_prefix_key"] = (
            chunk["country_norm"]
            + "|"
            + chunk["name_norm"].str[:6]
        )

        chunk["address_exact_key"] = (
            chunk["country_norm"]
            + "|"
            + chunk["address_norm"]
        )

        chunk["address_num_key"] = (
            chunk["country_norm"]
            + "|"
            + chunk["address_norm"].apply(get_numeric_tokens)
        )

        # Process each record
        for _, row in chunk.iterrows():

            matched_s1 = set()

            # Exact name
            key = row["name_exact_key"]

            if key in query_maps["name_exact"]:
                matched_s1.update(
                    query_maps["name_exact"][key]
                )

            # Name prefix
            key = row["name_prefix_key"]

            if key in query_maps["name_prefix"]:
                matched_s1.update(
                    query_maps["name_prefix"][key]
                )

            # Exact address
            key = row["address_exact_key"]

            if key in query_maps["address_exact"]:
                matched_s1.update(
                    query_maps["address_exact"][key]
                )

            # Numeric address
            key = row["address_num_key"]

            if key and key in query_maps["address_num"]:
                matched_s1.update(
                    query_maps["address_num"][key]
                )

            # Create candidate pair
            for s1_id in matched_s1:

                candidate_pairs.append({
                    "entity_id_s1": s1_id,
                    "entity_id_matched": row["entity_id"],
                    "business_name_matched": row["business_name"],
                    "business_address_matched": row["business_address"],
                    "country_matched": row["country"],
                    "source": source_name,
                    "name_norm": row["name_norm"],
                    "address_norm": row["address_norm"]
                })

        if chunk_number % 10 == 0:
            print(
                f"{source_name}: processed "
                f"{chunk_number} chunks | "
                f"candidate pairs so far: "
                f"{len(candidate_pairs):,}"
            )

    return pd.DataFrame(candidate_pairs)

In [ ]:
# STEP 25C — Run corrected blocking on S2 and S3

candidate_s2 = generate_candidate_pairs(
    "S2",
    "/content/train_source2.tsv",
    query_maps
)

candidate_s3 = generate_candidate_pairs(
    "S3",
    "/content/train_source3.tsv",
    query_maps
)

candidate_pairs_corrected = pd.concat(
    [candidate_s2, candidate_s3],
    ignore_index=True
)

candidate_pairs_corrected = (
    candidate_pairs_corrected
    .drop_duplicates(
        subset=["entity_id_s1", "entity_id_matched"]
    )
    .reset_index(drop=True)
)

print("\nCorrected candidate generation complete.")
print("S2 candidates:", len(candidate_s2))
print("S3 candidates:", len(candidate_s3))
print("Total unique candidates:", len(candidate_pairs_corrected))

display(candidate_pairs_corrected.head(10))

S2: processed 10 chunks | candidate pairs so far: 1,109,880
S2: processed 20 chunks | candidate pairs so far: 2,203,790
S2: processed 30 chunks | candidate pairs so far: 3,295,267
S2: processed 40 chunks | candidate pairs so far: 4,403,800
S2: processed 50 chunks | candidate pairs so far: 5,501,254
S3: processed 10 chunks | candidate pairs so far: 1,226,773
S3: processed 20 chunks | candidate pairs so far: 2,446,939
S3: processed 30 chunks | candidate pairs so far: 3,670,177
S3: processed 40 chunks | candidate pairs so far: 4,890,235
S3: processed 50 chunks | candidate pairs so far: 6,113,751

Corrected candidate generation complete.
S2 candidates: 5539119
S3 candidates: 6466562
Total unique candidates: 12005681


,entity_id_s1,entity_id_matched,business_name_matched,business_address_matched,country_matched,source,name_norm,address_norm
0,S1-859978210,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US,S2,holloway peak inc seafood,105 elm st morganton nc
1,S1-998404378,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US,S2,delta tetlecommunication inc,914 pierpont ave cleveland oh
2,S1-36208538,S2-138046867,Lee and Lawson,"1702 Pine Avenue, CITY OF MENOMONIE, WI",US,S2,lee and lawson,1702 pine avenue city of menomonie wi
3,S1-244723389,S2-277444929,Shree Infracon Private Ltd,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",India,S2,shree infracon private ltd,63 2275 7 alhind tower first floor jafferkhan ...
4,S1-556350997,S2-277444929,Shree Infracon Private Ltd,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",India,S2,shree infracon private ltd,63 2275 7 alhind tower first floor jafferkhan ...
5,S1-987498835,S2-277444929,Shree Infracon Private Ltd,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",India,S2,shree infracon private ltd,63 2275 7 alhind tower first floor jafferkhan ...
6,S1-590578012,S2-277444929,Shree Infracon Private Ltd,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",India,S2,shree infracon private ltd,63 2275 7 alhind tower first floor jafferkhan ...
7,S1-256294247,S2-684635925,Global Developers Medicals Private Limited,H.NO 008 GOVIND BUILDING OPP.HOTEL VRINDHAVAN ...,India,S2,global developers medicals private limited,h no 008 govind building opp hotel vrindhavan ...
8,S1-541010282,S2-684635925,Global Developers Medicals Private Limited,H.NO 008 GOVIND BUILDING OPP.HOTEL VRINDHAVAN ...,India,S2,global developers medicals private limited,h no 008 govind building opp hotel vrindhavan ...
9,S1-596542776,S2-414418413,CANDY'S-SERVICES,"313 FOXBOROUGH DRIVE, BRUNSWICK, OH",US,S2,candy s services,313 foxborough drive brunswick oh


In [ ]:
print(type(candidate_results_corrected))

NameError: name 'candidate_results_corrected' is not defined

In [ ]:
print([
    x for x in globals().keys()
    if "candidate" in x.lower()
])

['candidate_sample', 'negative_candidates', 'candidates', 'candidate_results', 'generate_candidate_pairs', 'candidate_s2', 'candidate_s3', 'candidate_pairs_corrected']


In [ ]:
# ============================================================
# STEP 25E — BLOCKING RECALL
# ============================================================

print("Calculating blocking recall...")

# ------------------------------------------------------------
# 1. Build TRUE matched pairs for our 500 S1 sample
# ------------------------------------------------------------

sample_s1_ids = set(training_s1_records["entity_id"])

true_pairs = set()

for _, row in ground_truth.iterrows():

    s1_id = row["source1_entity_id"]

    # Only our 500 development S1 entities
    if s1_id not in sample_s1_ids:
        continue

    matched = row["matched_entity_ids"]

    if pd.isna(matched) or str(matched).strip() == "":
        continue

    for matched_id in str(matched).split(","):

        matched_id = matched_id.strip()

        if matched_id:
            true_pairs.add((s1_id, matched_id))


print("Total true positive pairs:", len(true_pairs))


# ------------------------------------------------------------
# 2. Build candidate-pair set
# ------------------------------------------------------------

candidate_pair_set = set(
    zip(
        candidate_pairs_corrected["entity_id_s1"],
        candidate_pairs_corrected["entity_id_matched"]
    )
)

print("Total candidate pairs:", len(candidate_pair_set))


# ------------------------------------------------------------
# 3. Find recovered true matches
# ------------------------------------------------------------

recovered_pairs = true_pairs.intersection(candidate_pair_set)

print("Recovered true pairs:", len(recovered_pairs))


# ------------------------------------------------------------
# 4. Blocking recall
# ------------------------------------------------------------

blocking_recall = (
    len(recovered_pairs) / len(true_pairs)
    if len(true_pairs) > 0
    else 0
)

print(
    f"Blocking Recall: {blocking_recall:.4%}"
)


# ------------------------------------------------------------
# 5. Missed true matches
# ------------------------------------------------------------

missed_pairs = true_pairs - candidate_pair_set

print("Missed true pairs:", len(missed_pairs))


# Show examples if any were missed
if len(missed_pairs) > 0:

    print("\nExamples of missed true pairs:")

    for pair in list(missed_pairs)[:20]:
        print(pair)

Calculating blocking recall...
Total true positive pairs: 1762
Total candidate pairs: 12005681
Recovered true pairs: 1593
Blocking Recall: 90.4086%
Missed true pairs: 169

Examples of missed true pairs:
('S1-493193236', 'S3-464830319')
('S1-390796822', 'S3-247803230')
('S1-503957501', 'S3-181423094')
('S1-244723389', 'S3-646630077')
('S1-287541726', 'S2-988436899')
('S1-424812889', 'S2-665316026')
('S1-428041252', 'S2-360385940')
('S1-300794758', 'S2-47169456')
('S1-770273967', 'S3-816881088')
('S1-196201570', 'S2-29892224')
('S1-276101699', 'S3-860644724')
('S1-635981809', 'S2-867953926')
('S1-841275752', 'S3-467197335')
('S1-885693586', 'S3-155189632')
('S1-941860360', 'S2-447670025')
('S1-107267900', 'S3-26219628')
('S1-728814210', 'S3-580284534')
('S1-422604178', 'S2-259995308')
('S1-382399317', 'S2-71905214')
('S1-917750987', 'S2-690153191')


In [ ]:
# ============================================================
# STEP 26 — ANALYZE MISSED TRUE MATCHES
# ============================================================

import pandas as pd
import re

print("Analyzing missed true pairs...")
print("Missed pairs:", len(missed_pairs))


# ------------------------------------------------------------
# 1. Prepare the 500 S1 records
# ------------------------------------------------------------

s1_lookup = training_s1_records.set_index("entity_id")


# ------------------------------------------------------------
# 2. Convert missed pairs into a DataFrame
# ------------------------------------------------------------

missed_df = pd.DataFrame(
    list(missed_pairs),
    columns=["entity_id_s1", "entity_id_matched"]
)

missed_df["source"] = missed_df["entity_id_matched"].str[:2]

print("\nMissed by source:")
print(missed_df["source"].value_counts())


# ------------------------------------------------------------
# 3. Retrieve the 169 target records from S2/S3
# ------------------------------------------------------------

target_s2 = set(
    missed_df.loc[
        missed_df["source"] == "S2",
        "entity_id_matched"
    ]
)

target_s3 = set(
    missed_df.loc[
        missed_df["source"] == "S3",
        "entity_id_matched"
    ]
)

print("\nS2 missed targets:", len(target_s2))
print("S3 missed targets:", len(target_s3))


def retrieve_target_rows(file_path, target_ids):

    found = []

    if not target_ids:
        return pd.DataFrame()

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        chunksize=100000,
        dtype=str
    ):

        mask = chunk["entity_id"].isin(target_ids)

        if mask.any():
            found.append(chunk.loc[mask].copy())

    if found:
        return pd.concat(found, ignore_index=True)

    return pd.DataFrame()


missed_s2 = retrieve_target_rows(
    "/content/train_source2.tsv",
    target_s2
)

missed_s3 = retrieve_target_rows(
    "/content/train_source3.tsv",
    target_s3
)

missed_targets = pd.concat(
    [missed_s2, missed_s3],
    ignore_index=True
)

print("\nRetrieved missed target records:", len(missed_targets))


# ------------------------------------------------------------
# 4. Add normalized blocking keys to target records
# ------------------------------------------------------------

missed_targets["name_norm"] = (
    missed_targets["business_name"]
    .apply(normalize_name)
)

missed_targets["address_norm"] = (
    missed_targets["business_address"]
    .apply(normalize_address)
)

missed_targets["country_norm"] = (
    missed_targets["country"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

missed_targets["name_exact_key"] = (
    missed_targets["country_norm"]
    + "|"
    + missed_targets["name_norm"]
)

missed_targets["name_prefix_key"] = (
    missed_targets["country_norm"]
    + "|"
    + missed_targets["name_norm"].str[:6]
)

missed_targets["address_exact_key"] = (
    missed_targets["country_norm"]
    + "|"
    + missed_targets["address_norm"]
)

missed_targets["address_num_key"] = (
    missed_targets["country_norm"]
    + "|"
    + missed_targets["address_norm"].apply(get_numeric_tokens)
)


# ------------------------------------------------------------
# 5. Check which blocking rules SHOULD have caught each pair
# ------------------------------------------------------------

results = []

for _, row in missed_df.iterrows():

    s1_id = row["entity_id_s1"]
    target_id = row["entity_id_matched"]

    if s1_id not in s1_lookup.index:
        continue

    target_match = missed_targets[
        missed_targets["entity_id"] == target_id
    ]

    if target_match.empty:
        continue

    s1 = s1_lookup.loc[s1_id]
    target = target_match.iloc[0]

    s1_country = str(s1["country"]).lower().strip()
    s1_name = normalize_name(s1["business_name"])
    s1_address = normalize_address(s1["business_address"])

    s1_name_exact = (
        s1_country + "|" + s1_name
    )

    s1_name_prefix = (
        s1_country + "|" + s1_name[:6]
    )

    s1_address_exact = (
        s1_country + "|" + s1_address
    )

    s1_address_num = (
        s1_country + "|" + get_numeric_tokens(s1_address)
    )

    results.append({

        "s1_id": s1_id,
        "target_id": target_id,
        "source": row["source"],

        "s1_name": s1["business_name"],
        "target_name": target["business_name"],

        "s1_address": s1["business_address"],
        "target_address": target["business_address"],

        "s1_country": s1["country"],
        "target_country": target["country"],

        "name_exact": (
            s1_name_exact ==
            target["name_exact_key"]
        ),

        "name_prefix": (
            s1_name_prefix ==
            target["name_prefix_key"]
        ),

        "address_exact": (
            s1_address_exact ==
            target["address_exact_key"]
        ),

        "address_num": (
            s1_address_num != s1_country + "|"
            and
            s1_address_num ==
            target["address_num_key"]
        )
    })


missed_analysis = pd.DataFrame(results)


# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print("\n==============================")
print("MISSED MATCH ANALYSIS")
print("==============================")

print(
    "\nCould be recovered by name_exact:",
    missed_analysis["name_exact"].sum()
)

print(
    "Could be recovered by name_prefix:",
    missed_analysis["name_prefix"].sum()
)

print(
    "Could be recovered by address_exact:",
    missed_analysis["address_exact"].sum()
)

print(
    "Could be recovered by address_num:",
    missed_analysis["address_num"].sum()
)


# ------------------------------------------------------------
# 7. Display examples
# ------------------------------------------------------------

display(
    missed_analysis[
        [
            "s1_id",
            "target_id",
            "source",
            "s1_name",
            "target_name",
            "s1_address",
            "target_address",
            "name_exact",
            "name_prefix",
            "address_exact",
            "address_num"
        ]
    ].head(30)
)

Analyzing missed true pairs...
Missed pairs: 169

Missed by source:
source
S3    88
S2    81
Name: count, dtype: int64

S2 missed targets: 81
S3 missed targets: 88

Retrieved missed target records: 169

MISSED MATCH ANALYSIS

Could be recovered by name_exact: 0
Could be recovered by name_prefix: 0
Could be recovered by address_exact: 0
Could be recovered by address_num: 0


,s1_id,target_id,source,s1_name,target_name,s1_address,target_address,name_exact,name_prefix,address_exact,address_num
0,S1-493193236,S3-464830319,S3,Lahari Brothers Private Limited,Sri Lahari Private Limited Center,"Shop No. 2, Sharma Market Sector 37, Noida, Ga...","Shop No. 2, Noida, Gautam Buddha Nagar, UP",False,False,False,False
1,S1-390796822,S3-247803230,S3,Pmr India,Sri Pmr India Limited,"Flat No-1008, X Wing, Eiffel City, Khed, Pune,...","Flat No-1007, X Wing, Eiffel City, Pune, Khed, MH",False,False,False,False
2,S1-503957501,S3-181423094,S3,Automated Industrial Ventures Inc,Aut0mated Industrial Ventures Inc,"1061 Americana Lane, Unit 1063, Mesquite, TX","001061 Americana Ln, # 1063, Mesquite CITY, Texas",False,False,False,False
3,S1-244723389,S3-646630077,S3,Shree Services Limited,shreeservices.com,"1/831, S-4 Vasundhara, Ghaziabad, Uttar Pradesh","H.no 371 1/831, S-4 Vasundhara, Ghaziabad, UP",False,False,False,False
4,S1-287541726,S2-988436899,S2,Biseix P.C.,Biesnx P.C.,"TX, 3260 Galloway Avenue, Mesquite, Unit 1008","2260 GALLOWAY AVENUE, MESQUITE, TX",False,False,False,False
5,S1-424812889,S2-665316026,S2,Ontime & Brothers Private Limited,Mr Ontime and Brothers Private Limited,"484, G. Floor, Haveli Haider Quili, Chandni Ch...","#119 484, G. FLOOR, HAVELI HAIDER QUILI, CHAND...",False,False,False,False
6,S1-428041252,S2-360385940,S2,Thane Foundation Co,THANEFOUNDATION.COM,"Maharashtra, Thane, B-202 Tierra Casa, Thane, ...","PLOT 299 B-202 TIERRA CASA, UNIVIS LODHA, THAN...",False,False,False,False
7,S1-300794758,S2-47169456,S2,Sivika Sales Pvt Ltd,Sri Sivika Sales,"Punjab, Plot No F 93 & 102Phase Vii Focal Poin...",PLOT NO F 093 & 102PHASE VII FOCAL POINT LUDHI...,False,False,False,False
8,S1-770273967,S3-816881088,S3,Singhal & Co,S.Com,"G I D C Plot No. 43, Nr. Mahadev Engineering, ...","G I D C Plot No. 44, Nr. Mahadev Engineering, ...",False,False,False,False
9,S1-196201570,S2-29892224,S2,Red Enterprises Pvt Ltd,Réd Enterprises Pvt Ltd,"Plot At Khasra 609 & 610, Village Tusyana, Noi...","PLOT AT KHASRA 0609 & 610, VILLAGE TUSYANA, NO...",False,False,False,False


In [ ]:
# ============================================================
# STEP 27 — TOKEN OVERLAP DIAGNOSTIC
# ============================================================

import re
from collections import Counter

# Words that are too common to be useful for blocking
STOPWORDS = {
    "private", "limited", "ltd", "pvt", "llp", "llc",
    "inc", "incorporated", "company", "co", "corp",
    "corporation", "the", "and", "of",
    "india", "indian",
    "usa", "us",
    "no", "number",
    "road", "rd", "street", "st",
    "avenue", "ave",
    "floor", "fl",
    "plot", "building",
    "house", "hno",
    "near", "opp", "opposite",
    "sector", "block",
    "city", "town"
}


def useful_tokens(text):
    text = str(text).lower()

    # Keep alphanumeric words
    tokens = re.findall(r"[a-z0-9]+", text)

    # Remove very common/legal/address words
    tokens = [
        t for t in tokens
        if t not in STOPWORDS
        and len(t) >= 3
    ]

    return set(tokens)


diagnostic_rows = []

for _, row in missed_analysis.iterrows():

    s1_name_tokens = useful_tokens(row["s1_name"])
    target_name_tokens = useful_tokens(row["target_name"])

    s1_address_tokens = useful_tokens(row["s1_address"])
    target_address_tokens = useful_tokens(row["target_address"])

    name_overlap = (
        s1_name_tokens &
        target_name_tokens
    )

    address_overlap = (
        s1_address_tokens &
        target_address_tokens
    )

    diagnostic_rows.append({
        "s1_id": row["s1_id"],
        "target_id": row["target_id"],
        "source": row["source"],

        "s1_name": row["s1_name"],
        "target_name": row["target_name"],

        "name_overlap": ", ".join(
            sorted(name_overlap)
        ),

        "name_overlap_count": len(name_overlap),

        "address_overlap": ", ".join(
            sorted(address_overlap)
        ),

        "address_overlap_count": len(address_overlap)
    })


token_diagnostic = pd.DataFrame(diagnostic_rows)


print("================================================")
print("TOKEN OVERLAP RESULTS")
print("================================================")

print(
    "Name token overlap >= 1:",
    (
        token_diagnostic["name_overlap_count"] >= 1
    ).sum()
)

print(
    "Name token overlap >= 2:",
    (
        token_diagnostic["name_overlap_count"] >= 2
    ).sum()
)

print(
    "Address token overlap >= 1:",
    (
        token_diagnostic["address_overlap_count"] >= 1
    ).sum()
)

print(
    "Either name OR address token overlap:",
    (
        (token_diagnostic["name_overlap_count"] >= 1)
        |
        (token_diagnostic["address_overlap_count"] >= 1)
    ).sum()
)

print("\nExamples:")
display(
    token_diagnostic[
        [
            "s1_id",
            "target_id",
            "source",
            "s1_name",
            "target_name",
            "name_overlap",
            "address_overlap"
        ]
    ].head(40)
)

TOKEN OVERLAP RESULTS
Name token overlap >= 1: 70
Name token overlap >= 2: 45
Address token overlap >= 1: 159
Either name OR address token overlap: 166

Examples:


,s1_id,target_id,source,s1_name,target_name,name_overlap,address_overlap
0,S1-493193236,S3-464830319,S3,Lahari Brothers Private Limited,Sri Lahari Private Limited Center,lahari,"buddha, gautam, nagar, noida, shop"
1,S1-390796822,S3-247803230,S3,Pmr India,Sri Pmr India Limited,pmr,"eiffel, flat, khed, pune, wing"
2,S1-503957501,S3-181423094,S3,Automated Industrial Ventures Inc,Aut0mated Industrial Ventures Inc,"industrial, ventures","1063, americana, mesquite"
3,S1-244723389,S3-646630077,S3,Shree Services Limited,shreeservices.com,,"831, ghaziabad, vasundhara"
4,S1-287541726,S2-988436899,S2,Biseix P.C.,Biesnx P.C.,,"galloway, mesquite"
5,S1-424812889,S2-665316026,S2,Ontime & Brothers Private Limited,Mr Ontime and Brothers Private Limited,"brothers, ontime","484, chandni, chowk, delhi, haider, haveli, quili"
6,S1-428041252,S2-360385940,S2,Thane Foundation Co,THANEFOUNDATION.COM,,"202, casa, lodha, maharashtra, thane, tierra, ..."
7,S1-300794758,S2-47169456,S2,Sivika Sales Pvt Ltd,Sri Sivika Sales,"sales, sivika","102phase, focal, ludhiana, point, punjab, vii"
8,S1-770273967,S3-816881088,S3,Singhal & Co,S.Com,,"engineering, mahadev, porbandar"
9,S1-196201570,S2-29892224,S2,Red Enterprises Pvt Ltd,Réd Enterprises Pvt Ltd,enterprises,"610, khasra, noida, pradesh, tusyana, uttar, v..."


In [ ]:
# ============================================================
# STEP 28 — INSPECT THE 3 REMAINING CASES
# ============================================================

no_token_overlap = token_diagnostic[
    (token_diagnostic["name_overlap_count"] == 0) &
    (token_diagnostic["address_overlap_count"] == 0)
].copy()

print("Remaining missed pairs with NO useful token overlap:")
print("Count:", len(no_token_overlap))

display(
    no_token_overlap[
        [
            "s1_id",
            "target_id",
            "source",
            "s1_name",
            "target_name",
            "name_overlap",
            "address_overlap"
        ]
    ]
)

Remaining missed pairs with NO useful token overlap:
Count: 3


,s1_id,target_id,source,s1_name,target_name,name_overlap,address_overlap
36,S1-229574890,S2-266748285,S2,TE North LLC,TE Nórth LLC,,
41,S1-394619513,S3-851166923,S3,T+ Century LLC,The T+ Céntury LLC,,
157,S1-803997391,S2-107677081,S2,E/Z Clean,E/Z Cclon,,


In [ ]:
# ============================================================
# STEP 29 — INSPECT THE 3 MISSED PAIRS
# ============================================================

missed_ids = {
    "S1-229574890": "S2-266748285",
    "S1-394619513": "S3-851166923",
    "S1-803997391": "S2-107677081"
}

for s1_id, target_id in missed_ids.items():

    print("\n" + "=" * 80)
    print("S1:", s1_id, " -> TARGET:", target_id)
    print("=" * 80)

    # Find Source 1 record
    s1_row = training_s1_records[
        training_s1_records["entity_id"] == s1_id
    ]

    # Find target record
    if target_id.startswith("S2-"):
        target_row = train_source2[
            train_source2["entity_id"] == target_id
        ]
    else:
        target_row = train_source3[
            train_source3["entity_id"] == target_id
        ]

    print("\nSOURCE 1:")
    display(
        s1_row[
            ["entity_id", "business_name",
             "business_address", "country"]
        ]
    )

    print("\nTARGET:")
    display(
        target_row[
            ["entity_id", "business_name",
             "business_address", "country"]
        ]
    )


S1: S1-229574890  -> TARGET: S2-266748285


NameError: name 'train_source2' is not defined

In [ ]:
# Find existing DataFrame variables that contain Source 2 / Source 3 data

import pandas as pd

for name, obj in globals().items():
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)

        if {"entity_id", "business_name", "business_address", "country"}.issubset(cols):
            print(f"{name:30s} shape={obj.shape}")

RuntimeError: dictionary changed size during iteration

In [ ]:
import pandas as pd

# Make a snapshot first so globals() can change safely
all_vars = list(globals().items())

for name, obj in all_vars:
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)

        if {"entity_id", "business_name", "business_address", "country"}.issubset(cols):
            print(f"{name:30s} shape={obj.shape}")

sample_s2                      shape=(105, 4)
sample_s3                      shape=(83, 4)
sample_s1                      shape=(50, 4)
s1_small                       shape=(50, 5)
source_small                   shape=(188, 6)
positive_s2                    shape=(861, 4)
positive_s3                    shape=(901, 4)
training_s1_records            shape=(500, 4)
blocking_s1                    shape=(500, 11)
missed_s2                      shape=(81, 4)
missed_s3                      shape=(88, 4)
missed_targets                 shape=(169, 11)
target_match                   shape=(1, 11)
s1_row                         shape=(1, 4)


In [ ]:
# Inspect the 3 missed pairs that had no useful token overlap

three_missed = missed_targets[
    missed_targets["s1_id"].isin([
        "S1-229574890",
        "S1-394619513",
        "S1-803997391"
    ])
]

display(
    three_missed[
        [
            "s1_id",
            "target_id",
            "source",
            "s1_name",
            "target_name",
            "s1_address",
            "target_address",
            "name_exact",
            "name_prefix",
            "address_exact",
            "address_num",
            "name_overlap",
            "address_overlap"
        ]
    ]
)

KeyError: 's1_id'

In [ ]:
print("missed_targets columns:")
print(missed_targets.columns.tolist())

print("\nFirst 3 rows:")
display(missed_targets.head(3))

missed_targets columns:
['entity_id', 'business_name', 'business_address', 'country', 'name_norm', 'address_norm', 'country_norm', 'name_exact_key', 'name_prefix_key', 'address_exact_key', 'address_num_key']

First 3 rows:


,entity_id,business_name,business_address,country,name_norm,address_norm,country_norm,name_exact_key,name_prefix_key,address_exact_key,address_num_key
0,S2-83209191,New Lfe Cathedral,"HOMER, AK, MORNING STAR ROAD",US,new lfe cathedral,homer ak morning star road,us,us|new lfe cathedral,us|new lf,us|homer ak morning star road,us|
1,S2-259390999,M/s Snchsl & Co,"757 G I D C PLOT NO. 43, NR. MAHADEV ENGINEERI...",India,m s snchsl and co,757 g i d c plot no 43 nr mahadev engineering ...,india,india|m s snchsl and co,india|m s sn,india|757 g i d c plot no 43 nr mahadev engine...,india|757 43
2,S2-563434086,लक्ष्मी इंडियन मीडिया प्राइवेट लिमिटेड,"A-43, NEW DELHI, दिल्ली",India,लक्ष्मी इंडियन मीडिया प्राइवेट लिमिटेड,a 43 new delhi दिल्ली,india,india|लक्ष्मी इंडियन मीडिया प्राइवेट लिमिटेड,india|लक्ष्म,india|a 43 new delhi दिल्ली,india|43


In [ ]:
# Inspect the 3 difficult missed matches

target_ids = [
    "S2-266748285",
    "S3-851166923",
    "S2-107677081"
]

print("SOURCE 2 MISSED RECORDS")
display(
    missed_s2[
        missed_s2["entity_id"].isin([
            "S2-266748285",
            "S2-107677081"
        ])
    ][
        ["entity_id", "business_name", "business_address", "country"]
    ]
)

print("\nSOURCE 3 MISSED RECORD")
display(
    missed_s3[
        missed_s3["entity_id"].isin([
            "S3-851166923"
        ])
    ][
        ["entity_id", "business_name", "business_address", "country"]
    ]
)

SOURCE 2 MISSED RECORDS


,entity_id,business_name,business_address,country
10,S2-266748285,TE Nórth LLC,NaN,US
21,S2-107677081,E/Z Cclon,NaN,US



SOURCE 3 MISSED RECORD


,entity_id,business_name,business_address,country
62,S3-851166923,The T+ Céntury LLC,NaN,US


In [ ]:
# Compare Source 1 names with the 3 missed target names

s1_ids = [
    "S1-229574890",
    "S1-394619513",
    "S1-803997391"
]

s1_check = training_s1_records[
    training_s1_records["entity_id"].isin(s1_ids)
][
    ["entity_id", "business_name", "business_address", "country"]
].copy()

print("SOURCE 1 RECORDS")
display(s1_check)

SOURCE 1 RECORDS


,entity_id,business_name,business_address,country
146,S1-229574890,TE North LLC,"32 South Street, Panama, NY",US
243,S1-803997391,E/Z Clean,"Holyoke, MA, Unit 1, 50 Gates Street",US
437,S1-394619513,T+ Century LLC,"1853 Blackburn Way, Unit UNIT C, Badger, AK",US


In [ ]:
import unicodedata
from difflib import SequenceMatcher

pairs = [
    ("TE North LLC", "TE Nórth LLC"),
    ("T+ Century LLC", "The T+ Céntury LLC"),
    ("E/Z Clean", "E/Z Cclon")
]

def strip_accents(text):
    text = str(text)
    return "".join(
        c for c in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(c)
    )

for s1_name, target_name in pairs:

    current_s1 = normalize_name(s1_name)
    current_target = normalize_name(target_name)

    accent_s1 = strip_accents(current_s1)
    accent_target = strip_accents(current_target)

    similarity = SequenceMatcher(
        None,
        accent_s1,
        accent_target
    ).ratio()

    print("\n" + "=" * 70)
    print("Original:")
    print(" S1     :", s1_name)
    print(" Target :", target_name)

    print("\nCurrent normalize_name():")
    print(" S1     :", current_s1)
    print(" Target :", current_target)

    print("\nAfter accent removal:")
    print(" S1     :", accent_s1)
    print(" Target :", accent_target)

    print(f"\nCharacter similarity: {similarity:.3f}")


Original:
 S1     : TE North LLC
 Target : TE Nórth LLC

Current normalize_name():
 S1     : te north llc
 Target : te n rth llc

After accent removal:
 S1     : te north llc
 Target : te n rth llc

Character similarity: 0.917

Original:
 S1     : T+ Century LLC
 Target : The T+ Céntury LLC

Current normalize_name():
 S1     : t century llc
 Target : the t c ntury llc

After accent removal:
 S1     : t century llc
 Target : the t c ntury llc

Character similarity: 0.800

Original:
 S1     : E/Z Clean
 Target : E/Z Cclon

Current normalize_name():
 S1     : e z clean
 Target : e z cclon

After accent removal:
 S1     : e z clean
 Target : e z cclon

Character similarity: 0.778


In [ ]:
# ============================================================
# TEST IMPROVED NAME NORMALIZATION
# ============================================================

import re
import unicodedata

def normalize_name_v2(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    # Convert accented characters to their base characters
    # é -> e, ó -> o, ñ -> n, etc.
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    # Replace punctuation with spaces
    text = re.sub(r"[^a-z0-9]+", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Test the 3 difficult cases
pairs = [
    ("TE North LLC", "TE Nórth LLC"),
    ("T+ Century LLC", "The T+ Céntury LLC"),
    ("E/Z Clean", "E/Z Cclon")
]

for s1_name, target_name in pairs:

    a = normalize_name_v2(s1_name)
    b = normalize_name_v2(target_name)

    print("\n" + "=" * 60)
    print("Original:")
    print("S1     :", s1_name)
    print("Target :", target_name)

    print("\nNew normalization:")
    print("S1     :", a)
    print("Target :", b)

    print("Exact match:", a == b)
    print("Prefix 6   :", a[:6] == b[:6])


Original:
S1     : TE North LLC
Target : TE Nórth LLC

New normalization:
S1     : te north llc
Target : te north llc
Exact match: True
Prefix 6   : True

Original:
S1     : T+ Century LLC
Target : The T+ Céntury LLC

New normalization:
S1     : t century llc
Target : the t century llc
Exact match: False
Prefix 6   : False

Original:
S1     : E/Z Clean
Target : E/Z Cclon

New normalization:
S1     : e z clean
Target : e z cclon
Exact match: False
Prefix 6   : False


In [ ]:
# Check what variables contain our missed-pair analysis

all_vars = list(globals().items())

for name, obj in all_vars:
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)

        if (
            "target_id" in cols
            or "s1_id" in cols
            or "source" in cols
        ):
            print(
                f"{name:30s} shape={obj.shape}"
            )
            print(
                "Columns:",
                list(obj.columns)
            )
            print()

relationships                  shape=(188, 3)
Columns: ['source1_entity_id', 'matched_entity_id', 'source']

matched_examples               shape=(188, 9)
Columns: ['source1_entity_id', 'matched_entity_id', 'source', 's1_name', 's1_address', 's1_country', 'matched_name', 'matched_address', 'matched_country']

df                             shape=(188, 19)
Columns: ['source1_entity_id', 'matched_entity_id', 'source', 's1_name', 's1_address', 's1_country', 'matched_name', 'matched_address', 'matched_country', 's1_name_norm', 'matched_name_norm', 's1_address_norm', 'matched_address_norm', 's1_address_norm_v2', 'matched_address_norm_v2', 'name_ratio', 'name_token_ratio', 'address_ratio', 'address_token_ratio']

source_small                   shape=(188, 6)
Columns: ['entity_id', 'business_name', 'business_address', 'country', 'source', '_key']

candidate_sample               shape=(9400, 19)
Columns: ['entity_id_s1', 'business_name_s1', 'business_address_s1', 'country_s1', 'entity_id_match

In [ ]:
# ============================================================
# TEST V2 NORMALIZATION ON ALL 169 MISSED TRUE PAIRS
# ============================================================

missed_test = missed_analysis.copy()

# Apply improved normalization
missed_test["s1_name_v2"] = missed_test["s1_name"].apply(
    normalize_name_v2
)

missed_test["target_name_v2"] = missed_test["target_name"].apply(
    normalize_name_v2
)

# Exact normalized-name recovery
missed_test["name_exact_v2"] = (
    missed_test["s1_name_v2"]
    == missed_test["target_name_v2"]
)

# 6-character prefix recovery
missed_test["name_prefix_v2"] = (
    missed_test["s1_name_v2"].str[:6]
    == missed_test["target_name_v2"].str[:6]
)

print("Total missed true pairs:", len(missed_test))

print(
    "Recovered by improved exact name:",
    missed_test["name_exact_v2"].sum()
)

print(
    "Recovered by improved name prefix:",
    missed_test["name_prefix_v2"].sum()
)

print(
    "Recovered by either:",
    (
        missed_test["name_exact_v2"]
        | missed_test["name_prefix_v2"]
    ).sum()
)

print(
    "Still missed:",
    (
        ~(
            missed_test["name_exact_v2"]
            | missed_test["name_prefix_v2"]
        )
    ).sum()
)

print("\nPairs recovered by improved normalization:")

display(
    missed_test[
        missed_test["name_exact_v2"]
        | missed_test["name_prefix_v2"]
    ][
        [
            "s1_id",
            "target_id",
            "source",
            "s1_name",
            "target_name",
            "name_exact_v2",
            "name_prefix_v2"
        ]
    ].head(30)
)

Total missed true pairs: 169
Recovered by improved exact name: 3
Recovered by improved name prefix: 4
Recovered by either: 4
Still missed: 165

Pairs recovered by improved normalization:


,s1_id,target_id,source,s1_name,target_name,name_exact_v2,name_prefix_v2
9,S1-196201570,S2-29892224,S2,Red Enterprises Pvt Ltd,Réd Enterprises Pvt Ltd,True,True
23,S1-89545333,S3-499959015,S3,ZV Amc,ZV ámc,True,True
36,S1-229574890,S2-266748285,S2,TE North LLC,TE Nórth LLC,True,True
120,S1-472678371,S2-144001095,S2,Raks & Co Company,Raks + [Co],False,True


In [ ]:
# ============================================================
# ANALYZE THE 165 STILL-MISSED PAIRS
# ============================================================

still_missed = missed_test[
    ~(
        missed_test["name_exact_v2"]
        | missed_test["name_prefix_v2"]
    )
].copy()

print("Still missed:", len(still_missed))

# Show all remaining missed pairs
display(
    still_missed[
        [
            "s1_id",
            "target_id",
            "source",
            "s1_name",
            "target_name",
            "s1_address",
            "target_address",
            "s1_country",
            "target_country"
        ]
    ].head(50)
)

Still missed: 165


,s1_id,target_id,source,s1_name,target_name,s1_address,target_address,s1_country,target_country
0,S1-493193236,S3-464830319,S3,Lahari Brothers Private Limited,Sri Lahari Private Limited Center,"Shop No. 2, Sharma Market Sector 37, Noida, Ga...","Shop No. 2, Noida, Gautam Buddha Nagar, UP",India,India
1,S1-390796822,S3-247803230,S3,Pmr India,Sri Pmr India Limited,"Flat No-1008, X Wing, Eiffel City, Khed, Pune,...","Flat No-1007, X Wing, Eiffel City, Pune, Khed, MH",India,India
2,S1-503957501,S3-181423094,S3,Automated Industrial Ventures Inc,Aut0mated Industrial Ventures Inc,"1061 Americana Lane, Unit 1063, Mesquite, TX","001061 Americana Ln, # 1063, Mesquite CITY, Texas",US,US
3,S1-244723389,S3-646630077,S3,Shree Services Limited,shreeservices.com,"1/831, S-4 Vasundhara, Ghaziabad, Uttar Pradesh","H.no 371 1/831, S-4 Vasundhara, Ghaziabad, UP",India,India
4,S1-287541726,S2-988436899,S2,Biseix P.C.,Biesnx P.C.,"TX, 3260 Galloway Avenue, Mesquite, Unit 1008","2260 GALLOWAY AVENUE, MESQUITE, TX",US,US
5,S1-424812889,S2-665316026,S2,Ontime & Brothers Private Limited,Mr Ontime and Brothers Private Limited,"484, G. Floor, Haveli Haider Quili, Chandni Ch...","#119 484, G. FLOOR, HAVELI HAIDER QUILI, CHAND...",India,India
6,S1-428041252,S2-360385940,S2,Thane Foundation Co,THANEFOUNDATION.COM,"Maharashtra, Thane, B-202 Tierra Casa, Thane, ...","PLOT 299 B-202 TIERRA CASA, UNIVIS LODHA, THAN...",India,India
7,S1-300794758,S2-47169456,S2,Sivika Sales Pvt Ltd,Sri Sivika Sales,"Punjab, Plot No F 93 & 102Phase Vii Focal Poin...",PLOT NO F 093 & 102PHASE VII FOCAL POINT LUDHI...,India,India
8,S1-770273967,S3-816881088,S3,Singhal & Co,S.Com,"G I D C Plot No. 43, Nr. Mahadev Engineering, ...","G I D C Plot No. 44, Nr. Mahadev Engineering, ...",India,India
10,S1-276101699,S3-860644724,S3,Pauletta Medina Adr LLC,medinaadrpauletta.com,"3851 Cobble Ridge Drive, UT, Unit 5301, West J...","03851 Cobble Ridge Dr, # 5301, West Jordan, Utah",US,US


In [ ]:
# ============================================================
# CHECK HOW MUCH INFORMATION IS SHARED
# ============================================================

def token_set(text):
    if pd.isna(text):
        return set()

    return set(
        str(text).lower().split()
    )


def token_overlap_count(a, b):
    return len(token_set(a) & token_set(b))


still_missed["name_token_overlap"] = (
    still_missed.apply(
        lambda r: token_overlap_count(
            r["s1_name"],
            r["target_name"]
        ),
        axis=1
    )
)

still_missed["address_token_overlap"] = (
    still_missed.apply(
        lambda r: token_overlap_count(
            r["s1_address"],
            r["target_address"]
        ),
        axis=1
    )
)

print(
    "Name token overlap >= 1:",
    (still_missed["name_token_overlap"] >= 1).sum()
)

print(
    "Name token overlap >= 2:",
    (still_missed["name_token_overlap"] >= 2).sum()
)

print(
    "Address token overlap >= 1:",
    (still_missed["address_token_overlap"] >= 1).sum()
)

print(
    "Either name/address overlap:",
    (
        (still_missed["name_token_overlap"] >= 1)
        |
        (still_missed["address_token_overlap"] >= 1)
    ).sum()
)

Name token overlap >= 1: 82
Name token overlap >= 2: 64
Address token overlap >= 1: 157
Either name/address overlap: 165


In [ ]:
# ============================================================
# ANALYZE SHARED ADDRESS TOKENS IN THE 165 MISSED PAIRS
# ============================================================

import re
from collections import Counter

def get_tokens(text):
    if pd.isna(text):
        return set()

    text = str(text).lower()

    # Keep alphanumeric tokens
    return set(
        re.findall(r"[a-z0-9]+", text)
    )


common_address_words = {
    "road", "street", "st", "rd",
    "avenue", "ave",
    "lane", "ln",
    "floor", "fl",
    "building", "bldg",
    "house", "h",
    "no", "number",
    "plot", "sector",
    "block",
    "city",
    "district",
    "near",
    "opp", "opposite",
    "road",
    "india"
}

def useful_shared_tokens(row):

    s1_tokens = get_tokens(row["s1_address"])
    target_tokens = get_tokens(row["target_address"])

    shared = s1_tokens & target_tokens

    useful = {
        t for t in shared
        if (
            len(t) >= 4
            and t not in common_address_words
        )
    }

    return useful


still_missed["useful_address_tokens"] = (
    still_missed.apply(
        useful_shared_tokens,
        axis=1
    )
)

still_missed["useful_token_count"] = (
    still_missed["useful_address_tokens"]
    .apply(len)
)

print(
    "Pairs with >=1 useful address token:",
    (still_missed["useful_token_count"] >= 1).sum()
)

print(
    "Pairs with >=2 useful address tokens:",
    (still_missed["useful_token_count"] >= 2).sum()
)

print("\nExamples:")

display(
    still_missed[
        [
            "s1_id",
            "target_id",
            "source",
            "s1_address",
            "target_address",
            "useful_address_tokens"
        ]
    ].head(30)
)

Pairs with >=1 useful address token: 156
Pairs with >=2 useful address tokens: 130

Examples:


,s1_id,target_id,source,s1_address,target_address,useful_address_tokens
0,S1-493193236,S3-464830319,S3,"Shop No. 2, Sharma Market Sector 37, Noida, Ga...","Shop No. 2, Noida, Gautam Buddha Nagar, UP","{nagar, buddha, shop, noida, gautam}"
1,S1-390796822,S3-247803230,S3,"Flat No-1008, X Wing, Eiffel City, Khed, Pune,...","Flat No-1007, X Wing, Eiffel City, Pune, Khed, MH","{khed, wing, eiffel, flat, pune}"
2,S1-503957501,S3-181423094,S3,"1061 Americana Lane, Unit 1063, Mesquite, TX","001061 Americana Ln, # 1063, Mesquite CITY, Texas","{1063, mesquite, americana}"
3,S1-244723389,S3-646630077,S3,"1/831, S-4 Vasundhara, Ghaziabad, Uttar Pradesh","H.no 371 1/831, S-4 Vasundhara, Ghaziabad, UP","{vasundhara, ghaziabad}"
4,S1-287541726,S2-988436899,S2,"TX, 3260 Galloway Avenue, Mesquite, Unit 1008","2260 GALLOWAY AVENUE, MESQUITE, TX","{mesquite, galloway}"
5,S1-424812889,S2-665316026,S2,"484, G. Floor, Haveli Haider Quili, Chandni Ch...","#119 484, G. FLOOR, HAVELI HAIDER QUILI, CHAND...","{quili, haider, chowk, chandni, delhi, haveli}"
6,S1-428041252,S2-360385940,S2,"Maharashtra, Thane, B-202 Tierra Casa, Thane, ...","PLOT 299 B-202 TIERRA CASA, UNIVIS LODHA, THAN...","{lodha, thane, maharashtra, casa, tierra, univis}"
7,S1-300794758,S2-47169456,S2,"Punjab, Plot No F 93 & 102Phase Vii Focal Poin...",PLOT NO F 093 & 102PHASE VII FOCAL POINT LUDHI...,"{punjab, 102phase, point, focal, ludhiana}"
8,S1-770273967,S3-816881088,S3,"G I D C Plot No. 43, Nr. Mahadev Engineering, ...","G I D C Plot No. 44, Nr. Mahadev Engineering, ...","{porbandar, mahadev, engineering}"
10,S1-276101699,S3-860644724,S3,"3851 Cobble Ridge Drive, UT, Unit 5301, West J...","03851 Cobble Ridge Dr, # 5301, West Jordan, Utah","{5301, jordan, cobble, west, ridge}"


In [ ]:
# ============================================================
# COUNT USEFUL ADDRESS TOKENS IN THE MISSED TRUE PAIRS
# ============================================================

from collections import Counter

token_counter = Counter()

for tokens in still_missed["useful_address_tokens"]:
    token_counter.update(tokens)

print("Top useful shared address tokens:")
for token, count in token_counter.most_common(40):
    print(f"{token:25s} {count}")

Top useful shared address tokens:
delhi                     19
nagar                     14
bangalore                 11
hyderabad                 10
pune                      9
south                     9
mumbai                    9
maharashtra               8
main                      8
flat                      7
west                      7
chennai                   7
ward                      7
buddha                    6
gautam                    6
wing                      6
noida                     5
karnataka                 5
revenue                   5
north                     4
east                      4
purbasha                  4
mayur                     4
phase                     4
anandlok                  4
operative                 4
housing                   4
viha                      4
society                   4
second                    4
pradesh                   4
place                     4
2645                      4
gulbarga                  4
stage     

In [ ]:
# ============================================================
# GLOBAL ADDRESS TOKEN FREQUENCY
# ============================================================

from collections import Counter
import pandas as pd

def count_address_tokens(file_path, chunk_size=100_000):

    counter = Counter()

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        dtype=str,
        chunksize=chunk_size
    ):

        for value in chunk["business_address"].fillna(""):

            tokens = get_tokens(value)

            for token in tokens:
                if (
                    len(token) >= 4
                    and token not in common_address_words
                ):
                    counter[token] += 1

    return counter


print("Counting Source 2 address tokens...")

s2_token_freq = count_address_tokens(
    "/content/train_source2.tsv"
)

print("Source 2 done.")

print("Counting Source 3 address tokens...")

s3_token_freq = count_address_tokens(
    "/content/train_source3.tsv"
)

print("Source 3 done.")

print("Finished.")

Counting Source 2 address tokens...
Source 2 done.
Counting Source 3 address tokens...
Source 3 done.
Finished.


In [ ]:
# ============================================================
# DISPLAY GLOBAL FREQUENCY OF THE IMPORTANT MISSED-PAIR TOKENS
# ============================================================

important_tokens = [
    token
    for token, _ in token_counter.most_common(40)
]

frequency_table = pd.DataFrame({
    "token": important_tokens,
    "s2_frequency": [
        s2_token_freq.get(t, 0)
        for t in important_tokens
    ],
    "s3_frequency": [
        s3_token_freq.get(t, 0)
        for t in important_tokens
    ]
})

frequency_table["total_frequency"] = (
    frequency_table["s2_frequency"]
    + frequency_table["s3_frequency"]
)

display(
    frequency_table.sort_values(
        "total_frequency"
    )
)

,token,s2_frequency,s3_frequency,total_frequency
21,purbasha,48,33,81
24,anandlok,56,47,103
27,viha,126,78,204
32,2645,170,148,318
33,gulbarga,632,843,1475
18,revenue,1949,1575,3524
25,operative,2202,1758,3960
38,mesquite,2458,2623,5081
22,mayur,5101,4379,9480
26,housing,7822,6434,14256


In [ ]:
# ============================================================
# TEST ADDRESS-TOKEN BLOCKING THRESHOLDS
# ============================================================

thresholds = [
    100,
    250,
    500,
    1000,
    2000,
    5000,
    10000,
    25000,
    50000
]

results = []

for threshold in thresholds:

    recovered = 0

    for tokens in still_missed["useful_address_tokens"]:

        # A missed true pair is recoverable if it contains
        # at least one address token whose global frequency
        # is <= the threshold.
        found = False

        for token in tokens:

            freq = (
                s2_token_freq.get(token, 0)
                + s3_token_freq.get(token, 0)
            )

            if 0 < freq <= threshold:
                found = True
                break

        if found:
            recovered += 1

    results.append({
        "threshold": threshold,
        "missed_pairs_recovered": recovered,
        "missed_pairs_remaining": 169 - recovered,
        "recovery_%_of_missed": round(
            recovered / 169 * 100,
            2
        ),
        "overall_blocking_recall": round(
            (1593 + recovered) / 1762 * 100,
            2
        )
    })

threshold_results = pd.DataFrame(results)

display(threshold_results)

,threshold,missed_pairs_recovered,missed_pairs_remaining,recovery_%_of_missed,overall_blocking_recall
0,100,45,124,26.63,92.96
1,250,55,114,32.54,93.53
2,500,74,95,43.79,94.61
3,1000,86,83,50.89,95.29
4,2000,102,67,60.36,96.20
5,5000,112,57,66.27,96.77
6,10000,116,53,68.64,96.99
7,25000,121,48,71.60,97.28
8,50000,130,39,76.92,97.79


In [ ]:
# ============================================================
# ESTIMATE CANDIDATE GROWTH FROM ADDRESS-TOKEN BLOCKING
# ============================================================

import re
from collections import Counter

# ------------------------------------------------------------
# 1. Find the full Source 1 dataframe safely
# ------------------------------------------------------------

possible_s1 = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = set(obj.columns)

        if {
            "entity_id",
            "business_name",
            "business_address",
            "country"
        }.issubset(cols):

            # Source 1 should be around 2.2M rows
            if len(obj) > 1_000_000:
                possible_s1.append((name, obj))

print("Possible full Source 1 dataframes:")
for name, obj in possible_s1:
    print(name, obj.shape)


Possible full Source 1 dataframes:


In [ ]:
# Show all DataFrames currently in memory
# Safely use list(globals().items()) to avoid "dictionary changed size" error.

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"{name:35s} shape={obj.shape}")
        print("   columns:", list(obj.columns))
        print()

ground_truth                        shape=(2206821, 2)
   columns: ['source1_entity_id', 'matched_entity_ids']

_1                                  shape=(5, 2)
   columns: ['source1_entity_id', 'matched_entity_ids']

matched_gt                          shape=(50, 2)
   columns: ['source1_entity_id', 'matched_entity_ids']

sample_s2                           shape=(105, 4)
   columns: ['entity_id', 'business_name', 'business_address', 'country']

sample_s3                           shape=(83, 4)
   columns: ['entity_id', 'business_name', 'business_address', 'country']

sample_s1                           shape=(50, 4)
   columns: ['entity_id', 'business_name', 'business_address', 'country']

relationships                       shape=(188, 3)
   columns: ['source1_entity_id', 'matched_entity_id', 'source']

s1_lookup                           shape=(500, 3)
   columns: ['business_name', 'business_address', 'country']

s2_lookup                           shape=(105, 4)
   columns: ['matc

In [ ]:
import os

print(os.listdir("/content")[:50])

['.config', 'train_ground_truth.tsv', 'train_source3.tsv', 'train_source2.tsv', 'train_source1.tsv', 'sample_data']


In [ ]:
train_source1 = pd.read_csv(
    "/content/train_source1.tsv",
    sep="\t"
)

print("Source 1 shape:", train_source1.shape)
print(train_source1.columns.tolist())

Source 1 shape: (2206821, 4)
['entity_id', 'business_name', 'business_address', 'country']


In [ ]:
# ============================================================
# ESTIMATE ADDRESS-TOKEN BLOCKING SIZE
# ============================================================

from collections import Counter

# We already have frequency_table:
# columns = token, s2_frequency, s3_frequency, total_frequency

thresholds = [1000, 2000, 5000, 10000, 25000, 50000]

print("Number of address tokens allowed at each threshold:\n")

for t in thresholds:
    allowed = frequency_table[
        frequency_table["total_frequency"] <= t
    ]

    print(
        f"Threshold {t:>6}: "
        f"{len(allowed):>3} tokens"
    )

print("\nTop frequencies:")
display(
    frequency_table
    .sort_values("total_frequency", ascending=False)
    .head(20)
)

Number of address tokens allowed at each threshold:

Threshold   1000:   4 tokens
Threshold   2000:   5 tokens
Threshold   5000:   7 tokens
Threshold  10000:   9 tokens
Threshold  25000:  11 tokens
Threshold  50000:  18 tokens

Top frequencies:


,token,s2_frequency,s3_frequency,total_frequency
1,nagar,286965,267815,554780
0,delhi,262478,272877,535355
36,drive,215965,241783,457748
19,north,72593,327168,399761
10,west,200360,191943,392303
6,mumbai,162531,197983,360514
7,maharashtra,319026,23988,343014
30,pradesh,214708,43604,258312
2,bangalore,110613,130208,240821
5,south,76691,136754,213445


In [ ]:
# ============================================================
# ESTIMATE CANDIDATE PAIRS GENERATED BY ADDRESS TOKENS
# ============================================================

import re
from collections import Counter

# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------
def address_tokens(text):
    if pd.isna(text):
        return set()

    text = str(text).lower()

    # Keep letters/numbers and split everything else
    tokens = re.findall(r"[a-z0-9]+", text)

    return set(tokens)


# ------------------------------------------------------------
# 1. Count how many Source 1 records contain each token
# ------------------------------------------------------------

print("Counting Source 1 address tokens...")

s1_token_count = Counter()

for address in train_source1["business_address"]:
    tokens = address_tokens(address)

    for token in tokens:
        s1_token_count[token] += 1

print("Source 1 token counting complete.")


# ------------------------------------------------------------
# 2. Calculate estimated candidate pairs
# ------------------------------------------------------------

thresholds = [1000, 2000, 5000, 10000, 25000, 50000]

results = []

for threshold in thresholds:

    allowed = frequency_table[
        frequency_table["total_frequency"] <= threshold
    ]

    estimated_pairs = 0

    for _, row in allowed.iterrows():

        token = row["token"]

        s1_count = s1_token_count.get(token, 0)

        s2_count = int(row["s2_frequency"])
        s3_count = int(row["s3_frequency"])

        # Every S1 containing this token can pair with
        # every S2/S3 record containing this token.
        estimated_pairs += s1_count * (s2_count + s3_count)

    results.append({
        "threshold": threshold,
        "tokens": len(allowed),
        "estimated_pairs": estimated_pairs
    })


estimate_df = pd.DataFrame(results)

display(estimate_df)

Counting Source 1 address tokens...
Source 1 token counting complete.


,threshold,tokens,estimated_pairs
0,1000,4,44242
1,2000,5,631292
2,5000,7,8613016
3,10000,9,39399171
4,25000,11,241608486
5,50000,18,3744924587


In [ ]:
# ============================================================
# STEP 30 — INSPECT THE 2,000-FREQUENCY ADDRESS BLOCK
# ============================================================

threshold = 2000

recovered_2000 = []

for _, row in still_missed.iterrows():

    found = False
    matching_tokens = []

    for token in row["useful_address_tokens"]:

        freq = (
            s2_token_freq.get(token, 0)
            + s3_token_freq.get(token, 0)
        )

        if 0 < freq <= threshold:
            found = True
            matching_tokens.append(
                (token, freq)
            )

    if found:
        recovered_2000.append({
            "s1_id": row["s1_id"],
            "target_id": row["target_id"],
            "source": row["source"],
            "tokens": matching_tokens
        })

recovered_2000_df = pd.DataFrame(
    recovered_2000
)

print(
    "Recovered at threshold 2000:",
    len(recovered_2000_df)
)

print("\nBy source:")
print(
    recovered_2000_df["source"].value_counts()
)

print("\nExamples:")
display(
    recovered_2000_df.head(30)
)

Recovered at threshold 2000: 102

By source:
source
S3    57
S2    45
Name: count, dtype: int64

Examples:


,s1_id,target_id,source,tokens
0,S1-390796822,S3-247803230,S3,"[(eiffel, 63)]"
1,S1-503957501,S3-181423094,S3,"[(1063, 771), (americana, 270)]"
2,S1-287541726,S2-988436899,S2,"[(galloway, 1426)]"
3,S1-424812889,S2-665316026,S2,"[(quili, 6), (haider, 224), (chandni, 1791)]"
4,S1-428041252,S2-360385940,S2,"[(tierra, 958), (univis, 59)]"
5,S1-300794758,S2-47169456,S2,"[(102phase, 4), (focal, 1387)]"
6,S1-770273967,S3-816881088,S3,"[(porbandar, 577), (engineering, 1355)]"
7,S1-276101699,S3-860644724,S3,"[(5301, 520), (cobble, 428)]"
8,S1-635981809,S2-867953926,S2,"[(vepery, 371), (barracks, 157)]"
9,S1-841275752,S3-467197335,S3,"[(ksnr, 8)]"


In [ ]:
# STEP 31 — ADD SELECTIVE ADDRESS-TOKEN BLOCKING

ADDRESS_TOKEN_FREQ_THRESHOLD = 2000

# Tokens allowed for blocking
allowed_address_tokens = {
    token
    for token, freq in (
        frequency_table
        .set_index("token")["total_frequency"]
        .to_dict()
        .items()
    )
    if freq <= ADDRESS_TOKEN_FREQ_THRESHOLD
}

print("Allowed address tokens:", len(allowed_address_tokens))
print(sorted(list(allowed_address_tokens))[:50])

Allowed address tokens: 5
['2645', 'anandlok', 'gulbarga', 'purbasha', 'viha']


In [ ]:
# STEP 32 — SELECTIVE ADDRESS-TOKEN BLOCK

# Build inverted indexes for the allowed rare address tokens
s2_address_token_index = {}
s3_address_token_index = {}

for token in allowed_address_tokens:
    s2_address_token_index[token] = set()
    s3_address_token_index[token] = set()


def get_useful_address_tokens(address):
    if pd.isna(address):
        return set()

    tokens = set(re.findall(r"[a-z0-9]+", str(address).lower()))

    return {
        token
        for token in tokens
        if len(token) >= 4
        and token in allowed_address_tokens
    }


# Scan S2 in chunks
for chunk in pd.read_csv(
    "/content/train_source2.tsv",
    sep="\t",
    chunksize=100_000
):
    for entity_id, address in zip(
        chunk["entity_id"],
        chunk["business_address"]
    ):
        tokens = get_useful_address_tokens(address)

        for token in tokens:
            s2_address_token_index[token].add(entity_id)


# Scan S3 in chunks
for chunk in pd.read_csv(
    "/content/train_source3.tsv",
    sep="\t",
    chunksize=100_000
):
    for entity_id, address in zip(
        chunk["entity_id"],
        chunk["business_address"]
    ):
        tokens = get_useful_address_tokens(address)

        for token in tokens:
            s3_address_token_index[token].add(entity_id)


print("S2 index sizes:")
for token, ids in s2_address_token_index.items():
    print(token, len(ids))

print("\nS3 index sizes:")
for token, ids in s3_address_token_index.items():
    print(token, len(ids))

S2 index sizes:
purbasha 48
2645 170
gulbarga 632
anandlok 56
viha 126

S3 index sizes:
purbasha 33
2645 148
gulbarga 843
anandlok 47
viha 78


In [ ]:
# STEP 33 — GENERATE ADDRESS-TOKEN CANDIDATES

address_token_candidates = []

for _, row in blocking_s1.iterrows():

    s1_id = row["entity_id"]
    address = row["business_address"]

    s1_tokens = get_useful_address_tokens(address)

    # Only tokens that can actually produce candidates
    s1_tokens = s1_tokens.intersection(allowed_address_tokens)

    # S2 candidates
    for token in s1_tokens:
        for target_id in s2_address_token_index.get(token, set()):
            address_token_candidates.append({
                "entity_id_s1": s1_id,
                "entity_id_matched": target_id,
                "source": "S2",
                "blocking_key": token
            })

    # S3 candidates
    for token in s1_tokens:
        for target_id in s3_address_token_index.get(token, set()):
            address_token_candidates.append({
                "entity_id_s1": s1_id,
                "entity_id_matched": target_id,
                "source": "S3",
                "blocking_key": token
            })


address_token_candidates = pd.DataFrame(address_token_candidates)

print("Raw address-token candidates:", len(address_token_candidates))

# Remove duplicates caused by multiple shared tokens
address_token_candidates_unique = (
    address_token_candidates
    .drop_duplicates(
        subset=["entity_id_s1", "entity_id_matched", "source"]
    )
    .reset_index(drop=True)
)

print(
    "Unique address-token candidates:",
    len(address_token_candidates_unique)
)

print("\nBy source:")
print(
    address_token_candidates_unique["source"]
    .value_counts()
)

print("\nSample:")
display(address_token_candidates_unique.head(20))

Raw address-token candidates: 2181
Unique address-token candidates: 2165

By source:
source
S3    1141
S2    1024
Name: count, dtype: int64

Sample:


,entity_id_s1,entity_id_matched,source,blocking_key
0,S1-382399317,S2-755440138,S2,purbasha
1,S1-382399317,S2-771071769,S2,purbasha
2,S1-382399317,S2-841273643,S2,purbasha
3,S1-382399317,S2-808967278,S2,purbasha
4,S1-382399317,S2-615490440,S2,purbasha
5,S1-382399317,S2-36879102,S2,purbasha
6,S1-382399317,S2-389260693,S2,purbasha
7,S1-382399317,S2-174783934,S2,purbasha
8,S1-382399317,S2-737413273,S2,purbasha
9,S1-382399317,S2-166797693,S2,purbasha


In [ ]:
# STEP 34 — CHECK BLOCKING RECALL AFTER ADDRESS-TOKEN BLOCK

# Convert candidate pairs into a set for fast lookup
address_token_candidate_set = set(
    zip(
        address_token_candidates_unique["entity_id_s1"],
        address_token_candidates_unique["entity_id_matched"],
        address_token_candidates_unique["source"]
    )
)

# Check which previously missed true matches are now recovered
recovered_by_address_token = []

for _, row in missed_df.iterrows():

    pair = (
        row["entity_id_s1"],
        row["entity_id_matched"],
        row["source"]
    )

    if pair in address_token_candidate_set:
        recovered_by_address_token.append(row)


recovered_by_address_token = pd.DataFrame(
    recovered_by_address_token
)

print(
    "Previously missed pairs recovered:",
    len(recovered_by_address_token)
)

print(
    "Remaining missed pairs:",
    len(missed_df) - len(recovered_by_address_token)
)

print("\nRecovered by source:")
print(
    recovered_by_address_token["source"].value_counts()
)

print("\nRecovered examples:")
display(recovered_by_address_token.head(30))

Previously missed pairs recovered: 8
Remaining missed pairs: 161

Recovered by source:
source
S2    4
S3    4
Name: count, dtype: int64

Recovered examples:


,entity_id_s1,entity_id_matched,source
18,S1-382399317,S2-71905214,S2
39,S1-357424667,S3-399839281,S3
44,S1-357424667,S2-931202347,S2
67,S1-382399317,S2-389260693,S2
72,S1-357424667,S3-760468044,S3
144,S1-357424667,S2-71460422,S2
150,S1-382399317,S3-606109397,S3
160,S1-382399317,S3-457631003,S3


In [ ]:
# STEP 35 — DEBUG THE 102 vs 8 DISCREPANCY

# Find missed pairs that were predicted recoverable by the
# frequency diagnostic but NOT recovered by our actual index.

debug_rows = []

for _, row in still_missed.iterrows():

    s1_id = row["s1_id"]
    target_id = row["target_id"]
    source = row["source"]

    qualifying_tokens = []

    for token in row["useful_address_tokens"]:
        freq = (
            s2_token_freq.get(token, 0)
            + s3_token_freq.get(token, 0)
        )

        if 0 < freq <= 2000:
            qualifying_tokens.append(
                (token, freq)
            )

    if qualifying_tokens:

        actual_pair = (
            s1_id,
            target_id,
            source
        )

        found_in_actual_index = (
            actual_pair in address_token_candidate_set
        )

        debug_rows.append({
            "s1_id": s1_id,
            "target_id": target_id,
            "source": source,
            "qualifying_tokens": qualifying_tokens,
            "found_in_actual_index": found_in_actual_index
        })


debug_df = pd.DataFrame(debug_rows)

print("Diagnostic qualifying pairs:", len(debug_df))

print(
    "Actually recovered:",
    debug_df["found_in_actual_index"].sum()
)

print(
    "Not recovered:",
    (~debug_df["found_in_actual_index"]).sum()
)

display(
    debug_df[
        ~debug_df["found_in_actual_index"]
    ].head(30)
)

Diagnostic qualifying pairs: 102
Actually recovered: 8
Not recovered: 94


,s1_id,target_id,source,qualifying_tokens,found_in_actual_index
0,S1-390796822,S3-247803230,S3,"[(eiffel, 63)]",False
1,S1-503957501,S3-181423094,S3,"[(1063, 771), (americana, 270)]",False
2,S1-287541726,S2-988436899,S2,"[(galloway, 1426)]",False
3,S1-424812889,S2-665316026,S2,"[(quili, 6), (haider, 224), (chandni, 1791)]",False
4,S1-428041252,S2-360385940,S2,"[(tierra, 958), (univis, 59)]",False
5,S1-300794758,S2-47169456,S2,"[(102phase, 4), (focal, 1387)]",False
6,S1-770273967,S3-816881088,S3,"[(porbandar, 577), (engineering, 1355)]",False
7,S1-276101699,S3-860644724,S3,"[(5301, 520), (cobble, 428)]",False
8,S1-635981809,S2-867953926,S2,"[(vepery, 371), (barracks, 157)]",False
9,S1-841275752,S3-467197335,S3,"[(ksnr, 8)]",False


In [ ]:
# STEP 36 — BUILD CORRECT GLOBAL ADDRESS-TOKEN BLOCKING VOCABULARY

ADDRESS_TOKEN_FREQ_THRESHOLD = 2000

all_address_tokens = (
    set(s2_token_freq.keys())
    | set(s3_token_freq.keys())
)

allowed_address_tokens = {
    token
    for token in all_address_tokens
    if (
        s2_token_freq.get(token, 0)
        + s3_token_freq.get(token, 0)
    ) <= ADDRESS_TOKEN_FREQ_THRESHOLD
}

print("Total unique address tokens:", len(all_address_tokens))
print(
    "Allowed tokens (frequency <= 2000):",
    len(allowed_address_tokens)
)

print("\nSome allowed tokens:")
print(sorted(allowed_address_tokens)[:100])

Total unique address tokens: 721812
Allowed tokens (frequency <= 2000): 719631

Some allowed tokens:
['0000', '00000', '0000000', '0000000000000000000000', '0000000305', '000000305', '000000dsk', '00000134', '000001a', '00000bl', '00000ganga', '00000nr', '00000sn', '00000sri', '00000vadrdhaman', '00000vbdrdhaman', '00001', '0000108', '00002', '00003', '00004', '00005', '00006', '00007', '00008', '00009', '0000bld', '0000bt', '0000laxmi', '0000nyati', '0000pl', '0000sn', '0001', '00011', '00012', '00013', '00014', '00015', '00017', '00018', '0001ab', '0001dd', '0001h', '0001st', '0002', '00022', '00023', '00024', '00029', '0002a', '0002f', '0002nd', '0003', '00032', '00034', '00037', '00039', '0003gfb', '0004', '00040102', '00041', '00042', '00043', '00044', '00045', '00047', '00048', '00049', '0005', '00050002', '00052', '00053', '00055', '00056', '00058', '00059', '0005th', '0006', '00060', '0006003', '00061', '0007', '00070', '00072', '00073', '00076', '0007a', '0007th', '0008', '000

In [ ]:
# STEP 37 — FILTER ADDRESS TOKENS FOR BLOCKING

COMMON_ADDRESS_WORDS = {
    "road", "rd", "street", "st",
    "avenue", "ave", "lane", "ln",
    "drive", "dr", "boulevard", "blvd",
    "highway", "hwy",
    "building", "bldg",
    "floor", "fl",
    "house", "h",
    "number", "no",
    "plot",
    "sector", "sec",
    "block",
    "city", "district",
    "near", "opp", "opposite",
    "india",
    "state",
    "main", "north", "south",
    "east", "west",
    "first", "second", "third",
    "phase", "stage",
    "place", "roadway"
}

def is_good_blocking_token(token, max_freq=2000):

    token = str(token).lower()

    # Frequency condition
    freq = (
        s2_token_freq.get(token, 0)
        + s3_token_freq.get(token, 0)
    )

    if freq == 0 or freq > max_freq:
        return False

    # Length
    if len(token) < 4:
        return False

    # Common address words
    if token in COMMON_ADDRESS_WORDS:
        return False

    # Purely numeric tokens are too noisy
    if token.isdigit():
        return False

    # Require at least one alphabetic character
    if not any(c.isalpha() for c in token):
        return False

    return True


good_address_tokens = {
    token
    for token in all_address_tokens
    if is_good_blocking_token(token)
}

print(
    "Good blocking tokens:",
    len(good_address_tokens)
)

print("\nExamples:")
print(sorted(good_address_tokens)[:100])

Good blocking tokens: 631226

Examples:
['000000dsk', '000001a', '00000bl', '00000ganga', '00000nr', '00000sn', '00000sri', '00000vadrdhaman', '00000vbdrdhaman', '0000bld', '0000bt', '0000laxmi', '0000nyati', '0000pl', '0000sn', '0001ab', '0001dd', '0001h', '0001st', '0002a', '0002f', '0002nd', '0003gfb', '0005th', '0007a', '0007th', '0008okhla', '0009th', '000a103', '000b', '000b30', '000bela', '000d', '000d59', '000dashmesh', '000n', '000s', '000sn', '000sn1a', '001005a', '001008a', '001017a', '001019cwing219atriumnextto', '00101a', '00101b', '00101iind', '00101sathy', '001024p2', '001027a2', '001029c', '00102a', '00102g', '00102sarafapartment', '001045p4', '00104a', '00104b', '00104chandralok', '00105b', '00105bell', '00105e', '00105mittal', '001061m', '00106d', '00107a', '0010803jhandewalan', '00108a', '00108e', '00109f', '0010a', '0010b', '0010c', '0010d', '0010f', '0010gemini', '0010j', '0010kailash', '0010rd', '0010s', '0010st', '0010t', '0010th', '001103a', '00110a', '00110roya

In [ ]:
# STEP 38 — STRICT ADDRESS BLOCKING TOKEN FILTER

def is_strong_blocking_token(token, max_freq=2000):

    token = str(token).lower()

    freq = (
        s2_token_freq.get(token, 0)
        + s3_token_freq.get(token, 0)
    )

    # Frequency
    if freq == 0 or freq > max_freq:
        return False

    # Length
    if len(token) < 4:
        return False

    # Must start with a letter
    if not token[0].isalpha():
        return False

    # Must contain at least 3 alphabetic characters
    alpha_count = sum(c.isalpha() for c in token)

    if alpha_count < 3:
        return False

    # Reject tokens dominated by digits
    digit_count = sum(c.isdigit() for c in token)

    if digit_count / len(token) > 0.25:
        return False

    # Reject common address words
    if token in COMMON_ADDRESS_WORDS:
        return False

    return True


strong_address_tokens = {
    token
    for token in all_address_tokens
    if is_strong_blocking_token(token)
}

print(
    "Strong blocking tokens:",
    len(strong_address_tokens)
)

print("\nExamples:")
print(sorted(strong_address_tokens)[:100])

Strong blocking tokens: 541703

Examples:
['a001904samarpanchskanakiasp', 'a005patrakarcolonyrdshree', 'a02great', 'a0304kaladeepapp', 'a0one', 'a1003platinumamaltaskanak', 'a1008mantraessence', 'a1012platinumamaltaskanak', 'a101aadeshwars', 'a104aadeshwars', 'a10ganesh', 'a1302udaygirikandivalieast', 'a1309udaygirikandivalieast', 'a15moneyplantnrbsnl', 'a1802flr18thawingsiddhivinayakvastuvishnu', 'a1802flr23thawingsiddhivinayakvastuvishnu', 'a1904samarpanchskanakiasp', 'a1906samarpanchskanakiasp', 'a1906utkarshnagarbhandupw', 'a1block', 'a1corian', 'a1flat', 'a1floor', 'a1ganesh', 'a1st', 'a1thirumangai', 'a1to', 'a2002hiranandani', 'a2007hiranandani', 'a211moneyplantnrbsnl', 'a21krishnacomplexamrel', 'a22industrial', 'a2akatra', 'a2ir', 'a2nd', 'a2rbon', 'a2towers', 'a2wadmukhwadi', 'a302kothapet', 'a304kaladeepapp', 'a311kaladeepapp', 'a313kothapet', 'a31stacross', 'a3302towerasociety', 'a34krishnacomplexamrel', 'a36sector', 'a3epip', 'a3ir', 'a3kashkand', 'a3rcapartment14stfloorraj

In [ ]:
# STEP 38 — INSPECT MISSED PAIRS AND THEIR ADDRESSES

# Show the first 20 still-missed pairs with full address information

cols = [
    "s1_id",
    "target_id",
    "source",
    "s1_address",
    "target_address"
]

display(
    missed_analysis[cols].head(20)
)

,s1_id,target_id,source,s1_address,target_address
0,S1-493193236,S3-464830319,S3,"Shop No. 2, Sharma Market Sector 37, Noida, Ga...","Shop No. 2, Noida, Gautam Buddha Nagar, UP"
1,S1-390796822,S3-247803230,S3,"Flat No-1008, X Wing, Eiffel City, Khed, Pune,...","Flat No-1007, X Wing, Eiffel City, Pune, Khed, MH"
2,S1-503957501,S3-181423094,S3,"1061 Americana Lane, Unit 1063, Mesquite, TX","001061 Americana Ln, # 1063, Mesquite CITY, Texas"
3,S1-244723389,S3-646630077,S3,"1/831, S-4 Vasundhara, Ghaziabad, Uttar Pradesh","H.no 371 1/831, S-4 Vasundhara, Ghaziabad, UP"
4,S1-287541726,S2-988436899,S2,"TX, 3260 Galloway Avenue, Mesquite, Unit 1008","2260 GALLOWAY AVENUE, MESQUITE, TX"
5,S1-424812889,S2-665316026,S2,"484, G. Floor, Haveli Haider Quili, Chandni Ch...","#119 484, G. FLOOR, HAVELI HAIDER QUILI, CHAND..."
6,S1-428041252,S2-360385940,S2,"Maharashtra, Thane, B-202 Tierra Casa, Thane, ...","PLOT 299 B-202 TIERRA CASA, UNIVIS LODHA, THAN..."
7,S1-300794758,S2-47169456,S2,"Punjab, Plot No F 93 & 102Phase Vii Focal Poin...",PLOT NO F 093 & 102PHASE VII FOCAL POINT LUDHI...
8,S1-770273967,S3-816881088,S3,"G I D C Plot No. 43, Nr. Mahadev Engineering, ...","G I D C Plot No. 44, Nr. Mahadev Engineering, ..."
9,S1-196201570,S2-29892224,S2,"Plot At Khasra 609 & 610, Village Tusyana, Noi...","PLOT AT KHASRA 0609 & 610, VILLAGE TUSYANA, NO..."


In [ ]:
# STEP 39 — ANALYZE SHARED NORMALIZED ADDRESS WORDS

import re
from collections import Counter

ADDRESS_STOPWORDS = {
    "road", "rd", "street", "st",
    "avenue", "ave", "lane", "ln",
    "drive", "dr", "boulevard", "blvd",
    "highway", "hwy",
    "building", "bldg",
    "floor", "fl",
    "house", "h",
    "number", "no",
    "plot",
    "sector", "sec",
    "block",
    "city", "district",
    "near", "nr",
    "opp", "opposite",
    "india",
    "state",
    "town",
    "village",
    "roadway",
    "place",
    "main",
    "north", "south",
    "east", "west",
    "first", "second", "third",
    "fourth",
    "phase", "stage",
    "unit",
    "floor",
    "flat",
    "wing"
}


def address_words(address):
    if pd.isna(address):
        return set()

    text = str(address).lower()

    # Extract alphabetic words and mixed alphanumeric words
    tokens = re.findall(r"[a-z]+", text)

    return {
        token
        for token in tokens
        if len(token) >= 4
        and token not in ADDRESS_STOPWORDS
    }


shared_word_analysis = []

for _, row in missed_analysis.iterrows():

    s1_words = address_words(row["s1_address"])
    target_words = address_words(row["target_address"])

    shared = s1_words.intersection(target_words)

    shared_word_analysis.append({
        "s1_id": row["s1_id"],
        "target_id": row["target_id"],
        "source": row["source"],
        "shared_words": sorted(shared),
        "shared_count": len(shared)
    })


shared_word_analysis = pd.DataFrame(shared_word_analysis)

print(
    "Missed pairs:",
    len(shared_word_analysis)
)

print("\nShared-word distribution:")
print(
    shared_word_analysis["shared_count"]
    .value_counts()
    .sort_index()
)

print("\nPairs with at least 1 shared word:",
      (shared_word_analysis["shared_count"] >= 1).sum())

print("\nPairs with at least 2 shared words:",
      (shared_word_analysis["shared_count"] >= 2).sum())

display(
    shared_word_analysis
    .sort_values("shared_count")
    .head(30)
)

Missed pairs: 169

Shared-word distribution:
shared_count
0     12
1     37
2     37
3     24
4     25
5     13
6     11
7      2
8      6
9      1
11     1
Name: count, dtype: int64

Pairs with at least 1 shared word: 157

Pairs with at least 2 shared words: 120


,s1_id,target_id,source,shared_words,shared_count
31,S1-673544849,S2-878880978,S2,[],0
36,S1-229574890,S2-266748285,S2,[],0
41,S1-394619513,S3-851166923,S3,[],0
96,S1-232978292,S3-818147807,S3,[],0
120,S1-472678371,S2-144001095,S2,[],0
124,S1-422604178,S2-399569699,S2,[],0
114,S1-837759376,S3-760783884,S3,[],0
115,S1-194230101,S2-564707664,S2,[],0
166,S1-905533273,S2-585582209,S2,[],0
157,S1-803997391,S2-107677081,S2,[],0


In [ ]:
# STEP 40 — TEST PER-SOURCE ADDRESS WORD FREQUENCIES

# Build frequency tables using the same word extraction
# that we just used for the missed-pair analysis.

s2_word_freq = Counter()
s3_word_freq = Counter()

for chunk in pd.read_csv(
    "/content/train_source2.tsv",
    sep="\t",
    chunksize=100_000
):
    for address in chunk["business_address"]:
        s2_word_freq.update(address_words(address))


for chunk in pd.read_csv(
    "/content/train_source3.tsv",
    sep="\t",
    chunksize=100_000
):
    for address in chunk["business_address"]:
        s3_word_freq.update(address_words(address))


thresholds = [10, 25, 50, 100, 250, 500, 1000, 2000]

results = []

for threshold in thresholds:

    recovered = 0

    for _, row in shared_word_analysis.iterrows():

        shared_words = row["shared_words"]
        source = row["source"]

        if source == "S2":
            freq_table = s2_word_freq
        else:
            freq_table = s3_word_freq

        has_rare_word = any(
            freq_table.get(word, 0) <= threshold
            for word in shared_words
        )

        if has_rare_word:
            recovered += 1

    results.append({
        "threshold": threshold,
        "recovered": recovered,
        "recall_on_missed": recovered / len(shared_word_analysis)
    })


threshold_results_address = pd.DataFrame(results)

display(threshold_results_address)

,threshold,recovered,recall_on_missed
0,10,19,0.112426
1,25,30,0.177515
2,50,41,0.242604
3,100,50,0.295858
4,250,65,0.384615
5,500,80,0.473373
6,1000,98,0.579882
7,2000,107,0.633136


In [ ]:
# STEP 41 — ANALYZE NAME SIMILARITY OF MISSED PAIRS

from difflib import SequenceMatcher

name_similarity_analysis = []

for _, row in missed_analysis.iterrows():

    s1_name = str(row["s1_name"])
    target_name = str(row["target_name"])

    similarity = SequenceMatcher(
        None,
        normalize_name_v2(s1_name),
        normalize_name_v2(target_name)
    ).ratio()

    name_similarity_analysis.append({
        "s1_id": row["s1_id"],
        "target_id": row["target_id"],
        "source": row["source"],
        "s1_name": s1_name,
        "target_name": target_name,
        "name_similarity": similarity
    })

name_similarity_analysis = pd.DataFrame(
    name_similarity_analysis
)

print("Missed pairs:", len(name_similarity_analysis))

print("\nSimilarity ranges:")

bins = [0, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]

print(
    pd.cut(
        name_similarity_analysis["name_similarity"],
        bins=bins,
        include_lowest=True
    ).value_counts()
    .sort_index()
)

print("\nHighest similarities:")
display(
    name_similarity_analysis
    .sort_values(
        "name_similarity",
        ascending=False
    )
    .head(30)
)

Missed pairs: 169

Similarity ranges:
name_similarity
(-0.001, 0.5]    72
(0.5, 0.6]        9
(0.6, 0.7]       24
(0.7, 0.75]      10
(0.75, 0.8]      20
(0.8, 0.85]       8
(0.85, 0.9]       5
(0.9, 0.95]      12
(0.95, 1.0]       9
Name: count, dtype: int64

Highest similarities:


,s1_id,target_id,source,s1_name,target_name,name_similarity
9,S1-196201570,S2-29892224,S2,Red Enterprises Pvt Ltd,Réd Enterprises Pvt Ltd,1.000000
23,S1-89545333,S3-499959015,S3,ZV Amc,ZV ámc,1.000000
36,S1-229574890,S2-266748285,S2,TE North LLC,TE Nórth LLC,1.000000
168,S1-87252645,S2-83209191,S2,New Life Cathedral,New Lfe Cathedral,0.971429
2,S1-503957501,S3-181423094,S3,Automated Industrial Ventures Inc,Aut0mated Industrial Ventures Inc,0.969697
49,S1-483002269,S2-716335718,S2,Womens Health Medicine Inc,w0mens health medicine ínc,0.961538
12,S1-841275752,S3-467197335,S3,South Apna Private Limited,S0uth Apna Private (Limited),0.961538
141,S1-15554340,S3-573237417,S3,Powerline Foods Pvt Ltd,P0werline Foods Pvt Ltd,0.956522
105,S1-732374765,S3-144254332,S3,East Infotech Limited,EAST lNFOTECH LIMITED,0.952381
6,S1-428041252,S2-360385940,S2,Thane Foundation Co,THANEFOUNDATION.COM,0.947368


In [ ]:
# STEP 42 — CHARACTER N-GRAM DIAGNOSTIC

def char_ngrams(text, n=3):
    text = normalize_name_v2(text)

    if len(text) < n:
        return {text}

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }


ngram_results = []

for _, row in missed_analysis.iterrows():

    s1_ngrams = char_ngrams(row["s1_name"], 3)
    target_ngrams = char_ngrams(row["target_name"], 3)

    shared = s1_ngrams.intersection(target_ngrams)

    union = s1_ngrams.union(target_ngrams)

    jaccard = (
        len(shared) / len(union)
        if union
        else 0
    )

    ngram_results.append({
        "s1_id": row["s1_id"],
        "target_id": row["target_id"],
        "source": row["source"],
        "shared_ngrams": len(shared),
        "jaccard": jaccard
    })


ngram_results = pd.DataFrame(ngram_results)

print("Missed pairs:", len(ngram_results))

print("\nJaccard distribution:")

print(
    pd.cut(
        ngram_results["jaccard"],
        bins=[0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0],
        include_lowest=True
    )
    .value_counts()
    .sort_index()
)

print("\nHighest n-gram similarities:")

display(
    ngram_results
    .sort_values("jaccard", ascending=False)
    .head(30)
)

Missed pairs: 169

Jaccard distribution:
jaccard
(-0.001, 0.1]    65
(0.1, 0.2]        8
(0.2, 0.3]       11
(0.3, 0.4]       18
(0.4, 0.5]       15
(0.5, 0.6]        7
(0.6, 0.7]       16
(0.7, 0.8]        9
(0.8, 0.9]       16
(0.9, 1.0]        4
Name: count, dtype: int64

Highest n-gram similarities:


,s1_id,target_id,source,shared_ngrams,jaccard
9,S1-196201570,S2-29892224,S2,21,1.000000
23,S1-89545333,S3-499959015,S3,4,1.000000
36,S1-229574890,S2-266748285,S2,10,1.000000
98,S1-584898715,S3-850198825,S3,29,0.935484
91,S1-841275752,S3-43374200,S3,24,0.888889
97,S1-960938668,S3-951187495,S3,29,0.878788
142,S1-677991820,S2-127736375,S2,28,0.875000
45,S1-483002269,S2-702416722,S2,24,0.857143
117,S1-476474673,S2-745079672,S2,36,0.857143
70,S1-413878302,S2-724659872,S2,28,0.848485


In [ ]:
# STEP 43 — TEST CHARACTER N-GRAM BLOCKING THRESHOLDS

thresholds = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90
]

ngram_threshold_results = []

for threshold in thresholds:

    recovered = (
        ngram_results["jaccard"] >= threshold
    ).sum()

    ngram_threshold_results.append({
        "threshold": threshold,
        "recovered": int(recovered),
        "recall_on_missed": recovered / len(ngram_results)
    })


ngram_threshold_results = pd.DataFrame(
    ngram_threshold_results
)

display(ngram_threshold_results)

,threshold,recovered,recall_on_missed
0,0.30,87,0.514793
1,0.40,71,0.420118
2,0.50,56,0.331361
3,0.60,46,0.272189
4,0.70,34,0.201183
5,0.75,23,0.136095
6,0.80,20,0.118343
7,0.85,9,0.053254
8,0.90,4,0.023669


In [ ]:
# STEP 44 — CHARACTER N-GRAM FREQUENCY ANALYSIS

from collections import Counter

s2_ngram_freq = Counter()
s3_ngram_freq = Counter()

# Count 3-grams in S2 names
for chunk in pd.read_csv(
    "/content/train_source2.tsv",
    sep="\t",
    chunksize=100_000
):
    for name in chunk["business_name"]:
        s2_ngram_freq.update(
            char_ngrams(name, 3)
        )


# Count 3-grams in S3 names
for chunk in pd.read_csv(
    "/content/train_source3.tsv",
    sep="\t",
    chunksize=100_000
):
    for name in chunk["business_name"]:
        s3_ngram_freq.update(
            char_ngrams(name, 3)
        )


# Combined frequency
all_ngram_freq = Counter()

all_ngram_freq.update(s2_ngram_freq)
all_ngram_freq.update(s3_ngram_freq)


print(
    "Unique 3-grams:",
    len(all_ngram_freq)
)

print("\nMost common 3-grams:")
print(
    all_ngram_freq.most_common(30)
)

Unique 3-grams: 32523

Most common 3-grams:
[('ate', 1756701), (' co', 1634835), (' in', 1568789), (' pr', 1433010), ('pri', 1415043), ('ted', 1402068), ('mit', 1365997), ('ite', 1357810), ('vat', 1290568), (' li', 1279075), ('riv', 1274587), ('te ', 1260515), ('iva', 1259261), ('imi', 1240555), ('lim', 1240273), ('e l', 1216062), (' ll', 1152035), ('llc', 1150884), ('ent', 1087296), ('al ', 974900), ('ter', 962403), ('inc', 916532), ('ion', 914591), ('es ', 858040), ('ers', 846312), ('ltd', 826630), (' lt', 801068), ('nte', 793228), ('s p', 790965), ('ing', 780401)]


In [ ]:
# STEP 45 — FIND USEFUL RARE 3-GRAMS

ngram_thresholds = [
    10,
    25,
    50,
    100,
    250,
    500,
    1000,
    2000,
    5000,
    10000
]

ngram_vocab_results = []

for threshold in ngram_thresholds:

    count = sum(
        1
        for ngram, freq in all_ngram_freq.items()
        if freq <= threshold
    )

    ngram_vocab_results.append({
        "max_frequency": threshold,
        "usable_ngrams": count
    })


ngram_vocab_results = pd.DataFrame(
    ngram_vocab_results
)

display(ngram_vocab_results)

,max_frequency,usable_ngrams
0,10,9812
1,25,13600
2,50,16117
3,100,18228
4,250,20866
5,500,23093
6,1000,26054
7,2000,27710
8,5000,29372
9,10000,30188


In [ ]:
# Step 46: Test rare 3-gram blocking recall on missed pairs

thresholds = [10, 25, 50, 100, 250, 500, 1000]

def get_char_ngrams(text, n=3):
    text = normalize_name_v2(text)
    return set(text[i:i+n] for i in range(len(text) - n + 1))


results = []

for threshold in thresholds:
    recovered = 0

    for _, row in missed_analysis.iterrows():

        s1_ngrams = get_char_ngrams(row["s1_name"])
        target_ngrams = get_char_ngrams(row["target_name"])

        # Keep only n-grams that are rare enough globally
        shared_rare = [
            ng for ng in (s1_ngrams & target_ngrams)
            if all_ngram_freq.get(ng, 0) <= threshold
        ]

        if len(shared_rare) > 0:
            recovered += 1

    recall = recovered / len(missed_analysis)

    results.append({
        "max_frequency": threshold,
        "recovered": recovered,
        "missed_remaining": len(missed_analysis) - recovered,
        "recall_on_missed": recall
    })

pd.DataFrame(results)

,max_frequency,recovered,missed_remaining,recall_on_missed
0,10,0,169,0.000000
1,25,0,169,0.000000
2,50,0,169,0.000000
3,100,3,166,0.017751
4,250,5,164,0.029586
5,500,6,163,0.035503
6,1000,13,156,0.076923


In [ ]:
candidate_pairs_corrected.columns

Index(['entity_id_s1', 'entity_id_matched', 'business_name_matched',
       'business_address_matched', 'country_matched', 'source', 'name_norm',
       'address_norm'],
      dtype='object')

In [1]:
print("candidate_pairs_corrected:",
      candidate_pairs_corrected.shape if 'candidate_pairs_corrected' in globals() else "NOT FOUND")

print("missed_analysis:",
      missed_analysis.shape if 'missed_analysis' in globals() else "NOT FOUND")

candidate_pairs_corrected: NOT FOUND
missed_analysis: NOT FOUND


In [2]:
# Check which important variables currently exist

important_vars = [
    "train_source1",
    "train_source2",
    "train_source3",
    "train_ground_truth",
    "blocking_s1",
    "candidate_s2",
    "candidate_s3",
    "candidate_pairs_corrected",
    "missed_df",
    "missed_analysis",
    "training_pairs",
    "feature_data"
]

for var in important_vars:
    print(f"{var}: {'FOUND' if var in globals() else 'NOT FOUND'}")

train_source1: NOT FOUND
train_source2: NOT FOUND
train_source3: NOT FOUND
train_ground_truth: NOT FOUND
blocking_s1: NOT FOUND
candidate_s2: NOT FOUND
candidate_s3: NOT FOUND
candidate_pairs_corrected: NOT FOUND
missed_df: NOT FOUND
missed_analysis: NOT FOUND
training_pairs: NOT FOUND
feature_data: NOT FOUND


In [3]:
import os

files = [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv"
]

for f in files:
    print(f"{f}: {'FOUND' if os.path.exists(f) else 'NOT FOUND'}")

train_source1.tsv: NOT FOUND
train_source2.tsv: NOT FOUND
train_source3.tsv: NOT FOUND
train_ground_truth.tsv: NOT FOUND


In [4]:
import os
import pandas as pd

files = [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv"
]

for f in files:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 ** 2)
        print(f"✅ {f} — {size_mb:.1f} MB")
    else:
        print(f"❌ {f} — NOT FOUND")

❌ train_source1.tsv — NOT FOUND
❌ train_source2.tsv — NOT FOUND
❌ train_source3.tsv — NOT FOUND
❌ train_ground_truth.tsv — NOT FOUND


In [5]:
import os

files = [
    "/train_source1.tsv",
    "/train_source2.tsv",
    "/train_source3.tsv",
    "/train_ground_truth.tsv"
]

for f in files:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 ** 2)
        print(f"✅ {f} — {size_mb:.1f} MB")
    else:
        print(f"❌ {f} — NOT FOUND")

✅ /train_source1.tsv — 200.3 MB
✅ /train_source2.tsv — 466.6 MB
✅ /train_source3.tsv — 480.4 MB
✅ /train_ground_truth.tsv — 121.1 MB


In [7]:
from google.colab import drive

drive.mount('/content/drive')

print("✅ Google Drive mounted successfully")

Mounted at /content/drive
✅ Google Drive mounted successfully


In [8]:
import os
import shutil

# Create a permanent project folder in Google Drive
PROJECT_DIR = "/content/drive/MyDrive/Amazon_ML_Challenge"

os.makedirs(PROJECT_DIR, exist_ok=True)

# Copy the uploaded TSV files from the current Colab runtime to Drive
files = [
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv",
    "train_ground_truth.tsv"
]

for f in files:
    src = "/" + f
    dst = os.path.join(PROJECT_DIR, f)

    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"✅ Saved: {f}")
    else:
        print(f"❌ Not found: {src}")

print("\n📁 Project folder:")
print(PROJECT_DIR)

✅ Saved: train_source1.tsv
✅ Saved: train_source2.tsv
✅ Saved: train_source3.tsv
✅ Saved: train_ground_truth.tsv

📁 Project folder:
/content/drive/MyDrive/Amazon_ML_Challenge


In [9]:
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/Amazon_ML_Challenge"

# Load training datasets
train_source1 = pd.read_csv(
    f"{PROJECT_DIR}/train_source1.tsv",
    sep="\t"
)

train_source2 = pd.read_csv(
    f"{PROJECT_DIR}/train_source2.tsv",
    sep="\t"
)

train_source3 = pd.read_csv(
    f"{PROJECT_DIR}/train_source3.tsv",
    sep="\t"
)

train_ground_truth = pd.read_csv(
    f"{PROJECT_DIR}/train_ground_truth.tsv",
    sep="\t"
)

print("✅ All datasets loaded")

print("\nShapes:")
print("Source 1:", train_source1.shape)
print("Source 2:", train_source2.shape)
print("Source 3:", train_source3.shape)
print("Ground Truth:", train_ground_truth.shape)

✅ All datasets loaded

Shapes:
Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground Truth: (2206821, 2)


In [10]:
import os
import pandas as pd

# Development sample size
SAMPLE_SIZE = 500
RANDOM_STATE = 42

# Create development folder
DEV_DIR = os.path.join(PROJECT_DIR, "development")
os.makedirs(DEV_DIR, exist_ok=True)

# Reproducible 500 Source-1 sample
blocking_s1 = train_source1.sample(
    n=SAMPLE_SIZE,
    random_state=RANDOM_STATE
).copy()

# Get the ground-truth records for these 500 S1 entities
dev_s1_ids = set(blocking_s1["entity_id"])

dev_ground_truth = train_ground_truth[
    train_ground_truth["source1_entity_id"].isin(dev_s1_ids)
].copy()

# Save checkpoints to Drive
blocking_s1.to_csv(
    os.path.join(DEV_DIR, "blocking_s1.csv"),
    index=False
)

dev_ground_truth.to_csv(
    os.path.join(DEV_DIR, "dev_ground_truth.csv"),
    index=False
)

print("✅ Development sample created")
print("S1 sample:", blocking_s1.shape)
print("Ground-truth rows:", dev_ground_truth.shape)

print("\nSaved to:")
print(DEV_DIR)

✅ Development sample created
S1 sample: (500, 4)
Ground-truth rows: (500, 2)

Saved to:
/content/drive/MyDrive/Amazon_ML_Challenge/development


In [11]:
# Step 5: Recreate all known positive S1 -> S2/S3 pairs

positive_rows = []

for _, row in dev_ground_truth.iterrows():

    s1_id = row["source1_entity_id"]
    matched_ids = str(row["matched_entity_ids"]).split(",")

    for target_id in matched_ids:

        target_id = target_id.strip()

        if not target_id:
            continue

        # Determine whether the true match belongs to Source 2 or Source 3
        if target_id.startswith("S2-"):
            source = "S2"
        elif target_id.startswith("S3-"):
            source = "S3"
        else:
            continue

        positive_rows.append({
            "entity_id_s1": s1_id,
            "entity_id_matched": target_id,
            "source": source,
            "label": 1
        })

positive_pairs = pd.DataFrame(positive_rows)

# Remove accidental duplicates if any
positive_pairs = positive_pairs.drop_duplicates(
    subset=["entity_id_s1", "entity_id_matched"]
).reset_index(drop=True)

# Save checkpoint
positive_pairs.to_csv(
    os.path.join(DEV_DIR, "positive_pairs.csv"),
    index=False
)

print("✅ Positive pairs recreated")
print("Shape:", positive_pairs.shape)

print("\nBy source:")
print(positive_pairs["source"].value_counts())

print("\nSaved:")
print(os.path.join(DEV_DIR, "positive_pairs.csv"))

✅ Positive pairs recreated
Shape: (1705, 4)

By source:
source
S3    907
S2    798
Name: count, dtype: int64

Saved:
/content/drive/MyDrive/Amazon_ML_Challenge/development/positive_pairs.csv


In [12]:
# Step 5A: Validate positive pairs

print("Total positive pairs:", len(positive_pairs))

print("Unique S1 entities:",
      positive_pairs["entity_id_s1"].nunique())

print("Unique S1 -> target pairs:",
      positive_pairs[
          ["entity_id_s1", "entity_id_matched"]
      ].drop_duplicates().shape[0])

print("\nMissing values:")
print(positive_pairs.isna().sum())

print("\nMatches per S1:")
print(
    positive_pairs
    .groupby("entity_id_s1")
    .size()
    .describe()
)

Total positive pairs: 1705
Unique S1 entities: 462
Unique S1 -> target pairs: 1705

Missing values:
entity_id_s1         0
entity_id_matched    0
source               0
label                0
dtype: int64

Matches per S1:
count    462.000000
mean       3.690476
std        1.594961
min        1.000000
25%        2.000000
50%        4.000000
75%        5.000000
max       10.000000
dtype: float64


In [13]:
import re
import unicodedata


def normalize_name_v2(text):
    """
    Normalize business names:
    - lowercase
    - remove accents
    - replace punctuation with spaces
    - normalize whitespace
    """
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Unicode normalization
    text = unicodedata.normalize("NFKD", text)

    # Remove accent marks
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    # Replace non-alphanumeric characters with spaces
    text = re.sub(r"[^a-z0-9]+", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_address(text):
    """
    Normalize business addresses.
    """
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize("NFKD", text)

    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


print("✅ Normalization functions recreated")

# Quick sanity check
examples = [
    "TE North LLC",
    "TE Nórth LLC",
    "T+ Century LLC",
    "The T+ Céntury LLC"
]

for x in examples:
    print(f"{x}  ->  {normalize_name_v2(x)}")

✅ Normalization functions recreated
TE North LLC  ->  te north llc
TE Nórth LLC  ->  te north llc
T+ Century LLC  ->  t century llc
The T+ Céntury LLC  ->  the t century llc


In [14]:
from collections import defaultdict


def get_address_numbers(text):
    """Extract numeric tokens from a normalized address."""
    if not text:
        return set()

    return set(re.findall(r"\d+", text))


def build_query_maps(s1_df):
    """
    Build lookup keys for the Source-1 development entities.
    Each key maps to the S1 IDs that contain it.
    """

    maps = {
        "name_exact": defaultdict(set),
        "name_prefix": defaultdict(set),
        "address_exact": defaultdict(set),
        "address_num": defaultdict(set)
    }

    for _, row in s1_df.iterrows():

        s1_id = row["entity_id"]

        name = normalize_name_v2(row["business_name"])
        address = normalize_address(row["business_address"])

        # Exact normalized name
        if name:
            maps["name_exact"][name].add(s1_id)

            # First 4 normalized characters
            prefix = name[:4]

            if prefix:
                maps["name_prefix"][prefix].add(s1_id)

        # Exact normalized address
        if address:
            maps["address_exact"][address].add(s1_id)

        # Address numbers
        for number in get_address_numbers(address):
            maps["address_num"][number].add(s1_id)

    return maps


query_maps = build_query_maps(blocking_s1)

print("✅ Blocking maps created")

for key, mapping in query_maps.items():
    print(
        f"{key}: "
        f"{len(mapping):,} unique keys"
    )

✅ Blocking maps created
name_exact: 499 unique keys
name_prefix: 388 unique keys
address_exact: 500 unique keys
address_num: 444 unique keys


In [ ]:
def generate_candidate_pairs(source_name, file_path, query_maps):
    """
    Generate candidate S1 -> target pairs using the four blocking rules.
    Reads the large target file in chunks to avoid loading another
    5M-row dataframe into memory.
    """

    candidate_chunks = []

    usecols = [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            file_path,
            sep="\t",
            usecols=usecols,
            chunksize=200_000
        )
    ):

        # Normalize target records
        chunk["name_norm"] = (
            chunk["business_name"]
            .map(normalize_name_v2)
        )

        chunk["address_norm"] = (
            chunk["business_address"]
            .map(normalize_address)
        )

        # Collect target IDs that match any S1 blocking key
        candidate_rows = []

        for idx, row in chunk.iterrows():

            target_id = row["entity_id"]

            name = row["name_norm"]
            address = row["address_norm"]

            matched_s1 = set()

            # 1. Exact name
            if name in query_maps["name_exact"]:
                matched_s1.update(
                    query_maps["name_exact"][name]
                )

            # 2. Name prefix
            prefix = name[:4]

            if prefix in query_maps["name_prefix"]:
                matched_s1.update(
                    query_maps["name_prefix"][prefix]
                )

            # 3. Exact address
            if address in query_maps["address_exact"]:
                matched_s1.update(
                    query_maps["address_exact"][address]
                )

            # 4. Address number
            for number in get_address_numbers(address):

                if number in query_maps["address_num"]:
                    matched_s1.update(
                        query_maps["address_num"][number]
                    )

            # Create S1 -> target candidate rows
            for s1_id in matched_s1:

                candidate_rows.append({
                    "entity_id_s1": s1_id,
                    "entity_id_matched": target_id,
                    "business_name_matched": row["business_name"],
                    "business_address_matched": row["business_address"],
                    "country_matched": row["country"],
                    "source": source_name,
                    "name_norm": name,
                    "address_norm": address
                })

        if candidate_rows:
            candidate_chunks.append(
                pd.DataFrame(candidate_rows)
            )

        if (chunk_no + 1) % 5 == 0:
            print(
                f"{source_name}: "
                f"processed {(chunk_no + 1) * 200_000:,} rows"
            )

    if not candidate_chunks:
        return pd.DataFrame()

    return pd.concat(
        candidate_chunks,
        ignore_index=True
    )


print("Starting S2 candidate generation...")

candidate_s2 = generate_candidate_pairs(
    "S2",
    f"{PROJECT_DIR}/train_source2.tsv",
    query_maps
)

print("\nS2 candidates:", candidate_s2.shape)

print("\nStarting S3 candidate generation...")

candidate_s3 = generate_candidate_pairs(
    "S3",
    f"{PROJECT_DIR}/train_source3.tsv",
    query_maps
)

print("\nS3 candidates:", candidate_s3.shape)

Starting S2 candidate generation...
S2: processed 1,000,000 rows
S2: processed 2,000,000 rows
S2: processed 3,000,000 rows
S2: processed 4,000,000 rows
S2: processed 5,000,000 rows

S2 candidates: (49859159, 8)

Starting S3 candidate generation...
S3: processed 1,000,000 rows
S3: processed 2,000,000 rows
S3: processed 3,000,000 rows


In [1]:
# Remove the oversized candidate data

if "candidate_s2" in globals():
    del candidate_s2

if "candidate_s3" in globals():
    del candidate_s3

import gc
gc.collect()

print("✅ Oversized candidate data cleared")

✅ Oversized candidate data cleared


In [2]:
# Step 6A: Test different name-prefix lengths
# WITHOUT generating the full candidate set.

prefix_lengths = [4, 6, 8, 10]

# ---------------------------------------------------------
# Build S1 lookup information
# ---------------------------------------------------------

s1_lookup = {}

for _, row in blocking_s1.iterrows():

    s1_id = row["entity_id"]

    name = normalize_name_v2(row["business_name"])
    address = normalize_address(row["business_address"])

    s1_lookup[s1_id] = {
        "name": name,
        "address": address,
        "numbers": get_address_numbers(address)
    }


# ---------------------------------------------------------
# True target IDs from our 1,705 positive pairs
# ---------------------------------------------------------

true_targets_s2 = set(
    positive_pairs.loc[
        positive_pairs["source"] == "S2",
        "entity_id_matched"
    ]
)

true_targets_s3 = set(
    positive_pairs.loc[
        positive_pairs["source"] == "S3",
        "entity_id_matched"
    ]
)

print("True S2 targets:", len(true_targets_s2))
print("True S3 targets:", len(true_targets_s3))


# ---------------------------------------------------------
# Function to evaluate blockers against TRUE pairs
# ---------------------------------------------------------

def evaluate_blockers(source_name, file_path, true_target_ids):

    # Results:
    # each positive pair will be marked if any blocker recovers it
    recovered = {
        (row["entity_id_s1"], row["entity_id_matched"]): set()
        for _, row in positive_pairs[
            positive_pairs["source"] == source_name
        ].iterrows()
    }

    # Build target records ONLY for the true target IDs
    target_records = {}

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        usecols=[
            "entity_id",
            "business_name",
            "business_address"
        ],
        chunksize=200_000
    ):

        # Keep only known true targets
        chunk = chunk[
            chunk["entity_id"].isin(true_target_ids)
        ]

        for _, row in chunk.iterrows():

            target_id = row["entity_id"]

            target_records[target_id] = {
                "name": normalize_name_v2(row["business_name"]),
                "address": normalize_address(row["business_address"])
            }

    print(
        f"{source_name}: recovered "
        f"{len(target_records):,} true target records"
    )

    # Evaluate every true pair
    source_pairs = positive_pairs[
        positive_pairs["source"] == source_name
    ]

    for _, pair in source_pairs.iterrows():

        s1_id = pair["entity_id_s1"]
        target_id = pair["entity_id_matched"]

        s1 = s1_lookup[s1_id]
        target = target_records.get(target_id)

        if target is None:
            continue

        # Exact name
        if (
            s1["name"]
            and s1["name"] == target["name"]
        ):
            recovered[(s1_id, target_id)].add("name_exact")

        # Exact address
        if (
            s1["address"]
            and s1["address"] == target["address"]
        ):
            recovered[(s1_id, target_id)].add("address_exact")

        # Shared address number
        target_numbers = get_address_numbers(
            target["address"]
        )

        if s1["numbers"] & target_numbers:
            recovered[(s1_id, target_id)].add("address_num")

        # Name prefixes
        for length in prefix_lengths:

            if (
                len(s1["name"]) >= length
                and len(target["name"]) >= length
                and s1["name"][:length]
                == target["name"][:length]
            ):
                recovered[(s1_id, target_id)].add(
                    f"name_prefix_{length}"
                )

    return recovered


# ---------------------------------------------------------
# Evaluate S2 and S3
# ---------------------------------------------------------

print("\nEvaluating S2...")
recovered_s2 = evaluate_blockers(
    "S2",
    f"{PROJECT_DIR}/train_source2.tsv",
    true_targets_s2
)

print("\nEvaluating S3...")
recovered_s3 = evaluate_blockers(
    "S3",
    f"{PROJECT_DIR}/train_source3.tsv",
    true_targets_s3
)


# ---------------------------------------------------------
# Combine and calculate recall
# ---------------------------------------------------------

all_recovered = {
    key: recovered_s2.get(key, set())
    for key in recovered_s2
}

all_recovered.update(recovered_s3)


blockers = [
    "name_exact",
    "address_exact",
    "address_num",
    "name_prefix_4",
    "name_prefix_6",
    "name_prefix_8",
    "name_prefix_10"
]

results = []

total_pairs = len(positive_pairs)

for blocker in blockers:

    count = sum(
        blocker in methods
        for methods in all_recovered.values()
    )

    results.append({
        "blocker": blocker,
        "recovered": count,
        "total_positive_pairs": total_pairs,
        "recall": count / total_pairs
    })


# Combined recall for prefix + existing blockers
for prefix in prefix_lengths:

    blocker_name = f"name_prefix_{prefix}"

    count = sum(
        bool(
            methods.intersection({
                "name_exact",
                "address_exact",
                "address_num",
                blocker_name
            })
        )
        for methods in all_recovered.values()
    )

    results.append({
        "blocker": f"combined_up_to_prefix_{prefix}",
        "recovered": count,
        "total_positive_pairs": total_pairs,
        "recall": count / total_pairs
    })


blocker_results = pd.DataFrame(results)

print("\n========== BLOCKER RECALL ==========")
print(blocker_results.to_string(index=False))

NameError: name 'blocking_s1' is not defined

In [3]:
# Restore the development S1 sample from Google Drive

blocking_s1 = pd.read_csv(
    os.path.join(DEV_DIR, "blocking_s1.csv")
)

print("✅ blocking_s1 restored")
print("Shape:", blocking_s1.shape)

NameError: name 'pd' is not defined

In [4]:
import pandas as pd
import os

PROJECT_DIR = "/content/drive/MyDrive/Amazon_ML_Challenge"
DEV_DIR = os.path.join(PROJECT_DIR, "development")

print("Drive checkpoint exists:",
      os.path.exists(os.path.join(DEV_DIR, "blocking_s1.csv")))

blocking_s1 = pd.read_csv(
    os.path.join(DEV_DIR, "blocking_s1.csv")
)

print("✅ blocking_s1 restored")
print("Shape:", blocking_s1.shape)

Drive checkpoint exists: True
✅ blocking_s1 restored
Shape: (500, 4)


In [5]:
positive_pairs = pd.read_csv(
    os.path.join(DEV_DIR, "positive_pairs.csv")
)

print("✅ positive_pairs restored")
print("Shape:", positive_pairs.shape)

print("\nBy source:")
print(positive_pairs["source"].value_counts())

print("\nUnique S1 entities:",
      positive_pairs["entity_id_s1"].nunique())

✅ positive_pairs restored
Shape: (1705, 4)

By source:
source
S3    907
S2    798
Name: count, dtype: int64

Unique S1 entities: 462


In [6]:
import re
import unicodedata


def normalize_name_v2(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize("NFKD", text)

    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_address(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize("NFKD", text)

    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def get_address_numbers(text):
    if not text:
        return set()

    return set(re.findall(r"\d+", text))


print("✅ Helper functions restored")

# Quick test
print(normalize_name_v2("TE Nórth LLC"))
print(normalize_address("Flat No. 1008, X Wing, Pune"))
print(get_address_numbers("Flat No. 1008, X Wing"))

✅ Helper functions restored
te north llc
flat no 1008 x wing pune
{'1008'}


In [7]:
from collections import defaultdict

# Prefix lengths we want to test
prefix_lengths = [4, 6, 8, 10]


# ---------------------------------------------------------
# 1. Prepare S1 lookup
# ---------------------------------------------------------

s1_lookup = {}

for _, row in blocking_s1.iterrows():

    s1_id = row["entity_id"]

    s1_lookup[s1_id] = {
        "name": normalize_name_v2(row["business_name"]),
        "address": normalize_address(row["business_address"])
    }


# ---------------------------------------------------------
# 2. Get the TRUE target IDs
# ---------------------------------------------------------

s2_pairs = positive_pairs[
    positive_pairs["source"] == "S2"
]

s3_pairs = positive_pairs[
    positive_pairs["source"] == "S3"
]

true_targets_s2 = set(s2_pairs["entity_id_matched"])
true_targets_s3 = set(s3_pairs["entity_id_matched"])

print("True S2 targets:", len(true_targets_s2))
print("True S3 targets:", len(true_targets_s3))


# ---------------------------------------------------------
# 3. Load ONLY those true target records
# ---------------------------------------------------------

def load_true_targets(file_path, true_ids):

    target_records = {}

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        usecols=[
            "entity_id",
            "business_name",
            "business_address"
        ],
        chunksize=200_000
    ):

        # Keep ONLY the known positive target IDs
        chunk = chunk[
            chunk["entity_id"].isin(true_ids)
        ]

        for _, row in chunk.iterrows():

            target_records[row["entity_id"]] = {
                "name": normalize_name_v2(
                    row["business_name"]
                ),
                "address": normalize_address(
                    row["business_address"]
                )
            }

    return target_records


print("\nLoading true S2 targets...")

target_s2 = load_true_targets(
    f"{PROJECT_DIR}/train_source2.tsv",
    true_targets_s2
)

print("Loaded S2:", len(target_s2))


print("\nLoading true S3 targets...")

target_s3 = load_true_targets(
    f"{PROJECT_DIR}/train_source3.tsv",
    true_targets_s3
)

print("Loaded S3:", len(target_s3))


# ---------------------------------------------------------
# 4. Evaluate each true pair
# ---------------------------------------------------------

all_pairs = pd.concat(
    [s2_pairs, s3_pairs],
    ignore_index=True
)

recovered_methods = defaultdict(set)


for _, pair in all_pairs.iterrows():

    s1_id = pair["entity_id_s1"]
    target_id = pair["entity_id_matched"]
    source = pair["source"]

    s1 = s1_lookup[s1_id]

    target = (
        target_s2[target_id]
        if source == "S2"
        else target_s3[target_id]
    )

    s1_name = s1["name"]
    target_name = target["name"]

    s1_address = s1["address"]
    target_address = target["address"]


    # Exact normalized name
    if (
        s1_name
        and s1_name == target_name
    ):
        recovered_methods[
            (s1_id, target_id)
        ].add("name_exact")


    # Exact normalized address
    if (
        s1_address
        and s1_address == target_address
    ):
        recovered_methods[
            (s1_id, target_id)
        ].add("address_exact")


    # Shared address number
    s1_numbers = get_address_numbers(s1_address)
    target_numbers = get_address_numbers(target_address)

    if s1_numbers & target_numbers:
        recovered_methods[
            (s1_id, target_id)
        ].add("address_num")


    # Name prefixes
    for length in prefix_lengths:

        if (
            len(s1_name) >= length
            and len(target_name) >= length
            and s1_name[:length] == target_name[:length]
        ):
            recovered_methods[
                (s1_id, target_id)
            ].add(f"name_prefix_{length}")


# ---------------------------------------------------------
# 5. Calculate individual and combined recall
# ---------------------------------------------------------

blockers = [
    "name_exact",
    "address_exact",
    "address_num",
    "name_prefix_4",
    "name_prefix_6",
    "name_prefix_8",
    "name_prefix_10"
]

results = []

total = len(all_pairs)

for blocker in blockers:

    recovered = sum(
        blocker in methods
        for methods in recovered_methods.values()
    )

    results.append({
        "blocker": blocker,
        "recovered": recovered,
        "recall": recovered / total
    })


for prefix in prefix_lengths:

    prefix_blocker = f"name_prefix_{prefix}"

    recovered = sum(
        bool(
            methods.intersection({
                "name_exact",
                "address_exact",
                "address_num",
                prefix_blocker
            })
        )
        for methods in recovered_methods.values()
    )

    results.append({
        "blocker": f"combined_with_prefix_{prefix}",
        "recovered": recovered,
        "recall": recovered / total
    })


blocker_results = pd.DataFrame(results)

print("\n======================================")
print("        BLOCKER RECALL RESULTS")
print("======================================\n")

print(
    blocker_results.to_string(index=False)
)

True S2 targets: 798
True S3 targets: 907

Loading true S2 targets...
Loaded S2: 798

Loading true S3 targets...
Loaded S3: 907

        BLOCKER RECALL RESULTS

                blocker  recovered   recall
             name_exact        440 0.258065
          address_exact        134 0.078592
            address_num       1364 0.800000
          name_prefix_4       1289 0.756012
          name_prefix_6       1226 0.719062
          name_prefix_8       1129 0.662170
         name_prefix_10       1015 0.595308
 combined_with_prefix_4       1634 0.958358
 combined_with_prefix_6       1620 0.950147
 combined_with_prefix_8       1601 0.939003
combined_with_prefix_10       1583 0.928446


In [8]:
# Step 8: Measure candidate volume for different prefix lengths
# WITHOUT storing the candidate dataframe.

from collections import defaultdict

prefix_lengths_test = [6, 8, 10]

# Build S1 lookup maps
name_exact_map = defaultdict(set)
name_prefix_maps = {
    length: defaultdict(set)
    for length in prefix_lengths_test
}
address_exact_map = defaultdict(set)
address_num_map = defaultdict(set)

for _, row in blocking_s1.iterrows():

    s1_id = row["entity_id"]

    name = normalize_name_v2(row["business_name"])
    address = normalize_address(row["business_address"])

    # Exact name
    if name:
        name_exact_map[name].add(s1_id)

        for length in prefix_lengths_test:
            if len(name) >= length:
                name_prefix_maps[length][
                    name[:length]
                ].add(s1_id)

    # Exact address
    if address:
        address_exact_map[address].add(s1_id)

    # Address numbers
    for number in get_address_numbers(address):
        address_num_map[number].add(s1_id)


def count_candidates(
    file_path,
    prefix_length
):

    total_pairs = 0
    total_rows_with_candidates = 0

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            file_path,
            sep="\t",
            usecols=[
                "entity_id",
                "business_name",
                "business_address"
            ],
            chunksize=200_000
        )
    ):

        pair_count = 0
        rows_with_candidates = 0

        for _, row in chunk.iterrows():

            name = normalize_name_v2(
                row["business_name"]
            )

            address = normalize_address(
                row["business_address"]
            )

            matched_s1 = set()

            # Exact name
            if name in name_exact_map:
                matched_s1.update(
                    name_exact_map[name]
                )

            # Prefix
            if len(name) >= prefix_length:
                prefix = name[:prefix_length]

                if prefix in name_prefix_maps[prefix_length]:
                    matched_s1.update(
                        name_prefix_maps[prefix_length][prefix]
                    )

            # Exact address
            if address in address_exact_map:
                matched_s1.update(
                    address_exact_map[address]
                )

            # Address number
            for number in get_address_numbers(address):

                if number in address_num_map:
                    matched_s1.update(
                        address_num_map[number]
                    )

            if matched_s1:
                rows_with_candidates += 1
                pair_count += len(matched_s1)

        total_pairs += pair_count
        total_rows_with_candidates += rows_with_candidates

        if (chunk_no + 1) % 5 == 0:
            print(
                f"Processed {(chunk_no + 1) * 200_000:,} rows..."
            )

    return total_pairs, total_rows_with_candidates


volume_results = []

for prefix_length in prefix_lengths_test:

    print(
        f"\n========== Prefix {prefix_length} =========="
    )

    s2_pairs_count, s2_rows = count_candidates(
        f"{PROJECT_DIR}/train_source2.tsv",
        prefix_length
    )

    print(
        f"S2 candidate pairs: {s2_pairs_count:,}"
    )

    s3_pairs_count, s3_rows = count_candidates(
        f"{PROJECT_DIR}/train_source3.tsv",
        prefix_length
    )

    print(
        f"S3 candidate pairs: {s3_pairs_count:,}"
    )

    volume_results.append({
        "prefix_length": prefix_length,
        "s2_candidate_pairs": s2_pairs_count,
        "s3_candidate_pairs": s3_pairs_count,
        "total_candidate_pairs": (
            s2_pairs_count + s3_pairs_count
        )
    })


candidate_volume = pd.DataFrame(volume_results)

print("\n======================================")
print("       CANDIDATE VOLUME RESULTS")
print("======================================\n")

print(candidate_volume.to_string(index=False))


========== Prefix 6 ==========
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
S2 candidate pairs: 49,064,869
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
S3 candidate pairs: 50,840,464

========== Prefix 8 ==========
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
S2 candidate pairs: 48,410,225
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
S3 candidate pairs: 50,103,832

========== Prefix 10 ==========


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Amazon_ML_Challenge/train_source2.tsv'

In [9]:
# Step 8A: Inspect how many S1 entities share each address number

from collections import Counter

number_frequency = Counter()

for _, row in blocking_s1.iterrows():

    address = normalize_address(row["business_address"])

    numbers = get_address_numbers(address)

    for number in numbers:
        number_frequency[number] += 1


number_freq_df = pd.DataFrame(
    number_frequency.items(),
    columns=["number", "s1_frequency"]
).sort_values(
    "s1_frequency",
    ascending=False
)

print("Unique address numbers:", len(number_freq_df))

print("\nMost common address numbers:")
print(number_freq_df.head(30).to_string(index=False))

print("\nFrequency distribution:")
print(number_freq_df["s1_frequency"].describe())

Unique address numbers: 444

Most common address numbers:
number  s1_frequency
     1            53
     2            32
     4            23
     3            23
     8            14
     6            12
     5            12
     7            10
     9            10
    15             8
    12             8
    45             6
    14             6
    10             6
    41             5
   204             5
    28             5
    11             4
    22             4
    21             4
    16             4
    31             4
    20             4
    66             4
    40             4
    33             4
    19             3
   101             3
   609             3
   403             3

Frequency distribution:
count    444.000000
mean       1.747748
std        3.484810
min        1.000000
25%        1.000000
50%        1.000000
75%        1.000000
max       53.000000
Name: s1_frequency, dtype: float64


In [10]:
ADDRESS_STOPWORDS = {
    "road", "rd", "street", "st", "avenue", "ave",
    "lane", "ln", "drive", "dr", "boulevard", "blvd",
    "highway", "hwy", "building", "bldg",
    "floor", "fl", "house", "h", "number", "no",
    "plot", "sector", "sec", "block", "city", "district",
    "near", "nr", "opp", "opposite", "india", "state",
    "town", "village", "roadway", "place", "main",
    "north", "south", "east", "west",
    "first", "second", "third", "fourth",
    "phase", "stage", "unit", "flat", "wing"
}


def address_words(text):
    """Extract useful alphabetic address words."""

    if pd.isna(text):
        return set()

    text = str(text).lower()

    words = re.findall(r"[a-z]+", text)

    return {
        word
        for word in words
        if len(word) >= 4
        and word not in ADDRESS_STOPWORDS
    }


print("✅ Address-word function created")

print(
    address_words(
        "Flat No. 1008, X Wing, Eiffel City, Pune"
    )
)

✅ Address-word function created
{'eiffel', 'pune'}


In [11]:
# Step 9: Test stricter address blocking on the 1,705 true pairs

results = []

for _, pair in positive_pairs.iterrows():

    s1_id = pair["entity_id_s1"]
    target_id = pair["entity_id_matched"]
    source = pair["source"]

    # Get S1 record
    s1_row = blocking_s1[
        blocking_s1["entity_id"] == s1_id
    ].iloc[0]

    s1_address = normalize_address(
        s1_row["business_address"]
    )

    # Get target record
    target = (
        target_s2[target_id]
        if source == "S2"
        else target_s3[target_id]
    )

    target_address = target["address"]

    # Extract numbers
    s1_numbers = get_address_numbers(s1_address)
    target_numbers = get_address_numbers(target_address)

    shared_numbers = s1_numbers & target_numbers

    # Extract useful words
    s1_words = address_words(s1_address)
    target_words = address_words(target_address)

    shared_words = s1_words & target_words

    results.append({
        "entity_id_s1": s1_id,
        "entity_id_matched": target_id,
        "source": source,
        "shared_numbers": len(shared_numbers),
        "shared_words": len(shared_words),
        "number_and_word": (
            len(shared_numbers) > 0
            and len(shared_words) > 0
        ),
        "number_and_2_words": (
            len(shared_numbers) > 0
            and len(shared_words) >= 2
        )
    })


address_combo_results = pd.DataFrame(results)

total = len(address_combo_results)

print("Total true pairs:", total)

print("\nAddress-number recall:")
print(
    (
        address_combo_results["shared_numbers"] > 0
    ).sum(),
    "/",
    total
)

print(
    "Recall:",
    (
        address_combo_results["shared_numbers"] > 0
    ).mean()
)

print("\nNumber + at least 1 useful word:")
print(
    address_combo_results["number_and_word"].sum(),
    "/",
    total
)

print(
    "Recall:",
    address_combo_results["number_and_word"].mean()
)

print("\nNumber + at least 2 useful words:")
print(
    address_combo_results["number_and_2_words"].sum(),
    "/",
    total
)

print(
    "Recall:",
    address_combo_results["number_and_2_words"].mean()
)

Total true pairs: 1705

Address-number recall:
1364 / 1705
Recall: 0.8

Number + at least 1 useful word:
1340 / 1705
Recall: 0.7859237536656891

Number + at least 2 useful words:
1095 / 1705
Recall: 0.6422287390029325


In [12]:
from collections import Counter

# ---------------------------------------------------------
# Build address-word frequency across S2 and S3
# ---------------------------------------------------------

s2_word_freq = Counter()
s3_word_freq = Counter()


def collect_word_frequencies(file_path, counter, source_name):

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            file_path,
            sep="\t",
            usecols=["business_address"],
            chunksize=200_000
        )
    ):

        for address in chunk["business_address"]:

            words = address_words(address)

            for word in words:
                counter[word] += 1

        if (chunk_no + 1) % 5 == 0:
            print(
                f"{source_name}: "
                f"processed {(chunk_no + 1) * 200_000:,} rows"
            )


print("Building S2 address-word frequencies...")

collect_word_frequencies(
    "/train_source2.tsv",
    s2_word_freq,
    "S2"
)

print("\nBuilding S3 address-word frequencies...")

collect_word_frequencies(
    "/train_source3.tsv",
    s3_word_freq,
    "S3"
)

print("\nUnique S2 words:", len(s2_word_freq))
print("Unique S3 words:", len(s3_word_freq))

print("\nTop S2 words:")
print(s2_word_freq.most_common(20))

print("\nTop S3 words:")
print(s3_word_freq.most_common(20))

Building S2 address-word frequencies...
S2: processed 1,000,000 rows
S2: processed 2,000,000 rows
S2: processed 3,000,000 rows
S2: processed 4,000,000 rows
S2: processed 5,000,000 rows

Building S3 address-word frequencies...
S3: processed 1,000,000 rows
S3: processed 2,000,000 rows
S3: processed 3,000,000 rows
S3: processed 4,000,000 rows
S3: processed 5,000,000 rows

Unique S2 words: 459638
Unique S3 words: 429949

Top S2 words:
[('maharashtra', 319026), ('nagar', 287085), ('delhi', 262485), ('pradesh', 214708), ('mumbai', 162535), ('null', 131794), ('uttar', 123770), ('karnataka', 117453), ('door', 112337), ('bangalore', 110619), ('tamil', 105598), ('nadu', 105563), ('park', 104851), ('colony', 97719), ('gujarat', 95835), ('bengal', 95299), ('kolkata', 79145), ('pune', 78233), ('telangana', 75193), ('hyderabad', 68226)]

Top S3 words:
[('texas', 293313), ('delhi', 272880), ('nagar', 267920), ('york', 228655), ('carolina', 215878), ('mumbai', 197984), ('ohio', 188077), ('virginia', 1

In [13]:
# Step 10: Test rare address-word recall

thresholds = [10, 25, 50, 100, 250, 500, 1000, 2000, 5000]

results = []

for threshold in thresholds:

    recovered = 0

    for _, pair in positive_pairs.iterrows():

        s1_id = pair["entity_id_s1"]
        target_id = pair["entity_id_matched"]
        source = pair["source"]

        # S1 address
        s1_row = blocking_s1[
            blocking_s1["entity_id"] == s1_id
        ].iloc[0]

        s1_address = s1_row["business_address"]

        # Target address
        target = (
            target_s2[target_id]
            if source == "S2"
            else target_s3[target_id]
        )

        target_address = target["address"]

        # Address words
        s1_words = address_words(s1_address)
        target_words = address_words(target_address)

        shared_words = s1_words & target_words

        # Keep only words rare enough in the relevant source
        if source == "S2":
            freq_map = s2_word_freq
        else:
            freq_map = s3_word_freq

        rare_shared = [
            word
            for word in shared_words
            if freq_map.get(word, 0) <= threshold
        ]

        if rare_shared:
            recovered += 1

    results.append({
        "max_frequency": threshold,
        "recovered": recovered,
        "missed": len(positive_pairs) - recovered,
        "recall": recovered / len(positive_pairs)
    })


rare_word_results = pd.DataFrame(results)

print(
    rare_word_results.to_string(index=False)
)

 max_frequency  recovered  missed   recall
            10        187    1518 0.109677
            25        299    1406 0.175367
            50        415    1290 0.243402
           100        567    1138 0.332551
           250        755     950 0.442815
           500        913     792 0.535484
          1000       1112     593 0.652199
          2000       1285     420 0.753666
          5000       1417     288 0.831085


In [14]:
# Test: address number + rare address word blocking recall
# We do NOT generate candidates yet.
# We only test recall on the 1,705 known true pairs.

thresholds = [100, 250, 500, 1000, 2000, 5000]

results = []

for threshold in thresholds:
    recovered = 0

    for _, row in positive_pairs.iterrows():

        s1_addr = row["s1_address_norm"]

        if row["source"] == "S2":
            target_addr = row["s2_address_norm"]
            word_freq = s2_word_freq
        else:
            target_addr = row["s3_address_norm"]
            word_freq = s3_word_freq

        # 1. Address number overlap
        s1_numbers = get_address_numbers(s1_addr)
        target_numbers = get_address_numbers(target_addr)

        number_match = len(s1_numbers & target_numbers) > 0

        if not number_match:
            continue

        # 2. Rare shared address word
        s1_words = address_words(s1_addr)
        target_words = address_words(target_addr)

        shared_words = s1_words & target_words

        rare_word_match = any(
            word_freq.get(word, 0) <= threshold
            for word in shared_words
        )

        if rare_word_match:
            recovered += 1

    total = len(positive_pairs)

    results.append({
        "max_frequency": threshold,
        "recovered": recovered,
        "missed": total - recovered,
        "recall": recovered / total
    })

pd.DataFrame(results)

KeyError: 's1_address_norm'

In [15]:
print("positive_pairs shape:", positive_pairs.shape)
print("\nColumns:")
print(positive_pairs.columns.tolist())

print("\nFirst 3 rows:")
display(positive_pairs.head(3))

positive_pairs shape: (1705, 4)

Columns:
['entity_id_s1', 'entity_id_matched', 'source', 'label']

First 3 rows:


,entity_id_s1,entity_id_matched,source,label
0,S1-614166735,S2-836231990,S2,1
1,S1-614166735,S2-25291679,S2,1
2,S1-614166735,S3-609578756,S3,1


In [16]:
# Create lookup tables for the original training data

s1_lookup = train_source1.set_index("entity_id")
s2_lookup = train_source2.set_index("entity_id")
s3_lookup = train_source3.set_index("entity_id")

# Add normalized address information to positive_pairs
positive_test = positive_pairs.copy()

positive_test["s1_address_norm"] = positive_test["entity_id_s1"].map(
    s1_lookup["business_address"]
).apply(normalize_address)

positive_test["matched_address_norm"] = positive_test.apply(
    lambda row: normalize_address(
        s2_lookup.loc[row["entity_id_matched"], "business_address"]
        if row["source"] == "S2"
        else s3_lookup.loc[row["entity_id_matched"], "business_address"]
    ),
    axis=1
)

print("Shape:", positive_test.shape)
display(positive_test.head())

NameError: name 'train_source1' is not defined

In [17]:
import pandas as pd

# Load the original training files
train_source1 = pd.read_csv("/train_source1.tsv", sep="\t")
train_source2 = pd.read_csv("/train_source2.tsv", sep="\t")
train_source3 = pd.read_csv("/train_source3.tsv", sep="\t")

print("S1:", train_source1.shape)
print("S2:", train_source2.shape)
print("S3:", train_source3.shape)

S1: (2206821, 4)
S2: (5034616, 4)
S3: (5285603, 4)


In [18]:
# Create lookup tables
s1_lookup = train_source1.set_index("entity_id")
s2_lookup = train_source2.set_index("entity_id")
s3_lookup = train_source3.set_index("entity_id")

# Copy the 1,705 known true matches
positive_test = positive_pairs.copy()

# Get S1 address
positive_test["s1_address_norm"] = (
    positive_test["entity_id_s1"]
    .map(s1_lookup["business_address"])
    .apply(normalize_address)
)

# Get corresponding S2/S3 address
positive_test["matched_address_norm"] = positive_test.apply(
    lambda row: normalize_address(
        s2_lookup.loc[row["entity_id_matched"], "business_address"]
        if row["source"] == "S2"
        else s3_lookup.loc[row["entity_id_matched"], "business_address"]
    ),
    axis=1
)

print("Shape:", positive_test.shape)
print("\nColumns:")
print(positive_test.columns.tolist())

display(positive_test.head())

Shape: (1705, 6)

Columns:
['entity_id_s1', 'entity_id_matched', 'source', 'label', 's1_address_norm', 'matched_address_norm']


,entity_id_s1,entity_id_matched,source,label,s1_address_norm,matched_address_norm
0,S1-614166735,S2-836231990,S2,1,15001 howards mill road mulino or,or 15001 1 2 howards mill rd muino
1,S1-614166735,S2-25291679,S2,1,15001 howards mill road mulino or,15001 1 2 howards mill rd muino or
2,S1-614166735,S3-609578756,S3,1,15001 howards mill road mulino or,mulino oregon 15001 howards mill road
3,S1-614166735,S3-244407822,S3,1,15001 howards mill road mulino or,15001 howards mill road mulino oregon
4,S1-614166735,S3-974804011,S3,1,15001 howards mill road mulino or,mulino 15001 howards mill rd oregon


In [19]:
thresholds = [100, 250, 500, 1000, 2000, 5000]

results = []

for threshold in thresholds:
    recovered = 0

    for _, row in positive_test.iterrows():

        s1_numbers = get_address_numbers(row["s1_address_norm"])
        matched_numbers = get_address_numbers(row["matched_address_norm"])

        # Condition 1: shared address number
        if not (s1_numbers & matched_numbers):
            continue

        # Get words
        s1_words = address_words(row["s1_address_norm"])
        matched_words = address_words(row["matched_address_norm"])

        shared_words = s1_words & matched_words

        # Use the correct source's word-frequency dictionary
        freq_dict = (
            s2_word_freq
            if row["source"] == "S2"
            else s3_word_freq
        )

        # Condition 2: at least one rare shared word
        rare_word_match = any(
            freq_dict.get(word, 0) <= threshold
            for word in shared_words
        )

        if rare_word_match:
            recovered += 1

    total = len(positive_test)

    results.append({
        "max_frequency": threshold,
        "recovered": recovered,
        "missed": total - recovered,
        "recall": recovered / total
    })

results_df = pd.DataFrame(results)

display(results_df)

,max_frequency,recovered,missed,recall
0,100,463,1242,0.271554
1,250,608,1097,0.356598
2,500,734,971,0.430499
3,1000,892,813,0.523167
4,2000,1051,654,0.616422
5,5000,1166,539,0.683871


In [20]:
# Add country information to positive_test

positive_test["s1_country"] = positive_test["entity_id_s1"].map(
    s1_lookup["country"]
)

positive_test["matched_country"] = positive_test.apply(
    lambda row: (
        s2_lookup.loc[row["entity_id_matched"], "country"]
        if row["source"] == "S2"
        else s3_lookup.loc[row["entity_id_matched"], "country"]
    ),
    axis=1
)

recovered = 0

for _, row in positive_test.iterrows():

    # Country must match
    country_match = (
        str(row["s1_country"]).strip().lower()
        == str(row["matched_country"]).strip().lower()
    )

    if not country_match:
        continue

    # Address number must match
    s1_numbers = get_address_numbers(row["s1_address_norm"])
    matched_numbers = get_address_numbers(row["matched_address_norm"])

    if s1_numbers & matched_numbers:
        recovered += 1

total = len(positive_test)

print("Recovered:", recovered)
print("Missed:", total - recovered)
print("Recall:", recovered / total)

Recovered: 1364
Missed: 341
Recall: 0.8


In [21]:
from collections import Counter

# Count how many S1 development records contain each address number
number_freq = Counter()

for addr in blocking_s1["business_address"].apply(normalize_address):
    for num in get_address_numbers(addr):
        number_freq[num] += 1

print("Top address-number frequencies:")
print(number_freq.most_common(15))


# Test different frequency cutoffs
thresholds = [2, 3, 5, 10, 15, 20, 30, 50]

results = []

for threshold in thresholds:

    recovered = 0

    for _, row in positive_test.iterrows():

        s1_numbers = get_address_numbers(row["s1_address_norm"])
        matched_numbers = get_address_numbers(row["matched_address_norm"])

        shared_numbers = s1_numbers & matched_numbers

        # Keep only numbers that are not too common in S1
        useful_numbers = {
            num for num in shared_numbers
            if number_freq.get(num, 0) <= threshold
        }

        if useful_numbers:
            recovered += 1

    results.append({
        "max_number_frequency": threshold,
        "recovered": recovered,
        "missed": len(positive_test) - recovered,
        "recall": recovered / len(positive_test)
    })

display(pd.DataFrame(results))

Top address-number frequencies:
[('1', 53), ('2', 32), ('4', 23), ('3', 23), ('8', 14), ('5', 12), ('6', 12), ('7', 10), ('9', 10), ('12', 8), ('15', 8), ('14', 6), ('10', 6), ('45', 6), ('204', 5)]


,max_number_frequency,recovered,missed,recall
0,2,1009,696,0.591789
1,3,1093,612,0.641056
2,5,1163,542,0.682111
3,10,1239,466,0.726686
4,15,1250,455,0.733138
5,20,1250,455,0.733138
6,30,1308,397,0.767155
7,50,1339,366,0.785337


In [22]:
from collections import defaultdict

# Build selective address-number -> S2/S3 entity mappings
s2_num_map = defaultdict(list)
s3_num_map = defaultdict(list)

# S2
for _, row in train_source2.iterrows():
    nums = get_address_numbers(normalize_address(row["business_address"]))

    for num in nums:
        # Only use reasonably distinctive numbers
        if number_freq.get(num, 0) <= 50:
            s2_num_map[num].append(row["entity_id"])

# S3
for _, row in train_source3.iterrows():
    nums = get_address_numbers(normalize_address(row["business_address"]))

    for num in nums:
        if number_freq.get(num, 0) <= 50:
            s3_num_map[num].append(row["entity_id"])


# Count candidates for our 500 S1 development sample
s2_candidates = 0
s3_candidates = 0

for _, row in blocking_s1.iterrows():

    nums = get_address_numbers(
        normalize_address(row["business_address"])
    )

    useful_nums = [
        num for num in nums
        if number_freq.get(num, 0) <= 50
    ]

    # Number of S2/S3 records sharing at least one useful number
    s2_ids = set()
    s3_ids = set()

    for num in useful_nums:
        s2_ids.update(s2_num_map.get(num, []))
        s3_ids.update(s3_num_map.get(num, []))

    s2_candidates += len(s2_ids)
    s3_candidates += len(s3_ids)


print("S2 candidates:", f"{s2_candidates:,}")
print("S3 candidates:", f"{s3_candidates:,}")
print("Total candidates:", f"{s2_candidates + s3_candidates:,}")
print("Average candidates per S1:",
      (s2_candidates + s3_candidates) / len(blocking_s1))

S2 candidates: 29,451,700
S3 candidates: 29,806,317
Total candidates: 59,258,017
Average candidates per S1: 118516.034


In [23]:
from collections import defaultdict

# Build exact normalized-name maps
s2_name_map = defaultdict(list)
s3_name_map = defaultdict(list)

for _, row in train_source2.iterrows():
    name = normalize_name_v2(row["business_name"])
    if name:
        s2_name_map[name].append(row["entity_id"])

for _, row in train_source3.iterrows():
    name = normalize_name_v2(row["business_name"])
    if name:
        s3_name_map[name].append(row["entity_id"])


# Count candidates for the 500 S1 development entities
s2_count = 0
s3_count = 0

for _, row in blocking_s1.iterrows():

    name = normalize_name_v2(row["business_name"])

    s2_count += len(set(s2_name_map.get(name, [])))
    s3_count += len(set(s3_name_map.get(name, [])))


print("S2 candidates:", f"{s2_count:,}")
print("S3 candidates:", f"{s3_count:,}")
print("Total candidates:", f"{s2_count + s3_count:,}")
print(
    "Average candidates per S1:",
    (s2_count + s3_count) / len(blocking_s1)
)

S2 candidates: 3,159
S3 candidates: 3,517
Total candidates: 6,676
Average candidates per S1: 13.352


In [24]:
from collections import defaultdict

# Build exact normalized-address maps
s2_address_map = defaultdict(list)
s3_address_map = defaultdict(list)

for _, row in train_source2.iterrows():
    address = normalize_address(row["business_address"])
    if address:
        s2_address_map[address].append(row["entity_id"])

for _, row in train_source3.iterrows():
    address = normalize_address(row["business_address"])
    if address:
        s3_address_map[address].append(row["entity_id"])


# Count candidates for the 500 S1 development entities
s2_count = 0
s3_count = 0

for _, row in blocking_s1.iterrows():

    address = normalize_address(row["business_address"])

    s2_count += len(set(s2_address_map.get(address, [])))
    s3_count += len(set(s3_address_map.get(address, [])))


print("S2 candidates:", f"{s2_count:,}")
print("S3 candidates:", f"{s3_count:,}")
print("Total candidates:", f"{s2_count + s3_count:,}")
print(
    "Average candidates per S1:",
    (s2_count + s3_count) / len(blocking_s1)
)

S2 candidates: 118
S3 candidates: 41
Total candidates: 159
Average candidates per S1: 0.318


In [25]:
# Count exact-name + country candidates
s2_name_country_map = defaultdict(list)
s3_name_country_map = defaultdict(list)

for _, row in train_source2.iterrows():
    name = normalize_name_v2(row["business_name"])
    country = str(row["country"]).strip().lower()

    if name:
        s2_name_country_map[(name, country)].append(row["entity_id"])

for _, row in train_source3.iterrows():
    name = normalize_name_v2(row["business_name"])
    country = str(row["country"]).strip().lower()

    if name:
        s3_name_country_map[(name, country)].append(row["entity_id"])


# Candidate volume
s2_count = 0
s3_count = 0

for _, row in blocking_s1.iterrows():

    name = normalize_name_v2(row["business_name"])
    country = str(row["country"]).strip().lower()

    s2_count += len(
        set(s2_name_country_map.get((name, country), []))
    )

    s3_count += len(
        set(s3_name_country_map.get((name, country), []))
    )


print("S2 candidates:", f"{s2_count:,}")
print("S3 candidates:", f"{s3_count:,}")
print("Total candidates:", f"{s2_count + s3_count:,}")
print(
    "Average candidates per S1:",
    (s2_count + s3_count) / len(blocking_s1)
)


# Recall on the 1,705 true matches
recovered = 0

for _, row in positive_test.iterrows():

    s1_name = normalize_name_v2(
        s1_lookup.loc[row["entity_id_s1"], "business_name"]
    )

    matched_lookup = (
        s2_lookup
        if row["source"] == "S2"
        else s3_lookup
    )

    matched_name = normalize_name_v2(
        matched_lookup.loc[row["entity_id_matched"], "business_name"]
    )

    s1_country = str(
        s1_lookup.loc[row["entity_id_s1"], "country"]
    ).strip().lower()

    matched_country = str(
        matched_lookup.loc[row["entity_id_matched"], "country"]
    ).strip().lower()

    if (
        s1_name == matched_name
        and s1_country == matched_country
    ):
        recovered += 1

print("\nRecall test:")
print("Recovered:", recovered)
print("Missed:", len(positive_test) - recovered)
print("Recall:", recovered / len(positive_test))

S2 candidates: 3,152
S3 candidates: 3,497
Total candidates: 6,649
Average candidates per S1: 13.298

Recall test:
Recovered: 440
Missed: 1265
Recall: 0.25806451612903225


In [26]:
from collections import defaultdict

def first_name_token(name):
    name = normalize_name_v2(name)
    if not name:
        return ""
    return name.split()[0]


# Build first-token maps
s2_first_token_map = defaultdict(list)
s3_first_token_map = defaultdict(list)

for _, row in train_source2.iterrows():
    token = first_name_token(row["business_name"])
    if token:
        s2_first_token_map[token].append(row["entity_id"])

for _, row in train_source3.iterrows():
    token = first_name_token(row["business_name"])
    if token:
        s3_first_token_map[token].append(row["entity_id"])


# Candidate volume
s2_count = 0
s3_count = 0

for _, row in blocking_s1.iterrows():

    token = first_name_token(row["business_name"])

    s2_count += len(set(s2_first_token_map.get(token, [])))
    s3_count += len(set(s3_first_token_map.get(token, [])))


print("S2 candidates:", f"{s2_count:,}")
print("S3 candidates:", f"{s3_count:,}")
print("Total candidates:", f"{s2_count + s3_count:,}")
print(
    "Average candidates per S1:",
    (s2_count + s3_count) / len(blocking_s1)
)


# Recall on true matches
recovered = 0

for _, row in positive_test.iterrows():

    s1_token = first_name_token(
        s1_lookup.loc[row["entity_id_s1"], "business_name"]
    )

    matched_lookup = (
        s2_lookup if row["source"] == "S2"
        else s3_lookup
    )

    matched_token = first_name_token(
        matched_lookup.loc[row["entity_id_matched"], "business_name"]
    )

    if s1_token == matched_token:
        recovered += 1


print("\nRecall test:")
print("Recovered:", recovered)
print("Missed:", len(positive_test) - recovered)
print("Recall:", recovered / len(positive_test))

S2 candidates: 1,820,989
S3 candidates: 2,040,054
Total candidates: 3,861,043
Average candidates per S1: 7722.086

Recall test:
Recovered: 1214
Missed: 491
Recall: 0.7120234604105572


In [27]:
def first_two_name_tokens(name):
    name = normalize_name_v2(name)

    if not name:
        return ""

    tokens = name.split()

    if len(tokens) == 1:
        return tokens[0]

    return " ".join(tokens[:2])


# Build maps
s2_two_token_map = defaultdict(list)
s3_two_token_map = defaultdict(list)

for _, row in train_source2.iterrows():
    key = first_two_name_tokens(row["business_name"])
    if key:
        s2_two_token_map[key].append(row["entity_id"])

for _, row in train_source3.iterrows():
    key = first_two_name_tokens(row["business_name"])
    if key:
        s3_two_token_map[key].append(row["entity_id"])


# Candidate volume
s2_count = 0
s3_count = 0

for _, row in blocking_s1.iterrows():

    key = first_two_name_tokens(row["business_name"])

    s2_count += len(set(s2_two_token_map.get(key, [])))
    s3_count += len(set(s3_two_token_map.get(key, [])))


print("S2 candidates:", f"{s2_count:,}")
print("S3 candidates:", f"{s3_count:,}")
print("Total candidates:", f"{s2_count + s3_count:,}")
print(
    "Average candidates per S1:",
    (s2_count + s3_count) / len(blocking_s1)
)


# Recall
recovered = 0

for _, row in positive_test.iterrows():

    s1_key = first_two_name_tokens(
        s1_lookup.loc[row["entity_id_s1"], "business_name"]
    )

    matched_lookup = (
        s2_lookup if row["source"] == "S2"
        else s3_lookup
    )

    matched_key = first_two_name_tokens(
        matched_lookup.loc[row["entity_id_matched"], "business_name"]
    )

    if s1_key == matched_key:
        recovered += 1


print("\nRecall test:")
print("Recovered:", recovered)
print("Missed:", len(positive_test) - recovered)
print("Recall:", recovered / len(positive_test))

S2 candidates: 182,729
S3 candidates: 182,688
Total candidates: 365,417
Average candidates per S1: 730.834

Recall test:
Recovered: 976
Missed: 729
Recall: 0.5724340175953079


In [ ]:
from collections import defaultdict, Counter

# ---------------------------------------------------------
# 1. Count name-token frequency in the 500 S1 development set
# ---------------------------------------------------------

s1_name_token_freq = Counter()

for _, row in blocking_s1.iterrows():
    name = normalize_name_v2(row["business_name"])
    tokens = set(name.split())

    for token in tokens:
        if len(token) >= 3:
            s1_name_token_freq[token] += 1


# ---------------------------------------------------------
# 2. Build token -> S2/S3 entity maps
# ---------------------------------------------------------

s2_token_map = defaultdict(list)
s3_token_map = defaultdict(list)

for _, row in train_source2.iterrows():
    name = normalize_name_v2(row["business_name"])

    for token in set(name.split()):
        if len(token) >= 3:
            s2_token_map[token].append(row["entity_id"])

for _, row in train_source3.iterrows():
    name = normalize_name_v2(row["business_name"])

    for token in set(name.split()):
        if len(token) >= 3:
            s3_token_map[token].append(row["entity_id"])


# ---------------------------------------------------------
# 3. Test different frequency thresholds
# ---------------------------------------------------------

thresholds = [10, 25, 50, 100, 250, 500]

results = []

for threshold in thresholds:

    # Candidate volume
    s2_count = 0
    s3_count = 0

    for _, row in blocking_s1.iterrows():

        name = normalize_name_v2(row["business_name"])

        useful_tokens = [
            token
            for token in set(name.split())
            if len(token) >= 3
            and s1_name_token_freq.get(token, 0) <= threshold
        ]

        s2_ids = set()
        s3_ids = set()

        for token in useful_tokens:
            s2_ids.update(s2_token_map.get(token, []))
            s3_ids.update(s3_token_map.get(token, []))

        s2_count += len(s2_ids)
        s3_count += len(s3_ids)

    total_candidates = s2_count + s3_count


    # -----------------------------------------------------
    # Recall on the 1,705 true matches
    # -----------------------------------------------------

    recovered = 0

    for _, row in positive_test.iterrows():

        s1_name = normalize_name_v2(
            s1_lookup.loc[row["entity_id_s1"], "business_name"]
        )

        matched_lookup = (
            s2_lookup if row["source"] == "S2"
            else s3_lookup
        )

        matched_name = normalize_name_v2(
            matched_lookup.loc[
                row["entity_id_matched"],
                "business_name"
            ]
        )

        s1_tokens = {
            t for t in s1_name.split()
            if len(t) >= 3
        }

        matched_tokens = {
            t for t in matched_name.split()
            if len(t) >= 3
        }

        shared_tokens = s1_tokens & matched_tokens

        useful_shared_tokens = [
            token
            for token in shared_tokens
            if s1_name_token_freq.get(token, 0) <= threshold
        ]

        if useful_shared_tokens:
            recovered += 1


    results.append({
        "max_s1_token_frequency": threshold,
        "S2_candidates": s2_count,
        "S3_candidates": s3_count,
        "total_candidates": total_candidates,
        "avg_candidates_per_S1": total_candidates / len(blocking_s1),
        "recovered": recovered,
        "missed": len(positive_test) - recovered,
        "recall": recovered / len(positive_test)
    })


results_df = pd.DataFrame(results)

display(results_df)

In [1]:
import os

print("Files currently in /:")
print(os.listdir("/")[:50])

print("\nDataset files:")
for f in [
    "/train_source1.tsv",
    "/train_source2.tsv",
    "/train_source3.tsv",
    "/train_ground_truth.tsv"
]:
    print(f, "->", os.path.exists(f))

Files currently in /:
['lib64', 'etc', 'home', 'boot', 'dev', 'sbin', 'srv', 'sys', 'proc', 'bin', 'mnt', 'root', 'opt', 'usr', 'run', 'lib', 'media', 'tmp', 'var', 'train_ground_truth.tsv', 'train_source3.tsv', 'train_source2.tsv', 'content', 'kaggle', 'train_source1.tsv', '.dockerenv', 'datalab', 'tools', 'python-apt', 'python-apt.tar.xz', 'bin.usr-is-merged', 'sbin.usr-is-merged', 'lib.usr-is-merged']

Dataset files:
/train_source1.tsv -> True
/train_source2.tsv -> True
/train_source3.tsv -> True
/train_ground_truth.tsv -> True


In [2]:
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/Amazon_ML_Challenge"
DEV_DIR = os.path.join(PROJECT_DIR, "development")

print("Development folder exists:", os.path.exists(DEV_DIR))

for f in [
    "blocking_s1.csv",
    "dev_ground_truth.csv",
    "positive_pairs.csv"
]:
    path = os.path.join(DEV_DIR, f)
    print(f, "->", os.path.exists(path))

# Load the small development files
blocking_s1 = pd.read_csv(
    os.path.join(DEV_DIR, "blocking_s1.csv")
)

dev_ground_truth = pd.read_csv(
    os.path.join(DEV_DIR, "dev_ground_truth.csv")
)

positive_pairs = pd.read_csv(
    os.path.join(DEV_DIR, "positive_pairs.csv")
)

print("\nblocking_s1:", blocking_s1.shape)
print("dev_ground_truth:", dev_ground_truth.shape)
print("positive_pairs:", positive_pairs.shape)

Development folder exists: True
blocking_s1.csv -> True
dev_ground_truth.csv -> True
positive_pairs.csv -> True

blocking_s1: (500, 4)
dev_ground_truth: (500, 2)
positive_pairs: (1705, 4)


In [3]:
from collections import Counter
import pandas as pd
import re

# ---------------------------------------------------------
# 1. Get name tokens from our 500 S1 development sample
# ---------------------------------------------------------

s1_name_tokens = {}

for _, row in blocking_s1.iterrows():
    name = normalize_name_v2(row["business_name"])
    s1_name_tokens[row["entity_id"]] = {
        token for token in name.split()
        if len(token) >= 3
    }


# ---------------------------------------------------------
# 2. Count S1 token frequencies
# ---------------------------------------------------------

s1_token_freq = Counter()

for tokens in s1_name_tokens.values():
    for token in tokens:
        s1_token_freq[token] += 1

print("Unique S1 name tokens:", len(s1_token_freq))
print("Top tokens:", s1_token_freq.most_common(15))


# ---------------------------------------------------------
# 3. Test rare-token candidate volume
#    Process S2/S3 in chunks so RAM stays safe.
# ---------------------------------------------------------

thresholds = [10, 25, 50, 100, 250, 500]

# For each threshold, keep a set of S1 entities that
# actually receive at least one candidate.
candidate_counts_s2 = {t: 0 for t in thresholds}
candidate_counts_s3 = {t: 0 for t in thresholds}

# We need total candidate PAIRS, not just S1 entities.
total_candidates_s2 = {t: 0 for t in thresholds}
total_candidates_s3 = {t: 0 for t in thresholds}


def process_source(path, candidate_counts, total_candidates):

    # Process 100k rows at a time
    for chunk in pd.read_csv(
        path,
        sep="\t",
        chunksize=100_000,
        usecols=["entity_id", "business_name"]
    ):

        for _, row in chunk.iterrows():

            name = normalize_name_v2(row["business_name"])

            tokens = {
                token for token in name.split()
                if len(token) >= 3
            }

            if not tokens:
                continue

            # For each threshold, find S1 records sharing
            # at least one sufficiently rare token.
            for threshold in thresholds:

                useful_tokens = {
                    token for token in tokens
                    if s1_token_freq.get(token, 0) <= threshold
                }

                if not useful_tokens:
                    continue

                matched_s1 = 0

                for s1_id, s1_tokens in s1_name_tokens.items():

                    if useful_tokens & s1_tokens:
                        matched_s1 += 1

                if matched_s1:
                    candidate_counts[threshold] += matched_s1
                    total_candidates[threshold] += matched_s1


        print("Processed:", path)


process_source(
    "/train_source2.tsv",
    candidate_counts_s2,
    total_candidates_s2
)

process_source(
    "/train_source3.tsv",
    candidate_counts_s3,
    total_candidates_s3
)


# ---------------------------------------------------------
# 4. Display results
# ---------------------------------------------------------

results = []

for threshold in thresholds:

    total = (
        total_candidates_s2[threshold]
        + total_candidates_s3[threshold]
    )

    results.append({
        "max_s1_token_frequency": threshold,
        "S2_candidates": total_candidates_s2[threshold],
        "S3_candidates": total_candidates_s3[threshold],
        "total_candidates": total,
        "avg_candidates_per_S1": total / len(blocking_s1)
    })

display(pd.DataFrame(results))

NameError: name 'normalize_name_v2' is not defined

In [4]:
import re
import unicodedata

def normalize_name_v2(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Remove accents
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    # Keep only letters/numbers
    text = re.sub(r"[^a-z0-9]+", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_address(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Remove accents
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    # Keep only letters/numbers
    text = re.sub(r"[^a-z0-9]+", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def get_address_numbers(text):
    if not text:
        return set()

    return set(re.findall(r"\d+", text))


def address_words(text):
    if pd.isna(text):
        return set()

    text = str(text).lower()

    words = re.findall(r"[a-z]+", text)

    return {
        word
        for word in words
        if len(word) >= 4
    }


print("Normalization functions restored.")

# Quick test
print(normalize_name_v2("TE Nórth LLC"))
print(normalize_address("Flat No. 1008, X Wing, Pune"))

Normalization functions restored.
te north llc
flat no 1008 x wing pune


In [5]:
from collections import defaultdict, Counter

# Build name-token sets for the 500 S1 development records
s1_name_tokens = {}

for _, row in blocking_s1.iterrows():

    name = normalize_name_v2(row["business_name"])

    tokens = {
        token
        for token in name.split()
        if len(token) >= 3
    }

    s1_name_tokens[row["entity_id"]] = tokens


# Count how many S1 entities contain each token
s1_token_freq = Counter()

for tokens in s1_name_tokens.values():
    for token in tokens:
        s1_token_freq[token] += 1


# Inverted index:
# token -> S1 entity IDs containing that token
s1_token_index = defaultdict(set)

for s1_id, tokens in s1_name_tokens.items():
    for token in tokens:
        s1_token_index[token].add(s1_id)


print("S1 entities:", len(s1_name_tokens))
print("Unique name tokens:", len(s1_token_freq))
print("Top 15 tokens:")
print(s1_token_freq.most_common(15))

S1 entities: 500
Unique name tokens: 781
Top 15 tokens:
[('limited', 122), ('private', 95), ('llc', 66), ('inc', 59), ('ltd', 33), ('pvt', 28), ('care', 18), ('india', 15), ('group', 14), ('associates', 13), ('llp', 11), ('medicine', 10), ('center', 10), ('pediatric', 9), ('partners', 9)]


In [6]:
NAME_STOPWORDS = {
    "limited",
    "private",
    "llc",
    "inc",
    "ltd",
    "pvt",
    "llp",
    "incorporated",
    "corp",
    "corporation",
    "company",
    "co",
    "plc",
    "group",
    "holdings",
    "partners",
    "associates",
    "international",
    "india"
}

# Useful tokens = not legal/common business words
useful_token_freq = {
    token: freq
    for token, freq in s1_token_freq.items()
    if token not in NAME_STOPWORDS
}

print("Total unique tokens:", len(s1_token_freq))
print("Useful unique tokens:", len(useful_token_freq))

print("\nMost common useful tokens:")
print(
    sorted(
        useful_token_freq.items(),
        key=lambda x: x[1],
        reverse=True
    )[:30]
)

Total unique tokens: 781
Useful unique tokens: 765

Most common useful tokens:
[('care', 18), ('medicine', 10), ('center', 10), ('pediatric', 9), ('brothers', 8), ('and', 7), ('dental', 7), ('enterprises', 7), ('infra', 6), ('clinic', 6), ('management', 6), ('health', 6), ('finance', 6), ('foundation', 6), ('silver', 6), ('investments', 5), ('creative', 5), ('family', 5), ('apex', 5), ('investment', 5), ('premier', 5), ('services', 5), ('global', 5), ('trading', 5), ('perfect', 5), ('pllc', 4), ('national', 4), ('innovations', 4), ('construction', 4), ('consultants', 4)]


In [7]:
thresholds = [1, 2, 3, 5, 10, 15]

results = []

for threshold in thresholds:

    recovered = 0

    for _, row in positive_pairs.iterrows():

        # S1 name
        s1_name = normalize_name_v2(
            blocking_s1.loc[
                blocking_s1["entity_id"] == row["entity_id_s1"],
                "business_name"
            ].iloc[0]
        )

        # Matched S2/S3 name
        if row["source"] == "S2":
            matched_row = train_source2[
                train_source2["entity_id"] == row["entity_id_matched"]
            ].iloc[0]
        else:
            matched_row = train_source3[
                train_source3["entity_id"] == row["entity_id_matched"]
            ].iloc[0]

        matched_name = normalize_name_v2(
            matched_row["business_name"]
        )

        s1_tokens = {
            token for token in s1_name.split()
            if len(token) >= 3
            and token not in NAME_STOPWORDS
        }

        matched_tokens = {
            token for token in matched_name.split()
            if len(token) >= 3
            and token not in NAME_STOPWORDS
        }

        shared_tokens = s1_tokens & matched_tokens

        # At least one shared token that is rare
        # among our 500 S1 entities
        if any(
            s1_token_freq.get(token, 999999) <= threshold
            for token in shared_tokens
        ):
            recovered += 1

    results.append({
        "max_s1_token_frequency": threshold,
        "recovered": recovered,
        "missed": len(positive_pairs) - recovered,
        "recall": recovered / len(positive_pairs)
    })

display(pd.DataFrame(results))

NameError: name 'train_source2' is not defined

In [8]:
# IDs we need
s2_needed = set(
    positive_pairs.loc[
        positive_pairs["source"] == "S2",
        "entity_id_matched"
    ]
)

s3_needed = set(
    positive_pairs.loc[
        positive_pairs["source"] == "S3",
        "entity_id_matched"
    ]
)

print("S2 IDs needed:", len(s2_needed))
print("S3 IDs needed:", len(s3_needed))


# ---------------------------------------------------------
# Read S2 in chunks and keep ONLY the 798 needed records
# ---------------------------------------------------------

s2_positive_names = {}

for chunk in pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):
    matched = chunk[chunk["entity_id"].isin(s2_needed)]

    for _, row in matched.iterrows():
        s2_positive_names[row["entity_id"]] = row["business_name"]

print("S2 names recovered:", len(s2_positive_names))


# ---------------------------------------------------------
# Read S3 in chunks and keep ONLY the 907 needed records
# ---------------------------------------------------------

s3_positive_names = {}

for chunk in pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):
    matched = chunk[chunk["entity_id"].isin(s3_needed)]

    for _, row in matched.iterrows():
        s3_positive_names[row["entity_id"]] = row["business_name"]

print("S3 names recovered:", len(s3_positive_names))

S2 IDs needed: 798
S3 IDs needed: 907
S2 names recovered: 798
S3 names recovered: 907


In [9]:
thresholds = [1, 2, 3, 5, 10, 15]

# Fast lookup for the 500 S1 development names
s1_name_lookup = dict(
    zip(
        blocking_s1["entity_id"],
        blocking_s1["business_name"]
    )
)

results = []

for threshold in thresholds:

    recovered = 0

    for _, row in positive_pairs.iterrows():

        # S1 name
        s1_name = normalize_name_v2(
            s1_name_lookup[row["entity_id_s1"]]
        )

        # Matched S2/S3 name
        if row["source"] == "S2":
            matched_name = normalize_name_v2(
                s2_positive_names[row["entity_id_matched"]]
            )
        else:
            matched_name = normalize_name_v2(
                s3_positive_names[row["entity_id_matched"]]
            )

        # Remove common legal/business suffixes
        s1_tokens = {
            token
            for token in s1_name.split()
            if len(token) >= 3
            and token not in NAME_STOPWORDS
        }

        matched_tokens = {
            token
            for token in matched_name.split()
            if len(token) >= 3
            and token not in NAME_STOPWORDS
        }

        shared_tokens = s1_tokens & matched_tokens

        # At least one sufficiently rare shared token
        if any(
            s1_token_freq.get(token, 999999) <= threshold
            for token in shared_tokens
        ):
            recovered += 1

    results.append({
        "max_s1_token_frequency": threshold,
        "recovered": recovered,
        "missed": len(positive_pairs) - recovered,
        "recall": recovered / len(positive_pairs)
    })

display(pd.DataFrame(results))

,max_s1_token_frequency,recovered,missed,recall
0,1,1064,641,0.624047
1,2,1218,487,0.714370
2,3,1291,414,0.757185
3,5,1358,347,0.796481
4,10,1395,310,0.818182
5,15,1395,310,0.818182


In [10]:
thresholds = [3, 5, 10]

results = []

for threshold in thresholds:

    s2_candidates = 0
    s3_candidates = 0

    # -----------------------------------------------------
    # S2
    # -----------------------------------------------------
    for chunk in pd.read_csv(
        "/train_source2.tsv",
        sep="\t",
        chunksize=100_000,
        usecols=["entity_id", "business_name"]
    ):

        for _, row in chunk.iterrows():

            name = normalize_name_v2(row["business_name"])

            tokens = {
                token
                for token in name.split()
                if len(token) >= 3
                and token not in NAME_STOPWORDS
                and s1_token_freq.get(token, 999999) <= threshold
            }

            if not tokens:
                continue

            matched_s1 = set()

            for token in tokens:
                matched_s1.update(
                    s1_token_index.get(token, set())
                )

            s2_candidates += len(matched_s1)


    # -----------------------------------------------------
    # S3
    # -----------------------------------------------------
    for chunk in pd.read_csv(
        "/train_source3.tsv",
        sep="\t",
        chunksize=100_000,
        usecols=["entity_id", "business_name"]
    ):

        for _, row in chunk.iterrows():

            name = normalize_name_v2(row["business_name"])

            tokens = {
                token
                for token in name.split()
                if len(token) >= 3
                and token not in NAME_STOPWORDS
                and s1_token_freq.get(token, 999999) <= threshold
            }

            if not tokens:
                continue

            matched_s1 = set()

            for token in tokens:
                matched_s1.update(
                    s1_token_index.get(token, set())
                )

            s3_candidates += len(matched_s1)


    total = s2_candidates + s3_candidates

    results.append({
        "threshold": threshold,
        "S2_candidates": s2_candidates,
        "S3_candidates": s3_candidates,
        "total_candidates": total,
        "avg_candidates_per_S1": total / 500
    })

    print(
        f"Threshold {threshold} finished: "
        f"{total:,} candidates"
    )


display(pd.DataFrame(results))

Threshold 3 finished: 10,672,922 candidates
Threshold 5 finished: 17,501,240 candidates
Threshold 10 finished: 28,610,211 candidates


,threshold,S2_candidates,S3_candidates,total_candidates,avg_candidates_per_S1
0,3,5016565,5656357,10672922,21345.844
1,5,8112497,9388743,17501240,35002.480
2,10,13391607,15218604,28610211,57220.422


In [11]:
s2_positive_addresses = {}
s3_positive_addresses = {}

# S2
for chunk in pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_address"]
):
    matched = chunk[chunk["entity_id"].isin(s2_needed)]

    for _, row in matched.iterrows():
        s2_positive_addresses[row["entity_id"]] = row["business_address"]

# S3
for chunk in pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_address"]
):
    matched = chunk[chunk["entity_id"].isin(s3_needed)]

    for _, row in matched.iterrows():
        s3_positive_addresses[row["entity_id"]] = row["business_address"]

print("S2 addresses recovered:", len(s2_positive_addresses))
print("S3 addresses recovered:", len(s3_positive_addresses))

S2 addresses recovered: 798
S3 addresses recovered: 907


In [12]:
thresholds = [1, 2, 3, 5, 10, 15]

# Fast S1 lookups
s1_name_lookup = dict(
    zip(
        blocking_s1["entity_id"],
        blocking_s1["business_name"]
    )
)

s1_address_lookup = dict(
    zip(
        blocking_s1["entity_id"],
        blocking_s1["business_address"]
    )
)

results = []

for threshold in thresholds:

    recovered = 0

    for _, row in positive_pairs.iterrows():

        # -----------------------------
        # S1
        # -----------------------------
        s1_name = normalize_name_v2(
            s1_name_lookup[row["entity_id_s1"]]
        )

        s1_address = normalize_address(
            s1_address_lookup[row["entity_id_s1"]]
        )

        s1_tokens = {
            token
            for token in s1_name.split()
            if len(token) >= 3
            and token not in NAME_STOPWORDS
        }

        s1_numbers = get_address_numbers(s1_address)


        # -----------------------------
        # Matched S2/S3 record
        # -----------------------------
        if row["source"] == "S2":

            matched_name = normalize_name_v2(
                s2_positive_names[row["entity_id_matched"]]
            )

            matched_address = normalize_address(
                s2_positive_addresses[row["entity_id_matched"]]
            )

        else:

            matched_name = normalize_name_v2(
                s3_positive_names[row["entity_id_matched"]]
            )

            matched_address = normalize_address(
                s3_positive_addresses[row["entity_id_matched"]]
            )


        matched_tokens = {
            token
            for token in matched_name.split()
            if len(token) >= 3
            and token not in NAME_STOPWORDS
        }

        matched_numbers = get_address_numbers(matched_address)


        # -----------------------------
        # Two-field blocking condition
        # -----------------------------
        shared_name_tokens = s1_tokens & matched_tokens
        shared_numbers = s1_numbers & matched_numbers

        rare_shared_name = any(
            s1_token_freq.get(token, 999999) <= threshold
            for token in shared_name_tokens
        )

        number_match = bool(shared_numbers)

        if rare_shared_name and number_match:
            recovered += 1


    results.append({
        "max_name_token_frequency": threshold,
        "recovered": recovered,
        "missed": len(positive_pairs) - recovered,
        "recall": recovered / len(positive_pairs)
    })


display(pd.DataFrame(results))

,max_name_token_frequency,recovered,missed,recall
0,1,850,855,0.498534
1,2,971,734,0.569501
2,3,1031,674,0.604692
3,5,1082,623,0.634604
4,10,1114,591,0.653372
5,15,1114,591,0.653372


In [13]:
recovered = 0

for _, row in positive_pairs.iterrows():

    # -----------------------------
    # S1
    # -----------------------------
    s1_name = normalize_name_v2(
        s1_name_lookup[row["entity_id_s1"]]
    )

    s1_address = normalize_address(
        s1_address_lookup[row["entity_id_s1"]]
    )

    # -----------------------------
    # Matched record
    # -----------------------------
    if row["source"] == "S2":

        matched_name = normalize_name_v2(
            s2_positive_names[row["entity_id_matched"]]
        )

        matched_address = normalize_address(
            s2_positive_addresses[row["entity_id_matched"]]
        )

    else:

        matched_name = normalize_name_v2(
            s3_positive_names[row["entity_id_matched"]]
        )

        matched_address = normalize_address(
            s3_positive_addresses[row["entity_id_matched"]]
        )

    # -----------------------------
    # Block 1: exact name
    # -----------------------------
    exact_name = (
        s1_name == matched_name
        and s1_name != ""
    )

    # -----------------------------
    # Block 2: exact address
    # -----------------------------
    exact_address = (
        s1_address == matched_address
        and s1_address != ""
    )

    # -----------------------------
    # Block 3: first two name tokens
    # -----------------------------
    s1_tokens = s1_name.split()
    matched_tokens = matched_name.split()

    s1_two = " ".join(s1_tokens[:2])
    matched_two = " ".join(matched_tokens[:2])

    two_token_match = (
        s1_two != ""
        and s1_two == matched_two
    )

    if exact_name or exact_address or two_token_match:
        recovered += 1


total = len(positive_pairs)

print("Recovered:", recovered)
print("Missed:", total - recovered)
print("Recall:", recovered / total)

Recovered: 1044
Missed: 661
Recall: 0.612316715542522


In [14]:
from difflib import SequenceMatcher

def char_similarity(a, b):
    if not a or not b:
        return 0.0

    return SequenceMatcher(None, a, b).ratio()


# Recreate the 661 matches missed by our current basic blocker
missed_rows = []

for _, row in positive_pairs.iterrows():

    s1_name = normalize_name_v2(
        s1_name_lookup[row["entity_id_s1"]]
    )

    s1_address = normalize_address(
        s1_address_lookup[row["entity_id_s1"]]
    )

    if row["source"] == "S2":
        matched_name = normalize_name_v2(
            s2_positive_names[row["entity_id_matched"]]
        )

        matched_address = normalize_address(
            s2_positive_addresses[row["entity_id_matched"]]
        )

    else:
        matched_name = normalize_name_v2(
            s3_positive_names[row["entity_id_matched"]]
        )

        matched_address = normalize_address(
            s3_positive_addresses[row["entity_id_matched"]]
        )

    # Existing three blockers
    exact_name = (
        s1_name != "" and
        s1_name == matched_name
    )

    exact_address = (
        s1_address != "" and
        s1_address == matched_address
    )

    s1_two = " ".join(s1_name.split()[:2])
    matched_two = " ".join(matched_name.split()[:2])

    two_token_match = (
        s1_two != "" and
        s1_two == matched_two
    )

    if not (exact_name or exact_address or two_token_match):

        similarity = char_similarity(
            s1_name,
            matched_name
        )

        missed_rows.append({
            "entity_id_s1": row["entity_id_s1"],
            "entity_id_matched": row["entity_id_matched"],
            "source": row["source"],
            "s1_name": s1_name,
            "matched_name": matched_name,
            "char_similarity": similarity
        })


missed_df = pd.DataFrame(missed_rows)

print("Missed true matches:", len(missed_df))

print("\nSimilarity distribution:")
print(
    missed_df["char_similarity"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

Missed true matches: 661

Similarity distribution:
count    661.000000
mean       0.587292
std        0.345178
min        0.000000
25%        0.315789
50%        0.714286
75%        0.867925
90%        0.923077
95%        0.947368
max        0.987952
Name: char_similarity, dtype: float64


In [15]:
thresholds = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

results = []

for threshold in thresholds:

    recovered = (
        missed_df["char_similarity"] >= threshold
    ).sum()

    results.append({
        "similarity_threshold": threshold,
        "recovered_from_661": recovered,
        "missed_from_661": len(missed_df) - recovered,
        "recall_of_currently_missed": recovered / len(missed_df),
        "overall_recall": (
            1044 + recovered
        ) / len(positive_pairs)
    })

display(pd.DataFrame(results))

,similarity_threshold,recovered_from_661,missed_from_661,recall_of_currently_missed,overall_recall
0,0.50,473,188,0.715582,0.889736
1,0.60,438,223,0.662632,0.869208
2,0.70,352,309,0.532526,0.818768
3,0.75,296,365,0.447806,0.785924
4,0.80,250,411,0.378215,0.758944
5,0.85,187,474,0.282905,0.721994
6,0.90,95,566,0.143722,0.668035
7,0.95,29,632,0.043873,0.629326


In [16]:
def char_token_signature(name):
    """
    Create a character-based signature from meaningful name tokens.

    Example:
    'ABC Technologies Private Limited'
    -> ('abc', 'tec')
    """

    name = normalize_name_v2(name)

    tokens = [
        token
        for token in name.split()
        if len(token) >= 3
        and token not in NAME_STOPWORDS
    ]

    # First 3 characters of each meaningful token
    prefixes = {
        token[:3]
        for token in tokens
    }

    return prefixes


thresholds = [1, 2]

results = []

for min_shared in thresholds:

    recovered = 0

    for _, row in positive_pairs.iterrows():

        s1_name = normalize_name_v2(
            s1_name_lookup[row["entity_id_s1"]]
        )

        if row["source"] == "S2":
            matched_name = normalize_name_v2(
                s2_positive_names[row["entity_id_matched"]]
            )
        else:
            matched_name = normalize_name_v2(
                s3_positive_names[row["entity_id_matched"]]
            )

        s1_signature = char_token_signature(s1_name)
        matched_signature = char_token_signature(matched_name)

        shared = s1_signature & matched_signature

        if len(shared) >= min_shared:
            recovered += 1

    results.append({
        "minimum_shared_3char_prefixes": min_shared,
        "recovered": recovered,
        "missed": len(positive_pairs) - recovered,
        "recall": recovered / len(positive_pairs)
    })

display(pd.DataFrame(results))

,minimum_shared_3char_prefixes,recovered,missed,recall
0,1,1485,220,0.870968
1,2,1120,585,0.656891


In [17]:
from collections import defaultdict

# ---------------------------------------------------------
# Build 3-character prefix -> S1 entity IDs
# ---------------------------------------------------------

s1_prefix_index = defaultdict(set)

for s1_id, tokens in s1_name_tokens.items():

    for token in tokens:

        if len(token) >= 3:
            prefix = token[:3]
            s1_prefix_index[prefix].add(s1_id)


print("Unique 3-character prefixes:", len(s1_prefix_index))

print("\nMost common prefixes:")
prefix_frequency = sorted(
    [
        (prefix, len(ids))
        for prefix, ids in s1_prefix_index.items()
    ],
    key=lambda x: x[1],
    reverse=True
)

print(prefix_frequency[:20])

Unique 3-character prefixes: 531

Most common prefixes:
[('lim', 122), ('pri', 98), ('llc', 66), ('inc', 59), ('ltd', 33), ('pvt', 28), ('con', 24), ('ind', 24), ('car', 22), ('med', 14), ('ass', 14), ('gro', 14), ('tra', 13), ('int', 12), ('pro', 11), ('tec', 11), ('cen', 11), ('inf', 11), ('llp', 11), ('inv', 10)]


In [18]:
# Build a 3-character prefix index using only meaningful name tokens

s1_useful_prefix_index = defaultdict(set)

for s1_id, tokens in s1_name_tokens.items():

    for token in tokens:

        if (
            len(token) >= 3
            and token not in NAME_STOPWORDS
        ):
            prefix = token[:3]
            s1_useful_prefix_index[prefix].add(s1_id)


# Show prefix statistics
prefix_frequency = sorted(
    [
        (prefix, len(ids))
        for prefix, ids in s1_useful_prefix_index.items()
    ],
    key=lambda x: x[1],
    reverse=True
)

print(
    "Unique useful 3-character prefixes:",
    len(s1_useful_prefix_index)
)

print("\nMost common useful prefixes:")
print(prefix_frequency[:30])

Unique useful 3-character prefixes: 523

Most common useful prefixes:
[('con', 24), ('car', 22), ('med', 14), ('tra', 13), ('pro', 11), ('tec', 11), ('cen', 11), ('inf', 11), ('inv', 10), ('bro', 10), ('ped', 9), ('ind', 9), ('man', 9), ('and', 8), ('pre', 8), ('hea', 8), ('den', 8), ('per', 8), ('cha', 7), ('ent', 7), ('har', 7), ('fou', 7), ('adv', 7), ('sil', 7), ('des', 6), ('cre', 6), ('lak', 6), ('mar', 6), ('sta', 6), ('all', 6)]


In [20]:
# Measure candidate volume for the useful 3-character prefix blocker
# without storing candidate pairs.

s2_candidates = 0
s3_candidates = 0

# ---------------------------------------------------------
# S2
# ---------------------------------------------------------

for chunk in pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        name = normalize_name_v2(row["business_name"])

        tokens = {
            token
            for token in name.split()
            if (
                len(token) >= 3
                and token not in NAME_STOPWORDS
            )
        }

        prefixes = {
            token[:3]
            for token in tokens
        }

        matched_s1 = set()

        for prefix in prefixes:
            matched_s1.update(
                s1_useful_prefix_index.get(prefix, set())
            )

        s2_candidates += len(matched_s1)


# ---------------------------------------------------------
# S3
# ---------------------------------------------------------

for chunk in pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        name = normalize_name_v2(row["business_name"])

        tokens = {
            token
            for token in name.split()
            if (
                len(token) >= 3
                and token not in NAME_STOPWORDS
            )
        }

        prefixes = {
            token[:3]
            for token in tokens
        }

        matched_s1 = set()

        for prefix in prefixes:
            matched_s1.update(
                s1_useful_prefix_index.get(prefix, set())
            )

        s3_candidates += len(matched_s1)


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

total = s2_candidates + s3_candidates

print("S2 candidates:", f"{s2_candidates:,}")
print("S3 candidates:", f"{s3_candidates:,}")
print("Total candidates:", f"{total:,}")
print(
    "Average candidates per S1:",
    total / len(blocking_s1)
)

S2 candidates: 37,581,941
S3 candidates: 42,155,043
Total candidates: 79,736,984
Average candidates per S1: 159473.968


In [21]:
# Build a selective 2-token name-prefix index for S1

from collections import defaultdict

s1_two_prefix_index = defaultdict(set)

for s1_id, name in s1_name_lookup.items():
    tokens = [
        token for token in name.split()
        if len(token) >= 3 and token not in NAME_STOPWORDS
    ]

    # Need at least 2 useful tokens
    if len(tokens) >= 2:
        signature = (tokens[0][:3], tokens[1][:3])
        s1_two_prefix_index[signature].add(s1_id)

print("Unique 2-token prefix signatures:",
      len(s1_two_prefix_index))

print("Example signatures:")
for sig, ids in list(s1_two_prefix_index.items())[:10]:
    print(sig, "->", len(ids), "S1 entities")

Unique 2-token prefix signatures: 472
Example signatures:
('Ped', 'Med') -> 2 S1 entities
('Fet', 'Nat') -> 1 S1 entities
('Gen', 'Des') -> 1 S1 entities
('Con', 'Ide') -> 1 S1 entities
('Pen', 'and') -> 1 S1 entities
('Nat', 'Inv') -> 1 S1 entities
('Reu', 'Ida') -> 1 S1 entities
('Cre', 'Pro') -> 1 S1 entities
('Goa', 'Sto') -> 1 S1 entities
('Lak', 'Con') -> 2 S1 entities


In [22]:
# Measure recall + candidate volume for the
# 2-token, 3-character name-prefix blocker

# True positive lookup sets
true_s2_pairs = set(
    zip(
        positive_pairs.loc[positive_pairs["source"] == "S2", "entity_id_s1"],
        positive_pairs.loc[positive_pairs["source"] == "S2", "entity_id_matched"]
    )
)

true_s3_pairs = set(
    zip(
        positive_pairs.loc[positive_pairs["source"] == "S3", "entity_id_s1"],
        positive_pairs.loc[positive_pairs["source"] == "S3", "entity_id_matched"]
    )
)

s2_candidates = 0
s3_candidates = 0

s2_hits = 0
s3_hits = 0

# --------------------------------------------------
# S2
# --------------------------------------------------

for chunk in pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        name = normalize_name_v2(row["business_name"])

        tokens = [
            token for token in name.split()
            if len(token) >= 3 and token not in NAME_STOPWORDS
        ]

        if len(tokens) < 2:
            continue

        signature = (tokens[0][:3], tokens[1][:3])

        matched_s1 = s1_two_prefix_index.get(signature, set())

        s2_candidates += len(matched_s1)

        entity_id = row["entity_id"]

        for s1_id in matched_s1:
            if (s1_id, entity_id) in true_s2_pairs:
                s2_hits += 1


# --------------------------------------------------
# S3
# --------------------------------------------------

for chunk in pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        name = normalize_name_v2(row["business_name"])

        tokens = [
            token for token in name.split()
            if len(token) >= 3 and token not in NAME_STOPWORDS
        ]

        if len(tokens) < 2:
            continue

        signature = (tokens[0][:3], tokens[1][:3])

        matched_s1 = s1_two_prefix_index.get(signature, set())

        s3_candidates += len(matched_s1)

        entity_id = row["entity_id"]

        for s1_id in matched_s1:
            if (s1_id, entity_id) in true_s3_pairs:
                s3_hits += 1


# --------------------------------------------------
# Results
# --------------------------------------------------

total_candidates = s2_candidates + s3_candidates
total_hits = s2_hits + s3_hits
total_true = len(positive_pairs)

print("S2 candidates:", f"{s2_candidates:,}")
print("S3 candidates:", f"{s3_candidates:,}")
print("Total candidates:", f"{total_candidates:,}")

print()
print("S2 true matches recovered:", s2_hits)
print("S3 true matches recovered:", s3_hits)
print("Total true matches recovered:", total_hits)
print("Total true matches:", total_true)

print()
print(
    "Recall:",
    f"{total_hits / total_true:.4%}"
)

print(
    "Average candidates per S1:",
    f"{total_candidates / len(blocking_s1):,.2f}"
)

S2 candidates: 0
S3 candidates: 0
Total candidates: 0

S2 true matches recovered: 0
S3 true matches recovered: 0
Total true matches recovered: 0
Total true matches: 1705

Recall: 0.0000%
Average candidates per S1: 0.00


In [23]:
# Diagnose the 2-token prefix index

print("First 10 index keys:")
for key in list(s1_two_prefix_index.keys())[:10]:
    print(key)

print("\nFirst 10 S1 names:")
for s1_id, name in list(s1_name_lookup.items())[:10]:
    print(s1_id, "->", name)

print("\nTesting normalized signatures from those S1 names:")

for s1_id, name in list(s1_name_lookup.items())[:10]:
    tokens = [
        token for token in normalize_name_v2(name).split()
        if len(token) >= 3 and token not in NAME_STOPWORDS
    ]

    if len(tokens) >= 2:
        signature = (tokens[0][:3], tokens[1][:3])
        print(
            s1_id,
            "->",
            signature,
            "FOUND:", signature in s1_two_prefix_index
        )

First 10 index keys:
('Ped', 'Med')
('Fet', 'Nat')
('Gen', 'Des')
('Con', 'Ide')
('Pen', 'and')
('Nat', 'Inv')
('Reu', 'Ida')
('Cre', 'Pro')
('Goa', 'Sto')
('Lak', 'Con')

First 10 S1 names:
S1-53356671 -> Pediatric Medicine PLLC
S1-320151505 -> Fetech National Twin
S1-938947364 -> General Design Innovations LLC
S1-195839862 -> Construction Ideaz Papers Private Limited
S1-655046555 -> Penaloza and Bittle First Inc.
S1-758070915 -> National Investments LLC
S1-169338169 -> Reus Idaho Company
S1-796421498 -> Creative Projects Limited
S1-802535179 -> Goar Stone LLC
S1-97176033 -> Lakshmi Consultants Private Limited

Testing normalized signatures from those S1 names:
S1-53356671 -> ('ped', 'med') FOUND: False
S1-320151505 -> ('fet', 'nat') FOUND: False
S1-938947364 -> ('gen', 'des') FOUND: False
S1-195839862 -> ('con', 'ide') FOUND: False
S1-655046555 -> ('pen', 'and') FOUND: False
S1-758070915 -> ('nat', 'inv') FOUND: False
S1-169338169 -> ('reu', 'ida') FOUND: False
S1-796421498 -> ('cre'

In [24]:
# Rebuild the 2-token prefix index using normalized lowercase names

from collections import defaultdict

s1_two_prefix_index = defaultdict(set)

for s1_id, original_name in s1_name_lookup.items():

    # Normalize first
    name = normalize_name_v2(original_name)

    # Keep useful tokens only
    tokens = [
        token for token in name.split()
        if len(token) >= 3 and token not in NAME_STOPWORDS
    ]

    # Need at least two useful tokens
    if len(tokens) >= 2:

        signature = (
            tokens[0][:3],
            tokens[1][:3]
        )

        s1_two_prefix_index[signature].add(s1_id)


print("Unique 2-token prefix signatures:",
      len(s1_two_prefix_index))

print("\nFirst 10 signatures:")
for sig, ids in list(s1_two_prefix_index.items())[:10]:
    print(sig, "->", len(ids), "S1 entities")

Unique 2-token prefix signatures: 432

First 10 signatures:
('ped', 'med') -> 2 S1 entities
('fet', 'nat') -> 1 S1 entities
('gen', 'des') -> 1 S1 entities
('con', 'ide') -> 1 S1 entities
('pen', 'and') -> 1 S1 entities
('nat', 'inv') -> 1 S1 entities
('reu', 'ida') -> 1 S1 entities
('cre', 'pro') -> 1 S1 entities
('goa', 'sto') -> 1 S1 entities
('lak', 'con') -> 2 S1 entities


In [25]:
# Measure recall + candidate volume for the corrected
# 2-token, 3-character name-prefix blocker

true_s2_pairs = set(
    zip(
        positive_pairs.loc[
            positive_pairs["source"] == "S2",
            "entity_id_s1"
        ],
        positive_pairs.loc[
            positive_pairs["source"] == "S2",
            "entity_id_matched"
        ]
    )
)

true_s3_pairs = set(
    zip(
        positive_pairs.loc[
            positive_pairs["source"] == "S3",
            "entity_id_s1"
        ],
        positive_pairs.loc[
            positive_pairs["source"] == "S3",
            "entity_id_matched"
        ]
    )
)

s2_candidates = 0
s3_candidates = 0

s2_hits = 0
s3_hits = 0


# =========================
# S2
# =========================

for chunk in pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        name = normalize_name_v2(row["business_name"])

        tokens = [
            token
            for token in name.split()
            if len(token) >= 3
            and token not in NAME_STOPWORDS
        ]

        if len(tokens) < 2:
            continue

        signature = (
            tokens[0][:3],
            tokens[1][:3]
        )

        matched_s1 = s1_two_prefix_index.get(
            signature,
            set()
        )

        s2_candidates += len(matched_s1)

        entity_id = row["entity_id"]

        for s1_id in matched_s1:
            if (s1_id, entity_id) in true_s2_pairs:
                s2_hits += 1


# =========================
# S3
# =========================

for chunk in pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        name = normalize_name_v2(row["business_name"])

        tokens = [
            token
            for token in name.split()
            if len(token) >= 3
            and token not in NAME_STOPWORDS
        ]

        if len(tokens) < 2:
            continue

        signature = (
            tokens[0][:3],
            tokens[1][:3]
        )

        matched_s1 = s1_two_prefix_index.get(
            signature,
            set()
        )

        s3_candidates += len(matched_s1)

        entity_id = row["entity_id"]

        for s1_id in matched_s1:
            if (s1_id, entity_id) in true_s3_pairs:
                s3_hits += 1


# =========================
# Results
# =========================

total_candidates = s2_candidates + s3_candidates
total_hits = s2_hits + s3_hits
total_true = len(positive_pairs)

print("S2 candidates:", f"{s2_candidates:,}")
print("S3 candidates:", f"{s3_candidates:,}")
print("Total candidates:", f"{total_candidates:,}")

print()

print("S2 true matches recovered:", s2_hits)
print("S3 true matches recovered:", s3_hits)
print("Total true matches recovered:", total_hits)
print("Total true matches:", total_true)

print()

print(
    "Recall:",
    f"{total_hits / total_true:.4%}"
)

print(
    "Average candidates per S1:",
    f"{total_candidates / len(blocking_s1):,.2f}"
)

S2 candidates: 243,954
S3 candidates: 247,214
Total candidates: 491,168

S2 true matches recovered: 469
S3 true matches recovered: 508
Total true matches recovered: 977
Total true matches: 1705

Recall: 57.3021%
Average candidates per S1: 982.34


In [26]:
# Measure recall overlap between our selective blockers
# using the 1,705 known true matches.

def get_useful_tokens(name):
    name = normalize_name_v2(name)
    return [
        token
        for token in name.split()
        if len(token) >= 3
        and token not in NAME_STOPWORDS
    ]


# Build S1 signatures
s1_exact_name = {}
s1_exact_address = {}
s1_first_two_tokens = {}
s1_two_prefix = {}

for s1_id, name in s1_name_lookup.items():

    normalized_name = normalize_name_v2(name)
    s1_exact_name[s1_id] = normalized_name

    tokens = get_useful_tokens(name)

    if len(tokens) >= 2:
        s1_first_two_tokens[s1_id] = (
            tokens[0],
            tokens[1]
        )

    if len(tokens) >= 2:
        s1_two_prefix[s1_id] = (
            tokens[0][:3],
            tokens[1][:3]
        )


for s1_id, address in s1_address_lookup.items():
    s1_exact_address[s1_id] = normalize_address(address)


# -----------------------------------------
# Evaluate each true pair
# -----------------------------------------

results = []

for _, row in positive_pairs.iterrows():

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    if source == "S2":
        source_name = s2_positive_names.get(matched_id, "")
        source_address = s2_positive_addresses.get(matched_id, "")
    else:
        source_name = s3_positive_names.get(matched_id, "")
        source_address = s3_positive_addresses.get(matched_id, "")

    name = normalize_name_v2(source_name)
    address = normalize_address(source_address)

    tokens = get_useful_tokens(source_name)

    exact_name = (
        name != ""
        and name == s1_exact_name.get(s1_id, "")
    )

    exact_address = (
        address != ""
        and address == s1_exact_address.get(s1_id, "")
    )

    first_two = (
        len(tokens) >= 2
        and s1_first_two_tokens.get(s1_id) ==
            (tokens[0], tokens[1])
    )

    two_prefix = (
        len(tokens) >= 2
        and s1_two_prefix.get(s1_id) ==
            (tokens[0][:3], tokens[1][:3])
    )

    results.append({
        "exact_name": exact_name,
        "exact_address": exact_address,
        "first_two_tokens": first_two,
        "two_prefix": two_prefix
    })


blocker_results = pd.DataFrame(results)

print("Total true matches:", len(blocker_results))
print()

for col in blocker_results.columns:
    count = blocker_results[col].sum()
    print(
        f"{col:20s}: "
        f"{count:4d} / {len(blocker_results)} "
        f"({count / len(blocker_results):.2%})"
    )

print()

union = (
    blocker_results["exact_name"]
    | blocker_results["exact_address"]
    | blocker_results["first_two_tokens"]
    | blocker_results["two_prefix"]
)

print(
    "UNION recall:",
    f"{union.sum()} / {len(union)} "
    f"({union.mean():.2%})"
)

print(
    "Still missed:",
    (~union).sum()
)

Total true matches: 1705

exact_name          :  440 / 1705 (25.81%)
exact_address       :  134 / 1705 (7.86%)
first_two_tokens    :  935 / 1705 (54.84%)
two_prefix          :  977 / 1705 (57.30%)

UNION recall: 1092 / 1705 (64.05%)
Still missed: 613


In [27]:
# Analyze character 3-gram overlap for the 613 matches
# currently missed by our selective blockers.

def char_ngrams(text, n=3):
    text = normalize_name_v2(text)
    text = text.replace(" ", "")

    if len(text) < n:
        return set()

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }


# Get the currently missed true matches
missed_rows = []

for idx, row in positive_pairs.iterrows():

    blocker_row = blocker_results.iloc[idx]

    union_hit = (
        blocker_row["exact_name"]
        or blocker_row["exact_address"]
        or blocker_row["first_two_tokens"]
        or blocker_row["two_prefix"]
    )

    if not union_hit:
        missed_rows.append(row)


missed_df = pd.DataFrame(missed_rows)

print("Currently missed matches:", len(missed_df))


# Calculate maximum 3-gram Jaccard between the
# S1 name and its true matched source name.

jaccard_scores = []

for _, row in missed_df.iterrows():

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        source_name = s2_positive_names[matched_id]
    else:
        source_name = s3_positive_names[matched_id]

    grams1 = char_ngrams(s1_name)
    grams2 = char_ngrams(source_name)

    if not grams1 or not grams2:
        score = 0
    else:
        score = len(grams1 & grams2) / len(grams1 | grams2)

    jaccard_scores.append(score)


missed_df = missed_df.copy()
missed_df["char_3gram_jaccard"] = jaccard_scores


print()
print(missed_df["char_3gram_jaccard"].describe())

print()

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:

    recovered = (
        missed_df["char_3gram_jaccard"] >= threshold
    ).sum()

    total_recall = (
        1092 + recovered
    ) / 1705

    print(
        f"Threshold >= {threshold:.2f}: "
        f"recovers {recovered:3d} / 613 "
        f"missed | overall recall = {total_recall:.2%}"
    )

Currently missed matches: 613

count    613.000000
mean       0.396849
std        0.297970
min        0.000000
25%        0.000000
50%        0.440000
75%        0.642857
max        1.000000
Name: char_3gram_jaccard, dtype: float64

Threshold >= 0.20: recovers 423 / 613 missed | overall recall = 88.86%
Threshold >= 0.30: recovers 386 / 613 missed | overall recall = 86.69%
Threshold >= 0.40: recovers 339 / 613 missed | overall recall = 83.93%
Threshold >= 0.50: recovers 272 / 613 missed | overall recall = 80.00%
Threshold >= 0.60: recovers 182 / 613 missed | overall recall = 74.72%
Threshold >= 0.70: recovers 123 / 613 missed | overall recall = 71.26%


In [28]:
from collections import Counter, defaultdict

# --------------------------------------------------
# Build character 3-gram frequency across S1
# --------------------------------------------------

s1_chargram_freq = Counter()

for s1_id, name in s1_name_lookup.items():

    grams = char_ngrams(name, 3)

    for gram in grams:
        s1_chargram_freq[gram] += 1


print("Unique 3-grams:", len(s1_chargram_freq))

print("\nMost common 3-grams:")
for gram, count in s1_chargram_freq.most_common(20):
    print(gram, "->", count)


# --------------------------------------------------
# Build index using rare 3-grams
# --------------------------------------------------

s1_rare_chargram_index = defaultdict(set)

# A gram appearing in <= 3 S1 names
# is considered selective for this experiment.

for s1_id, name in s1_name_lookup.items():

    grams = char_ngrams(name, 3)

    for gram in grams:

        if s1_chargram_freq[gram] <= 3:
            s1_rare_chargram_index[gram].add(s1_id)


print("\nRare 3-gram index size:",
      len(s1_rare_chargram_index))

print(
    "S1 entities indexed:",
    len(set().union(*s1_rare_chargram_index.values()))
)

Unique 3-grams: 2436

Most common 3-grams:
mit -> 129
ite -> 126
ted -> 125
imi -> 122
lim -> 122
ate -> 117
pri -> 105
eli -> 102
vat -> 101
tel -> 98
iva -> 97
riv -> 96
llc -> 74
inc -> 60
ent -> 56
ion -> 49
ers -> 44
tio -> 39
spr -> 37
ati -> 36

Rare 3-gram index size: 1793
S1 entities indexed: 490


In [29]:
# Measure candidate volume and recall for the rare
# character 3-gram blocker.
#
# IMPORTANT:
# We only count candidates; we do NOT store them.

s2_candidates = 0
s3_candidates = 0

s2_hits = 0
s3_hits = 0


# --------------------------------------------------
# S2
# --------------------------------------------------

for chunk in pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        name = normalize_name_v2(row["business_name"])

        grams = char_ngrams(name, 3)

        matched_s1 = set()

        for gram in grams:

            if gram in s1_rare_chargram_index:
                matched_s1.update(
                    s1_rare_chargram_index[gram]
                )

        s2_candidates += len(matched_s1)

        entity_id = row["entity_id"]

        for s1_id in matched_s1:

            if (s1_id, entity_id) in true_s2_pairs:
                s2_hits += 1


# --------------------------------------------------
# S3
# --------------------------------------------------

for chunk in pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        name = normalize_name_v2(row["business_name"])

        grams = char_ngrams(name, 3)

        matched_s1 = set()

        for gram in grams:

            if gram in s1_rare_chargram_index:
                matched_s1.update(
                    s1_rare_chargram_index[gram]
                )

        s3_candidates += len(matched_s1)

        entity_id = row["entity_id"]

        for s1_id in matched_s1:

            if (s1_id, entity_id) in true_s3_pairs:
                s3_hits += 1


# --------------------------------------------------
# Results
# --------------------------------------------------

total_candidates = s2_candidates + s3_candidates
total_hits = s2_hits + s3_hits

print("S2 candidates:", f"{s2_candidates:,}")
print("S3 candidates:", f"{s3_candidates:,}")
print("Total candidates:", f"{total_candidates:,}")

print()

print("S2 true matches recovered:", s2_hits)
print("S3 true matches recovered:", s3_hits)

print(
    "Total true matches recovered:",
    total_hits
)

print(
    "Total true matches:",
    len(positive_pairs)
)

print()

print(
    "Recall:",
    f"{total_hits / len(positive_pairs):.4%}"
)

print(
    "Average candidates per S1:",
    f"{total_candidates / len(blocking_s1):,.2f}"
)

S2 candidates: 30,811,709
S3 candidates: 34,143,233
Total candidates: 64,954,942

S2 true matches recovered: 666
S3 true matches recovered: 787
Total true matches recovered: 1453
Total true matches: 1705

Recall: 85.2199%
Average candidates per S1: 129,909.88


In [30]:
# Test how many true matches have at least
# 2 shared rare character 3-grams.

def get_rare_grams(name):
    grams = char_ngrams(name, 3)

    return {
        gram for gram in grams
        if gram in s1_rare_chargram_index
    }


recovered_1 = 0
recovered_2 = 0
recovered_3 = 0

for _, row in positive_pairs.iterrows():

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        source_name = s2_positive_names[matched_id]
    else:
        source_name = s3_positive_names[matched_id]

    s1_grams = get_rare_grams(s1_name)
    source_grams = get_rare_grams(source_name)

    shared = len(s1_grams & source_grams)

    if shared >= 1:
        recovered_1 += 1

    if shared >= 2:
        recovered_2 += 1

    if shared >= 3:
        recovered_3 += 1


print("Total true matches:", len(positive_pairs))
print()

print(
    ">= 1 shared rare 3-gram:",
    recovered_1,
    f"({recovered_1 / len(positive_pairs):.2%})"
)

print(
    ">= 2 shared rare 3-grams:",
    recovered_2,
    f"({recovered_2 / len(positive_pairs):.2%})"
)

print(
    ">= 3 shared rare 3-grams:",
    recovered_3,
    f"({recovered_3 / len(positive_pairs):.2%})"
)

Total true matches: 1705

>= 1 shared rare 3-gram: 1453 (85.22%)
>= 2 shared rare 3-grams: 1297 (76.07%)
>= 3 shared rare 3-grams: 1114 (65.34%)


In [31]:
# Measure candidate volume for the
# >= 2 shared rare character 3-gram blocker.
#
# We only count candidates; we don't store them.

s2_candidates = 0
s3_candidates = 0

s2_hits = 0
s3_hits = 0


# --------------------------------------------------
# S2
# --------------------------------------------------

for chunk in pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        name = normalize_name_v2(row["business_name"])

        grams = {
            gram
            for gram in char_ngrams(name, 3)
            if gram in s1_rare_chargram_index
        }

        if not grams:
            continue

        # Count how many rare grams each S1 entity shares
        s1_shared_counts = Counter()

        for gram in grams:

            for s1_id in s1_rare_chargram_index[gram]:
                s1_shared_counts[s1_id] += 1

        # Keep only S1 entities sharing >= 2 grams
        matched_s1 = {
            s1_id
            for s1_id, count in s1_shared_counts.items()
            if count >= 2
        }

        s2_candidates += len(matched_s1)

        entity_id = row["entity_id"]

        for s1_id in matched_s1:

            if (s1_id, entity_id) in true_s2_pairs:
                s2_hits += 1


# --------------------------------------------------
# S3
# --------------------------------------------------

for chunk in pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        name = normalize_name_v2(row["business_name"])

        grams = {
            gram
            for gram in char_ngrams(name, 3)
            if gram in s1_rare_chargram_index
        }

        if not grams:
            continue

        s1_shared_counts = Counter()

        for gram in grams:

            for s1_id in s1_rare_chargram_index[gram]:
                s1_shared_counts[s1_id] += 1

        matched_s1 = {
            s1_id
            for s1_id, count in s1_shared_counts.items()
            if count >= 2
        }

        s3_candidates += len(matched_s1)

        entity_id = row["entity_id"]

        for s1_id in matched_s1:

            if (s1_id, entity_id) in true_s3_pairs:
                s3_hits += 1


# --------------------------------------------------
# Results
# --------------------------------------------------

total_candidates = s2_candidates + s3_candidates
total_hits = s2_hits + s3_hits

print("S2 candidates:", f"{s2_candidates:,}")
print("S3 candidates:", f"{s3_candidates:,}")
print("Total candidates:", f"{total_candidates:,}")

print()

print("S2 true matches recovered:", s2_hits)
print("S3 true matches recovered:", s3_hits)

print(
    "Total true matches recovered:",
    total_hits
)

print(
    "Total true matches:",
    len(positive_pairs)
)

print()

print(
    "Recall:",
    f"{total_hits / len(positive_pairs):.4%}"
)

print(
    "Average candidates per S1:",
    f"{total_candidates / len(blocking_s1):,.2f}"
)

KeyboardInterrupt: 

In [32]:
# Analyze the 613 true matches missed by our current blockers

missed_rows = []

for idx, row in positive_pairs.iterrows():

    blocker_row = blocker_results.iloc[idx]

    union_hit = (
        blocker_row["exact_name"]
        or blocker_row["exact_address"]
        or blocker_row["first_two_tokens"]
        or blocker_row["two_prefix"]
    )

    if not union_hit:
        missed_rows.append(row)


missed_df = pd.DataFrame(missed_rows)

print("Missed true matches:", len(missed_df))
print()

# Show 20 examples
for _, row in missed_df.head(20).iterrows():

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        matched_name = s2_positive_names[matched_id]
    else:
        matched_name = s3_positive_names[matched_id]

    print(
        f"{source} | {s1_id} -> {matched_id}"
    )
    print("S1 :", s1_name)
    print("SRC:", matched_name)
    print("-" * 70)

Missed true matches: 613

S3 | S1-614166735 -> S3-244407822
S1 : American Catalyst
SRC: American  Cetl
----------------------------------------------------------------------
S2 | S1-210935903 -> S2-388023430
S1 : Leann Colon Federal Acquisitions Inc
SRC: leanncolonfederal.com
----------------------------------------------------------------------
S2 | S1-210935903 -> S2-842875529
S1 : Leann Colon Federal Acquisitions Inc
SRC: LEANNCOLONFEDERAL.COM
----------------------------------------------------------------------
S2 | S1-983540069 -> S2-859028958
S1 : Ss Technology Private Limited
SRC: एसएस टेक्नोलॉजी प्राइवेट लिमिटेड
----------------------------------------------------------------------
S2 | S1-983540069 -> S2-4190133
S1 : Ss Technology Private Limited
SRC: एसएस टेक्नोलॉजी प्राइवेट लिमिटेड
----------------------------------------------------------------------
S3 | S1-983540069 -> S3-435171628
S1 : Ss Technology Private Limited
SRC: एसएस Technology Private Limited
------------------

In [33]:
# Test order-independent useful-token signatures
# on the 1,705 known true matches.

def useful_token_signature(text):
    name = normalize_name_v2(text)

    tokens = {
        token
        for token in name.split()
        if len(token) >= 3
        and token not in NAME_STOPWORDS
    }

    return tuple(sorted(tokens))


token_sig_recovered = 0

for _, row in positive_pairs.iterrows():

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        source_name = s2_positive_names[matched_id]
    else:
        source_name = s3_positive_names[matched_id]

    s1_sig = useful_token_signature(s1_name)
    source_sig = useful_token_signature(source_name)

    if s1_sig == source_sig:
        token_sig_recovered += 1


print("Exact token-set signature matches:",
      token_sig_recovered)

print(
    "Recall:",
    f"{token_sig_recovered / len(positive_pairs):.2%}"
)

Exact token-set signature matches: 926
Recall: 54.31%


In [34]:
# Test partial token-overlap blocking on the 1,705 true matches

def useful_token_set(text):
    name = normalize_name_v2(text)

    return {
        token
        for token in name.split()
        if len(token) >= 3
        and token not in NAME_STOPWORDS
    }


recovered_1 = 0
recovered_2 = 0
recovered_3 = 0

for _, row in positive_pairs.iterrows():

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        source_name = s2_positive_names[matched_id]
    else:
        source_name = s3_positive_names[matched_id]

    s1_tokens = useful_token_set(s1_name)
    source_tokens = useful_token_set(source_name)

    shared = len(s1_tokens & source_tokens)

    if shared >= 1:
        recovered_1 += 1

    if shared >= 2:
        recovered_2 += 1

    if shared >= 3:
        recovered_3 += 1


print("Total true matches:", len(positive_pairs))
print()

print(
    ">= 1 shared useful token:",
    recovered_1,
    f"({recovered_1 / len(positive_pairs):.2%})"
)

print(
    ">= 2 shared useful tokens:",
    recovered_2,
    f"({recovered_2 / len(positive_pairs):.2%})"
)

print(
    ">= 3 shared useful tokens:",
    recovered_3,
    f"({recovered_3 / len(positive_pairs):.2%})"
)

Total true matches: 1705

>= 1 shared useful token: 1395 (81.82%)
>= 2 shared useful tokens: 1085 (63.64%)
>= 3 shared useful tokens: 388 (22.76%)


In [35]:
# Check how much >=2 useful-token overlap
# adds to our existing blocker union.

additional_recovered = 0
combined_recovered = 0

for idx, row in positive_pairs.iterrows():

    blocker_row = blocker_results.iloc[idx]

    current_union = (
        blocker_row["exact_name"]
        or blocker_row["exact_address"]
        or blocker_row["first_two_tokens"]
        or blocker_row["two_prefix"]
    )

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        source_name = s2_positive_names[matched_id]
    else:
        source_name = s3_positive_names[matched_id]

    s1_tokens = useful_token_set(s1_name)
    source_tokens = useful_token_set(source_name)

    shared_tokens = len(s1_tokens & source_tokens)

    token_overlap_hit = shared_tokens >= 2

    if token_overlap_hit and not current_union:
        additional_recovered += 1

    if current_union or token_overlap_hit:
        combined_recovered += 1


print("Existing blocker union:")
print("1092 / 1705 = 64.05%")

print()

print(
    "Additional matches from >=2 token overlap:",
    additional_recovered
)

print()

print(
    "Combined recall:",
    combined_recovered,
    "/ 1705 =",
    f"{combined_recovered / 1705:.2%}"
)

Existing blocker union:
1092 / 1705 = 64.05%

Additional matches from >=2 token overlap: 115

Combined recall: 1207 / 1705 = 70.79%


In [36]:
# Measure candidate volume for >=2 shared useful-token blocker
# without storing candidate pairs.

s2_candidates = 0
s3_candidates = 0

s2_hits = 0
s3_hits = 0


# --------------------------------------------------
# S2
# --------------------------------------------------

for chunk in pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        tokens = useful_token_set(row["business_name"])

        if len(tokens) < 2:
            continue

        # Find S1 entities sharing tokens
        candidate_s1 = set()

        for token in tokens:

            if token in s1_token_index:
                candidate_s1.update(
                    s1_token_index[token]
                )

        # Count actual shared tokens
        matched_s1 = []

        for s1_id in candidate_s1:

            s1_tokens = s1_name_tokens[s1_id]

            if len(tokens & s1_tokens) >= 2:
                matched_s1.append(s1_id)

        s2_candidates += len(matched_s1)

        entity_id = row["entity_id"]

        for s1_id in matched_s1:

            if (s1_id, entity_id) in true_s2_pairs:
                s2_hits += 1


# --------------------------------------------------
# S3
# --------------------------------------------------

for chunk in pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        tokens = useful_token_set(row["business_name"])

        if len(tokens) < 2:
            continue

        candidate_s1 = set()

        for token in tokens:

            if token in s1_token_index:
                candidate_s1.update(
                    s1_token_index[token]
                )

        matched_s1 = []

        for s1_id in candidate_s1:

            s1_tokens = s1_name_tokens[s1_id]

            if len(tokens & s1_tokens) >= 2:
                matched_s1.append(s1_id)

        s3_candidates += len(matched_s1)

        entity_id = row["entity_id"]

        for s1_id in matched_s1:

            if (s1_id, entity_id) in true_s3_pairs:
                s3_hits += 1


# --------------------------------------------------
# Results
# --------------------------------------------------

total_candidates = s2_candidates + s3_candidates
total_hits = s2_hits + s3_hits

print("S2 candidates:", f"{s2_candidates:,}")
print("S3 candidates:", f"{s3_candidates:,}")
print("Total candidates:", f"{total_candidates:,}")

print()

print("S2 true matches recovered:", s2_hits)
print("S3 true matches recovered:", s3_hits)

print(
    "Total true matches recovered:",
    total_hits
)

print(
    "Recall:",
    f"{total_hits / len(positive_pairs):.4%}"
)

print()

print(
    "Average candidates per S1:",
    f"{total_candidates / len(blocking_s1):,.2f}"
)

S2 candidates: 387,238
S3 candidates: 408,898
Total candidates: 796,136

S2 true matches recovered: 500
S3 true matches recovered: 585
Total true matches recovered: 1085
Recall: 63.6364%

Average candidates per S1: 1,592.27


In [37]:
# Examine the remaining true matches after our
# current 70.79% blocker union.

remaining = []

for idx, row in positive_pairs.iterrows():

    current_union = (
        blocker_results.iloc[idx]["exact_name"]
        or blocker_results.iloc[idx]["exact_address"]
        or blocker_results.iloc[idx]["first_two_tokens"]
        or blocker_results.iloc[idx]["two_prefix"]
    )

    # Check >=2 useful-token overlap
    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        source_name = s2_positive_names[matched_id]
    else:
        source_name = s3_positive_names[matched_id]

    shared_tokens = len(
        useful_token_set(s1_name)
        & useful_token_set(source_name)
    )

    combined_hit = (
        current_union
        or shared_tokens >= 2
    )

    if not combined_hit:
        remaining.append(row)


remaining_df = pd.DataFrame(remaining)

print("Remaining missed matches:", len(remaining_df))
print()

# Show examples
for _, row in remaining_df.head(30).iterrows():

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        source_name = s2_positive_names[matched_id]
    else:
        source_name = s3_positive_names[matched_id]

    print(f"{source} | {s1_id} -> {matched_id}")
    print("S1 :", s1_name)
    print("SRC:", source_name)
    print("-" * 70)

Remaining missed matches: 498

S3 | S1-614166735 -> S3-244407822
S1 : American Catalyst
SRC: American  Cetl
----------------------------------------------------------------------
S2 | S1-210935903 -> S2-388023430
S1 : Leann Colon Federal Acquisitions Inc
SRC: leanncolonfederal.com
----------------------------------------------------------------------
S2 | S1-210935903 -> S2-842875529
S1 : Leann Colon Federal Acquisitions Inc
SRC: LEANNCOLONFEDERAL.COM
----------------------------------------------------------------------
S2 | S1-983540069 -> S2-859028958
S1 : Ss Technology Private Limited
SRC: एसएस टेक्नोलॉजी प्राइवेट लिमिटेड
----------------------------------------------------------------------
S2 | S1-983540069 -> S2-4190133
S1 : Ss Technology Private Limited
SRC: एसएस टेक्नोलॉजी प्राइवेट लिमिटेड
----------------------------------------------------------------------
S3 | S1-983540069 -> S3-435171628
S1 : Ss Technology Private Limited
SRC: एसएस Technology Private Limited
-------------

In [38]:
import re

def normalize_domain_style(text):
    """
    Convert domain-like text into a compact alphanumeric form.
    Example:
        DILIGENCE-TRADING.COM -> diligencetrading
    """
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Remove protocol if present
    text = re.sub(r"^https?://", "", text)

    # Remove www.
    text = re.sub(r"^www\.", "", text)

    # Remove common domain extensions
    text = re.sub(
        r"\.(com|org|net|co|in|biz|io|us|uk)$",
        "",
        text
    )

    # Keep only alphanumeric characters
    text = re.sub(r"[^a-z0-9]", "", text)

    return text


def compact_s1_name(text):
    """
    Create a compact version of an S1 business name.
    """
    name = normalize_name_v2(text)

    tokens = [
        token
        for token in name.split()
        if token not in NAME_STOPWORDS
    ]

    return "".join(tokens)


domain_recovered = 0

examples = []

for _, row in remaining_df.iterrows():

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        source_name = s2_positive_names[matched_id]
    else:
        source_name = s3_positive_names[matched_id]

    compact_s1 = compact_s1_name(s1_name)
    compact_source = normalize_domain_style(source_name)

    if (
        compact_source
        and len(compact_source) >= 5
        and (
            compact_source in compact_s1
            or compact_s1 in compact_source
        )
    ):
        domain_recovered += 1

        if len(examples) < 10:
            examples.append(
                (s1_name, source_name,
                 compact_s1, compact_source)
            )


print("Domain-style matches recovered:",
      domain_recovered)

print(
    "Recall among remaining 498:",
    f"{domain_recovered / len(remaining_df):.2%}"
)

print()

print("Examples:")

for ex in examples:
    print("S1 :", ex[0])
    print("SRC:", ex[1])
    print("S1 compact :", ex[2])
    print("SRC compact:", ex[3])
    print("-" * 60)

Domain-style matches recovered: 162
Recall among remaining 498: 32.53%

Examples:
S1 : Leann Colon Federal Acquisitions Inc
SRC: leanncolonfederal.com
S1 compact : leanncolonfederalacquisitions
SRC compact: leanncolonfederal
------------------------------------------------------------
S1 : Leann Colon Federal Acquisitions Inc
SRC: LEANNCOLONFEDERAL.COM
S1 compact : leanncolonfederalacquisitions
SRC compact: leanncolonfederal
------------------------------------------------------------
S1 : Ss Technology Private Limited
SRC: Ss Technology Private
S1 compact : sstechnology
SRC compact: sstechnologyprivate
------------------------------------------------------------
S1 : Diligence Trading Private Limited
SRC: DILIGENCETRADING.COM
S1 compact : diligencetrading
SRC compact: diligencetrading
------------------------------------------------------------
S1 : Mohammad Brothers Private Limited
SRC: privatemohammadbrothers.com
S1 compact : mohammadbrothers
SRC compact: privatemohammadbrothers
---

In [39]:
# More conservative legal-suffix removal for domain blocking

DOMAIN_STOPWORDS = {
    "limited",
    "private",
    "llc",
    "inc",
    "ltd",
    "pvt",
    "llp",
    "incorporated",
    "corp",
    "corporation",
    "company",
    "co",
    "plc"
}


def compact_s1_domain_name(text):
    name = normalize_name_v2(text)

    tokens = [
        token
        for token in name.split()
        if token not in DOMAIN_STOPWORDS
    ]

    return "".join(tokens)


# Re-test the 498 remaining matches

domain_recovered = 0

examples = []

for _, row in remaining_df.iterrows():

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        source_name = s2_positive_names[matched_id]
    else:
        source_name = s3_positive_names[matched_id]

    compact_s1 = compact_s1_domain_name(s1_name)
    compact_source = normalize_domain_style(source_name)

    if (
        compact_source
        and len(compact_source) >= 5
        and (
            compact_source in compact_s1
            or compact_s1 in compact_source
        )
    ):
        domain_recovered += 1

        if len(examples) < 10:
            examples.append(
                (s1_name, source_name,
                 compact_s1, compact_source)
            )


print("Domain-style matches recovered:",
      domain_recovered)

print(
    "Recall among remaining 498:",
    f"{domain_recovered / len(remaining_df):.2%}"
)

print()

for ex in examples:
    print("S1 :", ex[0])
    print("SRC:", ex[1])
    print("S1 compact :", ex[2])
    print("SRC compact:", ex[3])
    print("-" * 60)

Domain-style matches recovered: 151
Recall among remaining 498: 30.32%

S1 : Leann Colon Federal Acquisitions Inc
SRC: leanncolonfederal.com
S1 compact : leanncolonfederalacquisitions
SRC compact: leanncolonfederal
------------------------------------------------------------
S1 : Leann Colon Federal Acquisitions Inc
SRC: LEANNCOLONFEDERAL.COM
S1 compact : leanncolonfederalacquisitions
SRC compact: leanncolonfederal
------------------------------------------------------------
S1 : Ss Technology Private Limited
SRC: Ss Technology Private
S1 compact : sstechnology
SRC compact: sstechnologyprivate
------------------------------------------------------------
S1 : Diligence Trading Private Limited
SRC: DILIGENCETRADING.COM
S1 compact : diligencetrading
SRC compact: diligencetrading
------------------------------------------------------------
S1 : Mohammad Brothers Private Limited
SRC: privatemohammadbrothers.com
S1 compact : mohammadbrothers
SRC compact: privatemohammadbrothers
-------------

In [40]:
# Check how many additional true matches are recovered
# by the conservative domain-style blocker.

additional_domain = 0
combined_recovered = 0

for idx, row in positive_pairs.iterrows():

    # Existing blocker union
    current_union = (
        blocker_results.iloc[idx]["exact_name"]
        or blocker_results.iloc[idx]["exact_address"]
        or blocker_results.iloc[idx]["first_two_tokens"]
        or blocker_results.iloc[idx]["two_prefix"]
    )

    # >=2 useful token overlap
    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        source_name = s2_positive_names[matched_id]
    else:
        source_name = s3_positive_names[matched_id]

    shared_tokens = len(
        useful_token_set(s1_name)
        & useful_token_set(source_name)
    )

    current_union = (
        current_union
        or shared_tokens >= 2
    )

    # Domain-style blocker
    compact_s1 = compact_s1_domain_name(s1_name)
    compact_source = normalize_domain_style(source_name)

    domain_hit = (
        compact_source
        and len(compact_source) >= 5
        and (
            compact_source in compact_s1
            or compact_s1 in compact_source
        )
    )

    if domain_hit and not current_union:
        additional_domain += 1

    if current_union or domain_hit:
        combined_recovered += 1


print(
    "Additional matches from domain blocker:",
    additional_domain
)

print()

print(
    "Combined recovered:",
    combined_recovered,
    "/ 1705"
)

print(
    "Combined recall:",
    f"{combined_recovered / 1705:.2%}"
)

Additional matches from domain blocker: 151

Combined recovered: 1358 / 1705
Combined recall: 79.65%


In [41]:
# Measure candidate volume + recall for the
# conservative domain/compact-name blocker.
#
# We only count candidates; we do NOT store them.

s2_candidates = 0
s3_candidates = 0

s2_hits = 0
s3_hits = 0


# --------------------------------------------------
# S2
# --------------------------------------------------

for chunk in pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        source_compact = normalize_domain_style(
            row["business_name"]
        )

        if len(source_compact) < 5:
            continue

        candidate_s1 = set()

        # Check against the 500 S1 compact names
        for s1_id, s1_name in s1_name_lookup.items():

            s1_compact = compact_s1_domain_name(
                s1_name
            )

            if (
                source_compact in s1_compact
                or s1_compact in source_compact
            ):
                candidate_s1.add(s1_id)

        s2_candidates += len(candidate_s1)

        entity_id = row["entity_id"]

        for s1_id in candidate_s1:

            if (s1_id, entity_id) in true_s2_pairs:
                s2_hits += 1


# --------------------------------------------------
# S3
# --------------------------------------------------

for chunk in pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for _, row in chunk.iterrows():

        source_compact = normalize_domain_style(
            row["business_name"]
        )

        if len(source_compact) < 5:
            continue

        candidate_s1 = set()

        for s1_id, s1_name in s1_name_lookup.items():

            s1_compact = compact_s1_domain_name(
                s1_name
            )

            if (
                source_compact in s1_compact
                or s1_compact in source_compact
            ):
                candidate_s1.add(s1_id)

        s3_candidates += len(candidate_s1)

        entity_id = row["entity_id"]

        for s1_id in candidate_s1:

            if (s1_id, entity_id) in true_s3_pairs:
                s3_hits += 1


# --------------------------------------------------
# Results
# --------------------------------------------------

total_candidates = s2_candidates + s3_candidates
total_hits = s2_hits + s3_hits

print("S2 candidates:", f"{s2_candidates:,}")
print("S3 candidates:", f"{s3_candidates:,}")
print("Total candidates:", f"{total_candidates:,}")

print()

print("S2 true matches recovered:", s2_hits)
print("S3 true matches recovered:", s3_hits)

print(
    "Total true matches recovered:",
    total_hits
)

print(
    "Recall:",
    f"{total_hits / len(positive_pairs):.4%}"
)

print()

print(
    "Average candidates per S1:",
    f"{total_candidates / len(blocking_s1):,.2f}"
)

KeyboardInterrupt: 

In [42]:
print("Runtime is responsive.")
print("blocking_s1:", blocking_s1.shape)
print("positive_pairs:", positive_pairs.shape)

Runtime is responsive.
blocking_s1: (500, 4)
positive_pairs: (1705, 4)


In [43]:
from collections import defaultdict

# --------------------------------------------------
# Build compact S1 names
# --------------------------------------------------

s1_compact_lookup = {}

for s1_id, s1_name in s1_name_lookup.items():
    s1_compact_lookup[s1_id] = compact_s1_domain_name(s1_name)


# --------------------------------------------------
# Build inverted 5-character substring index
#
# Example:
# "diligencetrading"
# produces:
# dilig, ilige, lig en, ...
#
# Each substring points to the S1 entities
# containing it.
# --------------------------------------------------

s1_domain_5gram_index = defaultdict(set)

for s1_id, compact_name in s1_compact_lookup.items():

    if len(compact_name) < 5:
        continue

    grams = {
        compact_name[i:i+5]
        for i in range(len(compact_name) - 4)
    }

    for gram in grams:
        s1_domain_5gram_index[gram].add(s1_id)


print(
    "S1 compact names:",
    len(s1_compact_lookup)
)

print(
    "Unique 5-gram index keys:",
    len(s1_domain_5gram_index)
)

print(
    "Example index entries:"
)

for gram, ids in list(s1_domain_5gram_index.items())[:10]:
    print(
        gram,
        "->",
        len(ids),
        "S1 entities"
    )

S1 compact names: 500
Unique 5-gram index keys: 4559
Example index entries:
icine -> 10 S1 entities
icmed -> 3 S1 entities
cmedi -> 3 S1 entities
medic -> 12 S1 entities
pedia -> 9 S1 entities
nepll -> 2 S1 entities
inepl -> 2 S1 entities
tricm -> 2 S1 entities
epllc -> 3 S1 entities
cinep -> 2 S1 entities


In [44]:
# Fast domain/compact-name blocker
#
# Instead of comparing every source row against all 500 S1s,
# use the 5-character inverted index to retrieve only possible S1s.

s2_candidates = 0
s3_candidates = 0

s2_hits = 0
s3_hits = 0


def get_domain_candidates(source_compact):
    """
    Retrieve possible S1 entities using shared 5-character
    substrings, then perform the exact compact containment check.
    """

    if len(source_compact) < 5:
        return set()

    candidate_s1 = set()

    grams = {
        source_compact[i:i+5]
        for i in range(len(source_compact) - 4)
    }

    for gram in grams:
        ids = s1_domain_5gram_index.get(gram)

        if ids:
            candidate_s1.update(ids)

    # Final exact compact-name condition
    matched_s1 = set()

    for s1_id in candidate_s1:

        s1_compact = s1_compact_lookup[s1_id]

        if (
            source_compact in s1_compact
            or s1_compact in source_compact
        ):
            matched_s1.add(s1_id)

    return matched_s1


# ==================================================
# S2
# ==================================================

print("Scanning S2...")

for chunk in pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for entity_id, business_name in zip(
        chunk["entity_id"],
        chunk["business_name"]
    ):

        source_compact = normalize_domain_style(
            business_name
        )

        if len(source_compact) < 5:
            continue

        matched_s1 = get_domain_candidates(
            source_compact
        )

        s2_candidates += len(matched_s1)

        for s1_id in matched_s1:

            if (s1_id, entity_id) in true_s2_pairs:
                s2_hits += 1


print("S2 done.")


# ==================================================
# S3
# ==================================================

print("Scanning S3...")

for chunk in pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    chunksize=100_000,
    usecols=["entity_id", "business_name"]
):

    for entity_id, business_name in zip(
        chunk["entity_id"],
        chunk["business_name"]
    ):

        source_compact = normalize_domain_style(
            business_name
        )

        if len(source_compact) < 5:
            continue

        matched_s1 = get_domain_candidates(
            source_compact
        )

        s3_candidates += len(matched_s1)

        for s1_id in matched_s1:

            if (s1_id, entity_id) in true_s3_pairs:
                s3_hits += 1


print("S3 done.")


# ==================================================
# Results
# ==================================================

total_candidates = s2_candidates + s3_candidates
total_hits = s2_hits + s3_hits

print()
print("=" * 55)
print("FAST DOMAIN BLOCKER RESULTS")
print("=" * 55)

print("S2 candidates:", f"{s2_candidates:,}")
print("S3 candidates:", f"{s3_candidates:,}")
print("Total candidates:", f"{total_candidates:,}")

print()

print("S2 true matches recovered:", s2_hits)
print("S3 true matches recovered:", s3_hits)

print(
    "Total true matches recovered:",
    total_hits
)

print(
    "Recall:",
    f"{total_hits / len(positive_pairs):.4%}"
)

print(
    "Average candidates per S1:",
    f"{total_candidates / len(blocking_s1):,.2f}"
)

Scanning S2...
S2 done.
Scanning S3...
S3 done.

FAST DOMAIN BLOCKER RESULTS
S2 candidates: 79,964
S3 candidates: 133,514
Total candidates: 213,478

S2 true matches recovered: 494
S3 true matches recovered: 555
Total true matches recovered: 1049
Recall: 61.5249%
Average candidates per S1: 426.96


In [45]:
# Test shared 5-character substring overlap
# on the 1,705 known true matches.

def char_5grams(text):
    text = normalize_domain_style(text)

    if len(text) < 5:
        return set()

    return {
        text[i:i+5]
        for i in range(len(text) - 4)
    }


shared_1 = 0
shared_2 = 0
shared_3 = 0
shared_4 = 0

for _, row in positive_pairs.iterrows():

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    s1_name = s1_name_lookup[s1_id]

    if source == "S2":
        source_name = s2_positive_names[matched_id]
    else:
        source_name = s3_positive_names[matched_id]

    s1_grams = char_5grams(s1_name)
    source_grams = char_5grams(source_name)

    shared = len(s1_grams & source_grams)

    if shared >= 1:
        shared_1 += 1

    if shared >= 2:
        shared_2 += 1

    if shared >= 3:
        shared_3 += 1

    if shared >= 4:
        shared_4 += 1


print("Total true matches:", len(positive_pairs))
print()

print(
    ">= 1 shared 5-gram:",
    shared_1,
    f"({shared_1 / len(positive_pairs):.2%})"
)

print(
    ">= 2 shared 5-grams:",
    shared_2,
    f"({shared_2 / len(positive_pairs):.2%})"
)

print(
    ">= 3 shared 5-grams:",
    shared_3,
    f"({shared_3 / len(positive_pairs):.2%})"
)

print(
    ">= 4 shared 5-grams:",
    shared_4,
    f"({shared_4 / len(positive_pairs):.2%})"
)

Total true matches: 1705

>= 1 shared 5-gram: 1502 (88.09%)
>= 2 shared 5-grams: 1483 (86.98%)
>= 3 shared 5-grams: 1458 (85.51%)
>= 4 shared 5-grams: 1417 (83.11%)


In [46]:
# Estimate candidate volume for >=2 and >=3 shared 5-grams
# using only the first 200,000 rows of S2 and S3.

def get_5gram_candidates(source_name):

    source_compact = normalize_domain_style(source_name)

    if len(source_compact) < 5:
        return set()

    grams = {
        source_compact[i:i+5]
        for i in range(len(source_compact) - 4)
    }

    shared_counts = Counter()

    for gram in grams:

        ids = s1_domain_5gram_index.get(gram)

        if ids:
            for s1_id in ids:
                shared_counts[s1_id] += 1

    candidates_2 = {
        s1_id
        for s1_id, count in shared_counts.items()
        if count >= 2
    }

    candidates_3 = {
        s1_id
        for s1_id, count in shared_counts.items()
        if count >= 3
    }

    return candidates_2, candidates_3


s2_2 = 0
s2_3 = 0

s3_2 = 0
s3_3 = 0


# ------------------------------------------
# S2 sample
# ------------------------------------------

s2_sample = pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    nrows=200_000,
    usecols=["entity_id", "business_name"]
)

for name in s2_sample["business_name"]:

    c2, c3 = get_5gram_candidates(name)

    s2_2 += len(c2)
    s2_3 += len(c3)


# ------------------------------------------
# S3 sample
# ------------------------------------------

s3_sample = pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    nrows=200_000,
    usecols=["entity_id", "business_name"]
)

for name in s3_sample["business_name"]:

    c2, c3 = get_5gram_candidates(name)

    s3_2 += len(c2)
    s3_3 += len(c3)


sample_total = 400_000

candidates_2 = s2_2 + s3_2
candidates_3 = s2_3 + s3_3

print("===== FIRST 200K + 200K SAMPLE =====")

print()
print(">=2 shared 5-grams:")
print("S2:", f"{s2_2:,}")
print("S3:", f"{s3_2:,}")
print("Total:", f"{candidates_2:,}")
print(
    "Avg candidates/source row:",
    f"{candidates_2 / sample_total:.3f}"
)

print()

print(">=3 shared 5-grams:")
print("S2:", f"{s2_3:,}")
print("S3:", f"{s3_3:,}")
print("Total:", f"{candidates_3:,}")
print(
    "Avg candidates/source row:",
    f"{candidates_3 / sample_total:.3f}"
)

print()

print("Estimated full-dataset candidates:")

print(
    ">=2:",
    f"{candidates_2 / sample_total * 10_320_219:,.0f}"
)

print(
    ">=3:",
    f"{candidates_3 / sample_total * 10_320_219:,.0f}"
)

ValueError: not enough values to unpack (expected 2, got 0)

In [47]:
def get_5gram_candidates(source_name):

    source_compact = normalize_domain_style(source_name)

    if len(source_compact) < 5:
        return set(), set()

    grams = {
        source_compact[i:i+5]
        for i in range(len(source_compact) - 4)
    }

    shared_counts = Counter()

    for gram in grams:

        ids = s1_domain_5gram_index.get(gram)

        if ids:
            for s1_id in ids:
                shared_counts[s1_id] += 1

    candidates_2 = {
        s1_id
        for s1_id, count in shared_counts.items()
        if count >= 2
    }

    candidates_3 = {
        s1_id
        for s1_id, count in shared_counts.items()
        if count >= 3
    }

    return candidates_2, candidates_3


print("Function fixed.")

Function fixed.


In [48]:
# Estimate candidate volume for >=2 and >=3 shared 5-grams
# using only 200,000 S2 + 200,000 S3 rows.

s2_2 = 0
s2_3 = 0

s3_2 = 0
s3_3 = 0


# ------------------------------------------
# S2 sample
# ------------------------------------------

s2_sample = pd.read_csv(
    "/train_source2.tsv",
    sep="\t",
    nrows=200_000,
    usecols=["entity_id", "business_name"]
)

for name in s2_sample["business_name"]:

    c2, c3 = get_5gram_candidates(name)

    s2_2 += len(c2)
    s2_3 += len(c3)


# ------------------------------------------
# S3 sample
# ------------------------------------------

s3_sample = pd.read_csv(
    "/train_source3.tsv",
    sep="\t",
    nrows=200_000,
    usecols=["entity_id", "business_name"]
)

for name in s3_sample["business_name"]:

    c2, c3 = get_5gram_candidates(name)

    s3_2 += len(c2)
    s3_3 += len(c3)


sample_total = 400_000

candidates_2 = s2_2 + s3_2
candidates_3 = s2_3 + s3_3

print("===== FIRST 200K S2 + FIRST 200K S3 =====")

print()
print(">=2 shared 5-grams:")
print("S2:", f"{s2_2:,}")
print("S3:", f"{s3_3:,}" if False else f"{s3_2:,}")
print("Total:", f"{candidates_2:,}")
print(
    "Avg candidates/source row:",
    f"{candidates_2 / sample_total:.3f}"
)

print()

print(">=3 shared 5-grams:")
print("S2:", f"{s2_3:,}")
print("S3:", f"{s3_3:,}")
print("Total:", f"{candidates_3:,}")
print(
    "Avg candidates/source row:",
    f"{candidates_3 / sample_total:.3f}"
)

print()

print("Estimated full-dataset candidates:")

print(
    ">=2:",
    f"{candidates_2 / sample_total * 10_320_219:,.0f}"
)

print(
    ">=3:",
    f"{candidates_3 / sample_total * 10_320_219:,.0f}"
)

===== FIRST 200K S2 + FIRST 200K S3 =====

>=2 shared 5-grams:
S2: 742,430
S3: 793,110
Total: 1,535,540
Avg candidates/source row: 3.839

>=3 shared 5-grams:
S2: 515,853
S3: 551,112
Total: 1,066,965
Avg candidates/source row: 2.667

Estimated full-dataset candidates:
>=2: 39,617,773
>=3: 27,528,281


In [ ]:
# ============================================================
# STEP 1: COMBINED CANDIDATE GENERATOR
# Development set = 500 S1 entities
# ============================================================

from collections import defaultdict
import pandas as pd
import re

# ------------------------------------------------------------
# 1. Load S2 and S3 in chunks
# ------------------------------------------------------------

S2_PATH = "/train_source2.tsv"
S3_PATH = "/train_source3.tsv"

# ------------------------------------------------------------
# 2. Prepare S1-side lookup structures
# ------------------------------------------------------------

# Exact normalized name
s1_exact_name = defaultdict(set)

# Exact normalized address
s1_exact_address = defaultdict(set)

# Address number -> S1 IDs
s1_address_num_index = defaultdict(set)

# First 2 useful name tokens
s1_two_token_index = defaultdict(set)

# >=2 useful name tokens
s1_two_useful_token_index = defaultdict(set)

# Compact/domain-style name
s1_compact_name_index = defaultdict(set)


def useful_name_tokens(name):
    name = normalize_name_v2(name)
    return {
        t for t in name.split()
        if len(t) >= 3 and t not in NAME_STOPWORDS
    }


def compact_name_for_blocking(name):
    name = normalize_name_v2(name)
    tokens = [
        t for t in name.split()
        if t not in DOMAIN_STOPWORDS
    ]
    return "".join(tokens)


for _, row in blocking_s1.iterrows():

    s1_id = row["entity_id"]

    name = normalize_name_v2(row["business_name"])
    address = normalize_address(row["business_address"])

    # Exact name
    if name:
        s1_exact_name[name].add(s1_id)

    # Exact address
    if address:
        s1_exact_address[address].add(s1_id)

    # Address numbers
    for num in get_address_numbers(address):
        s1_address_num_index[num].add(s1_id)

    # Useful name tokens
    tokens = sorted(useful_name_tokens(row["business_name"]))

    # First 2 useful tokens
    if len(tokens) >= 2:
        key = tuple(tokens[:2])
        s1_two_token_index[key].add(s1_id)

    # Any 2 useful tokens
    if len(tokens) >= 2:
        for i in range(len(tokens)):
            for j in range(i + 1, len(tokens)):
                s1_two_useful_token_index[(tokens[i], tokens[j])].add(s1_id)

    # Compact/domain-style name
    compact = compact_name_for_blocking(row["business_name"])

    if compact:
        s1_compact_name_index[compact].add(s1_id)


print("S1 indexes ready:")
print("Exact names:", len(s1_exact_name))
print("Exact addresses:", len(s1_exact_address))
print("Address numbers:", len(s1_address_num_index))
print("First-2-token keys:", len(s1_two_token_index))
print("Two-useful-token keys:", len(s1_two_useful_token_index))
print("Compact names:", len(s1_compact_name_index))


# ------------------------------------------------------------
# 3. Candidate generation function
# ------------------------------------------------------------

def generate_s1_candidates(row):

    candidates = set()

    name = normalize_name_v2(row["business_name"])
    address = normalize_address(row["business_address"])

    # --------------------------------------------------------
    # Blocker 1: exact normalized name
    # --------------------------------------------------------
    if name:
        candidates.update(
            s1_exact_name.get(name, set())
        )

    # --------------------------------------------------------
    # Blocker 2: exact normalized address
    # --------------------------------------------------------
    if address:
        candidates.update(
            s1_exact_address.get(address, set())
        )

    # --------------------------------------------------------
    # Blocker 3: address number
    # --------------------------------------------------------
    for num in get_address_numbers(address):
        candidates.update(
            s1_address_num_index.get(num, set())
        )

    # --------------------------------------------------------
    # Blocker 4: first 2 useful name tokens
    # --------------------------------------------------------
    tokens = sorted(useful_name_tokens(row["business_name"]))

    if len(tokens) >= 2:
        candidates.update(
            s1_two_token_index.get(
                tuple(tokens[:2]),
                set()
            )
        )

    # --------------------------------------------------------
    # Blocker 5: any 2 useful name tokens
    # --------------------------------------------------------
    if len(tokens) >= 2:

        for i in range(len(tokens)):
            for j in range(i + 1, len(tokens)):

                candidates.update(
                    s1_two_useful_token_index.get(
                        (tokens[i], tokens[j]),
                        set()
                    )
                )

    # --------------------------------------------------------
    # Blocker 6: compact/domain-style name
    # --------------------------------------------------------
    compact = compact_name_for_blocking(
        row["business_name"]
    )

    if compact:
        candidates.update(
            s1_compact_name_index.get(
                compact,
                set()
            )
        )

    return candidates


# ------------------------------------------------------------
# 4. Generate candidates from S2 + S3
# ------------------------------------------------------------

candidate_pairs = set()

s2_rows = 0
s3_rows = 0

print("\nProcessing S2...")

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    chunksize=100_000,
    dtype=str
):

    s2_rows += len(chunk)

    for _, row in chunk.iterrows():

        matched_s1 = generate_s1_candidates(row)

        for s1_id in matched_s1:
            candidate_pairs.add(
                (
                    s1_id,
                    row["entity_id"],
                    "S2"
                )
            )

print("S2 rows processed:", s2_rows)


print("\nProcessing S3...")

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    chunksize=100_000,
    dtype=str
):

    s3_rows += len(chunk)

    for _, row in chunk.iterrows():

        matched_s1 = generate_s1_candidates(row)

        for s1_id in matched_s1:
            candidate_pairs.add(
                (
                    s1_id,
                    row["entity_id"],
                    "S3"
                )
            )

print("S3 rows processed:", s3_rows)


# ------------------------------------------------------------
# 5. Convert to DataFrame
# ------------------------------------------------------------

candidate_pairs = pd.DataFrame(
    list(candidate_pairs),
    columns=[
        "entity_id_s1",
        "entity_id_matched",
        "source"
    ]
)

print("\n====================================")
print("CANDIDATE GENERATION COMPLETE")
print("====================================")

print("Candidate pairs:", len(candidate_pairs))
print("S2 candidates:",
      (candidate_pairs["source"] == "S2").sum())
print("S3 candidates:",
      (candidate_pairs["source"] == "S3").sum())

print(
    "Average candidates per S1:",
    len(candidate_pairs) / len(blocking_s1)
)


# ------------------------------------------------------------
# 6. Calculate candidate recall
# ------------------------------------------------------------

true_pairs = set(
    zip(
        positive_pairs["entity_id_s1"],
        positive_pairs["entity_id_matched"],
        positive_pairs["source"]
    )
)

generated_pairs = set(
    zip(
        candidate_pairs["entity_id_s1"],
        candidate_pairs["entity_id_matched"],
        candidate_pairs["source"]
    )
)

recovered = len(true_pairs & generated_pairs)
total_true = len(true_pairs)

recall = recovered / total_true if total_true else 0

print("\n====================================")
print("CANDIDATE RECALL")
print("====================================")

print("True pairs:", total_true)
print("Recovered:", recovered)
print("Missed:", total_true - recovered)
print(f"Recall: {recall:.4%}")

S1 indexes ready:
Exact names: 499
Exact addresses: 500
Address numbers: 444
First-2-token keys: 439
Two-useful-token keys: 888
Compact names: 498

Processing S2...
S2 rows processed: 5034616

Processing S3...


In [1]:
import os

print("S2 exists:", os.path.exists("/train_source2.tsv"))
print("S3 exists:", os.path.exists("/train_source3.tsv"))
print("S1 exists:", os.path.exists("/train_source1.tsv"))
print("GT exists:", os.path.exists("/train_ground_truth.tsv"))

print("\nRuntime files:")
print([f for f in os.listdir("/") if f.endswith(".tsv")])

S2 exists: True
S3 exists: True
S1 exists: True
GT exists: True

Runtime files:
['train_ground_truth.tsv', 'train_source3.tsv', 'train_source2.tsv', 'train_source1.tsv']


In [2]:
# ============================================================
# STEP 1: RESTORE 500-S1 DEVELOPMENT DATA + BUILD INDEXES
# ============================================================

import pandas as pd
import re
import unicodedata
from collections import defaultdict

# Small development files saved earlier
DEV_DIR = "/content/drive/MyDrive/Amazon_ML_Challenge/development"

blocking_s1 = pd.read_csv(
    f"{DEV_DIR}/blocking_s1.csv",
    dtype=str
)

dev_ground_truth = pd.read_csv(
    f"{DEV_DIR}/dev_ground_truth.csv",
    dtype=str
)

positive_pairs = pd.read_csv(
    f"{DEV_DIR}/positive_pairs.csv",
    dtype=str
)

print("blocking_s1:", blocking_s1.shape)
print("dev_ground_truth:", dev_ground_truth.shape)
print("positive_pairs:", positive_pairs.shape)


# ------------------------------------------------------------
# Normalization functions
# ------------------------------------------------------------

def normalize_name_v2(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_address(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


NAME_STOPWORDS = {
    "limited","private","llc","inc","ltd","pvt","llp",
    "incorporated","corp","corporation","company","co","plc",
    "group","holdings","partners","associates","international","india"
}

DOMAIN_STOPWORDS = {
    "limited","private","llc","inc","ltd","pvt","llp",
    "incorporated","corp","corporation","company","co","plc"
}


def useful_name_tokens(name):
    name = normalize_name_v2(name)

    return {
        token
        for token in name.split()
        if len(token) >= 3
        and token not in NAME_STOPWORDS
    }


def compact_name_for_blocking(name):
    name = normalize_name_v2(name)

    tokens = [
        token
        for token in name.split()
        if token not in DOMAIN_STOPWORDS
    ]

    return "".join(tokens)


# ------------------------------------------------------------
# Build indexes
# ------------------------------------------------------------

s1_exact_name = defaultdict(set)
s1_exact_address = defaultdict(set)
s1_two_token_index = defaultdict(set)
s1_two_useful_token_index = defaultdict(set)
s1_compact_name_index = defaultdict(set)

for _, row in blocking_s1.iterrows():

    s1_id = row["entity_id"]

    name = normalize_name_v2(row["business_name"])
    address = normalize_address(row["business_address"])

    # Exact name
    if name:
        s1_exact_name[name].add(s1_id)

    # Exact address
    if address:
        s1_exact_address[address].add(s1_id)

    # Useful tokens
    tokens = sorted(
        useful_name_tokens(row["business_name"])
    )

    # First 2 useful tokens
    if len(tokens) >= 2:
        s1_two_token_index[
            tuple(tokens[:2])
        ].add(s1_id)

    # Any 2 useful tokens
    if len(tokens) >= 2:
        for i in range(len(tokens)):
            for j in range(i + 1, len(tokens)):
                s1_two_useful_token_index[
                    (tokens[i], tokens[j])
                ].add(s1_id)

    # Compact/domain-style name
    compact = compact_name_for_blocking(
        row["business_name"]
    )

    if compact:
        s1_compact_name_index[compact].add(s1_id)


print("\nIndexes rebuilt:")
print("Exact names:", len(s1_exact_name))
print("Exact addresses:", len(s1_exact_address))
print("First-2-token keys:", len(s1_two_token_index))
print("Two-useful-token keys:", len(s1_two_useful_token_index))
print("Compact names:", len(s1_compact_name_index))

blocking_s1: (500, 4)
dev_ground_truth: (500, 2)
positive_pairs: (1705, 4)

Indexes rebuilt:
Exact names: 499
Exact addresses: 500
First-2-token keys: 439
Two-useful-token keys: 888
Compact names: 498


In [3]:
# ============================================================
# STEP 2: MEMORY-SAFE CANDIDATE GENERATION
# Writes candidates directly to disk
# ============================================================

import os
import csv
import pandas as pd

S2_PATH = "/train_source2.tsv"
S3_PATH = "/train_source3.tsv"

OUTPUT_CANDIDATES = "/content/dev_candidate_pairs.csv"

# Remove old file if it exists
if os.path.exists(OUTPUT_CANDIDATES):
    os.remove(OUTPUT_CANDIDATES)

# ------------------------------------------------------------
# Candidate lookup function
# ------------------------------------------------------------

def get_candidates_for_record(name, address):

    candidates = set()

    # Normalize
    norm_name = normalize_name_v2(name)
    norm_address = normalize_address(address)

    # --------------------------------------------------------
    # 1. Exact name
    # --------------------------------------------------------
    if norm_name:
        candidates.update(
            s1_exact_name.get(norm_name, set())
        )

    # --------------------------------------------------------
    # 2. Exact address
    # --------------------------------------------------------
    if norm_address:
        candidates.update(
            s1_exact_address.get(norm_address, set())
        )

    # --------------------------------------------------------
    # 3. First 2 useful name tokens
    # --------------------------------------------------------
    tokens = sorted(useful_name_tokens(name))

    if len(tokens) >= 2:

        candidates.update(
            s1_two_token_index.get(
                tuple(tokens[:2]),
                set()
            )
        )

    # --------------------------------------------------------
    # 4. Any 2 useful name tokens
    # --------------------------------------------------------
    if len(tokens) >= 2:

        for i in range(len(tokens)):

            for j in range(i + 1, len(tokens)):

                candidates.update(
                    s1_two_useful_token_index.get(
                        (tokens[i], tokens[j]),
                        set()
                    )
                )

    # --------------------------------------------------------
    # 5. Compact/domain-style name
    # --------------------------------------------------------
    compact = compact_name_for_blocking(name)

    if compact:

        candidates.update(
            s1_compact_name_index.get(
                compact,
                set()
            )
        )

    return candidates


# ------------------------------------------------------------
# Streaming writer
# ------------------------------------------------------------

header_written = False

total_s2_rows = 0
total_s3_rows = 0

total_s2_candidates = 0
total_s3_candidates = 0


def process_source(path, source_name):

    global header_written
    global total_s2_rows, total_s3_rows
    global total_s2_candidates, total_s3_candidates

    rows_buffer = []

    print(f"\nProcessing {source_name}...")

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            path,
            sep="\t",
            chunksize=100_000,
            dtype=str
        ),
        start=1
    ):

        for row in chunk.itertuples(index=False):

            # Access by column position
            entity_id = row[0]
            business_name = row[1]
            business_address = row[2]

            matched_s1 = get_candidates_for_record(
                business_name,
                business_address
            )

            for s1_id in matched_s1:

                rows_buffer.append(
                    (
                        s1_id,
                        entity_id,
                        source_name
                    )
                )

            # Write every 25,000 candidate rows
            if len(rows_buffer) >= 25_000:

                write_header = not header_written

                with open(
                    OUTPUT_CANDIDATES,
                    "a",
                    newline="",
                    encoding="utf-8"
                ) as f:

                    writer = csv.writer(f)

                    if write_header:
                        writer.writerow([
                            "entity_id_s1",
                            "entity_id_matched",
                            "source"
                        ])
                        header_written = True

                    writer.writerows(rows_buffer)

                if source_name == "S2":
                    total_s2_candidates += len(rows_buffer)
                else:
                    total_s3_candidates += len(rows_buffer)

                rows_buffer.clear()

        # Progress
        if source_name == "S2":
            total_s2_rows += len(chunk)
            processed = total_s2_rows
        else:
            total_s3_rows += len(chunk)
            processed = total_s3_rows

        print(
            f"{source_name} chunk {chunk_no}: "
            f"{processed:,} rows processed"
        )

    # Write remaining candidates
    if rows_buffer:

        write_header = not header_written

        with open(
            OUTPUT_CANDIDATES,
            "a",
            newline="",
            encoding="utf-8"
        ) as f:

            writer = csv.writer(f)

            if write_header:
                writer.writerow([
                    "entity_id_s1",
                    "entity_id_matched",
                    "source"
                ])
                header_written = True

            writer.writerows(rows_buffer)

        if source_name == "S2":
            total_s2_candidates += len(rows_buffer)
        else:
            total_s3_candidates += len(rows_buffer)

        rows_buffer.clear()

    print(f"{source_name} complete.")


# ------------------------------------------------------------
# Run S2 and S3
# ------------------------------------------------------------

process_source(
    S2_PATH,
    "S2"
)

process_source(
    S3_PATH,
    "S3"
)


# ------------------------------------------------------------
# Check output
# ------------------------------------------------------------

print("\n========================================")
print("CANDIDATE GENERATION COMPLETE")
print("========================================")

print("Output:", OUTPUT_CANDIDATES)

print(
    "S2 rows processed:",
    f"{total_s2_rows:,}"
)

print(
    "S3 rows processed:",
    f"{total_s3_rows:,}"
)

print(
    "S2 candidate rows written:",
    f"{total_s2_candidates:,}"
)

print(
    "S3 candidate rows written:",
    f"{total_s3_candidates:,}"
)

print(
    "Total candidate rows:",
    f"{total_s2_candidates + total_s3_candidates:,}"
)

print(
    "Output file size:",
    f"{os.path.getsize(OUTPUT_CANDIDATES) / (1024**2):.2f} MB"
)


Processing S2...
S2 chunk 1: 100,000 rows processed
S2 chunk 2: 200,000 rows processed
S2 chunk 3: 300,000 rows processed
S2 chunk 4: 400,000 rows processed
S2 chunk 5: 500,000 rows processed
S2 chunk 6: 600,000 rows processed
S2 chunk 7: 700,000 rows processed
S2 chunk 8: 800,000 rows processed
S2 chunk 9: 900,000 rows processed
S2 chunk 10: 1,000,000 rows processed
S2 chunk 11: 1,100,000 rows processed
S2 chunk 12: 1,200,000 rows processed
S2 chunk 13: 1,300,000 rows processed
S2 chunk 14: 1,400,000 rows processed
S2 chunk 15: 1,500,000 rows processed
S2 chunk 16: 1,600,000 rows processed
S2 chunk 17: 1,700,000 rows processed
S2 chunk 18: 1,800,000 rows processed
S2 chunk 19: 1,900,000 rows processed
S2 chunk 20: 2,000,000 rows processed
S2 chunk 21: 2,100,000 rows processed
S2 chunk 22: 2,200,000 rows processed
S2 chunk 23: 2,300,000 rows processed
S2 chunk 24: 2,400,000 rows processed
S2 chunk 25: 2,500,000 rows processed
S2 chunk 26: 2,600,000 rows processed
S2 chunk 27: 2,700,00

In [4]:
# ============================================================
# STEP 3: MEASURE ACTUAL CANDIDATE RECALL
# ============================================================

candidate_df = pd.read_csv(
    "/content/dev_candidate_pairs.csv",
    dtype=str
)

print("Candidate dataframe:", candidate_df.shape)

# Convert true pairs to tuples
true_pairs = set(
    zip(
        positive_pairs["entity_id_s1"],
        positive_pairs["entity_id_matched"],
        positive_pairs["source"]
    )
)

# Convert generated candidates to tuples
generated_pairs = set(
    zip(
        candidate_df["entity_id_s1"],
        candidate_df["entity_id_matched"],
        candidate_df["source"]
    )
)

# Intersection
recovered_pairs = true_pairs & generated_pairs

recovered = len(recovered_pairs)
total_true = len(true_pairs)
missed = total_true - recovered

recall = recovered / total_true if total_true else 0

print("\n========================================")
print("CANDIDATE RECALL")
print("========================================")

print(f"Total candidate pairs : {len(generated_pairs):,}")
print(f"True pairs             : {total_true:,}")
print(f"Recovered true pairs   : {recovered:,}")
print(f"Missed true pairs      : {missed:,}")
print(f"Candidate recall       : {recall:.4%}")


# ------------------------------------------------------------
# Recall separately for S2 and S3
# ------------------------------------------------------------

true_s2 = {
    pair for pair in true_pairs
    if pair[2] == "S2"
}

true_s3 = {
    pair for pair in true_pairs
    if pair[2] == "S3"
}

recovered_s2 = len(
    true_s2 & generated_pairs
)

recovered_s3 = len(
    true_s3 & generated_pairs
)

print("\nS2 recall:")
print(
    f"{recovered_s2:,} / {len(true_s2):,} "
    f"= {recovered_s2 / len(true_s2):.4%}"
)

print("\nS3 recall:")
print(
    f"{recovered_s3:,} / {len(true_s3):,} "
    f"= {recovered_s3 / len(true_s3):.4%}"
)


# ------------------------------------------------------------
# Show missed true matches
# ------------------------------------------------------------

missed_pairs = true_pairs - generated_pairs

missed_df = positive_pairs[
    positive_pairs.apply(
        lambda r: (
            r["entity_id_s1"],
            r["entity_id_matched"],
            r["source"]
        ) in missed_pairs,
        axis=1
    )
].copy()

print("\nMissed true pairs:", len(missed_df))

print("\nFirst 20 missed pairs:")
display(missed_df.head(20))

Candidate dataframe: (803735, 3)

CANDIDATE RECALL
Total candidate pairs : 803,735
True pairs             : 1,705
Recovered true pairs   : 1,227
Missed true pairs      : 478
Candidate recall       : 71.9648%

S2 recall:
579 / 798 = 72.5564%

S3 recall:
648 / 907 = 71.4443%

Missed true pairs: 478

First 20 missed pairs:


,entity_id_s1,entity_id_matched,source,label
3,S1-614166735,S3-244407822,S3,1
10,S1-210935903,S2-388023430,S2,1
13,S1-210935903,S2-842875529,S2,1
15,S1-983540069,S2-859028958,S2,1
16,S1-983540069,S2-4190133,S2,1
17,S1-983540069,S3-435171628,S3,1
19,S1-983540069,S3-611345729,S3,1
20,S1-983540069,S3-128967763,S3,1
22,S1-513407139,S2-630586931,S2,1
27,S1-145276542,S2-593468467,S2,1


In [5]:
# ============================================================
# STEP 4: RETRIEVE DETAILS OF MISSED TRUE MATCHES
# ============================================================

import pandas as pd
import os

S2_PATH = "/train_source2.tsv"
S3_PATH = "/train_source3.tsv"

# IDs of the 478 true matches missed by our candidate blockers
missed_s2_ids = set(
    missed_df.loc[
        missed_df["source"] == "S2",
        "entity_id_matched"
    ]
)

missed_s3_ids = set(
    missed_df.loc[
        missed_df["source"] == "S3",
        "entity_id_matched"
    ]
)

print("Missed S2 IDs:", len(missed_s2_ids))
print("Missed S3 IDs:", len(missed_s3_ids))


# ------------------------------------------------------------
# Retrieve only the missed S2 records
# ------------------------------------------------------------

missed_s2_records = []

for chunk in pd.read_csv(
    S2_PATH,
    sep="\t",
    chunksize=100_000,
    dtype=str
):

    mask = chunk["entity_id"].isin(missed_s2_ids)

    if mask.any():
        missed_s2_records.append(
            chunk.loc[mask]
        )

missed_s2 = pd.concat(
    missed_s2_records,
    ignore_index=True
)

print("Retrieved S2 missed records:", len(missed_s2))


# ------------------------------------------------------------
# Retrieve only the missed S3 records
# ------------------------------------------------------------

missed_s3_records = []

for chunk in pd.read_csv(
    S3_PATH,
    sep="\t",
    chunksize=100_000,
    dtype=str
):

    mask = chunk["entity_id"].isin(missed_s3_ids)

    if mask.any():
        missed_s3_records.append(
            chunk.loc[mask]
        )

missed_s3 = pd.concat(
    missed_s3_records,
    ignore_index=True
)

print("Retrieved S3 missed records:", len(missed_s3))


# ------------------------------------------------------------
# Add the corresponding S1 information
# ------------------------------------------------------------

s1_lookup = blocking_s1.set_index("entity_id")

missed_s2_analysis = missed_df[
    missed_df["source"] == "S2"
].merge(
    missed_s2,
    left_on="entity_id_matched",
    right_on="entity_id",
    suffixes=("_truth", "_source")
).merge(
    blocking_s1[
        ["entity_id", "business_name", "business_address", "country"]
    ],
    left_on="entity_id_s1",
    right_on="entity_id",
    suffixes=("_match", "_s1")
)

missed_s3_analysis = missed_df[
    missed_df["source"] == "S3"
].merge(
    missed_s3,
    left_on="entity_id_matched",
    right_on="entity_id",
    suffixes=("_truth", "_source")
).merge(
    blocking_s1[
        ["entity_id", "business_name", "business_address", "country"]
    ],
    left_on="entity_id_s1",
    right_on="entity_id",
    suffixes=("_match", "_s1")
)

missed_analysis = pd.concat(
    [
        missed_s2_analysis,
        missed_s3_analysis
    ],
    ignore_index=True
)


print("\n========================================")
print("MISSED MATCH ANALYSIS")
print("========================================")

print("Total missed:", len(missed_analysis))

print("\nColumns:")
print(missed_analysis.columns.tolist())


# ------------------------------------------------------------
# Display useful columns
# ------------------------------------------------------------

display(
    missed_analysis[
        [
            "entity_id_s1",
            "business_name_s1",
            "business_address_s1",
            "country_s1",
            "entity_id_matched",
            "business_name_source",
            "business_address_source",
            "country_source",
            "source"
        ]
    ].head(50)
)

Missed S2 IDs: 219
Missed S3 IDs: 259
Retrieved S2 missed records: 219
Retrieved S3 missed records: 259

MISSED MATCH ANALYSIS
Total missed: 478

Columns:
['entity_id_s1', 'entity_id_matched', 'source', 'label', 'entity_id_match', 'business_name_match', 'business_address_match', 'country_match', 'entity_id_s1', 'business_name_s1', 'business_address_s1', 'country_s1']


KeyError: "['business_name_source', 'business_address_source', 'country_source'] not in index"

In [6]:
# ============================================================
# STEP 4A: CLEAN MISSED-MATCH ANALYSIS
# ============================================================

# Build clean S1 lookup
s1_info = blocking_s1[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].rename(
    columns={
        "entity_id": "s1_id",
        "business_name": "s1_name",
        "business_address": "s1_address",
        "country": "s1_country"
    }
)

# Build clean missed S2/S3 source records
s2_info = missed_s2[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].rename(
    columns={
        "entity_id": "matched_id",
        "business_name": "matched_name",
        "business_address": "matched_address",
        "country": "matched_country"
    }
)

s3_info = missed_s3[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].rename(
    columns={
        "entity_id": "matched_id",
        "business_name": "matched_name",
        "business_address": "matched_address",
        "country": "matched_country"
    }
)

# Combine S2 + S3
source_info = pd.concat(
    [
        s2_info.assign(source="S2"),
        s3_info.assign(source="S3")
    ],
    ignore_index=True
)

# Merge missed truth pairs with source record
missed_clean = missed_df[
    [
        "entity_id_s1",
        "entity_id_matched",
        "source"
    ]
].merge(
    source_info,
    left_on=["entity_id_matched", "source"],
    right_on=["matched_id", "source"],
    how="left"
)

# Merge S1 information
missed_clean = missed_clean.merge(
    s1_info,
    left_on="entity_id_s1",
    right_on="s1_id",
    how="left"
)

# Select clean columns
missed_clean = missed_clean[
    [
        "entity_id_s1",
        "s1_name",
        "s1_address",
        "s1_country",
        "entity_id_matched",
        "matched_name",
        "matched_address",
        "matched_country",
        "source"
    ]
]

print("Total missed:", len(missed_clean))

print("\nFirst 50 missed true matches:")
display(missed_clean.head(50).to_string(index=False))

Total missed: 478

First 50 missed true matches:


"entity_id_s1                                     s1_name                                                                                             s1_address s1_country entity_id_matched                                                 matched_name                                                                                              matched_address matched_country source\nS1-614166735                           American Catalyst                                                                    15001 Howards Mill Road, Mulino, OR         US      S3-244407822                                               American  Cetl                                                                      15001 Howards Mill Road, Mulino, Oregon              US     S3\nS1-210935903        Leann Colon Federal Acquisitions Inc                                                                   9229 106th Way, Fl 1, Scottsdale, AZ         US      S2-388023430                                        leann

In [7]:
# ============================================================
# STEP 5: TEST TARGETED ADDRESS BLOCKER
#
# Rule:
#   same country
#   + shared address number
#   + shared meaningful address word
# ============================================================

import re
from collections import defaultdict

def address_words_v2(text):
    text = normalize_address(text)

    words = re.findall(r"[a-z]+", text)

    # Ignore very short/common words
    stop = {
        "road", "rd", "street", "st", "avenue", "ave",
        "floor", "fl", "no", "number", "near", "the",
        "and", "of", "for", "block", "unit", "india"
    }

    return {
        w for w in words
        if len(w) >= 4 and w not in stop
    }


# ------------------------------------------------------------
# 1. Test recall on the 478 missed true matches
# ------------------------------------------------------------

# S1 lookup
s1_lookup = blocking_s1.set_index("entity_id")

recoverable = []

for _, row in missed_clean.iterrows():

    s1_row = s1_lookup.loc[row["entity_id_s1"]]

    s1_country = str(
        s1_row["country"]
    ).lower().strip()

    s1_address = normalize_address(
        s1_row["business_address"]
    )

    match_country = str(
        row["matched_country"]
    ).lower().strip()

    match_address = normalize_address(
        row["matched_address"]
    )

    # Country must match
    if s1_country != match_country:
        continue

    # Shared numbers
    s1_nums = set(
        re.findall(r"\d+", s1_address)
    )

    match_nums = set(
        re.findall(r"\d+", match_address)
    )

    if not (s1_nums & match_nums):
        continue

    # Shared meaningful address word
    s1_words = address_words_v2(
        s1_address
    )

    match_words = address_words_v2(
        match_address
    )

    if s1_words & match_words:

        recoverable.append(
            (
                row["entity_id_s1"],
                row["entity_id_matched"],
                row["source"]
            )
        )


print("========================================")
print("MISSED-MATCH RECOVERY")
print("========================================")

print(
    "478 missed true pairs:",
    len(missed_clean)
)

print(
    "Recoverable by "
    "country + address number + address word:",
    len(recoverable)
)

print(
    "Recovery:",
    f"{len(recoverable) / len(missed_clean):.2%}"
)

print(
    "Overall candidate recall if added:",
    f"{(1705 - 478 + len(recoverable)) / 1705:.2%}"
)


# ------------------------------------------------------------
# 2. Estimate candidate volume on 200K S2 + 200K S3
# ------------------------------------------------------------

# Build S1 blocker index:
# (country, address_number, address_word) -> S1 IDs

address_block_index = defaultdict(set)

for _, row in blocking_s1.iterrows():

    s1_id = row["entity_id"]

    country = str(
        row["country"]
    ).lower().strip()

    address = normalize_address(
        row["business_address"]
    )

    nums = set(
        re.findall(r"\d+", address)
    )

    words = address_words_v2(address)

    for num in nums:

        for word in words:

            address_block_index[
                (country, num, word)
            ].add(s1_id)


print(
    "\nAddress blocker keys:",
    len(address_block_index)
)


def address_block_candidates(row):

    country = str(
        row["country"]
    ).lower().strip()

    address = normalize_address(
        row["business_address"]
    )

    nums = set(
        re.findall(r"\d+", address)
    )

    words = address_words_v2(address)

    candidates = set()

    for num in nums:

        for word in words:

            candidates.update(
                address_block_index.get(
                    (country, num, word),
                    set()
                )
            )

    return candidates


# Sample volume
sample_s2 = pd.read_csv(
    S2_PATH,
    sep="\t",
    nrows=200_000,
    dtype=str
)

sample_s3 = pd.read_csv(
    S3_PATH,
    sep="\t",
    nrows=200_000,
    dtype=str
)

s2_count = 0
s3_count = 0

for row in sample_s2.itertuples(index=False):

    # columns: entity_id, business_name,
    # business_address, country

    fake_row = {
        "business_address": row[2],
        "country": row[3]
    }

    s2_count += len(
        address_block_candidates(fake_row)
    )


for row in sample_s3.itertuples(index=False):

    fake_row = {
        "business_address": row[2],
        "country": row[3]
    }

    s3_count += len(
        address_block_candidates(fake_row)
    )


print("\n========================================")
print("200K + 200K CANDIDATE VOLUME")
print("========================================")

print("S2:", f"{s2_count:,}")
print("S3:", f"{s3_count:,}")
print("Total:", f"{s2_count + s3_count:,}")

print(
    "Average per source row:",
    f"{(s2_count + s3_count) / 400_000:.2f}"
)

estimated_full = (
    (s2_count + s3_count)
    * (10_320_219 / 400_000)
)

print(
    "Estimated full candidate rows:",
    f"{estimated_full:,.0f}"
)

MISSED-MATCH RECOVERY
478 missed true pairs: 478
Recoverable by country + address number + address word: 377
Recovery: 78.87%
Overall candidate recall if added: 94.08%

Address blocker keys: 3538

200K + 200K CANDIDATE VOLUME
S2: 229,967
S3: 154,992
Total: 384,959
Average per source row: 0.96
Estimated full candidate rows: 9,932,153


In [8]:
# ============================================================
# STEP 5A: OPTIMIZE ADDRESS-WORD RARITY
# ============================================================

from collections import Counter

# ------------------------------------------------------------
# 1. Count how many S1 entities contain each address word
# ------------------------------------------------------------

s1_address_word_freq = Counter()

for _, row in blocking_s1.iterrows():

    words = address_words_v2(
        row["business_address"]
    )

    for word in words:
        s1_address_word_freq[word] += 1


print("Unique address words:", len(s1_address_word_freq))

print("\nMost common address words:")
print(
    s1_address_word_freq.most_common(30)
)


# ------------------------------------------------------------
# 2. Test recall of different rarity thresholds
# ------------------------------------------------------------

thresholds = [1, 2, 3, 5, 10, 15, 20, 30, 50]

print("\n========================================")
print("ADDRESS WORD RARITY TEST")
print("========================================")

for threshold in thresholds:

    recovered = 0

    for _, row in missed_clean.iterrows():

        s1_id = row["entity_id_s1"]

        s1_row = s1_lookup.loc[s1_id]

        s1_country = str(
            s1_row["country"]
        ).lower().strip()

        match_country = str(
            row["matched_country"]
        ).lower().strip()

        if s1_country != match_country:
            continue

        s1_address = normalize_address(
            s1_row["business_address"]
        )

        match_address = normalize_address(
            row["matched_address"]
        )

        # Shared number
        s1_nums = set(
            re.findall(r"\d+", s1_address)
        )

        match_nums = set(
            re.findall(r"\d+", match_address)
        )

        if not (s1_nums & match_nums):
            continue

        # Shared rare address word
        s1_words = address_words_v2(
            s1_address
        )

        match_words = address_words_v2(
            match_address
        )

        shared_words = s1_words & match_words

        if any(
            s1_address_word_freq[w] <= threshold
            for w in shared_words
        ):
            recovered += 1

    overall_recall = (
        1227 + recovered
    ) / 1705

    print(
        f"<= {threshold:2d} occurrences: "
        f"{recovered:3d}/478 missed recovered | "
        f"overall recall: {overall_recall:.2%}"
    )

Unique address words: 1300

Most common address words:
[('maharashtra', 51), ('drive', 42), ('city', 35), ('nagar', 35), ('lane', 31), ('west', 30), ('pradesh', 29), ('delhi', 24), ('plot', 23), ('mumbai', 21), ('park', 18), ('uttar', 18), ('court', 17), ('karnataka', 17), ('north', 16), ('flat', 14), ('bangalore', 14), ('telangana', 14), ('gujarat', 13), ('pune', 13), ('house', 13), ('colony', 13), ('bengal', 13), ('building', 12), ('kerala', 12), ('tamil', 12), ('nadu', 12), ('south', 11), ('kolkata', 11), ('village', 11)]

ADDRESS WORD RARITY TEST
<=  1 occurrences: 311/478 missed recovered | overall recall: 90.21%
<=  2 occurrences: 343/478 missed recovered | overall recall: 92.08%
<=  3 occurrences: 351/478 missed recovered | overall recall: 92.55%
<=  5 occurrences: 356/478 missed recovered | overall recall: 92.84%
<= 10 occurrences: 361/478 missed recovered | overall recall: 93.14%
<= 15 occurrences: 372/478 missed recovered | overall recall: 93.78%
<= 20 occurrences: 373/478 mi

In [9]:
# ============================================================
# STEP 6: ESTIMATE RARE-ADDRESS BLOCKER VOLUME
# Using address-word frequency <= 15
# ============================================================

ADDRESS_WORD_MAX_FREQ = 15

# ------------------------------------------------------------
# Build selective address blocker index
# ------------------------------------------------------------

selective_address_index = defaultdict(set)

for _, row in blocking_s1.iterrows():

    s1_id = row["entity_id"]

    country = str(
        row["country"]
    ).lower().strip()

    address = normalize_address(
        row["business_address"]
    )

    nums = set(
        re.findall(r"\d+", address)
    )

    words = address_words_v2(address)

    # Keep only rare words
    rare_words = {
        w for w in words
        if s1_address_word_freq[w] <= ADDRESS_WORD_MAX_FREQ
    }

    for num in nums:

        for word in rare_words:

            selective_address_index[
                (country, num, word)
            ].add(s1_id)


print(
    "Selective address index keys:",
    len(selective_address_index)
)


def selective_address_candidates(row):

    country = str(
        row["country"]
    ).lower().strip()

    address = normalize_address(
        row["business_address"]
    )

    nums = set(
        re.findall(r"\d+", address)
    )

    words = address_words_v2(address)

    candidates = set()

    rare_words = {
        w for w in words
        if s1_address_word_freq[w] <= ADDRESS_WORD_MAX_FREQ
    }

    for num in nums:

        for word in rare_words:

            candidates.update(
                selective_address_index.get(
                    (country, num, word),
                    set()
                )
            )

    return candidates


# ------------------------------------------------------------
# 200K S2
# ------------------------------------------------------------

s2_count = 0

for row in sample_s2.itertuples(index=False):

    fake_row = {
        "business_address": row[2],
        "country": row[3]
    }

    s2_count += len(
        selective_address_candidates(fake_row)
    )


# ------------------------------------------------------------
# 200K S3
# ------------------------------------------------------------

s3_count = 0

for row in sample_s3.itertuples(index=False):

    fake_row = {
        "business_address": row[2],
        "country": row[3]
    }

    s3_count += len(
        selective_address_candidates(fake_row)
    )


sample_total = s2_count + s3_count

estimated_full = (
    sample_total
    * (10_320_219 / 400_000)
)

print("\n========================================")
print("SELECTIVE ADDRESS BLOCKER VOLUME")
print("========================================")

print("Threshold:", ADDRESS_WORD_MAX_FREQ)

print("S2 sample:", f"{s2_count:,}")
print("S3 sample:", f"{s3_count:,}")
print("Sample total:", f"{sample_total:,}")

print(
    "Average candidates/source row:",
    f"{sample_total / 400_000:.2f}"
)

print(
    "Estimated full candidates:",
    f"{estimated_full:,.0f}"
)

Selective address index keys: 2997

SELECTIVE ADDRESS BLOCKER VOLUME
Threshold: 15
S2 sample: 102,257
S3 sample: 77,838
Sample total: 180,095
Average candidates/source row: 0.45
Estimated full candidates: 4,646,550


In [10]:
# ============================================================
# STEP 7: GENERATE SELECTIVE ADDRESS CANDIDATES
# Country + address number + rare address word <= 15
#
# We already have the other 803,735 candidates.
# ============================================================

import csv
import os

ADDRESS_CANDIDATES = "/content/dev_address_candidates.csv"

if os.path.exists(ADDRESS_CANDIDATES):
    os.remove(ADDRESS_CANDIDATES)

address_rows_written = 0
rows_buffer = []

HEADER = [
    "entity_id_s1",
    "entity_id_matched",
    "source"
]


def flush_address_buffer():

    global rows_buffer
    global address_rows_written

    if not rows_buffer:
        return

    write_header = not os.path.exists(
        ADDRESS_CANDIDATES
    )

    with open(
        ADDRESS_CANDIDATES,
        "a",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.writer(f)

        if write_header:
            writer.writerow(HEADER)

        writer.writerows(rows_buffer)

    address_rows_written += len(rows_buffer)

    rows_buffer.clear()


# ------------------------------------------------------------
# Process S2
# ------------------------------------------------------------

print("Processing S2 address blocker...")

s2_processed = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S2_PATH,
        sep="\t",
        chunksize=100_000,
        dtype=str
    ),
    start=1
):

    for row in chunk.itertuples(index=False):

        entity_id = row[0]
        address = row[2]
        country = row[3]

        fake_row = {
            "business_address": address,
            "country": country
        }

        candidates = selective_address_candidates(
            fake_row
        )

        for s1_id in candidates:

            rows_buffer.append(
                (
                    s1_id,
                    entity_id,
                    "S2"
                )
            )

        if len(rows_buffer) >= 25_000:
            flush_address_buffer()

    s2_processed += len(chunk)

    print(
        f"S2: {s2_processed:,} rows processed"
    )


# ------------------------------------------------------------
# Process S3
# ------------------------------------------------------------

print("\nProcessing S3 address blocker...")

s3_processed = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S3_PATH,
        sep="\t",
        chunksize=100_000,
        dtype=str
    ),
    start=1
):

    for row in chunk.itertuples(index=False):

        entity_id = row[0]
        address = row[2]
        country = row[3]

        fake_row = {
            "business_address": address,
            "country": country
        }

        candidates = selective_address_candidates(
            fake_row
        )

        for s1_id in candidates:

            rows_buffer.append(
                (
                    s1_id,
                    entity_id,
                    "S3"
                )
            )

        if len(rows_buffer) >= 25_000:
            flush_address_buffer()

    s3_processed += len(chunk)

    print(
        f"S3: {s3_processed:,} rows processed"
    )


# Flush remaining rows
flush_address_buffer()


# ------------------------------------------------------------
# Final statistics
# ------------------------------------------------------------

print("\n========================================")
print("ADDRESS CANDIDATE GENERATION COMPLETE")
print("========================================")

print(
    "S2 processed:",
    f"{s2_processed:,}"
)

print(
    "S3 processed:",
    f"{s3_processed:,}"
)

print(
    "Address candidate rows:",
    f"{address_rows_written:,}"
)

print(
    "File size:",
    f"{os.path.getsize(ADDRESS_CANDIDATES)/(1024**2):.2f} MB"
)

print(
    "File:",
    ADDRESS_CANDIDATES
)

Processing S2 address blocker...
S2: 100,000 rows processed
S2: 200,000 rows processed
S2: 300,000 rows processed
S2: 400,000 rows processed
S2: 500,000 rows processed
S2: 600,000 rows processed
S2: 700,000 rows processed
S2: 800,000 rows processed
S2: 900,000 rows processed
S2: 1,000,000 rows processed
S2: 1,100,000 rows processed
S2: 1,200,000 rows processed
S2: 1,300,000 rows processed
S2: 1,400,000 rows processed
S2: 1,500,000 rows processed
S2: 1,600,000 rows processed
S2: 1,700,000 rows processed
S2: 1,800,000 rows processed
S2: 1,900,000 rows processed
S2: 2,000,000 rows processed
S2: 2,100,000 rows processed
S2: 2,200,000 rows processed
S2: 2,300,000 rows processed
S2: 2,400,000 rows processed
S2: 2,500,000 rows processed
S2: 2,600,000 rows processed
S2: 2,700,000 rows processed
S2: 2,800,000 rows processed
S2: 2,900,000 rows processed
S2: 3,000,000 rows processed
S2: 3,100,000 rows processed
S2: 3,200,000 rows processed
S2: 3,300,000 rows processed
S2: 3,400,000 rows processed

In [11]:
# ============================================================
# STEP 8: COMBINE + DEDUPLICATE CANDIDATES
# ============================================================

import pandas as pd
import os

BASE_CANDIDATES = "/content/dev_candidate_pairs.csv"
ADDRESS_CANDIDATES = "/content/dev_address_candidates.csv"
FINAL_CANDIDATES = "/content/dev_final_candidates.csv"

# Remove previous output if any
if os.path.exists(FINAL_CANDIDATES):
    os.remove(FINAL_CANDIDATES)

# ------------------------------------------------------------
# Read base candidates
# ------------------------------------------------------------

print("Reading original candidates...")

base = pd.read_csv(
    BASE_CANDIDATES,
    dtype=str
)

print(
    "Original candidates:",
    f"{len(base):,}"
)

# ------------------------------------------------------------
# Read address candidates in chunks
# ------------------------------------------------------------

print("\nAdding address candidates...")

# First write base candidates
base.to_csv(
    FINAL_CANDIDATES,
    index=False
)

del base

# ------------------------------------------------------------
# Append address candidates
# ------------------------------------------------------------

for chunk_no, chunk in enumerate(
    pd.read_csv(
        ADDRESS_CANDIDATES,
        dtype=str,
        chunksize=250_000
    ),
    start=1
):

    chunk.to_csv(
        FINAL_CANDIDATES,
        mode="a",
        header=False,
        index=False
    )

    print(
        f"Address chunk {chunk_no} appended: "
        f"{len(chunk):,} rows"
    )


# ------------------------------------------------------------
# Deduplicate using pandas chunks
# ------------------------------------------------------------

print("\nNow deduplicating...")

DEDUPED = "/content/dev_final_candidates_dedup.csv"

if os.path.exists(DEDUPED):
    os.remove(DEDUPED)

seen = set()
first_write = True
total_before = 0
total_after = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        FINAL_CANDIDATES,
        dtype=str,
        chunksize=250_000
    ),
    start=1
):

    total_before += len(chunk)

    # Remove duplicates within the chunk
    chunk = chunk.drop_duplicates(
        subset=[
            "entity_id_s1",
            "entity_id_matched",
            "source"
        ]
    )

    # Remove pairs already seen
    keys = list(
        zip(
            chunk["entity_id_s1"],
            chunk["entity_id_matched"],
            chunk["source"]
        )
    )

    keep = []

    for key in keys:

        if key not in seen:
            seen.add(key)
            keep.append(True)
        else:
            keep.append(False)

    chunk = chunk.loc[keep]

    total_after += len(chunk)

    chunk.to_csv(
        DEDUPED,
        mode="a",
        header=first_write,
        index=False
    )

    first_write = False

    print(
        f"Dedup chunk {chunk_no}: "
        f"{total_after:,} unique candidates"
    )


print("\n========================================")
print("COMBINED CANDIDATES READY")
print("========================================")

print(
    "Before deduplication:",
    f"{total_before:,}"
)

print(
    "After deduplication:",
    f"{total_after:,}"
)

print(
    "File size:",
    f"{os.path.getsize(DEDUPED)/(1024**2):.2f} MB"
)

print(
    "Final file:",
    DEDUPED
)

Reading original candidates...
Original candidates: 803,735

Adding address candidates...
Address chunk 1 appended: 250,000 rows
Address chunk 2 appended: 250,000 rows
Address chunk 3 appended: 250,000 rows
Address chunk 4 appended: 250,000 rows
Address chunk 5 appended: 250,000 rows
Address chunk 6 appended: 250,000 rows
Address chunk 7 appended: 250,000 rows
Address chunk 8 appended: 250,000 rows
Address chunk 9 appended: 250,000 rows
Address chunk 10 appended: 250,000 rows
Address chunk 11 appended: 250,000 rows
Address chunk 12 appended: 250,000 rows
Address chunk 13 appended: 250,000 rows
Address chunk 14 appended: 250,000 rows
Address chunk 15 appended: 250,000 rows
Address chunk 16 appended: 250,000 rows
Address chunk 17 appended: 250,000 rows
Address chunk 18 appended: 250,000 rows
Address chunk 19 appended: 142,869 rows

Now deduplicating...
Dedup chunk 1: 250,000 unique candidates
Dedup chunk 2: 500,000 unique candidates
Dedup chunk 3: 750,000 unique candidates
Dedup chunk 4:

In [12]:
# ============================================================
# STEP 9: FINAL CANDIDATE RECALL
# ============================================================

final_candidates = pd.read_csv(
    "/content/dev_final_candidates_dedup.csv",
    dtype=str
)

print(
    "Final candidates:",
    f"{len(final_candidates):,}"
)

# True pairs
true_pairs = set(
    zip(
        positive_pairs["entity_id_s1"],
        positive_pairs["entity_id_matched"],
        positive_pairs["source"]
    )
)

# Generated pairs
generated_pairs = set(
    zip(
        final_candidates["entity_id_s1"],
        final_candidates["entity_id_matched"],
        final_candidates["source"]
    )
)

# Recall
recovered_pairs = true_pairs & generated_pairs

recovered = len(recovered_pairs)
total_true = len(true_pairs)
missed = total_true - recovered

recall = recovered / total_true

print("\n========================================")
print("FINAL CANDIDATE RECALL")
print("========================================")

print(f"True pairs       : {total_true:,}")
print(f"Recovered        : {recovered:,}")
print(f"Missed           : {missed:,}")
print(f"Recall           : {recall:.4%}")


# ------------------------------------------------------------
# S2 / S3 recall
# ------------------------------------------------------------

true_s2 = {
    x for x in true_pairs
    if x[2] == "S2"
}

true_s3 = {
    x for x in true_pairs
    if x[2] == "S3"
}

rec_s2 = len(true_s2 & generated_pairs)
rec_s3 = len(true_s3 & generated_pairs)

print("\nS2 recall:")
print(
    f"{rec_s2:,} / {len(true_s2):,} "
    f"= {rec_s2 / len(true_s2):.4%}"
)

print("\nS3 recall:")
print(
    f"{rec_s3:,} / {len(true_s3):,} "
    f"= {rec_s3 / len(true_s3):.4%}"
)

Final candidates: 5,444,721

FINAL CANDIDATE RECALL
True pairs       : 1,705
Recovered        : 1,599
Missed           : 106
Recall           : 93.7830%

S2 recall:
745 / 798 = 93.3584%

S3 recall:
854 / 907 = 94.1566%


In [13]:
# ============================================================
# STEP 10: BUILD RF TRAINING PAIRS
# ============================================================

import pandas as pd
import numpy as np

CANDIDATE_FILE = "/content/dev_final_candidates_dedup.csv"

# ------------------------------------------------------------
# 1. Load candidate pairs
# ------------------------------------------------------------

print("Loading candidate pairs...")

candidates = pd.read_csv(
    CANDIDATE_FILE,
    dtype=str
)

print(
    "Candidates loaded:",
    f"{len(candidates):,}"
)


# ------------------------------------------------------------
# 2. Create the set of known true matches
# ------------------------------------------------------------

true_pairs_df = positive_pairs[
    [
        "entity_id_s1",
        "entity_id_matched",
        "source"
    ]
].drop_duplicates()

true_pairs = set(
    zip(
        true_pairs_df["entity_id_s1"],
        true_pairs_df["entity_id_matched"],
        true_pairs_df["source"]
    )
)

print(
    "Known true pairs:",
    len(true_pairs)
)


# ------------------------------------------------------------
# 3. Identify positive candidates
# ------------------------------------------------------------

candidate_tuples = list(
    zip(
        candidates["entity_id_s1"],
        candidates["entity_id_matched"],
        candidates["source"]
    )
)

positive_mask = [
    pair in true_pairs
    for pair in candidate_tuples
]

positive_candidates = candidates[
    positive_mask
].copy()

negative_candidates = candidates[
    [not x for x in positive_mask]
].copy()

print("\n========================================")
print("LABEL DISTRIBUTION")
print("========================================")

print(
    "Positive candidates:",
    f"{len(positive_candidates):,}"
)

print(
    "Negative candidates:",
    f"{len(negative_candidates):,}"
)


# ------------------------------------------------------------
# 4. Sample negatives
# ------------------------------------------------------------

# Use approximately 5 negatives per positive.
NEGATIVE_MULTIPLIER = 5

negative_sample_size = min(
    len(negative_candidates),
    len(positive_candidates) * NEGATIVE_MULTIPLIER
)

negative_sample = negative_candidates.sample(
    n=negative_sample_size,
    random_state=42
)

# Labels
positive_candidates["label"] = 1
negative_sample["label"] = 0


# ------------------------------------------------------------
# 5. Combine
# ------------------------------------------------------------

training_pairs = pd.concat(
    [
        positive_candidates,
        negative_sample
    ],
    ignore_index=True
)

# Shuffle
training_pairs = training_pairs.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


print("\n========================================")
print("RF TRAINING DATASET")
print("========================================")

print(
    "Training pairs:",
    f"{len(training_pairs):,}"
)

print(
    "Positive:",
    (training_pairs["label"] == 1).sum()
)

print(
    "Negative:",
    (training_pairs["label"] == 0).sum()
)

print(
    "Positive ratio:",
    f"{training_pairs['label'].mean():.4f}"
)


# ------------------------------------------------------------
# 6. Save training pairs
# ------------------------------------------------------------

TRAINING_PAIRS_FILE = "/content/rf_training_pairs.csv"

training_pairs.to_csv(
    TRAINING_PAIRS_FILE,
    index=False
)

print(
    "\nSaved:",
    TRAINING_PAIRS_FILE
)

Loading candidate pairs...
Candidates loaded: 5,444,721
Known true pairs: 1705

LABEL DISTRIBUTION
Positive candidates: 1,599
Negative candidates: 5,443,122

RF TRAINING DATASET
Training pairs: 9,594
Positive: 1599
Negative: 7995
Positive ratio: 0.1667

Saved: /content/rf_training_pairs.csv


In [14]:
# ============================================================
# STEP 11: RETRIEVE S2/S3 RECORDS FOR RF TRAINING
# ============================================================

import pandas as pd

S2_PATH = "/train_source2.tsv"
S3_PATH = "/train_source3.tsv"

# ------------------------------------------------------------
# IDs required by the training pairs
# ------------------------------------------------------------

s2_needed = set(
    training_pairs.loc[
        training_pairs["source"] == "S2",
        "entity_id_matched"
    ]
)

s3_needed = set(
    training_pairs.loc[
        training_pairs["source"] == "S3",
        "entity_id_matched"
    ]
)

print("S2 IDs needed:", len(s2_needed))
print("S3 IDs needed:", len(s3_needed))


# ------------------------------------------------------------
# Retrieve S2 records
# ------------------------------------------------------------

s2_records = []

print("\nSearching S2...")

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S2_PATH,
        sep="\t",
        chunksize=100_000,
        dtype=str
    ),
    start=1
):

    matched = chunk[
        chunk["entity_id"].isin(s2_needed)
    ]

    if len(matched) > 0:
        s2_records.append(matched)

    if chunk_no % 10 == 0:
        print(
            f"S2 scanned: {chunk_no * 100_000:,} rows"
        )


s2_train = pd.concat(
    s2_records,
    ignore_index=True
)

print(
    "S2 training records found:",
    len(s2_train)
)


# ------------------------------------------------------------
# Retrieve S3 records
# ------------------------------------------------------------

s3_records = []

print("\nSearching S3...")

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S3_PATH,
        sep="\t",
        chunksize=100_000,
        dtype=str
    ),
    start=1
):

    matched = chunk[
        chunk["entity_id"].isin(s3_needed)
    ]

    if len(matched) > 0:
        s3_records.append(matched)

    if chunk_no % 10 == 0:
        print(
            f"S3 scanned: {chunk_no * 100_000:,} rows"
        )


s3_train = pd.concat(
    s3_records,
    ignore_index=True
)

print(
    "S3 training records found:",
    len(s3_train)
)


# ------------------------------------------------------------
# Build lookup dictionaries
# ------------------------------------------------------------

s2_lookup = {
    row["entity_id"]: row
    for _, row in s2_train.iterrows()
}

s3_lookup = {
    row["entity_id"]: row
    for _, row in s3_train.iterrows()
}


print("\n========================================")
print("TRAINING LOOKUPS READY")
print("========================================")

print("S2 lookup:", len(s2_lookup))
print("S3 lookup:", len(s3_lookup))

S2 IDs needed: 5035
S3 IDs needed: 4528

Searching S2...
S2 scanned: 1,000,000 rows
S2 scanned: 2,000,000 rows
S2 scanned: 3,000,000 rows
S2 scanned: 4,000,000 rows
S2 scanned: 5,000,000 rows
S2 training records found: 5035

Searching S3...
S3 scanned: 1,000,000 rows
S3 scanned: 2,000,000 rows
S3 scanned: 3,000,000 rows
S3 scanned: 4,000,000 rows
S3 scanned: 5,000,000 rows
S3 training records found: 4528

TRAINING LOOKUPS READY
S2 lookup: 5035
S3 lookup: 4528


In [16]:
import re

def get_address_numbers(text):
    if not text:
        return set()

    return set(re.findall(r"\d+", str(text)))

In [17]:
# ============================================
# STEP: BUILD FEATURES FOR RANDOM FOREST
# ============================================

from difflib import SequenceMatcher
import numpy as np
import pandas as pd

# -----------------------------
# Helper functions
# -----------------------------

def similarity(a, b):
    return SequenceMatcher(None, str(a), str(b)).ratio()


def sorted_token_string(text):
    if not text:
        return ""
    return " ".join(sorted(str(text).split()))


def numeric_overlap_ratio(a, b):
    nums_a = get_address_numbers(a)
    nums_b = get_address_numbers(b)

    if not nums_a or not nums_b:
        return 0.0

    intersection = len(nums_a & nums_b)
    union = len(nums_a | nums_b)

    return intersection / union if union else 0.0


# -----------------------------
# S1 lookup
# -----------------------------

s1_lookup = blocking_s1.set_index("entity_id").to_dict("index")


# -----------------------------
# Feature extraction
# -----------------------------

feature_rows = []

for idx, row in training_pairs.iterrows():

    s1_id = row["entity_id_s1"]
    matched_id = row["entity_id_matched"]
    source = row["source"]

    # S1 record
    s1 = s1_lookup.get(s1_id)

    if s1 is None:
        continue

    # S2 or S3 record
    if source == "S2":
        matched = s2_lookup.get(matched_id)
    else:
        matched = s3_lookup.get(matched_id)

    if matched is None:
        continue

    # -------------------------
    # Normalize fields
    # -------------------------

    s1_name = normalize_name_v2(s1["business_name"])
    s2_name = normalize_name_v2(matched["business_name"])

    s1_address = normalize_address(s1["business_address"])
    s2_address = normalize_address(matched["business_address"])

    s1_name_tokens = sorted_token_string(
        useful_name_tokens(s1["business_name"])
    )

    s2_name_tokens = sorted_token_string(
        useful_name_tokens(matched["business_name"])
    )

    s1_address_tokens = sorted_token_string(
        address_words_v2(s1["business_address"])
    )

    s2_address_tokens = sorted_token_string(
        address_words_v2(matched["business_address"])
    )

    # -------------------------
    # 10 features
    # -------------------------

    features = {
        # 1. Character-level name similarity
        "name_ratio":
            similarity(s1_name, s2_name),

        # 2. Token-level name similarity
        "name_token_ratio":
            similarity(s1_name_tokens, s2_name_tokens),

        # 3. Character-level address similarity
        "address_ratio":
            similarity(s1_address, s2_address),

        # 4. Token-level address similarity
        "address_token_ratio":
            similarity(s1_address_tokens, s2_address_tokens),

        # 5. Exact normalized name
        "name_exact":
            int(s1_name != "" and s1_name == s2_name),

        # 6. Exact normalized address
        "address_exact":
            int(s1_address != "" and s1_address == s2_address),

        # 7. Country match
        "country_match":
            int(
                str(s1["country"]).strip().lower()
                == str(matched["country"]).strip().lower()
            ),

        # 8. Name length difference
        "name_length_diff":
            abs(len(s1_name) - len(s2_name)),

        # 9. Address length difference
        "address_length_diff":
            abs(len(s1_address) - len(s2_address)),

        # 10. Numeric overlap in address
        "numeric_overlap":
            numeric_overlap_ratio(
                s1["business_address"],
                matched["business_address"]
            ),

        # Label
        "label":
            int(row["label"])
    }

    feature_rows.append(features)


# -----------------------------
# Create feature DataFrame
# -----------------------------

feature_data = pd.DataFrame(feature_rows)

print("=" * 50)
print("FEATURE DATA CREATED")
print("=" * 50)

print("Shape:", feature_data.shape)

print("\nColumns:")
print(feature_data.columns.tolist())

print("\nLabel distribution:")
print(feature_data["label"].value_counts())

print("\nFirst 5 rows:")
display(feature_data.head())

FEATURE DATA CREATED
Shape: (9594, 11)

Columns:
['name_ratio', 'name_token_ratio', 'address_ratio', 'address_token_ratio', 'name_exact', 'address_exact', 'country_match', 'name_length_diff', 'address_length_diff', 'numeric_overlap', 'label']

Label distribution:
label
0    7995
1    1599
Name: count, dtype: int64

First 5 rows:


,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,country_match,name_length_diff,address_length_diff,numeric_overlap,label
0,0.156250,0.271186,0.402439,0.458599,0,0,1,0,24,0.200000,0
1,0.181818,0.476190,0.365854,0.467153,0,0,1,9,20,0.250000,0
2,0.423077,0.333333,0.350000,0.606452,0,0,1,0,6,0.200000,0
3,0.250000,0.448980,0.476923,0.564516,0,0,1,2,20,0.250000,0
4,0.310345,0.388060,0.229508,0.490566,0,0,1,8,19,0.142857,0


In [18]:
# ============================================
# STEP: ENTITY-LEVEL TRAIN / VALIDATION SPLIT
# ============================================

from sklearn.model_selection import GroupShuffleSplit

FEATURE_COLUMNS = [
    "name_ratio",
    "name_token_ratio",
    "address_ratio",
    "address_token_ratio",
    "name_exact",
    "address_exact",
    "country_match",
    "name_length_diff",
    "address_length_diff",
    "numeric_overlap"
]

X = feature_data[FEATURE_COLUMNS]
y = feature_data["label"]

# training_pairs contains the S1 entity for every feature row
groups = training_pairs.loc[
    feature_data.index, "entity_id_s1"
]

# 80% entities -> training
# 20% entities -> validation
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_val = X.iloc[val_idx].copy()

y_train = y.iloc[train_idx].copy()
y_val = y.iloc[val_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_val = groups.iloc[val_idx].copy()

print("=" * 50)
print("ENTITY-LEVEL SPLIT COMPLETE")
print("=" * 50)

print("Training pairs:", len(X_train))
print("Validation pairs:", len(X_val))

print("\nTraining entities:", groups_train.nunique())
print("Validation entities:", groups_val.nunique())

print("\nTraining labels:")
print(y_train.value_counts())

print("\nValidation labels:")
print(y_val.value_counts())

# Verify there is NO entity overlap
overlap = set(groups_train) & set(groups_val)

print("\nEntity overlap:", len(overlap))

ENTITY-LEVEL SPLIT COMPLETE
Training pairs: 6981
Validation pairs: 2613

Training entities: 382
Validation entities: 96

Training labels:
label
0    5704
1    1277
Name: count, dtype: int64

Validation labels:
label
0    2291
1     322
Name: count, dtype: int64

Entity overlap: 0


In [19]:
# ============================================
# STEP: TRAIN RANDOM FOREST
# ============================================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")

rf_model.fit(X_train, y_train)

print("\n" + "=" * 50)
print("RANDOM FOREST TRAINING COMPLETE")
print("=" * 50)

print("Number of trees:", rf_model.n_estimators)
print("Number of features:", X_train.shape[1])

Training Random Forest...

RANDOM FOREST TRAINING COMPLETE
Number of trees: 300
Number of features: 10


In [20]:
# ============================================
# STEP: RANDOM FOREST VALIDATION PROBABILITIES
# ============================================

# Probability that each candidate pair is a true match
val_proba = rf_model.predict_proba(X_val)[:, 1]

print("=" * 50)
print("VALIDATION PROBABILITIES GENERATED")
print("=" * 50)

print("Number of validation pairs:", len(val_proba))

print("\nProbability statistics:")
print("Minimum:", val_proba.min())
print("Maximum:", val_proba.max())
print("Mean:", val_proba.mean())

print("\nFirst 10 probabilities:")
print(val_proba[:10])

VALIDATION PROBABILITIES GENERATED
Number of validation pairs: 2613

Probability statistics:
Minimum: 0.0
Maximum: 1.0
Mean: 0.1239010077816048

First 10 probabilities:
[0.         0.         0.         0.25333333 0.98333333 0.
 0.89666667 0.         0.15       0.        ]


In [21]:
# ============================================
# STEP: ENTITY-LEVEL F0.5 THRESHOLD TUNING
# ============================================

from collections import defaultdict

BETA = 0.5
BETA2 = BETA ** 2

# --------------------------------------------
# Build validation dataframe
# --------------------------------------------

val_eval = training_pairs.iloc[val_idx].copy()

val_eval["probability"] = val_proba

# --------------------------------------------
# True matches for each validation S1 entity
# --------------------------------------------

true_matches = defaultdict(set)

for _, row in val_eval.iterrows():
    if row["label"] == 1:
        true_matches[row["entity_id_s1"]].add(
            row["entity_id_matched"]
        )

# All validation entities, including zero-match entities
validation_entities = set(
    groups_val.unique()
)


# --------------------------------------------
# Function for entity-level F0.5
# --------------------------------------------

def entity_f05(true_set, predicted_set):

    tp = len(true_set & predicted_set)
    fp = len(predicted_set - true_set)
    fn = len(true_set - predicted_set)

    # Both empty = correct "no match" prediction
    if len(true_set) == 0 and len(predicted_set) == 0:
        return 1.0

    # False prediction for an entity with no matches
    if len(true_set) == 0 and len(predicted_set) > 0:
        return 0.0

    # No prediction when matches exist
    if len(predicted_set) == 0:
        return 0.0

    precision = tp / (tp + fp)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    denominator = (BETA2 * precision) + recall

    if denominator == 0:
        return 0.0

    return ((1 + BETA2) * precision * recall) / denominator


# --------------------------------------------
# Evaluate a threshold
# --------------------------------------------

def evaluate_threshold(threshold):

    predicted_matches = defaultdict(set)

    selected = val_eval[
        val_eval["probability"] >= threshold
    ]

    for _, row in selected.iterrows():
        predicted_matches[row["entity_id_s1"]].add(
            row["entity_id_matched"]
        )

    scores = []

    for entity_id in validation_entities:

        true_set = true_matches.get(
            entity_id,
            set()
        )

        predicted_set = predicted_matches.get(
            entity_id,
            set()
        )

        score = entity_f05(
            true_set,
            predicted_set
        )

        scores.append(score)

    return np.mean(scores)


# --------------------------------------------
# Test thresholds
# --------------------------------------------

thresholds = np.arange(
    0.10,
    1.00,
    0.05
)

results = []

for threshold in thresholds:

    score = evaluate_threshold(threshold)

    results.append({
        "threshold": round(threshold, 2),
        "entity_f05": score
    })


threshold_results = pd.DataFrame(results)

print("=" * 50)
print("ENTITY-LEVEL F0.5 RESULTS")
print("=" * 50)

display(threshold_results)

best_row = threshold_results.loc[
    threshold_results["entity_f05"].idxmax()
]

print("\nBest threshold:",
      best_row["threshold"])

print("Best entity-level F0.5:",
      best_row["entity_f05"])

ENTITY-LEVEL F0.5 RESULTS


,threshold,entity_f05
0,0.10,0.887808
1,0.15,0.923672
2,0.20,0.933666
3,0.25,0.951822
4,0.30,0.969062
5,0.35,0.968350
6,0.40,0.974288
7,0.45,0.977149
8,0.50,0.983808
9,0.55,0.982072



Best threshold: 0.6
Best entity-level F0.5: 0.985481886078103


In [22]:
# ============================================
# STEP: RF METRICS AT THRESHOLD 0.60
# ============================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

RF_THRESHOLD = 0.60

val_pred = (val_proba >= RF_THRESHOLD).astype(int)

print("=" * 50)
print("RANDOM FOREST @ THRESHOLD 0.60")
print("=" * 50)

print("\nPair-level metrics:")
print("Precision:", precision_score(y_val, val_pred))
print("Recall:   ", recall_score(y_val, val_pred))
print("F1:       ", f1_score(y_val, val_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_pred,
        target_names=["Non-match", "Match"],
        digits=4
    )
)

print("\nEntity-level F0.5:")
print(evaluate_threshold(RF_THRESHOLD))

RANDOM FOREST @ THRESHOLD 0.60

Pair-level metrics:
Precision: 0.9935275080906149
Recall:    0.953416149068323
F1:        0.9730586370839936

Confusion Matrix:
[[2289    2]
 [  15  307]]

Classification Report:
              precision    recall  f1-score   support

   Non-match     0.9935    0.9991    0.9963      2291
       Match     0.9935    0.9534    0.9731       322

    accuracy                         0.9935      2613
   macro avg     0.9935    0.9763    0.9847      2613
weighted avg     0.9935    0.9935    0.9934      2613


Entity-level F0.5:
0.985481886078103


In [ ]:
from google.colab import files

uploaded = files.upload()

In [1]:
import os

for f in [
    "/content/test_source1.tsv",
    "/content/test_source2.tsv",
    "/content/test_source3.tsv"
]:
    print(f, "EXISTS" if os.path.exists(f) else "MISSING")

/content/test_source1.tsv MISSING
/content/test_source2.tsv MISSING
/content/test_source3.tsv MISSING


In [2]:
import os

print(os.path.exists("/content/test_source1.tsv"))

False


In [4]:
from google.colab import files

uploaded = files.upload()

Saving test_source1.tsv to test_source1.tsv


In [5]:
from google.colab import files

uploaded = files.upload()

Saving test_source2.tsv to test_source2.tsv


In [6]:
from google.colab import files

uploaded = files.upload()

Saving test_source3.tsv to test_source3.tsv


In [7]:
import os

test_files = [
    "/content/test_source1.tsv",
    "/content/test_source2.tsv",
    "/content/test_source3.tsv"
]

print("=" * 60)
print("TEST FILE CHECK")
print("=" * 60)

for path in test_files:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"FOUND  : {os.path.basename(path)}")
        print(f"SIZE   : {size_mb:.2f} MB")
        print()
    else:
        print(f"MISSING: {os.path.basename(path)}")

TEST FILE CHECK
FOUND  : test_source1.tsv
SIZE   : 166.91 MB

FOUND  : test_source2.tsv
SIZE   : 485.86 MB

FOUND  : test_source3.tsv
SIZE   : 482.56 MB



In [8]:
import pandas as pd

TEST_FILES = {
    "S1": "/content/test_source1.tsv",
    "S2": "/content/test_source2.tsv",
    "S3": "/content/test_source3.tsv"
}

for source, path in TEST_FILES.items():

    print("\n" + "=" * 60)
    print(f"{source} TEST FILE")
    print("=" * 60)

    # Read only first 5 rows
    sample = pd.read_csv(
        path,
        sep="\t",
        nrows=5
    )

    print("Columns:")
    print(sample.columns.tolist())

    print("\nFirst 2 rows:")
    display(sample.head(2))

    # Count rows without loading entire file
    row_count = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        usecols=["entity_id"],
        chunksize=200_000
    ):
        row_count += len(chunk)

    print("\nTotal rows:", row_count)


S1 TEST FILE
Columns:
['entity_id', 'business_name', 'business_address', 'country']

First 2 rows:


,entity_id,business_name,business_address,country
0,S1-714132312,Zephay Labs Inc,"2621 Cotten Road, Tyler, TX",US
1,S1-106407869,Vision Partners Corp,"IA, Iowa City, 1064 Newton Rd, Unit 11",US



Total rows: 1732544

S2 TEST FILE
Columns:
['entity_id', 'business_name', 'business_address', 'country']

First 2 rows:


,entity_id,business_name,business_address,country
0,S2-192345572,Brahma Infosoft,"COIMATORE COLONY, HUNSUR TQMYSORE DIST., Karna...",India
1,S2-566025912,Marina Ecole France Sarl,"63 R. DE DIEPPE, LILLE, Hauts-de-France",France



Total rows: 4887273

S3 TEST FILE
Columns:
['entity_id', 'business_name', 'business_address', 'country']

First 2 rows:


,entity_id,business_name,business_address,country
0,S3-462677478,मॉडर्न फाइनेंस,"No 10 Enkay Square, 448A, Udyog Vihar Phase V,...",India
1,S3-374810425,Shri Sai Infratech Co,"3/115, East Delhi, DL",India



Total rows: 5082316


In [9]:
# ============================================
# STEP: SAVE RANDOM FOREST MODEL
# ============================================

import joblib
import os

MODEL_PATH = "/content/rf_model.pkl"

joblib.dump(rf_model, MODEL_PATH)

print("=" * 50)
print("RF MODEL SAVED")
print("=" * 50)

print("Path:", MODEL_PATH)
print("Size:", os.path.getsize(MODEL_PATH) / 1024, "KB")
print("Threshold:", RF_THRESHOLD)

NameError: name 'rf_model' is not defined

In [10]:
import os

files_to_check = [
    "/content/rf_training_pairs.csv",
    "/content/dev_final_candidates_dedup.csv",
    "/content/rf_model.pkl"
]

for path in files_to_check:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"FOUND   {path}  ({size_mb:.2f} MB)")
    else:
        print(f"MISSING {path}")

MISSING /content/rf_training_pairs.csv
MISSING /content/dev_final_candidates_dedup.csv
MISSING /content/rf_model.pkl


In [11]:
import os

print("CONTENT FILES:")
for name in os.listdir("/content"):
    print(name)

CONTENT FILES:
.config
test_source3.tsv
test_source1.tsv
test_source2.tsv
sample_data


In [12]:
# ============================================
# STEP 1: LOAD TEST S1
# ============================================

import pandas as pd
import gc

S1_TEST_PATH = "/content/test_source1.tsv"

test_s1 = pd.read_csv(
    S1_TEST_PATH,
    sep="\t",
    dtype={
        "entity_id": "string",
        "business_name": "string",
        "business_address": "string",
        "country": "string"
    }
)

print("=" * 50)
print("TEST S1 LOADED")
print("=" * 50)

print("Shape:", test_s1.shape)
print("Memory:", round(test_s1.memory_usage(deep=True).sum() / 1024**2, 2), "MB")

print("\nCountries:")
print(test_s1["country"].value_counts().head(20))

display(test_s1.head())

TEST S1 LOADED
Shape: (1732544, 4)
Memory: 485.89 MB

Countries:
country
India     809986
US        663106
France    259452
Name: count, dtype: Int64


,entity_id,business_name,business_address,country
0,S1-714132312,Zephay Labs Inc,"2621 Cotten Road, Tyler, TX",US
1,S1-106407869,Vision Partners Corp,"IA, Iowa City, 1064 Newton Rd, Unit 11",US
2,S1-156285671,<< Team Ecole,"175 Boulevard du Président Franklin Roosevelt,...",France
3,S1-689823050,Red Perfect Trading,"Mirzapur, Ews 12, Uttar Pradesh, Mirzapursadar...",India
4,S1-921369899,ZNB Club SARL,"Nouvelle-Aquitaine, La Teste-de-Buch, 5 bis Ru...",France


In [13]:
# ============================================
# STEP 2: BUILD TEST S1 BLOCKING INDEXES
# ============================================

import re
import unicodedata
from collections import defaultdict
import gc

# --------------------------------------------
# Normalization
# --------------------------------------------

def normalize_name_v2(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


DOMAIN_STOPWORDS = {
    "limited", "private", "llc", "inc", "ltd", "pvt", "llp",
    "incorporated", "corp", "corporation", "company", "co", "plc"
}

NAME_STOPWORDS = {
    "limited", "private", "llc", "inc", "ltd", "pvt", "llp",
    "incorporated", "corp", "corporation", "company", "co", "plc",
    "group", "holdings", "partners", "associates",
    "international", "india"
}


def useful_name_tokens(name):
    name = normalize_name_v2(name)

    return {
        token
        for token in name.split()
        if len(token) >= 3
        and token not in NAME_STOPWORDS
    }


def compact_name_for_blocking(name):
    name = normalize_name_v2(name)

    tokens = [
        token
        for token in name.split()
        if token not in DOMAIN_STOPWORDS
    ]

    return "".join(tokens)


def normalize_address(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


# --------------------------------------------
# Indexes
# --------------------------------------------

s1_exact_name = defaultdict(list)
s1_exact_address = defaultdict(list)
s1_two_useful_token_index = defaultdict(list)
s1_compact_name_index = defaultdict(list)


# --------------------------------------------
# Build indexes
# --------------------------------------------

for idx, row in test_s1.iterrows():

    entity_id = row["entity_id"]

    # Normalized name
    name = normalize_name_v2(row["business_name"])

    if name:
        s1_exact_name[name].append(entity_id)

    # Normalized address
    address = normalize_address(row["business_address"])

    if address:
        s1_exact_address[address].append(entity_id)

    # Useful name tokens
    tokens = sorted(
        useful_name_tokens(row["business_name"])
    )

    if len(tokens) >= 2:

        # First two alphabetically sorted useful tokens
        key = (
            tokens[0],
            tokens[1]
        )

        s1_two_useful_token_index[key].append(entity_id)

    # Compact name
    compact = compact_name_for_blocking(
        row["business_name"]
    )

    if compact:
        s1_compact_name_index[compact].append(entity_id)


gc.collect()


# --------------------------------------------
# Statistics
# --------------------------------------------

print("=" * 60)
print("TEST S1 BLOCKING INDEXES READY")
print("=" * 60)

print("Exact name keys:",
      len(s1_exact_name))

print("Exact address keys:",
      len(s1_exact_address))

print("Two-token keys:",
      len(s1_two_useful_token_index))

print("Compact name keys:",
      len(s1_compact_name_index))

print("\nS1 memory:",
      round(
          test_s1.memory_usage(deep=True).sum() / 1024**2,
          2
      ),
      "MB")

TEST S1 BLOCKING INDEXES READY
Exact name keys: 1227453
Exact address keys: 1673358
Two-token keys: 729129
Compact name keys: 1117119

S1 memory: 485.89 MB


In [14]:
# ============================================
# STEP 3: STREAM TEST S2 AND GENERATE CANDIDATES
# ============================================

import pandas as pd
import gc
import os

S2_TEST_PATH = "/content/test_source2.tsv"
S2_CANDIDATE_PATH = "/content/test_s2_candidates.tsv"

CHUNK_SIZE = 100_000

total_rows = 0
total_candidates = 0

# Remove old file if it exists
if os.path.exists(S2_CANDIDATE_PATH):
    os.remove(S2_CANDIDATE_PATH)

first_write = True

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S2_TEST_PATH,
        sep="\t",
        dtype={
            "entity_id": "string",
            "business_name": "string",
            "business_address": "string",
            "country": "string"
        },
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    candidate_rows = []

    for _, row in chunk.iterrows():

        s2_id = row["entity_id"]
        s2_name = normalize_name_v2(row["business_name"])
        s2_address = normalize_address(row["business_address"])

        # ------------------------------------
        # Block 1: exact normalized name
        # ------------------------------------

        if s2_name:
            for s1_id in s1_exact_name.get(s2_name, []):
                candidate_rows.append(
                    (s1_id, s2_id, "S2")
                )

        # ------------------------------------
        # Block 2: exact normalized address
        # ------------------------------------

        if s2_address:
            for s1_id in s1_exact_address.get(s2_address, []):
                candidate_rows.append(
                    (s1_id, s2_id, "S2")
                )

        # ------------------------------------
        # Block 3: two useful name tokens
        # ------------------------------------

        tokens = sorted(
            useful_name_tokens(row["business_name"])
        )

        if len(tokens) >= 2:

            key = (
                tokens[0],
                tokens[1]
            )

            for s1_id in s1_two_useful_token_index.get(key, []):
                candidate_rows.append(
                    (s1_id, s2_id, "S2")
                )

        # ------------------------------------
        # Block 4: compact name
        # ------------------------------------

        compact = compact_name_for_blocking(
            row["business_name"]
        )

        if compact:

            for s1_id in s1_compact_name_index.get(
                compact, []
            ):
                candidate_rows.append(
                    (s1_id, s2_id, "S2")
                )

    # ----------------------------------------
    # Save this chunk's candidates
    # ----------------------------------------

    if candidate_rows:

        candidate_df = pd.DataFrame(
            candidate_rows,
            columns=[
                "entity_id_s1",
                "entity_id_matched",
                "source"
            ]
        )

        # Remove duplicates within chunk
        candidate_df.drop_duplicates(
            inplace=True
        )

        candidate_df.to_csv(
            S2_CANDIDATE_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False

        total_candidates += len(candidate_df)

    total_rows += len(chunk)

    print(
        f"Chunk {chunk_no}: "
        f"S2 rows processed = {total_rows:,} | "
        f"candidates = {total_candidates:,}"
    )

    del chunk
    gc.collect()


print("\n" + "=" * 60)
print("S2 BLOCKING COMPLETE")
print("=" * 60)

print("S2 rows processed:", f"{total_rows:,}")
print("S2 candidates:", f"{total_candidates:,}")
print("Output:", S2_CANDIDATE_PATH)

Chunk 1: S2 rows processed = 100,000 | candidates = 3,140,317
Chunk 2: S2 rows processed = 200,000 | candidates = 6,327,400
Chunk 3: S2 rows processed = 300,000 | candidates = 9,480,277
Chunk 4: S2 rows processed = 400,000 | candidates = 12,611,908
Chunk 5: S2 rows processed = 500,000 | candidates = 15,836,874
Chunk 6: S2 rows processed = 600,000 | candidates = 18,976,175
Chunk 7: S2 rows processed = 700,000 | candidates = 22,252,514
Chunk 8: S2 rows processed = 800,000 | candidates = 25,410,089
Chunk 9: S2 rows processed = 900,000 | candidates = 28,543,908
Chunk 10: S2 rows processed = 1,000,000 | candidates = 31,716,175
Chunk 11: S2 rows processed = 1,100,000 | candidates = 34,976,345
Chunk 12: S2 rows processed = 1,200,000 | candidates = 38,173,067
Chunk 13: S2 rows processed = 1,300,000 | candidates = 41,396,006
Chunk 14: S2 rows processed = 1,400,000 | candidates = 44,585,388
Chunk 15: S2 rows processed = 1,500,000 | candidates = 47,754,942
Chunk 16: S2 rows processed = 1,600,000 

In [15]:
import os

path = "/content/test_s2_candidates.tsv"

size_gb = os.path.getsize(path) / (1024 ** 3)

print("S2 candidate file size:", round(size_gb, 2), "GB")

S2 candidate file size: 4.18 GB


In [16]:
import os

path = "/content/test_s2_candidates.tsv"

if os.path.exists(path):
    os.remove(path)
    print("Deleted:", path)
else:
    print("File already deleted")

Deleted: /content/test_s2_candidates.tsv


In [17]:
# ============================================
# STEP 4: SELECTIVE S2 BLOCKING
# Exact name/address + country
# ============================================

import pandas as pd
import os
import gc

S2_TEST_PATH = "/content/test_source2.tsv"
S2_EXACT_PATH = "/content/test_s2_exact_candidates.tsv"

CHUNK_SIZE = 100_000

if os.path.exists(S2_EXACT_PATH):
    os.remove(S2_EXACT_PATH)

total_rows = 0
total_candidates = 0
first_write = True

# ------------------------------------------------
# Build country-aware indexes from S1
# ------------------------------------------------

s1_name_country = {}
s1_address_country = {}

for idx, row in test_s1.iterrows():

    country = str(row["country"]).strip().lower()
    s1_id = row["entity_id"]

    name = normalize_name_v2(row["business_name"])

    if name:
        key = (country, name)

        if key not in s1_name_country:
            s1_name_country[key] = []

        s1_name_country[key].append(s1_id)

    address = normalize_address(row["business_address"])

    if address:
        key = (country, address)

        if key not in s1_address_country:
            s1_address_country[key] = []

        s1_address_country[key].append(s1_id)


print("Country-aware indexes ready:")
print("Name keys:", len(s1_name_country))
print("Address keys:", len(s1_address_country))


# ------------------------------------------------
# Stream S2
# ------------------------------------------------

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S2_TEST_PATH,
        sep="\t",
        dtype={
            "entity_id": "string",
            "business_name": "string",
            "business_address": "string",
            "country": "string"
        },
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    candidate_rows = []

    for _, row in chunk.iterrows():

        s2_id = row["entity_id"]
        country = str(row["country"]).strip().lower()

        # ----------------------------------------
        # Exact normalized name + country
        # ----------------------------------------

        name = normalize_name_v2(row["business_name"])

        if name:

            key = (country, name)

            for s1_id in s1_name_country.get(key, []):
                candidate_rows.append(
                    (s1_id, s2_id, "S2")
                )

        # ----------------------------------------
        # Exact normalized address + country
        # ----------------------------------------

        address = normalize_address(row["business_address"])

        if address:

            key = (country, address)

            for s1_id in s1_address_country.get(key, []):
                candidate_rows.append(
                    (s1_id, s2_id, "S2")
                )

    if candidate_rows:

        candidate_df = pd.DataFrame(
            candidate_rows,
            columns=[
                "entity_id_s1",
                "entity_id_matched",
                "source"
            ]
        ).drop_duplicates()

        candidate_df.to_csv(
            S2_EXACT_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False

        total_candidates += len(candidate_df)

    total_rows += len(chunk)

    print(
        f"Chunk {chunk_no}: "
        f"rows={total_rows:,} | "
        f"candidates={total_candidates:,}"
    )

    del chunk
    gc.collect()


print("\n" + "=" * 60)
print("SELECTIVE S2 BLOCKING COMPLETE")
print("=" * 60)

print("Rows processed:", f"{total_rows:,}")
print("Candidates:", f"{total_candidates:,}")
print("Output:", S2_EXACT_PATH)

if os.path.exists(S2_EXACT_PATH):
    print(
        "File size:",
        round(
            os.path.getsize(S2_EXACT_PATH) / (1024**2),
            2
        ),
        "MB"
    )

Country-aware indexes ready:
Name keys: 1227942
Address keys: 1673358
Chunk 1: rows=100,000 | candidates=122,982
Chunk 2: rows=200,000 | candidates=248,054
Chunk 3: rows=300,000 | candidates=371,993
Chunk 4: rows=400,000 | candidates=494,143
Chunk 5: rows=500,000 | candidates=616,901
Chunk 6: rows=600,000 | candidates=740,770
Chunk 7: rows=700,000 | candidates=866,568
Chunk 8: rows=800,000 | candidates=991,959
Chunk 9: rows=900,000 | candidates=1,112,746
Chunk 10: rows=1,000,000 | candidates=1,238,915
Chunk 11: rows=1,100,000 | candidates=1,365,559
Chunk 12: rows=1,200,000 | candidates=1,488,214
Chunk 13: rows=1,300,000 | candidates=1,612,334
Chunk 14: rows=1,400,000 | candidates=1,736,180
Chunk 15: rows=1,500,000 | candidates=1,858,852
Chunk 16: rows=1,600,000 | candidates=1,980,159
Chunk 17: rows=1,700,000 | candidates=2,105,080
Chunk 18: rows=1,800,000 | candidates=2,229,307
Chunk 19: rows=1,900,000 | candidates=2,352,154
Chunk 20: rows=2,000,000 | candidates=2,477,024
Chunk 21: row

In [18]:
import shutil

total, used, free = shutil.disk_usage("/content")

print("Total disk :", round(total / (1024**3), 2), "GB")
print("Used disk  :", round(used / (1024**3), 2), "GB")
print("Free disk  :", round(free / (1024**3), 2), "GB")

Total disk : 107.72 GB
Used disk  : 21.62 GB
Free disk  : 86.08 GB


In [19]:
# ============================================
# STEP 5: BUILD SELECTIVE ADDRESS INDEX
# ============================================

from collections import defaultdict
import re
import gc

def get_address_numbers(text):
    if not text:
        return set()

    return set(re.findall(r"\d+", str(text)))


def address_words_v2(text):
    text = normalize_address(text)

    words = re.findall(r"[a-z]+", text)

    stop = {
        "road", "rd", "street", "st",
        "avenue", "ave", "floor", "fl",
        "no", "number", "near", "the",
        "and", "of", "for", "block",
        "unit", "india"
    }

    return {
        w for w in words
        if len(w) >= 4 and w not in stop
    }


# --------------------------------------------
# Count address-word frequency across S1
# --------------------------------------------

address_word_frequency = defaultdict(int)

for _, row in test_s1.iterrows():

    words = address_words_v2(
        row["business_address"]
    )

    for word in words:
        address_word_frequency[word] += 1


# --------------------------------------------
# Keep only relatively rare words
# --------------------------------------------

MAX_WORD_FREQUENCY = 15

rare_address_words = {
    word
    for word, count in address_word_frequency.items()
    if count <= MAX_WORD_FREQUENCY
}


print("=" * 60)
print("ADDRESS WORD FREQUENCY")
print("=" * 60)

print("Unique address words:",
      len(address_word_frequency))

print(
    f"Rare words (frequency <= {MAX_WORD_FREQUENCY}):",
    len(rare_address_words)
)


# --------------------------------------------
# Build country + address-number + word index
# --------------------------------------------

address_block_index = defaultdict(list)

for _, row in test_s1.iterrows():

    s1_id = row["entity_id"]

    country = str(
        row["country"]
    ).strip().lower()

    numbers = get_address_numbers(
        row["business_address"]
    )

    words = address_words_v2(
        row["business_address"]
    )

    words = words & rare_address_words

    if not numbers or not words:
        continue

    for number in numbers:

        for word in words:

            key = (
                country,
                number,
                word
            )

            address_block_index[key].append(
                s1_id
            )


gc.collect()


print("\n" + "=" * 60)
print("SELECTIVE ADDRESS INDEX READY")
print("=" * 60)

print("Index keys:",
      len(address_block_index))

ADDRESS WORD FREQUENCY
Unique address words: 324557
Rare words (frequency <= 15): 296479

SELECTIVE ADDRESS INDEX READY
Index keys: 834857


In [21]:
def get_address_numbers(text):
    if pd.isna(text):
        return set()

    text = str(text)

    return set(re.findall(r"\d+", text))

In [22]:
# ============================================
# STEP 6: ADD SELECTIVE ADDRESS CANDIDATES TO S2
# ============================================

import pandas as pd
import os
import gc

S2_TEST_PATH = "/content/test_source2.tsv"
S2_ADDRESS_PATH = "/content/test_s2_address_candidates.tsv"

CHUNK_SIZE = 100_000

if os.path.exists(S2_ADDRESS_PATH):
    os.remove(S2_ADDRESS_PATH)

total_rows = 0
total_candidates = 0
first_write = True

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S2_TEST_PATH,
        sep="\t",
        dtype={
            "entity_id": "string",
            "business_name": "string",
            "business_address": "string",
            "country": "string"
        },
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    candidate_rows = []

    for _, row in chunk.iterrows():

        s2_id = row["entity_id"]

        country = str(
            row["country"]
        ).strip().lower()

        numbers = get_address_numbers(
            row["business_address"]
        )

        words = address_words_v2(
            row["business_address"]
        )

        # Only rare address words
        words = words & rare_address_words

        if not numbers or not words:
            continue

        # ----------------------------------------
        # Country + address number + rare word
        # ----------------------------------------

        for number in numbers:

            for word in words:

                key = (
                    country,
                    number,
                    word
                )

                for s1_id in address_block_index.get(
                    key, []
                ):
                    candidate_rows.append(
                        (s1_id, s2_id, "S2")
                    )

    if candidate_rows:

        candidate_df = pd.DataFrame(
            candidate_rows,
            columns=[
                "entity_id_s1",
                "entity_id_matched",
                "source"
            ]
        ).drop_duplicates()

        candidate_df.to_csv(
            S2_ADDRESS_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False

        total_candidates += len(candidate_df)

    total_rows += len(chunk)

    print(
        f"Chunk {chunk_no}: "
        f"rows={total_rows:,} | "
        f"address candidates={total_candidates:,}"
    )

    del chunk
    gc.collect()


print("\n" + "=" * 60)
print("S2 ADDRESS BLOCKING COMPLETE")
print("=" * 60)

print(
    "Rows processed:",
    f"{total_rows:,}"
)

print(
    "Address candidates:",
    f"{total_candidates:,}"
)

print(
    "Output:",
    S2_ADDRESS_PATH
)

if os.path.exists(S2_ADDRESS_PATH):
    print(
        "File size:",
        round(
            os.path.getsize(S2_ADDRESS_PATH)
            / (1024**2),
            2
        ),
        "MB"
    )

Chunk 1: rows=100,000 | address candidates=28,966
Chunk 2: rows=200,000 | address candidates=57,582
Chunk 3: rows=300,000 | address candidates=87,038
Chunk 4: rows=400,000 | address candidates=115,941
Chunk 5: rows=500,000 | address candidates=144,566
Chunk 6: rows=600,000 | address candidates=173,125
Chunk 7: rows=700,000 | address candidates=202,192
Chunk 8: rows=800,000 | address candidates=231,034
Chunk 9: rows=900,000 | address candidates=260,222
Chunk 10: rows=1,000,000 | address candidates=288,652
Chunk 11: rows=1,100,000 | address candidates=317,138
Chunk 12: rows=1,200,000 | address candidates=346,005
Chunk 13: rows=1,300,000 | address candidates=374,434
Chunk 14: rows=1,400,000 | address candidates=402,951
Chunk 15: rows=1,500,000 | address candidates=431,383
Chunk 16: rows=1,600,000 | address candidates=460,013
Chunk 17: rows=1,700,000 | address candidates=489,142
Chunk 18: rows=1,800,000 | address candidates=518,337
Chunk 19: rows=1,900,000 | address candidates=547,267
Chun

In [23]:
# ============================================
# STEP 7: COMBINE + DEDUPLICATE S2 CANDIDATES
# ============================================

import pandas as pd
import os
import gc

EXACT_PATH = "/content/test_s2_exact_candidates.tsv"
ADDRESS_PATH = "/content/test_s2_address_candidates.tsv"
S2_FINAL_PATH = "/content/test_s2_final_candidates.tsv"

print("Reading exact candidates...")

s2_exact = pd.read_csv(
    EXACT_PATH,
    sep="\t",
    dtype="string"
)

print("Exact candidates:", len(s2_exact))

print("Reading address candidates...")

s2_address = pd.read_csv(
    ADDRESS_PATH,
    sep="\t",
    dtype="string"
)

print("Address candidates:", len(s2_address))

# Combine
s2_candidates = pd.concat(
    [s2_exact, s2_address],
    ignore_index=True
)

print(
    "\nBefore deduplication:",
    len(s2_candidates)
)

# Deduplicate by the actual pair
s2_candidates = s2_candidates.drop_duplicates(
    subset=[
        "entity_id_s1",
        "entity_id_matched",
        "source"
    ]
).reset_index(drop=True)

print(
    "After deduplication:",
    len(s2_candidates)
)

# Save
s2_candidates.to_csv(
    S2_FINAL_PATH,
    sep="\t",
    index=False
)

print("\n" + "=" * 60)
print("S2 FINAL CANDIDATES READY")
print("=" * 60)

print("Rows:", len(s2_candidates))

print(
    "File size:",
    round(
        os.path.getsize(S2_FINAL_PATH)
        / (1024**2),
        2
    ),
    "MB"
)

del s2_exact
del s2_address
gc.collect()

Reading exact candidates...
Exact candidates: 6079910
Reading address candidates...
Address candidates: 1408537

Before deduplication: 7488447
After deduplication: 7265043

S2 FINAL CANDIDATES READY
Rows: 7265043
File size: 199.4 MB


0

In [24]:
# ============================================
# STEP 8: SELECTIVE S3 BLOCKING
# Exact name/address + country
# ============================================

import pandas as pd
import os
import gc

S3_TEST_PATH = "/content/test_source3.tsv"
S3_EXACT_PATH = "/content/test_s3_exact_candidates.tsv"

CHUNK_SIZE = 100_000

if os.path.exists(S3_EXACT_PATH):
    os.remove(S3_EXACT_PATH)

total_rows = 0
total_candidates = 0
first_write = True

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S3_TEST_PATH,
        sep="\t",
        dtype={
            "entity_id": "string",
            "business_name": "string",
            "business_address": "string",
            "country": "string"
        },
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    candidate_rows = []

    for _, row in chunk.iterrows():

        s3_id = row["entity_id"]

        country = str(
            row["country"]
        ).strip().lower()

        # ----------------------------------------
        # Exact normalized name + country
        # ----------------------------------------

        name = normalize_name_v2(
            row["business_name"]
        )

        if name:

            key = (country, name)

            for s1_id in s1_name_country.get(
                key, []
            ):
                candidate_rows.append(
                    (s1_id, s3_id, "S3")
                )

        # ----------------------------------------
        # Exact normalized address + country
        # ----------------------------------------

        address = normalize_address(
            row["business_address"]
        )

        if address:

            key = (country, address)

            for s1_id in s1_address_country.get(
                key, []
            ):
                candidate_rows.append(
                    (s1_id, s3_id, "S3")
                )

    if candidate_rows:

        candidate_df = pd.DataFrame(
            candidate_rows,
            columns=[
                "entity_id_s1",
                "entity_id_matched",
                "source"
            ]
        ).drop_duplicates()

        candidate_df.to_csv(
            S3_EXACT_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False

        total_candidates += len(candidate_df)

    total_rows += len(chunk)

    print(
        f"Chunk {chunk_no}: "
        f"rows={total_rows:,} | "
        f"candidates={total_candidates:,}"
    )

    del chunk
    gc.collect()


print("\n" + "=" * 60)
print("S3 EXACT BLOCKING COMPLETE")
print("=" * 60)

print("Rows processed:", f"{total_rows:,}")
print("Candidates:", f"{total_candidates:,}")
print("Output:", S3_EXACT_PATH)

if os.path.exists(S3_EXACT_PATH):
    print(
        "File size:",
        round(
            os.path.getsize(S3_EXACT_PATH)
            / (1024**2),
            2
        ),
        "MB"
    )

Chunk 1: rows=100,000 | candidates=140,248
Chunk 2: rows=200,000 | candidates=279,611
Chunk 3: rows=300,000 | candidates=414,151
Chunk 4: rows=400,000 | candidates=557,577
Chunk 5: rows=500,000 | candidates=691,066
Chunk 6: rows=600,000 | candidates=831,696
Chunk 7: rows=700,000 | candidates=968,987
Chunk 8: rows=800,000 | candidates=1,107,186
Chunk 9: rows=900,000 | candidates=1,247,886
Chunk 10: rows=1,000,000 | candidates=1,386,534
Chunk 11: rows=1,100,000 | candidates=1,523,585
Chunk 12: rows=1,200,000 | candidates=1,662,138
Chunk 13: rows=1,300,000 | candidates=1,798,553
Chunk 14: rows=1,400,000 | candidates=1,940,865
Chunk 15: rows=1,500,000 | candidates=2,079,252
Chunk 16: rows=1,600,000 | candidates=2,216,550
Chunk 17: rows=1,700,000 | candidates=2,356,857
Chunk 18: rows=1,800,000 | candidates=2,495,713
Chunk 19: rows=1,900,000 | candidates=2,630,877
Chunk 20: rows=2,000,000 | candidates=2,772,205
Chunk 21: rows=2,100,000 | candidates=2,907,762
Chunk 22: rows=2,200,000 | candid

In [25]:
# ============================================
# STEP 9: ADD SELECTIVE ADDRESS CANDIDATES TO S3
# ============================================

import pandas as pd
import os
import gc

S3_TEST_PATH = "/content/test_source3.tsv"
S3_ADDRESS_PATH = "/content/test_s3_address_candidates.tsv"

CHUNK_SIZE = 100_000

if os.path.exists(S3_ADDRESS_PATH):
    os.remove(S3_ADDRESS_PATH)

total_rows = 0
total_candidates = 0
first_write = True

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S3_TEST_PATH,
        sep="\t",
        dtype={
            "entity_id": "string",
            "business_name": "string",
            "business_address": "string",
            "country": "string"
        },
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    candidate_rows = []

    for _, row in chunk.iterrows():

        s3_id = row["entity_id"]

        country = str(
            row["country"]
        ).strip().lower()

        numbers = get_address_numbers(
            row["business_address"]
        )

        words = address_words_v2(
            row["business_address"]
        )

        words = words & rare_address_words

        if not numbers or not words:
            continue

        for number in numbers:

            for word in words:

                key = (
                    country,
                    number,
                    word
                )

                for s1_id in address_block_index.get(
                    key, []
                ):
                    candidate_rows.append(
                        (s1_id, s3_id, "S3")
                    )

    if candidate_rows:

        candidate_df = pd.DataFrame(
            candidate_rows,
            columns=[
                "entity_id_s1",
                "entity_id_matched",
                "source"
            ]
        ).drop_duplicates()

        candidate_df.to_csv(
            S3_ADDRESS_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False

        total_candidates += len(candidate_df)

    total_rows += len(chunk)

    print(
        f"Chunk {chunk_no}: "
        f"rows={total_rows:,} | "
        f"address candidates={total_candidates:,}"
    )

    del chunk
    gc.collect()


print("\n" + "=" * 60)
print("S3 ADDRESS BLOCKING COMPLETE")
print("=" * 60)

print("Rows processed:", f"{total_rows:,}")
print("Address candidates:", f"{total_candidates:,}")
print("Output:", S3_ADDRESS_PATH)

if os.path.exists(S3_ADDRESS_PATH):
    print(
        "File size:",
        round(
            os.path.getsize(S3_ADDRESS_PATH)
            / (1024**2),
            2
        ),
        "MB"
    )

Chunk 1: rows=100,000 | address candidates=24,046
Chunk 2: rows=200,000 | address candidates=48,762
Chunk 3: rows=300,000 | address candidates=73,472
Chunk 4: rows=400,000 | address candidates=98,031
Chunk 5: rows=500,000 | address candidates=122,974
Chunk 6: rows=600,000 | address candidates=147,456
Chunk 7: rows=700,000 | address candidates=171,982
Chunk 8: rows=800,000 | address candidates=196,792
Chunk 9: rows=900,000 | address candidates=221,322
Chunk 10: rows=1,000,000 | address candidates=245,901
Chunk 11: rows=1,100,000 | address candidates=270,225
Chunk 12: rows=1,200,000 | address candidates=294,625
Chunk 13: rows=1,300,000 | address candidates=318,942
Chunk 14: rows=1,400,000 | address candidates=343,213
Chunk 15: rows=1,500,000 | address candidates=367,817
Chunk 16: rows=1,600,000 | address candidates=392,206
Chunk 17: rows=1,700,000 | address candidates=416,977
Chunk 18: rows=1,800,000 | address candidates=441,587
Chunk 19: rows=1,900,000 | address candidates=466,159
Chunk

In [26]:
# ============================================
# STEP 10: COMBINE + DEDUPLICATE S3 CANDIDATES
# ============================================

import pandas as pd
import os
import gc

EXACT_PATH = "/content/test_s3_exact_candidates.tsv"
ADDRESS_PATH = "/content/test_s3_address_candidates.tsv"
S3_FINAL_PATH = "/content/test_s3_final_candidates.tsv"

print("Reading exact candidates...")

s3_exact = pd.read_csv(
    EXACT_PATH,
    sep="\t",
    dtype="string"
)

print("Exact candidates:", len(s3_exact))

print("Reading address candidates...")

s3_address = pd.read_csv(
    ADDRESS_PATH,
    sep="\t",
    dtype="string"
)

print("Address candidates:", len(s3_address))

s3_candidates = pd.concat(
    [s3_exact, s3_address],
    ignore_index=True
)

print("\nBefore deduplication:", len(s3_candidates))

s3_candidates = s3_candidates.drop_duplicates(
    subset=[
        "entity_id_s1",
        "entity_id_matched",
        "source"
    ]
).reset_index(drop=True)

print("After deduplication:", len(s3_candidates))

s3_candidates.to_csv(
    S3_FINAL_PATH,
    sep="\t",
    index=False
)

print("\n" + "=" * 60)
print("S3 FINAL CANDIDATES READY")
print("=" * 60)

print("Rows:", len(s3_candidates))

print(
    "File size:",
    round(
        os.path.getsize(S3_FINAL_PATH)
        / (1024**2),
        2
    ),
    "MB"
)

del s3_exact
del s3_address
gc.collect()

Reading exact candidates...
Exact candidates: 7065869
Reading address candidates...
Address candidates: 1246731

Before deduplication: 8312600
After deduplication: 8133221

S3 FINAL CANDIDATES READY
Rows: 8133221
File size: 223.21 MB


0

In [27]:
# ============================================
# STEP 11: INSPECT HIGH-CONFIDENCE CANDIDATES
# ============================================

import pandas as pd

s2_exact_path = "/content/test_s2_exact_candidates.tsv"
s3_exact_path = "/content/test_s3_exact_candidates.tsv"

s2_exact = pd.read_csv(
    s2_exact_path,
    sep="\t",
    dtype="string"
)

s3_exact = pd.read_csv(
    s3_exact_path,
    sep="\t",
    dtype="string"
)

s2_exact = s2_exact.drop_duplicates(
    ["entity_id_s1", "entity_id_matched"]
)

s3_exact = s3_exact.drop_duplicates(
    ["entity_id_s1", "entity_id_matched"]
)

print("=" * 60)
print("HIGH-CONFIDENCE EXACT CANDIDATES")
print("=" * 60)

print("S2 exact pairs:", len(s2_exact))
print("S3 exact pairs:", len(s3_exact))

print(
    "Unique S1 with S2 exact matches:",
    s2_exact["entity_id_s1"].nunique()
)

print(
    "Unique S1 with S3 exact matches:",
    s3_exact["entity_id_s1"].nunique()
)

print("\nS2 sample:")
display(s2_exact.head())

print("\nS3 sample:")
display(s3_exact.head())

HIGH-CONFIDENCE EXACT CANDIDATES
S2 exact pairs: 6079910
S3 exact pairs: 7065869
Unique S1 with S2 exact matches: 1102565
Unique S1 with S3 exact matches: 1108597

S2 sample:


,entity_id_s1,entity_id_matched,source
0,S1-607177411,S2-89663826,S2
1,S1-10991820,S2-884102769,S2
2,S1-402022266,S2-375151053,S2
3,S1-896984903,S2-375151053,S2
4,S1-651381537,S2-375151053,S2



S3 sample:


,entity_id_s1,entity_id_matched,source
0,S1-553894773,S3-934627663,S3
1,S1-224045947,S3-934627663,S3
2,S1-516719347,S3-934627663,S3
3,S1-4891216,S3-934627663,S3
4,S1-746625519,S3-934627663,S3


In [28]:
# How many exact candidates does each S1 entity have?

s2_counts = (
    s2_exact.groupby("entity_id_s1")["entity_id_matched"]
    .nunique()
)

s3_counts = (
    s3_exact.groupby("entity_id_s1")["entity_id_matched"]
    .nunique()
)

print("S2 candidate multiplicity:")
print(s2_counts.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]))

print("\nS3 candidate multiplicity:")
print(s3_counts.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]))

print("\nS2 entities with >1 exact candidate:", (s2_counts > 1).sum())
print("S2 entities with >5 exact candidates:", (s2_counts > 5).sum())
print("S2 entities with >10 exact candidates:", (s2_counts > 10).sum())

print("\nS3 entities with >1 exact candidate:", (s3_counts > 1).sum())
print("S3 entities with >5 exact candidates:", (s3_counts > 5).sum())
print("S3 entities with >10 exact candidates:", (s3_counts > 10).sum())

S2 candidate multiplicity:
count    1.102565e+06
mean     5.514332e+00
std      1.212287e+01
min      1.000000e+00
50%      2.000000e+00
75%      4.000000e+00
90%      1.100000e+01
95%      2.400000e+01
99%      5.900000e+01
99.9%    1.330000e+02
max      2.270000e+02
Name: entity_id_matched, dtype: float64

S3 candidate multiplicity:
count    1.108597e+06
mean     6.373704e+00
std      1.242397e+01
min      1.000000e+00
50%      2.000000e+00
75%      7.000000e+00
90%      1.500000e+01
95%      2.400000e+01
99%      6.000000e+01
99.9%    1.290000e+02
max      2.810000e+02
Name: entity_id_matched, dtype: float64

S2 entities with >1 exact candidate: 643692
S2 entities with >5 exact candidates: 224215
S2 entities with >10 exact candidates: 115480

S3 entities with >1 exact candidate: 660648
S3 entities with >5 exact candidates: 317816
S3 entities with >10 exact candidates: 187127


In [29]:
# Pick a few S1 entities that have many exact S2 candidates
sample_s1_s2 = (
    s2_counts[s2_counts >= 10]
    .sort_values(ascending=False)
    .head(5)
    .index
)

print("S2 examples:\n")

for s1_id in sample_s1_s2:
    print("=" * 80)
    print("S1:", s1_id)

    rows = s2_exact[s2_exact["entity_id_s1"] == s1_id]
    print("Number of candidates:", len(rows))
    display(rows.head(20))

S2 examples:

S1: S1-218608617
Number of candidates: 227


,entity_id_s1,entity_id_matched,source
19036,S1-218608617,S2-720833267,S2
33505,S1-218608617,S2-827565023,S2
46684,S1-218608617,S2-510757616,S2
56860,S1-218608617,S2-453801588,S2
60898,S1-218608617,S2-196703165,S2
86233,S1-218608617,S2-246445052,S2
90934,S1-218608617,S2-781435158,S2
96157,S1-218608617,S2-482474748,S2
104460,S1-218608617,S2-935167696,S2
147661,S1-218608617,S2-947079145,S2


S1: S1-985693120
Number of candidates: 227


,entity_id_s1,entity_id_matched,source
4298,S1-985693120,S2-137062847,S2
30149,S1-985693120,S2-841505563,S2
75297,S1-985693120,S2-583486470,S2
117209,S1-985693120,S2-547485991,S2
158024,S1-985693120,S2-380713828,S2
165814,S1-985693120,S2-990250376,S2
175987,S1-985693120,S2-391714648,S2
206864,S1-985693120,S2-946384623,S2
249249,S1-985693120,S2-953022490,S2
257804,S1-985693120,S2-854410942,S2


S1: S1-253154556
Number of candidates: 226


,entity_id_s1,entity_id_matched,source
4330,S1-253154556,S2-137062847,S2
30181,S1-253154556,S2-841505563,S2
75329,S1-253154556,S2-583486470,S2
117241,S1-253154556,S2-547485991,S2
158056,S1-253154556,S2-380713828,S2
165846,S1-253154556,S2-990250376,S2
176019,S1-253154556,S2-391714648,S2
206896,S1-253154556,S2-946384623,S2
249281,S1-253154556,S2-953022490,S2
257836,S1-253154556,S2-854410942,S2


S1: S1-830245681
Number of candidates: 226


,entity_id_s1,entity_id_matched,source
4352,S1-830245681,S2-137062847,S2
30203,S1-830245681,S2-841505563,S2
75351,S1-830245681,S2-583486470,S2
117263,S1-830245681,S2-547485991,S2
158078,S1-830245681,S2-380713828,S2
165868,S1-830245681,S2-990250376,S2
176041,S1-830245681,S2-391714648,S2
206918,S1-830245681,S2-946384623,S2
249303,S1-830245681,S2-953022490,S2
257858,S1-830245681,S2-854410942,S2


S1: S1-345321512
Number of candidates: 225


,entity_id_s1,entity_id_matched,source
4304,S1-345321512,S2-137062847,S2
30155,S1-345321512,S2-841505563,S2
75303,S1-345321512,S2-583486470,S2
117215,S1-345321512,S2-547485991,S2
158030,S1-345321512,S2-380713828,S2
165820,S1-345321512,S2-990250376,S2
175993,S1-345321512,S2-391714648,S2
206870,S1-345321512,S2-946384623,S2
249255,S1-345321512,S2-953022490,S2
257810,S1-345321512,S2-854410942,S2


In [30]:
# Inspect one S1 with many candidates
s1_id = "S1-985693120"

# Get the S1 record
s1_row = test_s1[test_s1["entity_id"] == s1_id]

print("=" * 100)
print("SOURCE 1 RECORD")
print("=" * 100)
display(s1_row)

# Get its S2 candidate IDs
candidate_ids = s2_exact.loc[
    s2_exact["entity_id_s1"] == s1_id,
    "entity_id_matched"
].tolist()

print("\nNumber of S2 candidates:", len(candidate_ids))

# Read only the relevant S2 records from the TSV
candidate_ids_set = set(candidate_ids)

s2_matches = []

for chunk in pd.read_csv(
    "/content/test_source2.tsv",
    sep="\t",
    dtype="string",
    chunksize=200_000
):
    matches = chunk[chunk["entity_id"].isin(candidate_ids_set)]
    if len(matches):
        s2_matches.append(matches)

s2_matches = pd.concat(s2_matches, ignore_index=True)

print("\nS2 candidate records found:", len(s2_matches))
display(s2_matches.head(30))

SOURCE 1 RECORD


,entity_id,business_name,business_address,country
17307,S1-985693120,Bordeaux Club,"17 Rue Planterose, Bordeaux, Nouvelle-Aquitaine",France



Number of S2 candidates: 227

S2 candidate records found: 227


,entity_id,business_name,business_address,country
0,S2-137062847,BORDEAUX (CLUB),"189 BOULEVARD JEAN-JACQUES BOSC, BORDEAUX, Gir...",France
1,S2-841505563,BORDEAUX CLUB,"98 RUE EMILE COMBES, BORDEAUX",France
2,S2-583486470,Bordeaux Club,"NO 56 AVENUE DU JEU DE PAUME, BORDEAUX",France
3,S2-547485991,Bordeaux Club,"118 RUE GUILLAUME LEBLANC, BORDEAUX, Nouvelle-...",France
4,S2-380713828,Bordeaux Club,"RUE CROIX DE SEGUEY, Gironde, BORDEAUX",France
5,S2-990250376,Bordeaux Club,"81 BD ALRED DANEY, BORDEAUX, Nouvelle-Aquitaine",France
6,S2-391714648,Bordeaux Club,"Gironde, RESIDENCE COTE DARGENT ENTREE D-APT 2...",France
7,S2-946384623,Bordeaux Club,"63 - R NAUJAC, BORDEAUX",France
8,S2-953022490,Bordeaux Club,"16 R DE ROSNY, BORDEAUX",France
9,S2-854410942,BORDEAUX CLUB,"28 RUE MICHEL FAVREAU, BORDEAUX",France


In [32]:
!pip install rapidfuzz -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.6 MB/s eta 0:00:00


In [33]:
from rapidfuzz.fuzz import ratio, token_set_ratio

# Get the S1 address
s1_address = s1_row.iloc[0]["business_address"]

print("S1 address:")
print(s1_address)
print("\n")

# Calculate address similarities
s2_matches["address_ratio"] = s2_matches["business_address"].apply(
    lambda x: ratio(
        normalize_address(s1_address),
        normalize_address(x)
    )
)

s2_matches["address_token_ratio"] = s2_matches["business_address"].apply(
    lambda x: token_set_ratio(
        normalize_address(s1_address),
        normalize_address(x)
    )
)

# Show highest scoring candidates
display(
    s2_matches[
        [
            "entity_id",
            "business_name",
            "business_address",
            "address_ratio",
            "address_token_ratio"
        ]
    ]
    .sort_values(
        ["address_ratio", "address_token_ratio"],
        ascending=False
    )
    .head(20)
)

S1 address:
17 Rue Planterose, Bordeaux, Nouvelle-Aquitaine




,entity_id,business_name,business_address,address_ratio,address_token_ratio
41,S2-40217109,Bordeaux Centre Sarl,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",100.000000,100.000000
117,S2-984170521,Bordeaux Club-SAS,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",100.000000,100.000000
173,S2-67221085,BORDEAUX SARL DÉVELOPPEMENT,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",100.000000,100.000000
135,S2-538330415,BORDEAUX-CLUB,"7 RUE LUFLADE, BORDEAUX, Nouvelle-Aquitaine",86.046512,86.111111
159,S2-193665614,Bordeaux Club,"11 RUE DES PONTETS, BORDEAUX, Nouvelle-Aquitaine",85.714286,85.714286
199,S2-647204554,Bordeaux Club,"11 RUE DES PONTETS, BORDEAUX, Nouvelle-Aquitaine",85.714286,85.714286
21,S2-128745056,Bordeaux Club,"73 RUE ANTOINE MONIER, BORDEAUX, Nouvelle-Aqui...",85.106383,85.106383
190,S2-177966355,Bordeaux Club,"51 RUE NOTRE DAME, BORDEAUX, Nouvelle-Aquitaine",84.444444,86.666667
19,S2-725118058,Bordeaux Club,"97 RUE MALBEC, BORDEAUX, Nouvelle-Aquitaine",83.720930,86.111111
58,S2-783625176,Bordeaux Club,"#30 RUE BASQUE, BORDEAUX, Nouvelle-Aquitaine",83.720930,86.111111


In [34]:
import os

files_to_check = [
    "/content/dev_final_candidates_dedup.csv",
    "/content/rf_training_pairs.csv",
    "/content/rf_model.pkl",
    "/content/test_source1.tsv",
    "/content/test_source2.tsv",
    "/content/test_source3.tsv"
]

for f in files_to_check:
    print(f"{f}: {'EXISTS' if os.path.exists(f) else 'MISSING'}")

/content/dev_final_candidates_dedup.csv: MISSING
/content/rf_training_pairs.csv: MISSING
/content/rf_model.pkl: MISSING
/content/test_source1.tsv: EXISTS
/content/test_source2.tsv: EXISTS
/content/test_source3.tsv: EXISTS


In [35]:
from rapidfuzz.fuzz import ratio, token_set_ratio

s1_name = s1_row.iloc[0]["business_name"]
s1_address = s1_row.iloc[0]["business_address"]

# Name similarities
s2_matches["name_ratio"] = s2_matches["business_name"].apply(
    lambda x: ratio(
        normalize_name_v2(s1_name),
        normalize_name_v2(x)
    )
)

s2_matches["name_token_ratio"] = s2_matches["business_name"].apply(
    lambda x: token_set_ratio(
        normalize_name_v2(s1_name),
        normalize_name_v2(x)
    )
)

# Address similarities
s2_matches["address_ratio"] = s2_matches["business_address"].apply(
    lambda x: ratio(
        normalize_address(s1_address),
        normalize_address(x)
    )
)

s2_matches["address_token_ratio"] = s2_matches["business_address"].apply(
    lambda x: token_set_ratio(
        normalize_address(s1_address),
        normalize_address(x)
    )
)

# Address number overlap
s1_numbers = get_address_numbers(normalize_address(s1_address))

def number_overlap(addr):
    candidate_numbers = get_address_numbers(normalize_address(addr))

    if not s1_numbers or not candidate_numbers:
        return 0

    return int(bool(s1_numbers & candidate_numbers))

s2_matches["numeric_overlap"] = s2_matches["business_address"].apply(
    number_overlap
)

# Combined score
s2_matches["combined_score"] = (
    0.50 * s2_matches["name_token_ratio"] +
    0.35 * s2_matches["address_token_ratio"] +
    0.15 * s2_matches["numeric_overlap"] * 100
)

display(
    s2_matches[
        [
            "entity_id",
            "business_name",
            "business_address",
            "name_ratio",
            "name_token_ratio",
            "address_ratio",
            "address_token_ratio",
            "numeric_overlap",
            "combined_score"
        ]
    ]
    .sort_values("combined_score", ascending=False)
    .head(20)
)

,entity_id,business_name,business_address,name_ratio,name_token_ratio,address_ratio,address_token_ratio,numeric_overlap,combined_score
117,S2-984170521,Bordeaux Club-SAS,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",86.666667,100.000000,100.000000,100.000000,1,100.000000
225,S2-212264540,bordeaux club,"17 RUE HENRI EXPERT, BORDEAUX, Nouvelle-Aquitaine",100.000000,100.000000,82.608696,86.075949,1,95.126582
213,S2-722862921,Bordeaux Club,"17 R. HENRI EXPERT, Nouvelle-Aquitaine, BORDEAUX",100.000000,100.000000,64.444444,82.222222,1,93.777778
26,S2-691647116,Bordeaux Club,"17 RUE ADOLPHE CHALES, BORDEAUX",100.000000,100.000000,53.333333,66.666667,1,88.333333
173,S2-67221085,BORDEAUX SARL DÉVELOPPEMENT,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",50.000000,76.190476,100.000000,100.000000,1,88.095238
41,S2-40217109,Bordeaux Centre Sarl,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",66.666667,76.190476,100.000000,100.000000,1,88.095238
166,S2-156627655,Bordeaux Club,"17 RUE HENRUI GUILLEMIN, BORDEAUX",100.000000,100.000000,49.350649,63.829787,1,87.340426
146,S2-540616362,BORDEAUX CLUB,"17 R. LATESTA, BORDEAUX, Gironde",100.000000,100.000000,62.162162,62.162162,1,86.756757
156,S2-111498358,Bordeaux Club,BORDEAUX,100.000000,100.000000,30.188679,100.000000,0,85.000000
207,S2-831589965,Bordeaux Club,"17 RUE CROIX DE SEGUEY, BORDEAUX, Gironde",100.000000,100.000000,57.142857,57.142857,1,85.000000


In [36]:
s2_matches["strict_score"] = (
    0.55 * s2_matches["name_token_ratio"] +
    0.45 * s2_matches["address_token_ratio"]
)

# Strong bonus when the complete normalized address is identical
s2_matches.loc[
    s2_matches["address_ratio"] >= 99,
    "strict_score"
] += 10

display(
    s2_matches[
        [
            "entity_id",
            "business_name",
            "business_address",
            "name_token_ratio",
            "address_token_ratio",
            "address_ratio",
            "numeric_overlap",
            "strict_score"
        ]
    ]
    .sort_values("strict_score", ascending=False)
    .head(20)
)

,entity_id,business_name,business_address,name_token_ratio,address_token_ratio,address_ratio,numeric_overlap,strict_score
117,S2-984170521,Bordeaux Club-SAS,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",100.000000,100.000000,100.000000,1,110.000000
156,S2-111498358,Bordeaux Club,BORDEAUX,100.000000,100.000000,30.188679,0,100.000000
41,S2-40217109,Bordeaux Centre Sarl,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",76.190476,100.000000,100.000000,1,96.904762
173,S2-67221085,BORDEAUX SARL DÉVELOPPEMENT,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",76.190476,100.000000,100.000000,1,96.904762
190,S2-177966355,Bordeaux Club,"51 RUE NOTRE DAME, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.666667,84.444444,0,94.000000
187,S2-72616839,Bordeaux Club,"13 RUE NAUJAC, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.111111,81.395349,0,93.750000
135,S2-538330415,BORDEAUX-CLUB,"7 RUE LUFLADE, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.111111,86.046512,0,93.750000
58,S2-783625176,Bordeaux Club,"#30 RUE BASQUE, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.111111,83.720930,0,93.750000
19,S2-725118058,Bordeaux Club,"97 RUE MALBEC, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.111111,83.720930,0,93.750000
225,S2-212264540,bordeaux club,"17 RUE HENRI EXPERT, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.075949,82.608696,1,93.734177


In [37]:
# New combined score
s2_matches["final_test_score"] = (
    0.50 * s2_matches["name_token_ratio"] +
    0.50 * s2_matches["address_token_ratio"]
)

# Strong bonus for an exactly matching normalized address
s2_matches.loc[
    s2_matches["address_ratio"] >= 99,
    "final_test_score"
] += 20

display(
    s2_matches[
        [
            "entity_id",
            "business_name",
            "business_address",
            "name_token_ratio",
            "address_token_ratio",
            "address_ratio",
            "final_test_score"
        ]
    ]
    .sort_values("final_test_score", ascending=False)
    .head(20)
)

,entity_id,business_name,business_address,name_token_ratio,address_token_ratio,address_ratio,final_test_score
117,S2-984170521,Bordeaux Club-SAS,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",100.000000,100.000000,100.000000,120.000000
173,S2-67221085,BORDEAUX SARL DÉVELOPPEMENT,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",76.190476,100.000000,100.000000,108.095238
41,S2-40217109,Bordeaux Centre Sarl,"17 RUE PLANTEROSE, BORDEAUX, Nouvelle-Aquitaine",76.190476,100.000000,100.000000,108.095238
156,S2-111498358,Bordeaux Club,BORDEAUX,100.000000,100.000000,30.188679,100.000000
190,S2-177966355,Bordeaux Club,"51 RUE NOTRE DAME, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.666667,84.444444,93.333333
187,S2-72616839,Bordeaux Club,"13 RUE NAUJAC, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.111111,81.395349,93.055556
135,S2-538330415,BORDEAUX-CLUB,"7 RUE LUFLADE, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.111111,86.046512,93.055556
58,S2-783625176,Bordeaux Club,"#30 RUE BASQUE, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.111111,83.720930,93.055556
19,S2-725118058,Bordeaux Club,"97 RUE MALBEC, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.111111,83.720930,93.055556
225,S2-212264540,bordeaux club,"17 RUE HENRI EXPERT, BORDEAUX, Nouvelle-Aquitaine",100.000000,86.075949,82.608696,93.037975


In [38]:
import os

for f in [
    "/content/train_source1.tsv",
    "/content/train_source2.tsv",
    "/content/train_source3.tsv",
    "/content/train_ground_truth.tsv"
]:
    print(f"{f}: {'EXISTS' if os.path.exists(f) else 'MISSING'}")

/content/train_source1.tsv: MISSING
/content/train_source2.tsv: MISSING
/content/train_source3.tsv: MISSING
/content/train_ground_truth.tsv: MISSING


In [39]:
print("test_s1:", test_s1.shape if "test_s1" in globals() else "MISSING")
print("s2_exact:", s2_exact.shape if "s2_exact" in globals() else "MISSING")
print("s3_exact:", s3_exact.shape if "s3_exact" in globals() else "MISSING")
print("s2_matches:", s2_matches.shape if "s2_matches" in globals() else "MISSING")

test_s1: (1732544, 4)
s2_exact: (6079910, 3)
s3_exact: (7065869, 3)
s2_matches: (227, 12)


In [40]:
import pandas as pd
import os

S2_PATH = "/content/test_source2.tsv"
S2_LOOKUP_PATH = "/content/s2_exact_lookup.tsv"

# Only the S2 IDs that appear in our exact candidates
needed_s2_ids = set(s2_exact["entity_id_matched"].unique())

print("Unique S2 IDs needed:", len(needed_s2_ids))

# Remove old file if it exists
if os.path.exists(S2_LOOKUP_PATH):
    os.remove(S2_LOOKUP_PATH)

first_write = True
found = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S2_PATH,
        sep="\t",
        dtype="string",
        chunksize=200_000
    ),
    start=1
):
    matched = chunk[chunk["entity_id"].isin(needed_s2_ids)]

    if len(matched) > 0:
        matched.to_csv(
            S2_LOOKUP_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False
        found += len(matched)

    if chunk_no % 5 == 0:
        print(
            f"Processed {(chunk_no * 200_000):,} rows | "
            f"Found {found:,}"
        )

print("\nDone.")
print("S2 records found:", found)
print("Lookup file exists:", os.path.exists(S2_LOOKUP_PATH))

Unique S2 IDs needed: 1202265
Processed 1,000,000 rows | Found 245,889
Processed 2,000,000 rows | Found 491,412
Processed 3,000,000 rows | Found 737,575
Processed 4,000,000 rows | Found 983,996
Processed 5,000,000 rows | Found 1,202,265

Done.
S2 records found: 1202265
Lookup file exists: True


In [41]:
from rapidfuzz.fuzz import ratio, token_set_ratio
import pandas as pd
import os

CANDIDATE_PATH = "/content/test_s2_exact_candidates.tsv"
LOOKUP_PATH = "/content/s2_exact_lookup.tsv"
OUTPUT_PATH = "/content/s2_high_confidence_matches.tsv"

# ---------------------------------------------------------
# Load only the S2 records needed for our candidates
# ---------------------------------------------------------
s2_lookup = pd.read_csv(
    LOOKUP_PATH,
    sep="\t",
    dtype="string"
)

s2_lookup = s2_lookup.drop_duplicates("entity_id")

print("S2 lookup:", s2_lookup.shape)

# ---------------------------------------------------------
# Create small S1 lookup containing only required columns
# ---------------------------------------------------------
s1_lookup = test_s1[
    ["entity_id", "business_name", "business_address"]
].copy()

s1_lookup = s1_lookup.rename(
    columns={
        "entity_id": "entity_id_s1",
        "business_name": "s1_name",
        "business_address": "s1_address"
    }
)

print("S1 lookup:", s1_lookup.shape)

# ---------------------------------------------------------
# Remove previous output
# ---------------------------------------------------------
if os.path.exists(OUTPUT_PATH):
    os.remove(OUTPUT_PATH)

first_write = True
total_candidates = 0
total_kept = 0

# ---------------------------------------------------------
# Process candidate file in chunks
# ---------------------------------------------------------
for chunk_no, candidates in enumerate(
    pd.read_csv(
        CANDIDATE_PATH,
        sep="\t",
        dtype="string",
        chunksize=200_000
    ),
    start=1
):

    # Join S1 information
    candidates = candidates.merge(
        s1_lookup,
        on="entity_id_s1",
        how="left"
    )

    # Join S2 information
    candidates = candidates.merge(
        s2_lookup[
            ["entity_id", "business_name", "business_address"]
        ],
        left_on="entity_id_matched",
        right_on="entity_id",
        how="left"
    )

    candidates = candidates.rename(
        columns={
            "business_name": "s2_name",
            "business_address": "s2_address"
        }
    )

    # -----------------------------------------------------
    # Normalize names and addresses
    # -----------------------------------------------------
    candidates["s1_name_norm"] = candidates["s1_name"].map(
        normalize_name_v2
    )

    candidates["s2_name_norm"] = candidates["s2_name"].map(
        normalize_name_v2
    )

    candidates["s1_addr_norm"] = candidates["s1_address"].map(
        normalize_address
    )

    candidates["s2_addr_norm"] = candidates["s2_address"].map(
        normalize_address
    )

    # -----------------------------------------------------
    # Similarities
    # -----------------------------------------------------
    candidates["name_ratio"] = [
        ratio(a, b)
        for a, b in zip(
            candidates["s1_name_norm"],
            candidates["s2_name_norm"]
        )
    ]

    candidates["name_token_ratio"] = [
        token_set_ratio(a, b)
        for a, b in zip(
            candidates["s1_name_norm"],
            candidates["s2_name_norm"]
        )
    ]

    candidates["address_ratio"] = [
        ratio(a, b)
        for a, b in zip(
            candidates["s1_addr_norm"],
            candidates["s2_addr_norm"]
        )
    ]

    candidates["address_token_ratio"] = [
        token_set_ratio(a, b)
        for a, b in zip(
            candidates["s1_addr_norm"],
            candidates["s2_addr_norm"]
        )
    ]

    # -----------------------------------------------------
    # Exact signals
    # -----------------------------------------------------
    candidates["name_exact"] = (
        candidates["s1_name_norm"] ==
        candidates["s2_name_norm"]
    )

    candidates["address_exact"] = (
        candidates["s1_addr_norm"] ==
        candidates["s2_addr_norm"]
    )

    # -----------------------------------------------------
    # Address-number overlap
    # -----------------------------------------------------
    candidates["s1_numbers"] = candidates["s1_addr_norm"].map(
        get_address_numbers
    )

    candidates["s2_numbers"] = candidates["s2_addr_norm"].map(
        get_address_numbers
    )

    candidates["numeric_overlap"] = [
        int(bool(a & b))
        for a, b in zip(
            candidates["s1_numbers"],
            candidates["s2_numbers"]
        )
    ]

    # -----------------------------------------------------
    # HIGH-CONFIDENCE RULE
    #
    # Rule A:
    # exact address + name similarity >= 85
    #
    # OR
    #
    # Rule B:
    # exact name + address similarity >= 95
    # -----------------------------------------------------
    keep = (
        (
            candidates["address_exact"] &
            (candidates["name_token_ratio"] >= 85)
        )
        |
        (
            candidates["name_exact"] &
            (candidates["address_token_ratio"] >= 95)
        )
    )

    kept = candidates.loc[
        keep,
        [
            "entity_id_s1",
            "entity_id_matched",
            "name_ratio",
            "name_token_ratio",
            "address_ratio",
            "address_token_ratio",
            "name_exact",
            "address_exact",
            "numeric_overlap"
        ]
    ].copy()

    # Write immediately
    if len(kept) > 0:
        kept.to_csv(
            OUTPUT_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )
        first_write = False

    total_candidates += len(candidates)
    total_kept += len(kept)

    if chunk_no % 5 == 0:
        print(
            f"Processed: {total_candidates:,} | "
            f"High-confidence: {total_kept:,}"
        )

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)
print("Total candidates:", total_candidates)
print("High-confidence matches:", total_kept)
print("Output:", OUTPUT_PATH)

S2 lookup: (1202265, 4)
S1 lookup: (1732544, 3)
Processed: 1,000,000 | High-confidence: 103,903
Processed: 2,000,000 | High-confidence: 207,999
Processed: 3,000,000 | High-confidence: 311,072
Processed: 4,000,000 | High-confidence: 415,267
Processed: 5,000,000 | High-confidence: 517,714
Processed: 6,000,000 | High-confidence: 620,960

DONE
Total candidates: 6079910
High-confidence matches: 629025
Output: /content/s2_high_confidence_matches.tsv


In [42]:
id="r8s2m1"
# Inspect multiplicity of high-confidence S2 matches

s2_hc = pd.read_csv(
    "/content/s2_high_confidence_matches.tsv",
    sep="\t",
    dtype="string"
)

s2_hc_counts = (
    s2_hc.groupby("entity_id_s1")["entity_id_matched"]
    .nunique()
)

print("=" * 70)
print("S2 HIGH-CONFIDENCE MATCH MULTIPLICITY")
print("=" * 70)

print("Total high-confidence pairs:", len(s2_hc))
print("S1 entities with matches:", s2_hc_counts.size)

print("\nDistribution:")
print(
    s2_hc_counts.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )
)

print("\nS1 entities with >1 match:", (s2_hc_counts > 1).sum())
print("S1 entities with >5 matches:", (s2_hc_counts > 5).sum())
print("S1 entities with >10 matches:", (s2_hc_counts > 10).sum())
print("S1 entities with >20 matches:", (s2_hc_counts > 20).sum())
print("Maximum matches for one S1:", s2_hc_counts.max())

S2 HIGH-CONFIDENCE MATCH MULTIPLICITY
Total high-confidence pairs: 629025
S1 entities with matches: 496780

Distribution:
count    496780.000000
mean          1.266204
std           0.545050
min           1.000000
50%           1.000000
75%           1.000000
90%           2.000000
95%           2.000000
99%           3.000000
99.9%         4.000000
max          22.000000
Name: entity_id_matched, dtype: float64

S1 entities with >1 match: 110288
S1 entities with >5 matches: 8
S1 entities with >10 matches: 3
S1 entities with >20 matches: 2
Maximum matches for one S1: 22


In [43]:
# Inspect the S1 entities with the most high-confidence S2 matches

top_s2_entities = (
    s2_hc_counts
    .sort_values(ascending=False)
    .head(10)
)

print("Top S2 entities by number of retained matches:")
print(top_s2_entities)

for s1_id, count in top_s2_entities.items():
    print("\n" + "=" * 100)
    print("S1:", s1_id, "| Retained matches:", count)

    display(
        s2_hc[
            s2_hc["entity_id_s1"] == s1_id
        ].sort_values(
            ["address_exact", "name_token_ratio"],
            ascending=False
        )
    )

Top S2 entities by number of retained matches:
entity_id_s1
S1-960907519    22
S1-697537894    22
S1-729011057    12
S1-822474985     6
S1-608458814     6
S1-86074280      6
S1-781302444     6
S1-657133346     6
S1-827882455     5
S1-429691212     5
Name: entity_id_matched, dtype: int64

S1: S1-960907519 | Retained matches: 22


,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
22798,S1-960907519,S2-655192631,100.0,100.0,45.714285714285715,100.0,True,False,0
24232,S1-960907519,S2-209730769,100.0,100.0,76.05633802816901,100.0,True,False,0
37032,S1-960907519,S2-646576576,100.0,100.0,71.05263157894737,100.0,True,False,0
60642,S1-960907519,S2-84811825,100.0,100.0,81.81818181818181,100.0,True,False,0
114966,S1-960907519,S2-244115877,100.0,100.0,41.379310344827594,100.0,True,False,0
115084,S1-960907519,S2-805035542,100.0,100.0,75.0,100.0,True,False,0
123512,S1-960907519,S2-494610107,100.0,100.0,65.06024096385542,100.0,True,False,0
125256,S1-960907519,S2-498188863,100.0,100.0,85.71428571428572,100.0,True,False,0
155126,S1-960907519,S2-218345852,100.0,100.0,45.714285714285715,100.0,True,False,0
210140,S1-960907519,S2-886839479,100.0,100.0,73.97260273972603,100.0,True,False,0



S1: S1-697537894 | Retained matches: 22


,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
2019,S1-697537894,S2-638926005,100.0,100.0,63.76811594202898,100.0,True,False,0
38577,S1-697537894,S2-295018599,100.0,100.0,79.41176470588236,100.0,True,False,0
66251,S1-697537894,S2-766124912,100.0,100.0,45.714285714285715,100.0,True,False,0
140903,S1-697537894,S2-250382844,100.0,100.0,75.0,100.0,True,False,0
173294,S1-697537894,S2-115966667,100.0,100.0,70.12987012987013,100.0,True,False,0
201641,S1-697537894,S2-781240382,100.0,100.0,73.97260273972603,100.0,True,False,0
212480,S1-697537894,S2-267625265,100.0,100.0,73.97260273972603,100.0,True,False,0
236331,S1-697537894,S2-426412077,100.0,100.0,75.0,100.0,True,False,0
296736,S1-697537894,S2-464630053,100.0,100.0,70.12987012987013,100.0,True,False,0
313439,S1-697537894,S2-111756239,100.0,100.0,69.23076923076923,100.0,True,False,0



S1: S1-729011057 | Retained matches: 12


...

,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
219269,S1-657133346,S2-694966664,85.71428571428572,85.71428571428571,100.0,100.0,False,True,1
171756,S1-657133346,S2-602195499,100.0,100.0,100.0,100.0,True,True,1
100897,S1-657133346,S2-172069511,100.0,100.0,97.82608695652173,97.82608695652173,True,False,1
217230,S1-657133346,S2-449564544,100.0,100.0,96.62921348314607,96.62921348314607,True,False,1
304225,S1-657133346,S2-537923760,100.0,100.0,54.347826086956516,100.0,True,False,1
338538,S1-657133346,S2-801185057,100.0,100.0,95.45454545454545,96.29629629629629,True,False,1



S1: S1-827882455 | Retained matches: 5


,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
614689,S1-827882455,S2-116126489,93.02325581395348,92.3076923076923,100.0,100.0,False,True,1
535580,S1-827882455,S2-296633471,95.45454545454545,90.47619047619048,100.0,100.0,False,True,1
171674,S1-827882455,S2-538882701,77.27272727272727,89.47368421052632,100.0,100.0,False,True,1
358082,S1-827882455,S2-992577932,64.70588235294117,100.0,100.0,100.0,False,True,1
451518,S1-827882455,S2-893629805,100.0,100.0,100.0,100.0,True,True,1



S1: S1-429691212 | Retained matches: 5


,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
482520,S1-429691212,S2-894539762,89.79591836734694,89.79591836734694,100.0,100.0,False,True,1
447014,S1-429691212,S2-66600090,72.72727272727273,86.48648648648648,100.0,100.0,False,True,1
251420,S1-429691212,S2-545790062,100.0,100.0,100.0,100.0,True,True,1
297064,S1-429691212,S2-255320688,100.0,100.0,100.0,100.0,True,True,1
586474,S1-429691212,S2-521883412,100.0,100.0,97.5609756097561,97.5609756097561,True,False,1


In [44]:
print("\n" + "=" * 100)
print("BORDEAUX EXAMPLE")
print("=" * 100)

display(
    s2_hc[
        s2_hc["entity_id_s1"] == "S1-985693120"
    ]
)


BORDEAUX EXAMPLE


,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
305941,S1-985693120,S2-984170521,86.66666666666667,100.0,100.0,100.0,False,True,1
430887,S1-985693120,S2-111498358,100.0,100.0,30.188679245283023,100.0,True,False,0


In [45]:
# Apply corrected high-precision rule to existing S2 results

s2_hc_strict = s2_hc[
    (
        s2_hc["address_exact"].eq("True") &
        (s2_hc["name_token_ratio"].astype(float) >= 85)
    )
    |
    (
        s2_hc["name_exact"].eq("True") &
        (s2_hc["address_ratio"].astype(float) >= 95)
    )
].copy()

# Remove duplicate S1-S2 pairs just in case
s2_hc_strict = s2_hc_strict.drop_duplicates(
    ["entity_id_s1", "entity_id_matched"]
)

print("=" * 70)
print("STRICT S2 RESULTS")
print("=" * 70)
print("Pairs:", len(s2_hc_strict))
print("S1 entities:", s2_hc_strict["entity_id_s1"].nunique())

strict_counts = (
    s2_hc_strict
    .groupby("entity_id_s1")["entity_id_matched"]
    .nunique()
)

print("\nMultiplicity:")
print(
    strict_counts.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )
)

print("\n>1:", (strict_counts > 1).sum())
print(">5:", (strict_counts > 5).sum())
print(">10:", (strict_counts > 10).sum())
print("Max:", strict_counts.max())

print("\nBordeaux:")
display(
    s2_hc_strict[
        s2_hc_strict["entity_id_s1"] == "S1-985693120"
    ]
)

STRICT S2 RESULTS
Pairs: 344663
S1 entities: 270438

Multiplicity:
count    270438.000000
mean          1.274462
std           0.548646
min           1.000000
50%           1.000000
75%           1.000000
90%           2.000000
95%           2.000000
99%           3.000000
99.9%         4.000000
max           6.000000
Name: entity_id_matched, dtype: float64

>1: 61966
>5: 1
>10: 0
Max: 6

Bordeaux:


,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
305941,S1-985693120,S2-984170521,86.66666666666667,100.0,100.0,100.0,False,True,1


In [46]:
# Find the one S1 with 6 strict matches

top_strict = (
    strict_counts
    .sort_values(ascending=False)
    .head(5)
)

print(top_strict)

for s1_id, count in top_strict.items():
    print("\n" + "=" * 100)
    print("S1:", s1_id, "| Strict matches:", count)

    display(
        s2_hc_strict[
            s2_hc_strict["entity_id_s1"] == s1_id
        ]
    )

entity_id_s1
S1-781302444    6
S1-657133346    5
S1-320756739    5
S1-279686903    5
S1-657083385    5
Name: entity_id_matched, dtype: int64

S1: S1-781302444 | Strict matches: 6


,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
24160,S1-781302444,S2-466731347,73.6842105263158,91.2280701754386,100.0,100.0,False,True,1
279515,S1-781302444,S2-157367766,100.0,100.0,97.95918367346938,97.95918367346938,True,False,1
287395,S1-781302444,S2-699274166,100.0,100.0,97.95918367346938,97.95918367346938,True,False,1
342807,S1-781302444,S2-968592851,87.71929824561404,87.71929824561403,100.0,100.0,False,True,1
537495,S1-781302444,S2-737943698,100.0,100.0,97.91666666666666,97.91666666666667,True,False,1
566871,S1-781302444,S2-387235822,100.0,100.0,100.0,100.0,True,True,1



S1: S1-657133346 | Strict matches: 5


,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
100897,S1-657133346,S2-172069511,100.0,100.0,97.82608695652173,97.82608695652173,True,False,1
171756,S1-657133346,S2-602195499,100.0,100.0,100.0,100.0,True,True,1
217230,S1-657133346,S2-449564544,100.0,100.0,96.62921348314607,96.62921348314607,True,False,1
219269,S1-657133346,S2-694966664,85.71428571428572,85.71428571428571,100.0,100.0,False,True,1
338538,S1-657133346,S2-801185057,100.0,100.0,95.45454545454545,96.29629629629629,True,False,1



S1: S1-320756739 | Strict matches: 5


,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
64119,S1-320756739,S2-158907458,100.0,100.0,100.0,100.0,True,True,1
85429,S1-320756739,S2-117740612,100.0,100.0,100.0,100.0,True,True,1
123985,S1-320756739,S2-957678574,92.3076923076923,100.0,100.0,100.0,False,True,1
413142,S1-320756739,S2-36746968,75.36231884057972,92.3076923076923,100.0,100.0,False,True,1
443102,S1-320756739,S2-562406547,92.3076923076923,100.0,100.0,100.0,False,True,1



S1: S1-279686903 | Strict matches: 5


,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
328313,S1-279686903,S2-550172922,80.0,100.0,100.0,100.0,False,True,1
454779,S1-279686903,S2-35197470,81.63265306122449,100.0,100.0,100.0,False,True,1
475521,S1-279686903,S2-287636756,68.96551724137932,88.88888888888889,100.0,100.0,False,True,1
494539,S1-279686903,S2-779507834,100.0,100.0,100.0,100.0,True,True,1
535664,S1-279686903,S2-469499247,90.0,90.0,100.0,100.0,False,True,1



S1: S1-657083385 | Strict matches: 5


,entity_id_s1,entity_id_matched,name_ratio,name_token_ratio,address_ratio,address_token_ratio,name_exact,address_exact,numeric_overlap
177236,S1-657083385,S2-264264996,95.23809523809523,95.23809523809524,100.0,100.0,False,True,1
265971,S1-657083385,S2-701672065,95.23809523809523,95.23809523809524,100.0,100.0,False,True,1
278638,S1-657083385,S2-590652643,100.0,100.0,100.0,100.0,True,True,1
386790,S1-657083385,S2-72354390,100.0,100.0,100.0,100.0,True,True,1
407162,S1-657083385,S2-678046559,79.24528301886792,100.0,100.0,100.0,False,True,1


In [47]:
import pandas as pd
import os

S3_PATH = "/content/test_source3.tsv"
S3_CANDIDATE_PATH = "/content/test_s3_exact_candidates.tsv"
S3_LOOKUP_PATH = "/content/s3_exact_lookup.tsv"

needed_s3_ids = set(
    s3_exact["entity_id_matched"].unique()
)

print("Unique S3 IDs needed:", len(needed_s3_ids))

if os.path.exists(S3_LOOKUP_PATH):
    os.remove(S3_LOOKUP_PATH)

first_write = True
found = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S3_PATH,
        sep="\t",
        dtype="string",
        chunksize=200_000
    ),
    start=1
):
    matched = chunk[
        chunk["entity_id"].isin(needed_s3_ids)
    ]

    if len(matched) > 0:
        matched.to_csv(
            S3_LOOKUP_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False
        found += len(matched)

    if chunk_no % 5 == 0:
        print(
            f"Processed {(chunk_no * 200_000):,} rows | "
            f"Found {found:,}"
        )

print("\nDone.")
print("S3 records found:", found)
print("Lookup file exists:", os.path.exists(S3_LOOKUP_PATH))

Unique S3 IDs needed: 1210410
Processed 1,000,000 rows | Found 238,449
Processed 2,000,000 rows | Found 476,146
Processed 3,000,000 rows | Found 714,722
Processed 4,000,000 rows | Found 952,971
Processed 5,000,000 rows | Found 1,190,782

Done.
S3 records found: 1210410
Lookup file exists: True


In [48]:
import pandas as pd
from rapidfuzz.fuzz import ratio, token_set_ratio

S3_CANDIDATE_PATH = "/content/test_s3_exact_candidates.tsv"
S3_LOOKUP_PATH = "/content/s3_exact_lookup.tsv"

# Load S1 data needed for the candidates
s3_s1_ids = pd.read_csv(
    S3_CANDIDATE_PATH,
    sep="\t",
    usecols=["entity_id_s1"],
    dtype="string"
)["entity_id_s1"].unique()

s3_s1_needed = test_s1[
    test_s1["entity_id"].isin(s3_s1_ids)
][["entity_id", "business_name", "business_address", "country"]].copy()

s3_s1_needed = s3_s1_needed.rename(columns={
    "entity_id": "entity_id_s1",
    "business_name": "s1_name",
    "business_address": "s1_address",
    "country": "s1_country"
})

# Load the S3 lookup
s3_lookup = pd.read_csv(
    S3_LOOKUP_PATH,
    sep="\t",
    dtype="string"
)

s3_lookup = s3_lookup.rename(columns={
    "entity_id": "entity_id_matched",
    "business_name": "s3_name",
    "business_address": "s3_address",
    "country": "s3_country"
})

print("S1 records:", len(s3_s1_needed))
print("S3 records:", len(s3_lookup))

S1 records: 1108597
S3 records: 1210410


In [49]:
import pandas as pd
from rapidfuzz.fuzz import ratio, token_set_ratio

S3_CANDIDATE_PATH = "/content/test_s3_exact_candidates.tsv"

s3_matches = []

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S3_CANDIDATE_PATH,
        sep="\t",
        dtype="string",
        chunksize=200_000
    ),
    start=1
):
    # Join S1 information
    df = chunk.merge(
        s3_s1_needed,
        on="entity_id_s1",
        how="left"
    )

    # Join S3 information
    df = df.merge(
        s3_lookup,
        on="entity_id_matched",
        how="left"
    )

    # Calculate similarities
    df["name_ratio"] = [
        ratio(
            normalize_name_v2(a),
            normalize_name_v2(b)
        )
        for a, b in zip(df["s1_name"], df["s3_name"])
    ]

    df["name_token_ratio"] = [
        token_set_ratio(
            normalize_name_v2(a),
            normalize_name_v2(b)
        )
        for a, b in zip(df["s1_name"], df["s3_name"])
    ]

    df["address_ratio"] = [
        ratio(
            normalize_address(a),
            normalize_address(b)
        )
        for a, b in zip(df["s1_address"], df["s3_address"])
    ]

    df["name_exact"] = (
        df["s1_name"].map(normalize_name_v2)
        ==
        df["s3_name"].map(normalize_name_v2)
    )

    df["address_exact"] = (
        df["s1_address"].map(normalize_address)
        ==
        df["s3_address"].map(normalize_address)
    )

    # STRICT HIGH-CONFIDENCE RULE
    keep = (
        (df["address_exact"] & (df["name_token_ratio"] >= 85))
        |
        (df["name_exact"] & (df["address_ratio"] >= 95))
    )

    matched = df.loc[
        keep,
        ["entity_id_s1", "entity_id_matched", "source"]
    ].copy()

    if len(matched) > 0:
        s3_matches.append(matched)

    if chunk_no % 5 == 0:
        print(
            f"Processed {(chunk_no * 200_000):,} rows | "
            f"S3 matches so far: {sum(len(x) for x in s3_matches):,}"
        )

s3_hc_strict = pd.concat(
    s3_matches,
    ignore_index=True
)

# Remove any accidental duplicate pairs
s3_hc_strict = s3_hc_strict.drop_duplicates(
    subset=["entity_id_s1", "entity_id_matched"]
).reset_index(drop=True)

print("\n========== S3 STRICT RESULTS ==========")
print("Total strict S3 matches:", len(s3_hc_strict))
print(
    "S1 entities with S3 matches:",
    s3_hc_strict["entity_id_s1"].nunique()
)

counts = s3_hc_strict.groupby("entity_id_s1").size()

print("\nMatch multiplicity:")
print("Mean:", counts.mean())
print("Median:", counts.median())
print("90th percentile:", counts.quantile(0.90))
print("95th percentile:", counts.quantile(0.95))
print("99th percentile:", counts.quantile(0.99))
print("99.9th percentile:", counts.quantile(0.999))
print("Maximum:", counts.max())

print("\nEntities with >1 match:", (counts > 1).sum())
print("Entities with >5 matches:", (counts > 5).sum())
print("Entities with >10 matches:", (counts > 10).sum())

Processed 1,000,000 rows | S3 matches so far: 26,992
Processed 2,000,000 rows | S3 matches so far: 53,754
Processed 3,000,000 rows | S3 matches so far: 80,884
Processed 4,000,000 rows | S3 matches so far: 107,733
Processed 5,000,000 rows | S3 matches so far: 134,975
Processed 6,000,000 rows | S3 matches so far: 161,867
Processed 7,000,000 rows | S3 matches so far: 188,928

========== S3 STRICT RESULTS ==========
Total strict S3 matches: 190654
S1 entities with S3 matches: 168962

Match multiplicity:
Mean: 1.1283838969709166
Median: 1.0
90th percentile: 2.0
95th percentile: 2.0
99th percentile: 3.0
99.9th percentile: 4.0
Maximum: 6

Entities with >1 match: 17849
Entities with >5 matches: 2
Entities with >10 matches: 0


In [50]:
# Save strict S3 matches
S3_STRICT_PATH = "/content/s3_high_confidence_strict.tsv"

s3_hc_strict.to_csv(
    S3_STRICT_PATH,
    sep="\t",
    index=False
)

print("Saved:", S3_STRICT_PATH)
print("Rows:", len(s3_hc_strict))

Saved: /content/s3_high_confidence_strict.tsv
Rows: 190654


In [51]:
# ============================================
# BUILD FINAL matching_results.tsv
# ============================================

import pandas as pd

MATCHING_OUTPUT = "/content/matching_results.tsv"

# Keep only the two required columns from each source
s2_final = s2_hc_strict[
    ["entity_id_s1", "entity_id_matched"]
].copy()

s3_final = s3_hc_strict[
    ["entity_id_s1", "entity_id_matched"]
].copy()

# Combine S2 + S3
all_matches = pd.concat(
    [s2_final, s3_final],
    ignore_index=True
)

# Remove duplicate pairs just in case
all_matches = all_matches.drop_duplicates(
    subset=["entity_id_s1", "entity_id_matched"]
)

print("Total unique matched pairs:", len(all_matches))

# Group matched IDs by S1
grouped_matches = (
    all_matches
    .groupby("entity_id_s1")["entity_id_matched"]
    .apply(lambda x: ",".join(pd.unique(x)))
)

# Start from EVERY test S1 entity
matching_results = test_s1[["entity_id"]].copy()

matching_results = matching_results.rename(
    columns={"entity_id": "source1_entity_id"}
)

# Attach matches
matching_results["matched_entity_ids"] = (
    matching_results["source1_entity_id"]
    .map(grouped_matches)
    .fillna("")
)

# Save
matching_results.to_csv(
    MATCHING_OUTPUT,
    sep="\t",
    index=False
)

print("\n========== FINAL MATCHING RESULTS ==========")
print("Rows:", len(matching_results))
print("Expected S1 rows:", len(test_s1))
print(
    "Rows with at least one match:",
    (matching_results["matched_entity_ids"] != "").sum()
)
print(
    "Rows with NO match:",
    (matching_results["matched_entity_ids"] == "").sum()
)
print("File:", MATCHING_OUTPUT)

print("\nFirst 10 rows:")
print(matching_results.head(10))

Total unique matched pairs: 535317

========== FINAL MATCHING RESULTS ==========
Rows: 1732544
Expected S1 rows: 1732544
Rows with at least one match: 405328
Rows with NO match: 1327216
File: /content/matching_results.tsv

First 10 rows:
  source1_entity_id matched_entity_ids
0      S1-714132312                   
1      S1-106407869                   
2      S1-156285671                   
3      S1-689823050                   
4      S1-921369899                   
5      S1-481669221                   
6      S1-909865979       S2-528901108
7      S1-280204013                   
8      S1-742053041                   
9      S1-913506265                   


In [52]:
# ============================================
# BUILD FINAL candidate_pairs.tsv
# ============================================

import shutil
import os

S2_CANDIDATE_PATH = "/content/test_s2_exact_candidates.tsv"
S3_CANDIDATE_PATH = "/content/test_s3_exact_candidates.tsv"

CANDIDATE_OUTPUT = "/content/candidate_pairs.tsv"

# Remove old file if it exists
if os.path.exists(CANDIDATE_OUTPUT):
    os.remove(CANDIDATE_OUTPUT)

# Copy S2 candidates first
shutil.copyfile(
    S2_CANDIDATE_PATH,
    CANDIDATE_OUTPUT
)

# Append S3 candidates
with open(S3_CANDIDATE_PATH, "rb") as src:
    with open(CANDIDATE_OUTPUT, "ab") as dst:
        # Skip the header from S3
        src.readline()
        shutil.copyfileobj(src, dst)

print("Candidate file created.")
print("Path:", CANDIDATE_OUTPUT)

# Check size
size_mb = os.path.getsize(CANDIDATE_OUTPUT) / (1024 ** 2)
print(f"Size: {size_mb:.2f} MB")

Candidate file created.
Path: /content/candidate_pairs.tsv
Size: 360.79 MB


In [53]:
!python3 utils/validate_submission.py \
    --matching /content/matching_results.tsv \
    --candidate /content/candidate_pairs.tsv \
    --test-dir /content

python3: can't open file '/content/utils/validate_submission.py': [Errno 2] No such file or directory


In [55]:
!find /content -name "validate_submission.py" -type f

In [56]:
!ls -lah /content

total 2.6G
drwxr-xr-x 1 root root 4.0K Sep 27 12:00 .
drwxr-xr-x 1 root root 4.0K Sep 27 06:32 ..
-rw-r--r-- 1 root root 361M Sep 27 12:00 candidate_pairs.tsv
drwxr-xr-x 4 root root 4.0K Sep 16 13:26 .config
-rw-r--r-- 1 root root  30M Sep 27 11:59 matching_results.tsv
-rw-r--r-- 1 root root 106M Sep 27 11:15 s2_exact_lookup.tsv
-rw-r--r-- 1 root root  46M Sep 27 11:28 s2_high_confidence_matches.tsv
-rw-r--r-- 1 root root 105M Sep 27 11:36 s3_exact_lookup.tsv
-rw-r--r-- 1 root root 5.3M Sep 27 11:57 s3_high_confidence_strict.tsv
drwxr-xr-x 1 root root 4.0K Sep 16 13:26 sample_data
-rw-r--r-- 1 root root  39M Sep 27 10:13 test_s2_address_candidates.tsv
-rw-r--r-- 1 root root 167M Sep 27 09:51 test_s2_exact_candidates.tsv
-rw-r--r-- 1 root root 200M Sep 27 10:16 test_s2_final_candidates.tsv
-rw-r--r-- 1 root root  35M Sep 27 10:44 test_s3_address_candidates.tsv
-rw-r--r-- 1 root root 194M Sep 27 10:31 test_s3_exact_candidates.tsv
-rw-r--r-- 1 root root 224M Sep 27 10:54 test_s3_final_can

In [57]:
import pandas as pd
import os

MATCHING = "/content/matching_results.tsv"
CANDIDATES = "/content/candidate_pairs.tsv"

print("========== BASIC FILE CHECK ==========")

matching = pd.read_csv(
    MATCHING,
    sep="\t",
    dtype="string",
    keep_default_na=False
)

print("Matching rows:", len(matching))
print("Expected rows:", len(test_s1))

print("\nColumns:", list(matching.columns))

# --------------------------------------------------
# 1. Check every S1 entity appears exactly once
# --------------------------------------------------

print("\n========== S1 COVERAGE ==========")

duplicate_s1 = matching["source1_entity_id"].duplicated().sum()

expected_s1 = set(test_s1["entity_id"])
submitted_s1 = set(matching["source1_entity_id"])

missing_s1 = expected_s1 - submitted_s1
extra_s1 = submitted_s1 - expected_s1

print("Duplicate S1 rows:", duplicate_s1)
print("Missing S1 entities:", len(missing_s1))
print("Extra S1 entities:", len(extra_s1))

# --------------------------------------------------
# 2. Check duplicate matched IDs inside each row
# --------------------------------------------------

print("\n========== MATCH DUPLICATE CHECK ==========")

duplicate_match_rows = 0

for value in matching["matched_entity_ids"]:
    if value:
        ids = value.split(",")
        if len(ids) != len(set(ids)):
            duplicate_match_rows += 1

print("Rows containing duplicate matched IDs:", duplicate_match_rows)

# --------------------------------------------------
# 3. Load valid S2/S3 IDs
# --------------------------------------------------

print("\n========== VALID ID CHECK ==========")

valid_s2 = set()

for chunk in pd.read_csv(
    "/content/test_source2.tsv",
    sep="\t",
    usecols=["entity_id"],
    dtype="string",
    chunksize=500_000
):
    valid_s2.update(chunk["entity_id"])

print("Valid S2 IDs:", len(valid_s2))

valid_s3 = set()

for chunk in pd.read_csv(
    "/content/test_source3.tsv",
    sep="\t",
    usecols=["entity_id"],
    dtype="string",
    chunksize=500_000
):
    valid_s3.update(chunk["entity_id"])

print("Valid S3 IDs:", len(valid_s3))

valid_ids = valid_s2 | valid_s3

invalid_ids = set()

for value in matching["matched_entity_ids"]:
    if value:
        for entity_id in value.split(","):
            if entity_id not in valid_ids:
                invalid_ids.add(entity_id)

print("Invalid matched IDs:", len(invalid_ids))

# --------------------------------------------------
# 4. Check final matches are inside candidate_pairs
# --------------------------------------------------

print("\n========== CANDIDATE COVERAGE ==========")

# Build set of final submitted pairs
submitted_pairs = set()

for _, row in matching.iterrows():
    if row["matched_entity_ids"]:
        for matched_id in row["matched_entity_ids"].split(","):
            submitted_pairs.add(
                (row["source1_entity_id"], matched_id)
            )

print("Submitted match pairs:", len(submitted_pairs))

candidate_pairs_found = set()

for chunk_no, chunk in enumerate(
    pd.read_csv(
        CANDIDATES,
        sep="\t",
        dtype="string",
        chunksize=500_000
    ),
    start=1
):
    for s1, matched in zip(
        chunk["entity_id_s1"],
        chunk["entity_id_matched"]
    ):
        pair = (s1, matched)

        if pair in submitted_pairs:
            candidate_pairs_found.add(pair)

    print(
        f"Candidate rows checked: "
        f"{chunk_no * 500_000:,} | "
        f"Submitted pairs found: "
        f"{len(candidate_pairs_found):,}"
    )

missing_from_candidates = submitted_pairs - candidate_pairs_found

print("\n========== FINAL VALIDATION ==========")

print("Submitted pairs:", len(submitted_pairs))
print("Pairs found in candidate file:", len(candidate_pairs_found))
print("Submitted pairs missing from candidate file:",
      len(missing_from_candidates))

print("\n--------------------------------------")

if (
    len(missing_s1) == 0
    and duplicate_s1 == 0
    and duplicate_match_rows == 0
    and len(invalid_ids) == 0
    and len(missing_from_candidates) == 0
):
    print("✅ ALL IMPORTANT VALIDATION CHECKS PASSED")
else:
    print("❌ VALIDATION FAILED")

    if missing_s1:
        print("Missing S1:", len(missing_s1))
    if duplicate_s1:
        print("Duplicate S1 rows:", duplicate_s1)
    if duplicate_match_rows:
        print("Duplicate match rows:", duplicate_match_rows)
    if invalid_ids:
        print("Invalid IDs:", len(invalid_ids))
    if missing_from_candidates:
        print("Missing candidate pairs:", len(missing_from_candidates))

========== BASIC FILE CHECK ==========
Matching rows: 1732544
Expected rows: 1732544

Columns: ['source1_entity_id', 'matched_entity_ids']

========== S1 COVERAGE ==========
Duplicate S1 rows: 0
Missing S1 entities: 0
Extra S1 entities: 0

========== MATCH DUPLICATE CHECK ==========
Rows containing duplicate matched IDs: 0

========== VALID ID CHECK ==========
Valid S2 IDs: 4887273
Valid S3 IDs: 5082316
Invalid matched IDs: 0

========== CANDIDATE COVERAGE ==========
Submitted match pairs: 535317
Candidate rows checked: 500,000 | Submitted pairs found: 28,478
Candidate rows checked: 1,000,000 | Submitted pairs found: 57,095
Candidate rows checked: 1,500,000 | Submitted pairs found: 85,481
Candidate rows checked: 2,000,000 | Submitted pairs found: 114,081
Candidate rows checked: 2,500,000 | Submitted pairs found: 142,132
Candidate rows checked: 3,000,000 | Submitted pairs found: 170,256
Candidate rows checked: 3,500,000 | Submitted pairs found: 198,921
Candidate rows checked: 4,000,000 

In [58]:
from google.colab import files

files.download('/content/matching_results.tsv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
from google.colab import files

files.download('/content/candidate_pairs.tsv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
!ls -lh /content/matching_results.tsv /content/candidate_pairs.tsv

-rw-r--r-- 1 root root 361M Sep 27 12:00 /content/candidate_pairs.tsv
-rw-r--r-- 1 root root  30M Sep 27 11:59 /content/matching_results.tsv


In [61]:
import pandas as pd
import os

S2_ADDRESS_CANDIDATE_PATH = "/content/test_s2_address_candidates.tsv"
S2_ADDRESS_LOOKUP_PATH = "/content/s2_address_lookup.tsv"

# Get only the S2 IDs that occur in the broader address candidates
needed_s2_address_ids = set()

for chunk in pd.read_csv(
    S2_ADDRESS_CANDIDATE_PATH,
    sep="\t",
    dtype="string",
    chunksize=500_000
):
    needed_s2_address_ids.update(
        chunk["entity_id_matched"].dropna().unique()
    )

print("Unique S2 IDs needed:", len(needed_s2_address_ids))

# Scan Source 2 and extract only those records
if os.path.exists(S2_ADDRESS_LOOKUP_PATH):
    os.remove(S2_ADDRESS_LOOKUP_PATH)

first_write = True
found = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        "/content/test_source2.tsv",
        sep="\t",
        dtype="string",
        chunksize=500_000
    ),
    start=1
):
    matched = chunk[
        chunk["entity_id"].isin(needed_s2_address_ids)
    ]

    if len(matched) > 0:
        matched.to_csv(
            S2_ADDRESS_LOOKUP_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False
        found += len(matched)

    print(
        f"Processed {(chunk_no * 500_000):,} rows | "
        f"Found {found:,}"
    )

print("\n========== DONE ==========")
print("S2 records found:", found)
print("S2 IDs needed:", len(needed_s2_address_ids))
print("Lookup exists:", os.path.exists(S2_ADDRESS_LOOKUP_PATH))

Unique S2 IDs needed: 801969
Processed 500,000 rows | Found 81,968
Processed 1,000,000 rows | Found 163,921
Processed 1,500,000 rows | Found 245,750
Processed 2,000,000 rows | Found 327,866
Processed 2,500,000 rows | Found 409,842
Processed 3,000,000 rows | Found 492,202
Processed 3,500,000 rows | Found 574,243
Processed 4,000,000 rows | Found 655,949
Processed 4,500,000 rows | Found 738,145
Processed 5,000,000 rows | Found 801,969

========== DONE ==========
S2 records found: 801969
S2 IDs needed: 801969
Lookup exists: True


In [62]:
import pandas as pd
import os

S3_ADDRESS_CANDIDATE_PATH = "/content/test_s3_address_candidates.tsv"
S3_ADDRESS_LOOKUP_PATH = "/content/s3_address_lookup.tsv"

# Get all S3 IDs appearing in the broader address candidates
needed_s3_address_ids = set()

for chunk in pd.read_csv(
    S3_ADDRESS_CANDIDATE_PATH,
    sep="\t",
    dtype="string",
    chunksize=500_000
):
    needed_s3_address_ids.update(
        chunk["entity_id_matched"].dropna().unique()
    )

print("Unique S3 IDs needed:", len(needed_s3_address_ids))

# Extract those records from Source 3
if os.path.exists(S3_ADDRESS_LOOKUP_PATH):
    os.remove(S3_ADDRESS_LOOKUP_PATH)

first_write = True
found = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        "/content/test_source3.tsv",
        sep="\t",
        dtype="string",
        chunksize=500_000
    ),
    start=1
):
    matched = chunk[
        chunk["entity_id"].isin(needed_s3_address_ids)
    ]

    if len(matched) > 0:
        matched.to_csv(
            S3_ADDRESS_LOOKUP_PATH,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False
        found += len(matched)

    print(
        f"Processed {(chunk_no * 500_000):,} rows | "
        f"Found {found:,}"
    )

print("\n========== DONE ==========")
print("S3 records found:", found)
print("S3 IDs needed:", len(needed_s3_address_ids))
print("Lookup exists:", os.path.exists(S3_ADDRESS_LOOKUP_PATH))

Unique S3 IDs needed: 735497
Processed 500,000 rows | Found 72,666
Processed 1,000,000 rows | Found 144,905
Processed 1,500,000 rows | Found 216,956
Processed 2,000,000 rows | Found 289,510
Processed 2,500,000 rows | Found 362,077
Processed 3,000,000 rows | Found 434,637
Processed 3,500,000 rows | Found 506,744
Processed 4,000,000 rows | Found 578,948
Processed 4,500,000 rows | Found 651,416
Processed 5,000,000 rows | Found 723,650
Processed 5,500,000 rows | Found 735,497

========== DONE ==========
S3 records found: 735497
S3 IDs needed: 735497
Lookup exists: True


In [63]:
import pandas as pd
from rapidfuzz.fuzz import ratio, token_set_ratio

S2_ADDRESS_CANDIDATE_PATH = "/content/test_s2_address_candidates.tsv"

# S1 records needed by the address candidates
s2_addr_s1_ids = pd.read_csv(
    S2_ADDRESS_CANDIDATE_PATH,
    sep="\t",
    usecols=["entity_id_s1"],
    dtype="string"
)["entity_id_s1"].unique()

s2_addr_s1 = test_s1[
    test_s1["entity_id"].isin(s2_addr_s1_ids)
][
    ["entity_id", "business_name", "business_address", "country"]
].copy()

s2_addr_s1 = s2_addr_s1.rename(columns={
    "entity_id": "entity_id_s1",
    "business_name": "s1_name",
    "business_address": "s1_address",
    "country": "s1_country"
})

s2_addr_lookup = pd.read_csv(
    "/content/s2_address_lookup.tsv",
    sep="\t",
    dtype="string"
)

s2_addr_lookup = s2_addr_lookup.rename(columns={
    "entity_id": "entity_id_matched",
    "business_name": "s2_name",
    "business_address": "s2_address",
    "country": "s2_country"
})

print("S1 records:", len(s2_addr_s1))
print("S2 records:", len(s2_addr_lookup))

S1 records: 377453
S2 records: 801969


In [64]:
import pandas as pd
from rapidfuzz.fuzz import ratio, token_set_ratio

S2_ADDRESS_CANDIDATE_PATH = "/content/test_s2_address_candidates.tsv"

results = []

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S2_ADDRESS_CANDIDATE_PATH,
        sep="\t",
        dtype="string",
        chunksize=200_000
    ),
    start=1
):
    df = chunk.merge(
        s2_addr_s1,
        on="entity_id_s1",
        how="left"
    )

    df = df.merge(
        s2_addr_lookup,
        on="entity_id_matched",
        how="left"
    )

    # Normalize once
    s1_names = df["s1_name"].map(normalize_name_v2)
    s2_names = df["s2_name"].map(normalize_name_v2)

    s1_addresses = df["s1_address"].map(normalize_address)
    s2_addresses = df["s2_address"].map(normalize_address)

    # Similarities
    df["name_ratio"] = [
        ratio(a, b)
        for a, b in zip(s1_names, s2_names)
    ]

    df["name_token_ratio"] = [
        token_set_ratio(a, b)
        for a, b in zip(s1_names, s2_names)
    ]

    df["address_ratio"] = [
        ratio(a, b)
        for a, b in zip(s1_addresses, s2_addresses)
    ]

    df["address_token_ratio"] = [
        token_set_ratio(a, b)
        for a, b in zip(s1_addresses, s2_addresses)
    ]

    df["name_exact"] = s1_names == s2_names
    df["address_exact"] = s1_addresses == s2_addresses

    # Keep only useful columns for later inspection
    results.append(
        df[
            [
                "entity_id_s1",
                "entity_id_matched",
                "source",
                "name_ratio",
                "name_token_ratio",
                "address_ratio",
                "address_token_ratio",
                "name_exact",
                "address_exact"
            ]
        ]
    )

    if chunk_no % 5 == 0:
        print(
            f"Processed {(chunk_no * 200_000):,} rows | "
            f"Chunks stored: {len(results)}"
        )

s2_address_scored = pd.concat(
    results,
    ignore_index=True
)

print("\n========== S2 ADDRESS SCORING COMPLETE ==========")
print("Scored candidates:", len(s2_address_scored))

# Show how many candidates meet progressively strict rules
rules = {
    "Name >= 95 AND Address >= 90":
        (s2_address_scored["name_ratio"] >= 95) &
        (s2_address_scored["address_ratio"] >= 90),

    "Name >= 90 AND Address >= 90":
        (s2_address_scored["name_ratio"] >= 90) &
        (s2_address_scored["address_ratio"] >= 90),

    "Name token >= 95 AND Address >= 90":
        (s2_address_scored["name_token_ratio"] >= 95) &
        (s2_address_scored["address_ratio"] >= 90),

    "Name token >= 90 AND Address >= 90":
        (s2_address_scored["name_token_ratio"] >= 90) &
        (s2_address_scored["address_ratio"] >= 90),

    "Name >= 85 AND Address >= 95":
        (s2_address_scored["name_ratio"] >= 85) &
        (s2_address_scored["address_ratio"] >= 95),
}

print("\nCandidate counts by strictness:")

for name, mask in rules.items():
    print(f"{name}: {mask.sum():,}")

Processed 1,000,000 rows | Chunks stored: 5

========== S2 ADDRESS SCORING COMPLETE ==========
Scored candidates: 1408537

Candidate counts by strictness:
Name >= 95 AND Address >= 90: 82,867
Name >= 90 AND Address >= 90: 129,279
Name token >= 95 AND Address >= 90: 195,026
Name token >= 90 AND Address >= 90: 232,812
Name >= 85 AND Address >= 95: 97,764


In [65]:
# ============================================
# ANALYZE NEW S2 HIGH-CONFIDENCE CANDIDATES
# ============================================

# Existing matches from our 0.219911 solution
existing_s2_pairs = set(
    zip(
        s2_hc_strict["entity_id_s1"],
        s2_hc_strict["entity_id_matched"]
    )
)

# Very strict additional candidates
s2_new_strict = s2_address_scored[
    (s2_address_scored["name_ratio"] >= 95) &
    (s2_address_scored["address_ratio"] >= 90)
].copy()

# Remove pairs we already have
s2_new_strict["pair"] = list(
    zip(
        s2_new_strict["entity_id_s1"],
        s2_new_strict["entity_id_matched"]
    )
)

s2_new_strict = s2_new_strict[
    ~s2_new_strict["pair"].isin(existing_s2_pairs)
].copy()

print("========== NEW S2 CANDIDATES ==========")
print("New candidates:", len(s2_new_strict))
print(
    "S1 entities with new candidates:",
    s2_new_strict["entity_id_s1"].nunique()
)

# Multiplicity per S1
new_counts = s2_new_strict.groupby("entity_id_s1").size()

print("\nMultiplicity:")
print("Mean:", new_counts.mean())
print("Median:", new_counts.median())
print("90th percentile:", new_counts.quantile(0.90))
print("95th percentile:", new_counts.quantile(0.95))
print("99th percentile:", new_counts.quantile(0.99))
print("Maximum:", new_counts.max())

print("\nEntities with:")
print(">1:", (new_counts > 1).sum())
print(">2:", (new_counts > 2).sum())
print(">3:", (new_counts > 3).sum())

# Show examples with multiple new candidates
multi_ids = new_counts[
    new_counts > 1
].sort_values(ascending=False).head(10).index

print("\n========== EXAMPLES WITH MULTIPLE NEW MATCHES ==========")

for s1_id in multi_ids:
    rows = s2_new_strict[
        s2_new_strict["entity_id_s1"] == s1_id
    ]

    print("\nS1:", s1_id)

    print(
        rows[
            [
                "entity_id_matched",
                "name_ratio",
                "name_token_ratio",
                "address_ratio",
                "address_token_ratio"
            ]
        ].to_string(index=False)
    )

========== NEW S2 CANDIDATES ==========
New candidates: 38283
S1 entities with new candidates: 35031

Multiplicity:
Mean: 1.0928320630298878
Median: 1.0
90th percentile: 1.0
95th percentile: 2.0
99th percentile: 2.0
Maximum: 4

Entities with:
>1: 3045
>2: 203
>3: 4

========== EXAMPLES WITH MULTIPLE NEW MATCHES ==========

S1: S1-316151892
entity_id_matched  name_ratio  name_token_ratio  address_ratio  address_token_ratio
      S2-76564389  100.000000        100.000000      94.166667            99.107143
     S2-626250594   98.795181         98.795181      94.166667            99.107143
     S2-593476994   95.000000         95.000000      94.166667            99.107143
     S2-555923135  100.000000        100.000000      94.166667            99.107143

S1: S1-944950611
entity_id_matched  name_ratio  name_token_ratio  address_ratio  address_token_ratio
     S2-811735878  100.000000        100.000000      93.567251           100.000000
     S2-344275163   98.901099         98.901099     

In [66]:
# ============================================
# TEST A STRICTER S2 ADDITION RULE
# ============================================

s2_add_rule = (
    (s2_address_scored["name_ratio"] >= 95) &
    (s2_address_scored["address_ratio"] >= 93)
)

s2_add = s2_address_scored[s2_add_rule].copy()

# Remove pairs already in the baseline
s2_add["pair"] = list(zip(
    s2_add["entity_id_s1"],
    s2_add["entity_id_matched"]
))

s2_add = s2_add[
    ~s2_add["pair"].isin(existing_s2_pairs)
].copy()

print("========== S2 ADDITION CANDIDATES ==========")
print("New candidates:", len(s2_add))
print(
    "S1 entities:",
    s2_add["entity_id_s1"].nunique()
)

counts = s2_add.groupby("entity_id_s1").size()

print("\nMultiplicity:")
print("Mean:", counts.mean())
print("Median:", counts.median())
print("90th:", counts.quantile(.90))
print("95th:", counts.quantile(.95))
print("99th:", counts.quantile(.99))
print("Maximum:", counts.max())

print("\nEntities with >1:", (counts > 1).sum())
print("Entities with >2:", (counts > 2).sum())

========== S2 ADDITION CANDIDATES ==========
New candidates: 19467
S1 entities: 18079

Multiplicity:
Mean: 1.0767741578627137
Median: 1.0
90th: 1.0
95th: 2.0
99th: 2.0
Maximum: 4

Entities with >1: 1303
Entities with >2: 83


In [67]:
import pandas as pd
from rapidfuzz.fuzz import ratio, token_set_ratio

S3_ADDRESS_CANDIDATE_PATH = "/content/test_s3_address_candidates.tsv"

# S1 records needed by S3 address candidates
s3_addr_s1_ids = pd.read_csv(
    S3_ADDRESS_CANDIDATE_PATH,
    sep="\t",
    usecols=["entity_id_s1"],
    dtype="string"
)["entity_id_s1"].unique()

s3_addr_s1 = test_s1[
    test_s1["entity_id"].isin(s3_addr_s1_ids)
][
    ["entity_id", "business_name", "business_address", "country"]
].copy()

s3_addr_s1 = s3_addr_s1.rename(columns={
    "entity_id": "entity_id_s1",
    "business_name": "s1_name",
    "business_address": "s1_address",
    "country": "s1_country"
})

s3_addr_lookup = pd.read_csv(
    "/content/s3_address_lookup.tsv",
    sep="\t",
    dtype="string"
)

s3_addr_lookup = s3_addr_lookup.rename(columns={
    "entity_id": "entity_id_matched",
    "business_name": "s3_name",
    "business_address": "s3_address",
    "country": "s3_country"
})

print("S1 records:", len(s3_addr_s1))
print("S3 records:", len(s3_addr_lookup))

S1 records: 369691
S3 records: 735497


In [68]:
import pandas as pd
from rapidfuzz.fuzz import ratio, token_set_ratio

S3_ADDRESS_CANDIDATE_PATH = "/content/test_s3_address_candidates.tsv"

s3_results = []

for chunk_no, chunk in enumerate(
    pd.read_csv(
        S3_ADDRESS_CANDIDATE_PATH,
        sep="\t",
        dtype="string",
        chunksize=200_000
    ),
    start=1
):
    df = chunk.merge(
        s3_addr_s1,
        on="entity_id_s1",
        how="left"
    )

    df = df.merge(
        s3_addr_lookup,
        on="entity_id_matched",
        how="left"
    )

    s1_names = df["s1_name"].map(normalize_name_v2)
    s3_names = df["s3_name"].map(normalize_name_v2)

    s1_addresses = df["s1_address"].map(normalize_address)
    s3_addresses = df["s3_address"].map(normalize_address)

    df["name_ratio"] = [
        ratio(a, b)
        for a, b in zip(s1_names, s3_names)
    ]

    df["name_token_ratio"] = [
        token_set_ratio(a, b)
        for a, b in zip(s1_names, s3_names)
    ]

    df["address_ratio"] = [
        ratio(a, b)
        for a, b in zip(s1_addresses, s3_addresses)
    ]

    df["address_token_ratio"] = [
        token_set_ratio(a, b)
        for a, b in zip(s1_addresses, s3_addresses)
    ]

    df["name_exact"] = s1_names == s3_names
    df["address_exact"] = s1_addresses == s3_addresses

    s3_results.append(
        df[
            [
                "entity_id_s1",
                "entity_id_matched",
                "source",
                "name_ratio",
                "name_token_ratio",
                "address_ratio",
                "address_token_ratio",
                "name_exact",
                "address_exact"
            ]
        ]
    )

    if chunk_no % 5 == 0:
        print(
            f"Processed {(chunk_no * 200_000):,} rows | "
            f"Chunks stored: {len(s3_results)}"
        )

s3_address_scored = pd.concat(
    s3_results,
    ignore_index=True
)

print("\n========== S3 ADDRESS SCORING COMPLETE ==========")
print("Scored candidates:", len(s3_address_scored))

print("\nCandidate counts:")

rules = {
    "Name >= 95 AND Address >= 90":
        (s3_address_scored["name_ratio"] >= 95) &
        (s3_address_scored["address_ratio"] >= 90),

    "Name >= 95 AND Address >= 93":
        (s3_address_scored["name_ratio"] >= 95) &
        (s3_address_scored["address_ratio"] >= 93),

    "Name >= 90 AND Address >= 93":
        (s3_address_scored["name_ratio"] >= 90) &
        (s3_address_scored["address_ratio"] >= 93),

    "Name token >= 95 AND Address >= 93":
        (s3_address_scored["name_token_ratio"] >= 95) &
        (s3_address_scored["address_ratio"] >= 93),
}

for name, mask in rules.items():
    print(f"{name}: {mask.sum():,}")

Processed 1,000,000 rows | Chunks stored: 5

========== S3 ADDRESS SCORING COMPLETE ==========
Scored candidates: 1246731

Candidate counts:
Name >= 95 AND Address >= 90: 45,319
Name >= 95 AND Address >= 93: 25,390
Name >= 90 AND Address >= 93: 37,609
Name token >= 95 AND Address >= 93: 82,891


In [69]:
# ============================================
# ANALYZE NEW S3 ADDITION CANDIDATES
# ============================================

existing_s3_pairs = set(
    zip(
        s3_hc_strict["entity_id_s1"],
        s3_hc_strict["entity_id_matched"]
    )
)

# Same rule used for S2
s3_add = s3_address_scored[
    (s3_address_scored["name_ratio"] >= 95) &
    (s3_address_scored["address_ratio"] >= 93)
].copy()

s3_add["pair"] = list(zip(
    s3_add["entity_id_s1"],
    s3_add["entity_id_matched"]
))

# Remove matches already in baseline
s3_add = s3_add[
    ~s3_add["pair"].isin(existing_s3_pairs)
].copy()

print("========== S3 ADDITION CANDIDATES ==========")
print("New candidates:", len(s3_add))
print(
    "S1 entities:",
    s3_add["entity_id_s1"].nunique()
)

counts = s3_add.groupby("entity_id_s1").size()

print("\nMultiplicity:")
print("Mean:", counts.mean())
print("Median:", counts.median())
print("90th:", counts.quantile(.90))
print("95th:", counts.quantile(.95))
print("99th:", counts.quantile(.99))
print("Maximum:", counts.max())

print("\nEntities with >1:", (counts > 1).sum())
print("Entities with >2:", (counts > 2).sum())
print("Entities with >3:", (counts > 3).sum())

# Show examples with multiple candidates
multi_ids = counts[
    counts > 1
].sort_values(ascending=False).head(10).index

print("\n========== EXAMPLES WITH MULTIPLE NEW MATCHES ==========")

for s1_id in multi_ids:
    rows = s3_add[
        s3_add["entity_id_s1"] == s1_id
    ]

    print("\nS1:", s1_id)

    print(
        rows[
            [
                "entity_id_matched",
                "name_ratio",
                "name_token_ratio",
                "address_ratio",
                "address_token_ratio"
            ]
        ].to_string(index=False)
    )

========== S3 ADDITION CANDIDATES ==========
New candidates: 12431
S1 entities: 11777

Multiplicity:
Mean: 1.0555319690922986
Median: 1.0
90th: 1.0
95th: 2.0
99th: 2.0
Maximum: 3

Entities with >1: 620
Entities with >2: 34
Entities with >3: 0

========== EXAMPLES WITH MULTIPLE NEW MATCHES ==========

S1: S1-950754887
entity_id_matched  name_ratio  name_token_ratio  address_ratio  address_token_ratio
     S3-287896475       100.0             100.0      94.736842            94.736842
     S3-163762748       100.0             100.0      94.736842            94.736842
     S3-936260280       100.0             100.0      94.736842            94.736842

S1: S1-132698630
entity_id_matched  name_ratio  name_token_ratio  address_ratio  address_token_ratio
     S3-862845777  100.000000        100.000000      94.339623            94.339623
      S3-72585623   97.435897         97.435897      94.339623            94.339623
     S3-323331514  100.000000        100.000000      94.339623            9

In [70]:
# ============================================
# SELECT UNAMBIGUOUS S2 + S3 ADDITIONS
# ============================================

# ---------- S2 ----------
s2_counts = s2_add.groupby("entity_id_s1").size()

s2_unambiguous = s2_add[
    s2_add["entity_id_s1"].map(s2_counts) == 1
].copy()

# ---------- S3 ----------
s3_counts = s3_add.groupby("entity_id_s1").size()

s3_unambiguous = s3_add[
    s3_add["entity_id_s1"].map(s3_counts) == 1
].copy()

print("========== UNAMBIGUOUS ADDITIONS ==========")

print("\nS2:")
print("Pairs:", len(s2_unambiguous))
print("S1 entities:", s2_unambiguous["entity_id_s1"].nunique())

print("\nS3:")
print("Pairs:", len(s3_unambiguous))
print("S1 entities:", s3_unambiguous["entity_id_s1"].nunique())

print("\nTotal new pairs:",
      len(s2_unambiguous) + len(s3_unambiguous))

print(
    "Total new S1 entities:",
    len(
        set(s2_unambiguous["entity_id_s1"])
        |
        set(s3_unambiguous["entity_id_s1"])
    )
)

========== UNAMBIGUOUS ADDITIONS ==========

S2:
Pairs: 16776
S1 entities: 16776

S3:
Pairs: 11157
S1 entities: 11157

Total new pairs: 27933
Total new S1 entities: 27309


In [71]:
# ============================================
# BUILD IMPROVED MATCHING RESULTS
# ============================================

# Start from the original strict matches
baseline_matches = pd.concat([
    s2_hc_strict[["entity_id_s1", "entity_id_matched"]],
    s3_hc_strict[["entity_id_s1", "entity_id_matched"]]
], ignore_index=True)

# Add only unambiguous high-confidence address matches
additional_matches = pd.concat([
    s2_unambiguous[["entity_id_s1", "entity_id_matched"]],
    s3_unambiguous[["entity_id_s1", "entity_id_matched"]]
], ignore_index=True)

# Combine
improved_matches = pd.concat(
    [baseline_matches, additional_matches],
    ignore_index=True
)

# Remove any accidental duplicate pair
improved_matches = improved_matches.drop_duplicates(
    subset=["entity_id_s1", "entity_id_matched"]
).reset_index(drop=True)

print("========== IMPROVED MATCH SET ==========")
print("Baseline pairs:", len(baseline_matches))
print("Additional pairs:", len(additional_matches))
print("Final unique pairs:", len(improved_matches))

# Group by S1
improved_grouped = (
    improved_matches
    .groupby("entity_id_s1")["entity_id_matched"]
    .apply(lambda x: ",".join(pd.unique(x)))
)

# Start from EVERY test S1
improved_results = test_s1[["entity_id"]].copy()

improved_results = improved_results.rename(
    columns={"entity_id": "source1_entity_id"}
)

improved_results["matched_entity_ids"] = (
    improved_results["source1_entity_id"]
    .map(improved_grouped)
    .fillna("")
)

IMPROVED_OUTPUT = "/content/matching_results_improved.tsv"

improved_results.to_csv(
    IMPROVED_OUTPUT,
    sep="\t",
    index=False
)

print("\nRows:", len(improved_results))
print(
    "S1 with matches:",
    (improved_results["matched_entity_ids"] != "").sum()
)
print(
    "S1 with no matches:",
    (improved_results["matched_entity_ids"] == "").sum()
)

print("\nSaved:", IMPROVED_OUTPUT)

========== IMPROVED MATCH SET ==========
Baseline pairs: 535317
Additional pairs: 27933
Final unique pairs: 563250

Rows: 1732544
S1 with matches: 425109
S1 with no matches: 1307435

Saved: /content/matching_results_improved.tsv


In [ ]:
# ============================================
# BUILD IMPROVED CANDIDATE PAIRS
# ============================================

candidate_files = [
    "/content/test_s2_exact_candidates.tsv",
    "/content/test_s2_address_candidates.tsv",
    "/content/test_s3_exact_candidates.tsv",
    "/content/test_s3_address_candidates.tsv"
]

candidate_parts = []

for path in candidate_files:
    print("Reading:", path)

    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str
    )

    candidate_parts.append(
        df[["entity_id_s1", "entity_id_matched", "source"]]
    )

    print("Rows:", len(df))

# Combine all candidates
improved_candidates = pd.concat(
    candidate_parts,
    ignore_index=True
)

print("\nCombined candidates:", len(improved_candidates))

# Remove duplicate candidate pairs
improved_candidates = improved_candidates.drop_duplicates(
    subset=["entity_id_s1", "entity_id_matched", "source"]
).reset_index(drop=True)

print("Unique candidates:", len(improved_candidates))

# Save
IMPROVED_CANDIDATE_OUTPUT = "/content/candidate_pairs_improved.tsv"

improved_candidates.to_csv(
    IMPROVED_CANDIDATE_OUTPUT,
    sep="\t",
    index=False
)

print("\nSaved:", IMPROVED_CANDIDATE_OUTPUT)

Reading: /content/test_s2_exact_candidates.tsv
Rows: 6079910
Reading: /content/test_s2_address_candidates.tsv
Rows: 1408537
Reading: /content/test_s3_exact_candidates.tsv
Rows: 7065869
Reading: /content/test_s3_address_candidates.tsv
Rows: 1246731


In [1]:
# Check whether the candidate files survived the runtime crash

!ls -lh \
/content/test_s2_exact_candidates.tsv \
/content/test_s2_address_candidates.tsv \
/content/test_s3_exact_candidates.tsv \
/content/test_s3_address_candidates.tsv \
/content/matching_results_improved.tsv

-rw-r--r-- 1 root root  30M Sep 27 14:36 /content/matching_results_improved.tsv
-rw-r--r-- 1 root root  39M Sep 27 10:13 /content/test_s2_address_candidates.tsv
-rw-r--r-- 1 root root 167M Sep 27 09:51 /content/test_s2_exact_candidates.tsv
-rw-r--r-- 1 root root  35M Sep 27 10:44 /content/test_s3_address_candidates.tsv
-rw-r--r-- 1 root root 194M Sep 27 10:31 /content/test_s3_exact_candidates.tsv


In [2]:
%%bash

# ============================================
# BUILD IMPROVED CANDIDATE FILE - LOW RAM
# ============================================

OUT="/content/candidate_pairs_improved.tsv"
TMP="/content/candidate_pairs_data.tmp"

# Remove old partial files if any
rm -f "$OUT" "$TMP"

# Keep header only once
head -n 1 /content/test_s2_exact_candidates.tsv > "$OUT"

# Combine all candidate rows, skipping headers
tail -n +2 /content/test_s2_exact_candidates.tsv >> "$TMP"
tail -n +2 /content/test_s2_address_candidates.tsv >> "$TMP"
tail -n +2 /content/test_s3_exact_candidates.tsv >> "$TMP"
tail -n +2 /content/test_s3_address_candidates.tsv >> "$TMP"

echo "Combined raw candidate rows:"
wc -l "$TMP"

# Remove duplicate candidate rows using disk-based sorting
# -T specifies /content as temporary storage
LC_ALL=C sort -u -T /content "$TMP" >> "$OUT"

# Remove temporary file
rm -f "$TMP"

echo ""
echo "Final candidate file:"
ls -lh "$OUT"

echo ""
echo "Final candidate rows including header:"
wc -l "$OUT"

Combined raw candidate rows:
15801047 /content/candidate_pairs_data.tmp

Final candidate file:
-rw-r--r-- 1 root root 423M Sep 27 14:40 /content/candidate_pairs_improved.tsv

Final candidate rows including header:
15398265 /content/candidate_pairs_improved.tsv


In [3]:
%%bash

echo "============================================"
echo "1. CHECK MATCHING RESULTS STRUCTURE"
echo "============================================"

head -n 1 /content/matching_results_improved.tsv
echo "Matching rows including header:"
wc -l /content/matching_results_improved.tsv

echo ""
echo "============================================"
echo "2. CHECK CANDIDATE STRUCTURE"
echo "============================================"

head -n 1 /content/candidate_pairs_improved.tsv
echo "Candidate rows including header:"
wc -l /content/candidate_pairs_improved.tsv

echo ""
echo "============================================"
echo "3. CHECK DUPLICATE S1 ROWS"
echo "============================================"

awk -F'\t' 'NR>1 {print $1}' \
/content/matching_results_improved.tsv \
| sort -T /content \
| uniq -d \
| wc -l

echo "(Expected: 0)"

echo ""
echo "============================================"
echo "4. CHECK DUPLICATE MATCHED IDs WITHIN ROW"
echo "============================================"

awk -F'\t' '
NR > 1 && $2 != "" {
    n = split($2, a, ",")
    for(i=1; i<=n; i++) {
        if(a[i] in seen) {
            bad++
            break
        }
        seen[a[i]] = 1
    }
    delete seen
}
END {
    print "Rows with duplicate matched IDs:", bad+0
}' /content/matching_results_improved.tsv

echo ""
echo "============================================"
echo "5. CHECK MATCHED PAIRS ARE IN CANDIDATE FILE"
echo "============================================"

# Convert matching results into:
# entity_id_s1 <TAB> entity_id_matched <TAB> source
awk -F'\t' '
NR > 1 && $2 != "" {
    n = split($2, a, ",")
    for(i=1; i<=n; i++) {
        if(a[i] ~ /^S2-/)
            print $1 "\t" a[i] "\tS2"
        else if(a[i] ~ /^S3-/)
            print $1 "\t" a[i] "\tS3"
    }
}' /content/matching_results_improved.tsv \
| sort -u -T /content \
> /content/submitted_pairs_check.tsv

echo "Submitted unique pairs:"
wc -l /content/submitted_pairs_check.tsv

# Extract candidate pairs and sort
tail -n +2 /content/candidate_pairs_improved.tsv \
| sort -u -T /content \
> /content/candidate_pairs_check.tsv

echo "Candidate unique pairs:"
wc -l /content/candidate_pairs_check.tsv

# Find submitted pairs that are NOT candidates
comm -23 \
    /content/submitted_pairs_check.tsv \
    /content/candidate_pairs_check.tsv \
> /content/missing_candidate_pairs.txt

echo "Submitted pairs missing from candidate file:"
wc -l /content/missing_candidate_pairs.txt

echo ""
echo "============================================"
echo "VALIDATION COMPLETE"
echo "============================================"

1. CHECK MATCHING RESULTS STRUCTURE
source1_entity_id	matched_entity_ids
Matching rows including header:
1732545 /content/matching_results_improved.tsv

2. CHECK CANDIDATE STRUCTURE
entity_id_s1	entity_id_matched	source
Candidate rows including header:
15398265 /content/candidate_pairs_improved.tsv

3. CHECK DUPLICATE S1 ROWS
0
(Expected: 0)

4. CHECK DUPLICATE MATCHED IDs WITHIN ROW
Rows with duplicate matched IDs: 0

5. CHECK MATCHED PAIRS ARE IN CANDIDATE FILE
Submitted unique pairs:
563250 /content/submitted_pairs_check.tsv
Candidate unique pairs:
15398264 /content/candidate_pairs_check.tsv
Submitted pairs missing from candidate file:
0 /content/missing_candidate_pairs.txt

VALIDATION COMPLETE


In [4]:
%%bash

cp /content/matching_results_improved.tsv \
   /content/matching_results.tsv

cp /content/candidate_pairs_improved.tsv \
   /content/candidate_pairs.tsv

echo "Submission files:"
ls -lh /content/matching_results.tsv /content/candidate_pairs.tsv

Submission files:
-rw-r--r-- 1 root root 423M Sep 27 14:45 /content/candidate_pairs.tsv
-rw-r--r-- 1 root root  30M Sep 27 14:45 /content/matching_results.tsv


In [5]:
from google.colab import files

files.download("/content/matching_results.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
from google.colab import files

files.download("/content/candidate_pairs.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
!pip install Unidecode -q

from unidecode import unidecode

print("Unidecode imported successfully")

# Check our important files
import os

for f in [
    "/content/test_source1.tsv",
    "/content/test_source2.tsv",
    "/content/test_source3.tsv",
    "/content/matching_results.tsv",
    "/content/candidate_pairs.tsv"
]:
    print(f, "->", os.path.exists(f))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.6 MB/s eta 0:00:00
Unidecode imported successfully
/content/test_source1.tsv -> True
/content/test_source2.tsv -> True
/content/test_source3.tsv -> True
/content/matching_results.tsv -> True
/content/candidate_pairs.tsv -> True


In [8]:
# ============================================
# STEP 2 — INDIA S1 RECORDS
# ============================================

import pandas as pd
import numpy as np
from unidecode import unidecode

# Load only S1
test_s1_india = test_s1[
    test_s1["country"].fillna("").str.lower().eq("india")
].copy()

print("India S1 records:", len(test_s1_india))

# Create transliterated normalized name
def transliterate_name(text):
    if pd.isna(text):
        return ""

    text = unidecode(str(text))
    return normalize_name_v2(text)

test_s1_india["trans_name"] = (
    test_s1_india["business_name"]
    .map(transliterate_name)
)

# Only useful non-empty names
test_s1_india = test_s1_india[
    test_s1_india["trans_name"] != ""
].copy()

print("India S1 with usable names:", len(test_s1_india))
print("Unique transliterated names:",
      test_s1_india["trans_name"].nunique())

print("\nSample:")
print(
    test_s1_india[
        ["entity_id", "business_name", "trans_name"]
    ].head(15).to_string(index=False)
)

NameError: name 'test_s1' is not defined

In [9]:
# ============================================
# RELOAD TEST SOURCE 1
# ============================================

import pandas as pd
import numpy as np
from unidecode import unidecode

test_s1 = pd.read_csv(
    "/content/test_source1.tsv",
    sep="\t",
    dtype=str
)

print("test_s1 shape:", test_s1.shape)
print("\nColumns:")
print(test_s1.columns.tolist())

print("\nCountry distribution:")
print(test_s1["country"].value_counts())

test_s1 shape: (1732544, 4)

Columns:
['entity_id', 'business_name', 'business_address', 'country']

Country distribution:
country
India     809986
US        663106
France    259452
Name: count, dtype: int64


In [10]:
# ============================================
# STEP 3 — PREPARE INDIA S1 TRANSLITERATION
# ============================================

test_s1_india = test_s1[
    test_s1["country"].fillna("").str.lower().eq("india")
].copy()

def transliterate_name(text):
    if pd.isna(text):
        return ""

    text = unidecode(str(text))
    return normalize_name_v2(text)

test_s1_india["trans_name"] = (
    test_s1_india["business_name"]
    .map(transliterate_name)
)

# Keep only usable names
test_s1_india = test_s1_india[
    test_s1_india["trans_name"] != ""
].copy()

print("India S1 records:", len(test_s1_india))
print("Usable names:", len(test_s1_india))
print(
    "Unique transliterated names:",
    test_s1_india["trans_name"].nunique()
)

print("\nSample transliterations:")
print(
    test_s1_india[
        ["entity_id", "business_name", "trans_name"]
    ].head(20).to_string(index=False)
)

NameError: name 'normalize_name_v2' is not defined

In [11]:
# ============================================
# RESTORE NAME NORMALIZATION + TRANSLITERATION
# ============================================

import re
import unicodedata
import pandas as pd
from unidecode import unidecode

def normalize_name_v2(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def transliterate_name(text):
    if pd.isna(text):
        return ""

    text = unidecode(str(text))
    return normalize_name_v2(text)


# ============================================
# CREATE INDIA S1 TRANSLITERATED NAMES
# ============================================

test_s1_india = test_s1[
    test_s1["country"].fillna("").str.lower().eq("india")
].copy()

test_s1_india["trans_name"] = (
    test_s1_india["business_name"]
    .map(transliterate_name)
)

test_s1_india = test_s1_india[
    test_s1_india["trans_name"] != ""
].copy()

print("India S1 records:", len(test_s1_india))
print("Usable names:", len(test_s1_india))
print(
    "Unique transliterated names:",
    test_s1_india["trans_name"].nunique()
)

print("\nSample:")
print(
    test_s1_india[
        ["entity_id", "business_name", "trans_name"]
    ].head(20).to_string(index=False)
)

India S1 records: 809986
Usable names: 809986
Unique transliterated names: 518593

Sample:
   entity_id                        business_name                           trans_name
S1-689823050                  Red Perfect Trading                  red perfect trading
S1-481669221                    Nandlal Kisan LLP                    nandlal kisan llp
S1-280204013             Om Constructions Pvt Ltd             om constructions pvt ltd
S1-742053041                        Roongta Sangh                        roongta sangh
S1-479574617              Anupama Estates Pvt Ltd              anupama estates pvt ltd
S1-680637447               Siliguri Media Pvt Ltd               siliguri media pvt ltd
S1-870190130                          Naman Trust                          naman trust
S1-365170694 Consulting Sai Nanak Private Limited consulting sai nanak private limited
S1-378191895       East Marketing Private Limited       east marketing private limited
S1-750279127                           

In [12]:
# ============================================
# STEP 4 — CHECK ACTUAL NON-LATIN NAMES
# ============================================

# Original normalized name without transliteration
test_s1_india["normal_name"] = (
    test_s1_india["business_name"]
    .map(normalize_name_v2)
)

# Names where transliteration produces something different
changed = test_s1_india[
    test_s1_india["trans_name"] != test_s1_india["normal_name"]
].copy()

print("Total India S1:", len(test_s1_india))
print("Names changed by transliteration:", len(changed))
print(
    "Percentage changed:",
    round(len(changed) / len(test_s1_india) * 100, 3),
    "%"
)

print(
    "\nUnique changed transliterated names:",
    changed["trans_name"].nunique()
)

print("\nExamples:")
print(
    changed[
        ["entity_id", "business_name", "normal_name", "trans_name"]
    ].head(40).to_string(index=False)
)

Total India S1: 809986
Names changed by transliteration: 0
Percentage changed: 0.0 %

Unique changed transliterated names: 0

Examples:
Empty DataFrame
Columns: [entity_id, business_name, normal_name, trans_name]
Index: []


In [13]:
# ============================================
# EXPERIMENT: STRICT TYPO-TOLERANT MATCHES
# ============================================

import pandas as pd
from rapidfuzz.fuzz import ratio, token_set_ratio

print("RapidFuzz ready.")
print("Next: we will score the existing address candidates.")

RapidFuzz ready.
Next: we will score the existing address candidates.


In [14]:
# ============================================
# S2 STRICT TYPO-TOLERANT EXPERIMENT
# Rule:
#   name_ratio >= 90
#   AND
#   address_ratio >= 95
# ============================================

import pandas as pd
from rapidfuzz.fuzz import ratio

S2_ADDRESS_FILE = "/content/test_s2_address_candidates.tsv"
S2_OUTPUT_FILE = "/content/s2_typo_candidates.tsv"

# Load S1 records needed by the address candidates
s1_lookup = test_s1[
    ["entity_id", "business_name", "business_address", "country"]
].copy()

s1_lookup = s1_lookup.set_index("entity_id")

# Remove previous output
import os
if os.path.exists(S2_OUTPUT_FILE):
    os.remove(S2_OUTPUT_FILE)

first_write = True
total_scored = 0
total_kept = 0

for chunk in pd.read_csv(
    S2_ADDRESS_FILE,
    sep="\t",
    dtype=str,
    chunksize=100_000
):

    # Attach S1 information
    chunk = chunk.join(
        s1_lookup,
        on="entity_id_s1",
        rsuffix="_s1"
    )

    # Normalize names and addresses
    chunk["s1_name"] = chunk["business_name"].fillna("").map(normalize_name_v2)
    chunk["s2_name"] = chunk["entity_id_matched"].fillna("")

    # We need the actual S2 name/address.
    # These are not stored in the candidate file, so retrieve them
    # from the S2 source only for IDs in this chunk.

    ids = chunk["entity_id_matched"].dropna().unique()

    s2_part = pd.read_csv(
        "/content/test_source2.tsv",
        sep="\t",
        dtype=str,
        usecols=["entity_id", "business_name", "business_address"],
        nrows=0
    )

    # This placeholder is replaced in the next optimized step.
    # Stop before doing an expensive full-source scan.
    break

print("S2 candidate file is available.")
print("We will use an optimized lookup approach next.")

S2 candidate file is available.
We will use an optimized lookup approach next.


In [15]:
# ============================================
# BUILD S2 LOOKUP FOR ADDRESS CANDIDATES
# ============================================

import pandas as pd
import os

CANDIDATE_FILE = "/content/test_s2_address_candidates.tsv"
S2_FILE = "/content/test_source2.tsv"
LOOKUP_FILE = "/content/s2_typo_lookup.tsv"

# --------------------------------------------
# 1. Get the S2 IDs that actually appear
#    in our address candidate file
# --------------------------------------------

needed_ids = set()

for chunk in pd.read_csv(
    CANDIDATE_FILE,
    sep="\t",
    dtype=str,
    usecols=["entity_id_matched"],
    chunksize=200_000
):
    needed_ids.update(
        chunk["entity_id_matched"].dropna()
    )

print("S2 IDs needed:", len(needed_ids))

# --------------------------------------------
# 2. Scan S2 source ONCE and extract them
# --------------------------------------------

if os.path.exists(LOOKUP_FILE):
    os.remove(LOOKUP_FILE)

first_write = True
found = 0

for chunk in pd.read_csv(
    S2_FILE,
    sep="\t",
    dtype=str,
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ],
    chunksize=200_000
):

    matched = chunk[
        chunk["entity_id"].isin(needed_ids)
    ]

    if len(matched) > 0:
        matched.to_csv(
            LOOKUP_FILE,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False
        found += len(matched)

print("\nS2 records found:", found)
print("Missing S2 IDs:", len(needed_ids) - found)
print("Lookup file:", LOOKUP_FILE)

if os.path.exists(LOOKUP_FILE):
    print(
        "Lookup size:",
        round(os.path.getsize(LOOKUP_FILE) / (1024**2), 2),
        "MB"
    )

S2 IDs needed: 801969

S2 records found: 801969
Missing S2 IDs: 0
Lookup file: /content/s2_typo_lookup.tsv
Lookup size: 95.69 MB


In [16]:
# ============================================
# SCORE S2 ADDRESS CANDIDATES
# ============================================

from rapidfuzz.fuzz import ratio
import pandas as pd
import os

# ---------- Load S2 lookup ----------
s2_lookup = pd.read_csv(
    "/content/s2_typo_lookup.tsv",
    sep="\t",
    dtype=str
)

s2_lookup = s2_lookup.rename(
    columns={
        "entity_id": "entity_id_matched",
        "business_name": "s2_business_name",
        "business_address": "s2_business_address",
        "country": "s2_country"
    }
)

# ---------- Load candidate file ----------
candidates = pd.read_csv(
    "/content/test_s2_address_candidates.tsv",
    sep="\t",
    dtype=str
)

print("Candidate rows:", len(candidates))

# ---------- Attach S2 data ----------
candidates = candidates.merge(
    s2_lookup,
    on="entity_id_matched",
    how="left"
)

# ---------- Attach S1 data ----------
s1_data = test_s1[
    ["entity_id", "business_name", "business_address", "country"]
].rename(
    columns={
        "entity_id": "entity_id_s1",
        "business_name": "s1_business_name",
        "business_address": "s1_business_address",
        "country": "s1_country"
    }
)

candidates = candidates.merge(
    s1_data,
    on="entity_id_s1",
    how="left"
)

print("After lookup:", len(candidates))

# ---------- Normalize ----------
candidates["s1_name_norm"] = (
    candidates["s1_business_name"]
    .fillna("")
    .map(normalize_name_v2)
)

candidates["s2_name_norm"] = (
    candidates["s2_business_name"]
    .fillna("")
    .map(normalize_name_v2)
)

candidates["s1_address_norm"] = (
    candidates["s1_business_address"]
    .fillna("")
    .map(normalize_address)
)

candidates["s2_address_norm"] = (
    candidates["s2_business_address"]
    .fillna("")
    .map(normalize_address)
)

# ---------- Calculate ratios ----------
candidates["name_ratio"] = [
    ratio(a, b)
    for a, b in zip(
        candidates["s1_name_norm"],
        candidates["s2_name_norm"]
    )
]

candidates["address_ratio"] = [
    ratio(a, b)
    for a, b in zip(
        candidates["s1_address_norm"],
        candidates["s2_address_norm"]
    )
]

print("\nScoring complete.")

# ---------- Strict typo-tolerant rule ----------
s2_typo = candidates[
    (candidates["name_ratio"] >= 90) &
    (candidates["address_ratio"] >= 95)
].copy()

print("\n========== STRICT TYPO CANDIDATES ==========")
print("Candidates kept:", len(s2_typo))
print(
    "S1 entities:",
    s2_typo["entity_id_s1"].nunique()
)

# ---------- Remove existing baseline pairs ----------
existing_s2 = set(
    zip(
        s2_hc_strict["entity_id_s1"],
        s2_hc_strict["entity_id_matched"]
    )
)

s2_typo["pair"] = list(
    zip(
        s2_typo["entity_id_s1"],
        s2_typo["entity_id_matched"]
    )
)

s2_typo_new = s2_typo[
    ~s2_typo["pair"].isin(existing_s2)
].copy()

print("\nNew pairs after removing baseline:")
print("Pairs:", len(s2_typo_new))
print(
    "S1 entities:",
    s2_typo_new["entity_id_s1"].nunique()
)

# ---------- Multiplicity ----------
counts = s2_typo_new.groupby("entity_id_s1").size()

print("\nMultiplicity of NEW candidates:")
print("Mean:", round(counts.mean(), 4))
print("Median:", counts.median())
print(">1:", (counts > 1).sum())
print(">2:", (counts > 2).sum())
print(">3:", (counts > 3).sum())
print("Max:", counts.max())

print("\nTop examples:")
print(
    s2_typo_new[
        [
            "entity_id_s1",
            "entity_id_matched",
            "s1_business_name",
            "s2_business_name",
            "name_ratio",
            "address_ratio"
        ]
    ]
    .sort_values(
        ["name_ratio", "address_ratio"],
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

Candidate rows: 1408537
After lookup: 1408537


NameError: name 'normalize_address' is not defined

In [17]:
# ============================================
# RESTORE ADDRESS NORMALIZATION
# ============================================

import re
import unicodedata

def normalize_address(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ============================================
# CONTINUE S2 SCORING
# ============================================

candidates["s1_address_norm"] = (
    candidates["s1_business_address"]
    .fillna("")
    .map(normalize_address)
)

candidates["s2_address_norm"] = (
    candidates["s2_business_address"]
    .fillna("")
    .map(normalize_address)
)

# Calculate address similarity
candidates["address_ratio"] = [
    ratio(a, b)
    for a, b in zip(
        candidates["s1_address_norm"],
        candidates["s2_address_norm"]
    )
]

print("Address scoring complete.")

# ============================================
# STRICT TYPO-TOLERANT RULE
# ============================================

s2_typo = candidates[
    (candidates["name_ratio"] >= 90) &
    (candidates["address_ratio"] >= 95)
].copy()

print("\n========== STRICT TYPO CANDIDATES ==========")
print("Candidates kept:", len(s2_typo))
print("S1 entities:", s2_typo["entity_id_s1"].nunique())

# ============================================
# REMOVE EXISTING BASELINE PAIRS
# ============================================

existing_s2 = set(
    zip(
        s2_hc_strict["entity_id_s1"],
        s2_hc_strict["entity_id_matched"]
    )
)

s2_typo["pair"] = list(
    zip(
        s2_typo["entity_id_s1"],
        s2_typo["entity_id_matched"]
    )
)

s2_typo_new = s2_typo[
    ~s2_typo["pair"].isin(existing_s2)
].copy()

print("\n========== NEW PAIRS ==========")
print("Pairs:", len(s2_typo_new))
print("S1 entities:", s2_typo_new["entity_id_s1"].nunique())

# ============================================
# MULTIPLICITY
# ============================================

if len(s2_typo_new) > 0:

    counts = s2_typo_new.groupby("entity_id_s1").size()

    print("\nMultiplicity:")
    print("Mean:", round(counts.mean(), 4))
    print("Median:", counts.median())
    print(">1:", (counts > 1).sum())
    print(">2:", (counts > 2).sum())
    print(">3:", (counts > 3).sum())
    print("Max:", counts.max())

    print("\nTop examples:")
    print(
        s2_typo_new[
            [
                "entity_id_s1",
                "entity_id_matched",
                "s1_business_name",
                "s2_business_name",
                "name_ratio",
                "address_ratio"
            ]
        ]
        .sort_values(
            ["name_ratio", "address_ratio"],
            ascending=False
        )
        .head(20)
        .to_string(index=False)
    )
else:
    print("\nNo new candidates found.")

Address scoring complete.


KeyError: 'name_ratio'

In [18]:
# ============================================
# FIX: CALCULATE NAME RATIO
# ============================================

from rapidfuzz.fuzz import ratio

candidates["name_ratio"] = [
    ratio(a, b)
    for a, b in zip(
        candidates["s1_name_norm"],
        candidates["s2_name_norm"]
    )
]

print("Name scoring complete.")

# ============================================
# STRICT TYPO-TOLERANT RULE
# ============================================

s2_typo = candidates[
    (candidates["name_ratio"] >= 90) &
    (candidates["address_ratio"] >= 95)
].copy()

print("\n========== STRICT TYPO CANDIDATES ==========")
print("Candidates kept:", len(s2_typo))
print("S1 entities:", s2_typo["entity_id_s1"].nunique())

# ============================================
# REMOVE EXISTING BASELINE PAIRS
# ============================================

existing_s2 = set(
    zip(
        s2_hc_strict["entity_id_s1"],
        s2_hc_strict["entity_id_matched"]
    )
)

s2_typo["pair"] = list(
    zip(
        s2_typo["entity_id_s1"],
        s2_typo["entity_id_matched"]
    )
)

s2_typo_new = s2_typo[
    ~s2_typo["pair"].isin(existing_s2)
].copy()

print("\n========== NEW PAIRS ==========")
print("Pairs:", len(s2_typo_new))
print("S1 entities:", s2_typo_new["entity_id_s1"].nunique())

# ============================================
# MULTIPLICITY
# ============================================

if len(s2_typo_new) > 0:

    counts = s2_typo_new.groupby("entity_id_s1").size()

    print("\nMultiplicity:")
    print("Mean:", round(counts.mean(), 4))
    print("Median:", counts.median())
    print(">1:", (counts > 1).sum())
    print(">2:", (counts > 2).sum())
    print(">3:", (counts > 3).sum())
    print("Max:", counts.max())

    print("\nTop examples:")
    print(
        s2_typo_new[
            [
                "entity_id_s1",
                "entity_id_matched",
                "s1_business_name",
                "s2_business_name",
                "name_ratio",
                "address_ratio"
            ]
        ]
        .sort_values(
            ["name_ratio", "address_ratio"],
            ascending=False
        )
        .head(20)
        .to_string(index=False)
    )
else:
    print("\nNo new candidates found.")

Name scoring complete.

========== STRICT TYPO CANDIDATES ==========
Candidates kept: 74203
S1 entities: 61309


NameError: name 's2_hc_strict' is not defined

In [19]:
# ============================================
# RECONSTRUCT S2 BASELINE FROM 0.228 SUBMISSION
# ============================================

print("Reading successful matching_results.tsv...")

submitted = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

# Build baseline S2 pair set
existing_s2 = set()

for _, row in submitted.iterrows():
    matches = row["matched_entity_ids"]

    if pd.isna(matches) or matches == "":
        continue

    for entity_id in matches.split(","):
        entity_id = entity_id.strip()

        if entity_id.startswith("S2-"):
            existing_s2.add(
                (row["source1_entity_id"], entity_id)
            )

print("Existing S2 pairs in 0.228 submission:", len(existing_s2))

# ============================================
# REMOVE EXISTING PAIRS
# ============================================

s2_typo["pair"] = list(
    zip(
        s2_typo["entity_id_s1"],
        s2_typo["entity_id_matched"]
    )
)

s2_typo_new = s2_typo[
    ~s2_typo["pair"].isin(existing_s2)
].copy()

print("\n========== NEW S2 TYPO PAIRS ==========")
print("New pairs:", len(s2_typo_new))
print("S1 entities:", s2_typo_new["entity_id_s1"].nunique())

# ============================================
# MULTIPLICITY
# ============================================

if len(s2_typo_new) > 0:

    counts = s2_typo_new.groupby("entity_id_s1").size()

    print("\nMultiplicity:")
    print("Mean:", round(counts.mean(), 4))
    print("Median:", counts.median())
    print(">1:", (counts > 1).sum())
    print(">2:", (counts > 2).sum())
    print(">3:", (counts > 3).sum())
    print("Max:", counts.max())

    print("\nTop examples:")
    print(
        s2_typo_new[
            [
                "entity_id_s1",
                "entity_id_matched",
                "s1_business_name",
                "s2_business_name",
                "name_ratio",
                "address_ratio"
            ]
        ]
        .sort_values(
            ["name_ratio", "address_ratio"],
            ascending=False
        )
        .head(20)
        .to_string(index=False)
    )
else:
    print("No new pairs.")

Reading successful matching_results.tsv...
Existing S2 pairs in 0.228 submission: 361439

========== NEW S2 TYPO PAIRS ==========
New pairs: 13560
S1 entities: 12721

Multiplicity:
Mean: 1.066
Median: 1.0
>1: 799
>2: 40
>3: 0
Max: 3

Top examples:
entity_id_s1 entity_id_matched                         s1_business_name                          s2_business_name  name_ratio  address_ratio
 S1-38592824      S2-831909784                      Southern Pediatrics                       southern pediatrics  100.000000      98.701299
S1-820290062       S2-73310778       Gandhi Nagar India Private Limited       Gandhi Nagar India Private  Limited  100.000000      98.630137
S1-605085827      S2-133865276                     Lewis Capital Thayer                      Lewis Capital Thayer  100.000000      96.000000
S1-605085827      S2-154000714                     Lewis Capital Thayer                     LEWIS  CAPITAL THAYER  100.000000      96.000000
S1-341697870      S2-623077565            Ranso

In [20]:
# ============================================
# SELECT UNAMBIGUOUS TYPO MATCHES
# ============================================

s2_typo_counts = (
    s2_typo_new
    .groupby("entity_id_s1")
    .size()
)

s2_typo_unambiguous = s2_typo_new[
    s2_typo_new["entity_id_s1"].map(s2_typo_counts) == 1
].copy()

print("========== UNAMBIGUOUS TYPO MATCHES ==========")
print("Pairs:", len(s2_typo_unambiguous))
print(
    "S1 entities:",
    s2_typo_unambiguous["entity_id_s1"].nunique()
)

print("\nExcluded because ambiguous:")
print(
    "Pairs:",
    len(s2_typo_new) - len(s2_typo_unambiguous)
)
print(
    "S1 entities:",
    s2_typo_new["entity_id_s1"].nunique()
    - s2_typo_unambiguous["entity_id_s1"].nunique()
)

print("\nScore distribution:")
print(
    s2_typo_unambiguous[
        ["name_ratio", "address_ratio"]
    ].describe()
)

print("\nSample unambiguous matches:")
print(
    s2_typo_unambiguous[
        [
            "entity_id_s1",
            "entity_id_matched",
            "s1_business_name",
            "s2_business_name",
            "name_ratio",
            "address_ratio"
        ]
    ]
    .sort_values(
        ["name_ratio", "address_ratio"],
        ascending=False
    )
    .head(30)
    .to_string(index=False)
)

========== UNAMBIGUOUS TYPO MATCHES ==========
Pairs: 11922
S1 entities: 11922

Excluded because ambiguous:
Pairs: 1638
S1 entities: 799

Score distribution:
         name_ratio  address_ratio
count  11922.000000   11922.000000
mean      92.353674      96.861325
std        1.436451       1.301704
min       90.000000      95.000000
25%       91.176471      95.774648
50%       92.307692      96.666667
75%       93.548387      97.894737
max      100.000000     100.000000

Sample unambiguous matches:
entity_id_s1 entity_id_matched                                 s1_business_name                                       s2_business_name  name_ratio  address_ratio
 S1-38592824      S2-831909784                              Southern Pediatrics                                    southern pediatrics  100.000000      98.701299
 S1-57248857      S2-685357144            Ahmedabad Consultants Private Limited                   AHMEDABAD CONSULTATS-PRIVATE LIMITED   98.630137      95.890411
S1-974646931

In [21]:
# ============================================
# S2 TYPO CONFIDENCE TIERS
# ============================================

print("========== CONFIDENCE TIERS ==========")

tiers = {
    "Very High: name >= 96, address >= 97":
        (96, 97),

    "High: name >= 95, address >= 96":
        (95, 96),

    "Medium-High: name >= 94, address >= 96":
        (94, 96),

    "Current: name >= 90, address >= 95":
        (90, 95),
}

for label, (name_thr, addr_thr) in tiers.items():

    subset = s2_typo_unambiguous[
        (s2_typo_unambiguous["name_ratio"] >= name_thr) &
        (s2_typo_unambiguous["address_ratio"] >= addr_thr)
    ]

    print(f"\n{label}")
    print("Pairs:", len(subset))
    print("S1 entities:", subset["entity_id_s1"].nunique())

========== CONFIDENCE TIERS ==========

Very High: name >= 96, address >= 97
Pairs: 26
S1 entities: 26

High: name >= 95, address >= 96
Pairs: 51
S1 entities: 51

Medium-High: name >= 94, address >= 96
Pairs: 1219
S1 entities: 1219

Current: name >= 90, address >= 95
Pairs: 11922
S1 entities: 11922


In [22]:
# ============================================
# BUILD TYPO EXPERIMENT
# ============================================

s2_typo_selected = s2_typo_unambiguous[
    (s2_typo_unambiguous["name_ratio"] >= 94) &
    (s2_typo_unambiguous["address_ratio"] >= 96)
].copy()

print("Selected new S2 pairs:", len(s2_typo_selected))
print(
    "Selected S1 entities:",
    s2_typo_selected["entity_id_s1"].nunique()
)

# Make sure there are no duplicates
s2_typo_selected = s2_typo_selected.drop_duplicates(
    subset=["entity_id_s1", "entity_id_matched"]
)

print("After pair deduplication:", len(s2_typo_selected))

# Show the weakest selected matches
print("\nLowest selected scores:")
print(
    s2_typo_selected[
        [
            "entity_id_s1",
            "entity_id_matched",
            "s1_business_name",
            "s2_business_name",
            "name_ratio",
            "address_ratio"
        ]
    ]
    .sort_values(
        ["name_ratio", "address_ratio"]
    )
    .head(20)
    .to_string(index=False)
)

Selected new S2 pairs: 1219
Selected S1 entities: 1219
After pair deduplication: 1219

Lowest selected scores:
entity_id_s1 entity_id_matched                                   s1_business_name                                   s2_business_name  name_ratio  address_ratio
S1-292301896      S2-941693911 Padma Technology of Pathanamthitta Private Limited Padma Techno1ogy of Pathanamthitta Poihate Limited   94.000000      99.259259
S1-225938589      S2-671253441               Integrated Marketing Innovations LLC                   Integrated Marketing Innovations   94.117647      96.000000
S1-718932256      S2-733815621                          Environmental Society LLC                         Environmental Sacoiety LLC   94.117647      96.000000
S1-415313483      S2-273165520                                   SRP Ventures LLP                                SRP Ventures L.L.P.   94.117647      96.000000
S1-961196293      S2-350352707                           Pediatric Care of Zillah        

In [23]:
# ============================================
# BUILD VERSION 2 — 0.228 + 1,219 TYPO MATCHES
# ============================================

import pandas as pd

BASELINE_FILE = "/content/matching_results.tsv"
NEW_OUTPUT = "/content/matching_results_v2.tsv"

# Load the successful 0.228 submission
baseline = pd.read_csv(
    BASELINE_FILE,
    sep="\t",
    dtype=str
)

# Create lookup of the new S2 matches
new_pairs = (
    s2_typo_selected
    [["entity_id_s1", "entity_id_matched"]]
    .drop_duplicates()
)

new_lookup = (
    new_pairs
    .groupby("entity_id_s1")["entity_id_matched"]
    .apply(lambda x: ",".join(x))
)

# Add the new S2 IDs to the existing matching results
def add_matches(row):
    existing = row["matched_entity_ids"]
    new = new_lookup.get(row["source1_entity_id"], "")

    if not new:
        return existing

    if pd.isna(existing) or existing == "":
        return new

    existing_ids = existing.split(",")
    new_ids = new.split(",")

    combined = existing_ids + [
        x for x in new_ids
        if x not in existing_ids
    ]

    return ",".join(combined)

baseline["matched_entity_ids"] = baseline.apply(
    add_matches,
    axis=1
)

# Save
baseline.to_csv(
    NEW_OUTPUT,
    sep="\t",
    index=False
)

print("Saved:", NEW_OUTPUT)
print("Rows:", len(baseline))

# Count total submitted pairs
total_pairs = 0

for value in baseline["matched_entity_ids"]:
    if pd.notna(value) and value != "":
        total_pairs += len(value.split(","))

print("Total matched pairs:", total_pairs)

print(
    "S1 entities with matches:",
    (baseline["matched_entity_ids"] != "").sum()
)

Saved: /content/matching_results_v2.tsv
Rows: 1732544
Total matched pairs: 564469
S1 entities with matches: 1732544


In [24]:
from google.colab import files

files.download("/content/matching_results_v2.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
from google.colab import files

files.download("/content/matching_results_v2.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
# ============================================
# VERIFY 1,219 NEW MATCHES ARE IN CANDIDATES
# ============================================

# Save the 1,219 selected pairs to a small file
check_file = "/content/s2_typo_selected_check.tsv"

s2_typo_selected[
    ["entity_id_s1", "entity_id_matched"]
].drop_duplicates().to_csv(
    check_file,
    sep="\t",
    index=False
)

print("New pairs to verify:", len(s2_typo_selected))

# Use disk-based sorting; do NOT load the 423 MB
# candidate file into pandas.
!tail -n +2 /content/candidate_pairs.tsv \
    | cut -f1,2 \
    | sort -u -T /content \
    > /content/all_candidate_pairs_check.tsv

!tail -n +2 /content/s2_typo_selected_check.tsv \
    | sort -u -T /content \
    > /content/new_typo_pairs_check.tsv

# Find selected pairs missing from candidate file
!comm -23 \
    /content/new_typo_pairs_check.tsv \
    /content/all_candidate_pairs_check.tsv \
    > /content/missing_typo_candidates.tsv

print(
    "New pairs missing from candidate file:",
    sum(1 for _ in open("/content/missing_typo_candidates.tsv"))
)

New pairs to verify: 1219
New pairs missing from candidate file: 0


In [27]:
# ============================================
# FINAL VALIDATION — VERSION 2
# ============================================

import pandas as pd

v2 = pd.read_csv(
    "/content/matching_results_v2.tsv",
    sep="\t",
    dtype=str
)

print("========== BASIC CHECKS ==========")

print("Rows:", len(v2))
print("Expected rows:", 1732544)

print("\nColumns:", v2.columns.tolist())

# Duplicate S1 rows
duplicate_s1 = v2["source1_entity_id"].duplicated().sum()
print("\nDuplicate S1 rows:", duplicate_s1)

# Empty / non-empty
matched_count = (v2["matched_entity_ids"].fillna("") != "").sum()
empty_count = len(v2) - matched_count

print("S1 with matches:", matched_count)
print("S1 with no matches:", empty_count)

# Duplicate matched IDs inside individual rows
duplicate_match_rows = 0

for value in v2["matched_entity_ids"].fillna(""):
    if value == "":
        continue

    ids = value.split(",")

    if len(ids) != len(set(ids)):
        duplicate_match_rows += 1

print("Rows with duplicate matched IDs:", duplicate_match_rows)

# Total pairs
total_pairs = sum(
    len(value.split(","))
    for value in v2["matched_entity_ids"].fillna("")
    if value != ""
)

print("\nTotal matched pairs:", total_pairs)

print("\n========== EXPECTED ==========")
print("Rows: 1732544")
print("Duplicate S1 rows: 0")
print("Duplicate matched IDs: 0")
print("Total pairs: 564469")

========== BASIC CHECKS ==========
Rows: 1732544
Expected rows: 1732544

Columns: ['source1_entity_id', 'matched_entity_ids']

Duplicate S1 rows: 0
S1 with matches: 425843
S1 with no matches: 1306701
Rows with duplicate matched IDs: 0

Total matched pairs: 564469

========== EXPECTED ==========
Rows: 1732544
Duplicate S1 rows: 0
Duplicate matched IDs: 0
Total pairs: 564469


In [28]:
import shutil

shutil.copy(
    "/content/matching_results_v2.tsv",
    "/content/matching_results.tsv"
)

print("Ready:")
!ls -lh /content/matching_results.tsv /content/candidate_pairs.tsv

Ready:
-rw-r--r-- 1 root root 423M Sep 27 14:45 /content/candidate_pairs.tsv
-rw-r--r-- 1 root root  30M Sep 27 15:58 /content/matching_results.tsv


In [29]:
from google.colab import files

files.download("/content/matching_results.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:
import os

files_to_check = [
    "/content/test_s3_address_candidates.tsv",
    "/content/s3_address_lookup.tsv",
    "/content/test_source3.tsv",
]

for f in files_to_check:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"{f} -> {size_mb:.2f} MB")
    else:
        print(f"{f} -> NOT FOUND")

/content/test_s3_address_candidates.tsv -> 34.21 MB
/content/s3_address_lookup.tsv -> 83.01 MB
/content/test_source3.tsv -> 482.56 MB


In [31]:
# ============================================
# S3 TYPO-TOLERANT EXPERIMENT
# ============================================

import pandas as pd
from rapidfuzz.fuzz import ratio

# ---------- Load S3 lookup ----------
s3_lookup = pd.read_csv(
    "/content/s3_address_lookup.tsv",
    sep="\t",
    dtype=str
)

s3_lookup = s3_lookup.rename(
    columns={
        "entity_id": "entity_id_matched",
        "business_name": "s3_business_name",
        "business_address": "s3_business_address",
        "country": "s3_country"
    }
)

print("S3 lookup rows:", len(s3_lookup))

# ---------- Load address candidates ----------
s3_candidates = pd.read_csv(
    "/content/test_s3_address_candidates.tsv",
    sep="\t",
    dtype=str
)

print("S3 candidate rows:", len(s3_candidates))

# ---------- Attach S3 data ----------
s3_candidates = s3_candidates.merge(
    s3_lookup,
    on="entity_id_matched",
    how="left"
)

# ---------- Attach S1 data ----------
s1_data = test_s1[
    ["entity_id", "business_name", "business_address", "country"]
].rename(
    columns={
        "entity_id": "entity_id_s1",
        "business_name": "s1_business_name",
        "business_address": "s1_business_address",
        "country": "s1_country"
    }
)

s3_candidates = s3_candidates.merge(
    s1_data,
    on="entity_id_s1",
    how="left"
)

# ---------- Normalize ----------
s3_candidates["s1_name_norm"] = (
    s3_candidates["s1_business_name"]
    .fillna("")
    .map(normalize_name_v2)
)

s3_candidates["s3_name_norm"] = (
    s3_candidates["s3_business_name"]
    .fillna("")
    .map(normalize_name_v2)
)

s3_candidates["s1_address_norm"] = (
    s3_candidates["s1_business_address"]
    .fillna("")
    .map(normalize_address)
)

s3_candidates["s3_address_norm"] = (
    s3_candidates["s3_business_address"]
    .fillna("")
    .map(normalize_address)
)

# ---------- Similarities ----------
s3_candidates["name_ratio"] = [
    ratio(a, b)
    for a, b in zip(
        s3_candidates["s1_name_norm"],
        s3_candidates["s3_name_norm"]
    )
]

s3_candidates["address_ratio"] = [
    ratio(a, b)
    for a, b in zip(
        s3_candidates["s1_address_norm"],
        s3_candidates["s3_address_norm"]
    )
]

print("\nScoring complete.")

# ---------- Strict rule ----------
s3_typo = s3_candidates[
    (s3_candidates["name_ratio"] >= 94) &
    (s3_candidates["address_ratio"] >= 96)
].copy()

print("\n========== S3 TYPO CANDIDATES ==========")
print("Candidates kept:", len(s3_typo))
print("S1 entities:", s3_typo["entity_id_s1"].nunique())

S3 lookup rows: 735497
S3 candidate rows: 1246731

Scoring complete.

========== S3 TYPO CANDIDATES ==========
Candidates kept: 10817
S1 entities: 9556


In [32]:
# ============================================
# REMOVE EXISTING S3 MATCHES + AMBIGUITIES
# ============================================

# Read the successful 0.228 submission
baseline = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

# Build existing S3 pair set
existing_s3 = set()

for _, row in baseline.iterrows():
    matches = row["matched_entity_ids"]

    if pd.isna(matches) or matches == "":
        continue

    for entity_id in matches.split(","):
        entity_id = entity_id.strip()

        if entity_id.startswith("S3-"):
            existing_s3.add(
                (row["source1_entity_id"], entity_id)
            )

print("Existing S3 pairs:", len(existing_s3))

# Create pair identifier
s3_typo["pair"] = list(
    zip(
        s3_typo["entity_id_s1"],
        s3_typo["entity_id_matched"]
    )
)

# Remove existing matches
s3_typo_new = s3_typo[
    ~s3_typo["pair"].isin(existing_s3)
].copy()

print("\n========== NEW S3 TYPO CANDIDATES ==========")
print("New pairs:", len(s3_typo_new))
print("S1 entities:", s3_typo_new["entity_id_s1"].nunique())

# Check ambiguity
counts = s3_typo_new.groupby("entity_id_s1").size()

print("\nMultiplicity:")
print("Mean:", round(counts.mean(), 4))
print("Median:", counts.median())
print(">1:", (counts > 1).sum())
print(">2:", (counts > 2).sum())
print(">3:", (counts > 3).sum())
print("Max:", counts.max())

Existing S3 pairs: 201811

========== NEW S3 TYPO CANDIDATES ==========
New pairs: 678
S1 entities: 652

Multiplicity:
Mean: 1.0399
Median: 1.0
>1: 25
>2: 1
>3: 0
Max: 3


In [33]:
# ============================================
# SELECT UNAMBIGUOUS S3 TYPO MATCHES
# ============================================

s3_counts = s3_typo_new.groupby("entity_id_s1").size()

s3_typo_unambiguous = s3_typo_new[
    s3_typo_new["entity_id_s1"].map(s3_counts) == 1
].copy()

print("========== UNAMBIGUOUS S3 TYPO MATCHES ==========")
print("Pairs:", len(s3_typo_unambiguous))
print("S1 entities:", s3_typo_unambiguous["entity_id_s1"].nunique())

print("\nScore distribution:")
print(
    s3_typo_unambiguous[
        ["name_ratio", "address_ratio"]
    ].describe()
)

print("\nLowest-scoring selected matches:")
print(
    s3_typo_unambiguous[
        [
            "entity_id_s1",
            "entity_id_matched",
            "s1_business_name",
            "s3_business_name",
            "name_ratio",
            "address_ratio"
        ]
    ]
    .sort_values(
        ["name_ratio", "address_ratio"]
    )
    .head(30)
    .to_string(index=False)
)

========== UNAMBIGUOUS S3 TYPO MATCHES ==========
Pairs: 627
S1 entities: 627

Score distribution:
       name_ratio  address_ratio
count  627.000000     627.000000
mean    94.557612      97.078849
std      0.580086       0.784619
min     94.000000      96.000000
25%     94.117647      96.437636
50%     94.444444      96.969697
75%     94.736842      97.560976
max     98.305085      99.585062

Lowest-scoring selected matches:
entity_id_s1 entity_id_matched                                  s1_business_name                                    s3_business_name  name_ratio  address_ratio
S1-129444755       S3-66807567 Technologies Prathamesh Buildwell Private Limited Technologies Phrahtafmshs Buildwell Private Limited   94.000000      99.186992
S1-730800414      S3-630975284               Solutions Akl Farms Private Limited                   Solutions Akl Farms Pribe Limited   94.117647      96.000000
S1-397673172      S3-400309171                Krishna Baba Power Private Limited          

In [34]:
# ============================================
# S3 TYPO CONFIDENCE TIERS
# ============================================

print("========== S3 CONFIDENCE TIERS ==========")

tiers = {
    "Very High: name >= 96, address >= 97":
        (96, 97),

    "High: name >= 95, address >= 96":
        (95, 96),

    "Medium-High: name >= 94.5, address >= 96":
        (94.5, 96),

    "Current: name >= 94, address >= 96":
        (94, 96),
}

for label, (name_thr, addr_thr) in tiers.items():

    subset = s3_typo_unambiguous[
        (s3_typo_unambiguous["name_ratio"] >= name_thr) &
        (s3_typo_unambiguous["address_ratio"] >= addr_thr)
    ]

    print(f"\n{label}")
    print("Pairs:", len(subset))
    print("S1 entities:", subset["entity_id_s1"].nunique())

========== S3 CONFIDENCE TIERS ==========

Very High: name >= 96, address >= 97
Pairs: 11
S1 entities: 11

High: name >= 95, address >= 96
Pairs: 36
S1 entities: 36

Medium-High: name >= 94.5, address >= 96
Pairs: 272
S1 entities: 272

Current: name >= 94, address >= 96
Pairs: 627
S1 entities: 627


In [35]:
# ============================================
# INSPECT S3 MEDIUM-HIGH CONFIDENCE MATCHES
# ============================================

s3_typo_selected = s3_typo_unambiguous[
    (s3_typo_unambiguous["name_ratio"] >= 94.5) &
    (s3_typo_unambiguous["address_ratio"] >= 96)
].copy()

print("Selected:", len(s3_typo_selected))

print("\nLowest 30 selected matches:")
print(
    s3_typo_selected[
        [
            "entity_id_s1",
            "entity_id_matched",
            "s1_business_name",
            "s3_business_name",
            "name_ratio",
            "address_ratio"
        ]
    ]
    .sort_values(
        ["name_ratio", "address_ratio"]
    )
    .head(30)
    .to_string(index=False)
)

Selected: 272

Lowest 30 selected matches:
entity_id_s1 entity_id_matched                        s1_business_name                      s3_business_name  name_ratio  address_ratio
S1-182962303      S3-409173186            Spatial (India) Hills Center       Spatial (irlIndia) Hills Center   94.545455      96.045198
S1-671195711      S3-547264799            Colonial Regional Media PLLC           Colonial Rgioanl Media PLLC   94.545455      96.202532
S1-130054603      S3-724413530           Tennis Pharmacie (France) SAS        Tennis-Pharmacie (Frafcne) SAS   94.545455      96.296296
S1-893859375      S3-841761299            Britannia & Brothers Pvt Ltd       Dr Britannia & Bróthers Pvt Ltd   94.545455      96.296296
S1-458572530      S3-445868652              Greens Educational Society        Mr Greens Educational  Society   94.545455      96.296296
S1-774068829      S3-228206413           Shivalik Developers Pvt. Ltd.        Shivalik Develollers Pvt. Ltd.   94.545455      96.402878
 S1-9

In [36]:
# ============================================
# STRICTER S3 TYPO TIER
# ============================================

s3_strict_selected = s3_typo_unambiguous[
    (s3_typo_unambiguous["name_ratio"] >= 95) &
    (s3_typo_unambiguous["address_ratio"] >= 96)
].copy()

print("S3 matches with name >= 95 and address >= 96:")
print("Pairs:", len(s3_strict_selected))
print(
    "S1 entities:",
    s3_strict_selected["entity_id_s1"].nunique()
)

print("\nLowest selected:")
print(
    s3_strict_selected[
        [
            "entity_id_s1",
            "entity_id_matched",
            "s1_business_name",
            "s3_business_name",
            "name_ratio",
            "address_ratio"
        ]
    ]
    .sort_values(["name_ratio", "address_ratio"])
    .head(20)
    .to_string(index=False)
)

S3 matches with name >= 95 and address >= 96:
Pairs: 36
S1 entities: 36

Lowest selected:
entity_id_s1 entity_id_matched                            s1_business_name                                s3_business_name  name_ratio  address_ratio
S1-698838510      S3-152522942      Inventive Nursing Home Private Limited      Sri Inventive Nursing Home Private Limited   95.000000      96.350365
S1-481360516       S3-66196212                        Shree Global Limited                            Shree Ghobal Limited   95.000000      96.491228
S1-140324752      S3-663540317                        Delta Health Pvt Ltd                            De1ta Héalth Pvt Ltd   95.000000      96.932515
S1-104559742       S3-12239361      Department of Environmental Protection      Department Of Environmental Protection Ltd   95.000000      97.500000
S1-894426947      S3-469167605               Triathlon Visite Amicale EURL               Triathlon Visite Amicale E.U.R.L.   95.081967      97.916667
S1-6296960

In [37]:
# ============================================
# ZERO-MATCH S1s WITH ADDRESS CANDIDATES
# ============================================

# S1s with no matches in the successful 0.228 submission
zero_match_s1 = set(
    baseline[
        baseline["matched_entity_ids"].fillna("") == ""
    ]["source1_entity_id"]
)

print("S1 entities with zero matches:", len(zero_match_s1))

# S1s represented in the S2 address candidates
s2_addr_s1 = set(
    pd.read_csv(
        "/content/test_s2_address_candidates.tsv",
        sep="\t",
        dtype=str,
        usecols=["entity_id_s1"]
    )["entity_id_s1"]
)

# S1s represented in S3 address candidates
s3_addr_s1 = set(
    pd.read_csv(
        "/content/test_s3_address_candidates.tsv",
        sep="\t",
        dtype=str,
        usecols=["entity_id_s1"]
    )["entity_id_s1"]
)

zero_with_address = (
    zero_match_s1 &
    (s2_addr_s1 | s3_addr_s1)
)

print(
    "Zero-match S1s with address candidates:",
    len(zero_with_address)
)

S1 entities with zero matches: 1306701
Zero-match S1s with address candidates: 305949


In [38]:
# ============================================
# ZERO-MATCH S1 + STRONG NAME CANDIDATES
# ============================================

# Work only with S1s that currently have zero matches
s2_zero = s2_candidates[
    s2_candidates["entity_id_s1"].isin(zero_match_s1)
].copy()

s3_zero = s3_candidates[
    s3_candidates["entity_id_s1"].isin(zero_match_s1)
].copy()

print("S2 zero-match candidates:", len(s2_zero))
print("S3 zero-match candidates:", len(s3_zero))

# ============================================
# STRONG NAME + REASONABLE ADDRESS
# ============================================

s2_zero_strong = s2_zero[
    (s2_zero["name_ratio"] >= 95) &
    (s2_zero["address_ratio"] >= 90)
].copy()

s3_zero_strong = s3_zero[
    (s3_zero["name_ratio"] >= 95) &
    (s3_zero["address_ratio"] >= 90)
].copy()

print("\n========== STRONG ZERO-MATCH CANDIDATES ==========")

print("\nS2:")
print("Pairs:", len(s2_zero_strong))
print("S1 entities:", s2_zero_strong["entity_id_s1"].nunique())

print("\nS3:")
print("Pairs:", len(s3_zero_strong))
print("S1 entities:", s3_zero_strong["entity_id_s1"].nunique())

NameError: name 's2_candidates' is not defined

In [39]:
# ============================================
# RELOAD ADDRESS CANDIDATES FOR ZERO-MATCH S1s
# ============================================

import pandas as pd

# --------------------------------------------
# S2 address candidates
# --------------------------------------------
s2_zero = pd.read_csv(
    "/content/test_s2_address_candidates.tsv",
    sep="\t",
    dtype=str
)

print("Loaded S2:", s2_zero.shape)

# Keep only zero-match S1s
s2_zero = s2_zero[
    s2_zero["entity_id_s1"].isin(zero_match_s1)
].copy()

print("S2 zero-match candidates:", len(s2_zero))


# --------------------------------------------
# S3 address candidates
# --------------------------------------------
s3_zero = pd.read_csv(
    "/content/test_s3_address_candidates.tsv",
    sep="\t",
    dtype=str
)

print("Loaded S3:", s3_zero.shape)

# Keep only zero-match S1s
s3_zero = s3_zero[
    s3_zero["entity_id_s1"].isin(zero_match_s1)
].copy()

print("S3 zero-match candidates:", len(s3_zero))

Loaded S2: (1408537, 3)
S2 zero-match candidates: 995466
Loaded S3: (1246731, 3)
S3 zero-match candidates: 892384


In [40]:
# ============================================
# SCORE ZERO-MATCH ADDRESS CANDIDATES
# ============================================

from rapidfuzz.fuzz import ratio, token_set_ratio

# --------------------------------------------
# Load required S1 columns
# --------------------------------------------
s1_info = test_s1[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
].copy()

s1_info = s1_info.rename(columns={
    "entity_id": "entity_id_s1",
    "business_name": "s1_business_name",
    "business_address": "s1_business_address",
    "country": "s1_country"
})

# --------------------------------------------
# Load S2 required columns
# --------------------------------------------
s2_info = pd.read_csv(
    "/content/test_source2.tsv",
    sep="\t",
    dtype=str,
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

s2_info = s2_info.rename(columns={
    "entity_id": "entity_id_matched",
    "business_name": "s2_business_name",
    "business_address": "s2_business_address",
    "country": "s2_country"
})

# --------------------------------------------
# Load S3 required columns
# --------------------------------------------
s3_info = pd.read_csv(
    "/content/test_source3.tsv",
    sep="\t",
    dtype=str,
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

s3_info = s3_info.rename(columns={
    "entity_id": "entity_id_matched",
    "business_name": "s3_business_name",
    "business_address": "s3_business_address",
    "country": "s3_country"
})

print("S1:", s1_info.shape)
print("S2:", s2_info.shape)
print("S3:", s3_info.shape)

S1: (1732544, 4)
S2: (4887273, 4)
S3: (5082316, 4)


In [41]:
# ============================================
# SCORE S2 ZERO-MATCH CANDIDATES
# ============================================

# Join S1 information
s2_zero_scored = s2_zero.merge(
    s1_info,
    on="entity_id_s1",
    how="left"
)

# Join S2 information
s2_zero_scored = s2_zero_scored.merge(
    s2_info,
    on="entity_id_matched",
    how="left"
)

print("Joined S2 shape:", s2_zero_scored.shape)

# --------------------------------------------
# Calculate fuzzy scores
# --------------------------------------------

s2_zero_scored["name_ratio"] = [
    ratio(
        str(a),
        str(b)
    )
    for a, b in zip(
        s2_zero_scored["s1_business_name"],
        s2_zero_scored["s2_business_name"]
    )
]

s2_zero_scored["address_ratio"] = [
    ratio(
        str(a),
        str(b)
    )
    for a, b in zip(
        s2_zero_scored["s1_business_address"],
        s2_zero_scored["s2_business_address"]
    )
]

print("Scoring complete.")

print(
    s2_zero_scored[
        ["name_ratio", "address_ratio"]
    ].describe()
)

Joined S2 shape: (995466, 9)
Scoring complete.
          name_ratio  address_ratio
count  995466.000000  995466.000000
mean       43.330654      36.151452
std        29.866862      10.280257
min         0.000000       7.317073
25%        17.021277      28.205128
50%        34.285714      35.483871
75%        71.428571      43.243243
max       100.000000     100.000000


In [42]:
# ============================================
# S2 ZERO-MATCH SCORE THRESHOLDS
# ============================================

print("S2 zero-match candidates:", len(s2_zero_scored))

thresholds = [
    (95, 90),
    (95, 93),
    (95, 95),
    (96, 95),
    (97, 95),
    (98, 95),
    (99, 95),
    (95, 97),
    (97, 97),
    (98, 98),
    (99, 99),
]

print("\nname_ratio >= X AND address_ratio >= Y\n")

for name_t, addr_t in thresholds:
    x = s2_zero_scored[
        (s2_zero_scored["name_ratio"] >= name_t) &
        (s2_zero_scored["address_ratio"] >= addr_t)
    ]

    print(
        f"name >= {name_t:>2}, "
        f"address >= {addr_t:>2}: "
        f"{len(x):>6} pairs | "
        f"{x['entity_id_s1'].nunique():>6} S1"
    )

S2 zero-match candidates: 995466

name_ratio >= X AND address_ratio >= Y

name >= 95, address >= 90:      0 pairs |      0 S1
name >= 95, address >= 93:      0 pairs |      0 S1
name >= 95, address >= 95:      0 pairs |      0 S1
name >= 96, address >= 95:      0 pairs |      0 S1
name >= 97, address >= 95:      0 pairs |      0 S1
name >= 98, address >= 95:      0 pairs |      0 S1
name >= 99, address >= 95:      0 pairs |      0 S1
name >= 95, address >= 97:      0 pairs |      0 S1
name >= 97, address >= 97:      0 pairs |      0 S1
name >= 98, address >= 98:      0 pairs |      0 S1
name >= 99, address >= 99:      0 pairs |      0 S1


In [43]:
# ============================================
# SCORE S3 ZERO-MATCH CANDIDATES
# ============================================

s3_zero_scored = s3_zero.merge(
    s1_info,
    on="entity_id_s1",
    how="left"
)

s3_zero_scored = s3_zero_scored.merge(
    s3_info,
    on="entity_id_matched",
    how="left"
)

print("Joined S3 shape:", s3_zero_scored.shape)

# --------------------------------------------
# Calculate fuzzy scores
# --------------------------------------------

s3_zero_scored["name_ratio"] = [
    ratio(str(a), str(b))
    for a, b in zip(
        s3_zero_scored["s1_business_name"],
        s3_zero_scored["s3_business_name"]
    )
]

s3_zero_scored["address_ratio"] = [
    ratio(str(a), str(b))
    for a, b in zip(
        s3_zero_scored["s1_business_address"],
        s3_zero_scored["s3_business_address"]
    )
]

print("Scoring complete.")

print(
    s3_zero_scored[
        ["name_ratio", "address_ratio"]
    ].describe()
)

Joined S3 shape: (892384, 9)
Scoring complete.
          name_ratio  address_ratio
count  892384.000000  892384.000000
mean       52.547021      68.057951
std        29.322629      18.361136
min         0.000000      15.238095
25%        27.027027      52.413793
50%        50.847458      70.833333
75%        80.519481      84.090909
max       100.000000     100.000000


In [44]:
# ============================================
# S3 ZERO-MATCH SCORE THRESHOLDS
# ============================================

thresholds = [
    (95, 90),
    (95, 93),
    (95, 95),
    (95, 97),
    (96, 95),
    (97, 95),
    (97, 97),
    (98, 95),
    (98, 97),
    (98, 98),
    (99, 95),
    (99, 97),
    (99, 99),
]

print("S3 zero-match candidates:", len(s3_zero_scored))
print("\nname_ratio >= X AND address_ratio >= Y\n")

for name_t, addr_t in thresholds:
    x = s3_zero_scored[
        (s3_zero_scored["name_ratio"] >= name_t) &
        (s3_zero_scored["address_ratio"] >= addr_t)
    ]

    print(
        f"name >= {name_t:>2}, "
        f"address >= {addr_t:>2}: "
        f"{len(x):>6} pairs | "
        f"{x['entity_id_s1'].nunique():>6} S1"
    )

S3 zero-match candidates: 892384

name_ratio >= X AND address_ratio >= Y

name >= 95, address >= 90:   3899 pairs |   3616 S1
name >= 95, address >= 93:    357 pairs |    287 S1
name >= 95, address >= 95:     30 pairs |     26 S1
name >= 95, address >= 97:      1 pairs |      1 S1
name >= 96, address >= 95:     18 pairs |     15 S1
name >= 97, address >= 95:     11 pairs |      8 S1
name >= 97, address >= 97:      0 pairs |      0 S1
name >= 98, address >= 95:      4 pairs |      4 S1
name >= 98, address >= 97:      0 pairs |      0 S1
name >= 98, address >= 98:      0 pairs |      0 S1
name >= 99, address >= 95:      2 pairs |      2 S1
name >= 99, address >= 97:      0 pairs |      0 S1
name >= 99, address >= 99:      0 pairs |      0 S1


In [45]:
# ============================================
# INSPECT HIGH-CONFIDENCE S3 ZERO-MATCH PAIRS
# ============================================

s3_zero_high = s3_zero_scored[
    (s3_zero_scored["name_ratio"] >= 95) &
    (s3_zero_scored["address_ratio"] >= 95)
].copy()

print("Candidates:", len(s3_zero_high))
print("S1 entities:", s3_zero_high["entity_id_s1"].nunique())

cols = [
    "entity_id_s1",
    "entity_id_matched",
    "s1_business_name",
    "s3_business_name",
    "s1_business_address",
    "s3_business_address",
    "s1_country",
    "s3_country",
    "name_ratio",
    "address_ratio"
]

print(
    s3_zero_high[
        cols
    ]
    .sort_values(
        ["name_ratio", "address_ratio"],
        ascending=False
    )
    .to_string(index=False)
)

Candidates: 30
S1 entities: 26
entity_id_s1 entity_id_matched                                    s1_business_name                                s3_business_name                                                                                                                         s1_business_address                                                                                                              s3_business_address s1_country s3_country  name_ratio  address_ratio
S1-223460245      S3-131254550                                   Sandeep Processor                               Sandeep Processor                                        Gate No 3, 7Th Floor, Room No P Poodar Court, Rabindra Sarani, Kolkata, Kolkata, Howrah, West Bengal                                      Gate No 3, 7Th Floor, Room No P Poodar Court, Rabindra Sarani, Kolkata, Kolkata, Howrah, WB      India      India  100.000000      95.287958
S1-375576068      S3-525917262                            Shree Brands 

In [46]:
# ============================================
# SELECT UNAMBIGUOUS HIGH-CONFIDENCE S3 PAIRS
# ============================================

# High-confidence zero-match candidates
s3_high = s3_zero_scored[
    (s3_zero_scored["name_ratio"] >= 95) &
    (s3_zero_scored["address_ratio"] >= 95)
].copy()

# Count candidates per S1
s1_candidate_counts = (
    s3_high.groupby("entity_id_s1")
    .size()
)

# Keep only S1s having exactly ONE candidate
s3_unambiguous_zero = s3_high[
    s3_high["entity_id_s1"].map(s1_candidate_counts) == 1
].copy()

print("High-confidence candidates:", len(s3_high))
print("High-confidence S1s:", s3_high["entity_id_s1"].nunique())

print("\nUnambiguous pairs:", len(s3_unambiguous_zero))
print(
    "Unambiguous S1s:",
    s3_unambiguous_zero["entity_id_s1"].nunique()
)

print("\nSelected pairs:")
print(
    s3_unambiguous_zero[
        [
            "entity_id_s1",
            "entity_id_matched",
            "s1_business_name",
            "s3_business_name",
            "name_ratio",
            "address_ratio"
        ]
    ]
    .sort_values(["name_ratio", "address_ratio"], ascending=False)
    .to_string(index=False)
)

High-confidence candidates: 30
High-confidence S1s: 26

Unambiguous pairs: 22
Unambiguous S1s: 22

Selected pairs:
entity_id_s1 entity_id_matched                                    s1_business_name                                s3_business_name  name_ratio  address_ratio
S1-375576068      S3-525917262                            Shree Brands Corporation                        Shree Brands Corporation  100.000000      95.061728
 S1-66082566      S3-654457795                        Rishabh Law Chambers Pvt Ltd                     Rishabh Law Chambes Pvt Ltd   98.181818      95.714286
S1-831978162      S3-469717063                             Rsm Development Pvt Ltd                        Rsm Development Pvt Ltd.   97.872340      95.135135
S1-163665978      S3-965527172                              CM Packaging Pvt. Ltd.                         WCM Packaging Pvt. Ltd.   97.777778      95.880150
S1-809638961      S3-204479762                   Versatile Exports Private Limited             

In [47]:
# ============================================
# ADD 22 HIGH-CONFIDENCE S3 ZERO-MATCH PAIRS
# ============================================

# Load current submission
current = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

# Build lookup of the 22 selected pairs
s3_additions = dict(
    zip(
        s3_unambiguous_zero["entity_id_s1"],
        s3_unambiguous_zero["entity_id_matched"]
    )
)

print("Current rows:", len(current))
print("S3 additions:", len(s3_additions))

# Add each new S3 match to its S1 row
for s1_id, s3_id in s3_additions.items():

    idx = current.index[
        current["source1_entity_id"] == s1_id
    ]

    if len(idx) != 1:
        print("WARNING:", s1_id, "rows =", len(idx))
        continue

    i = idx[0]

    existing = current.at[i, "matched_entity_ids"]

    if pd.isna(existing) or existing == "":
        current.at[i, "matched_entity_ids"] = s3_id
    else:
        existing_ids = set(existing.split(","))

        if s3_id not in existing_ids:
            current.at[i, "matched_entity_ids"] = (
                existing + "," + s3_id
            )

# Save as a NEW file first
output_v3 = "/content/matching_results_v3.tsv"

current.to_csv(
    output_v3,
    sep="\t",
    index=False
)

print("\nSaved:", output_v3)

# --------------------------------------------
# Basic validation
# --------------------------------------------

print("\nRows:", len(current))
print(
    "S1 with matches:",
    (current["matched_entity_ids"].fillna("") != "").sum()
)
print(
    "S1 without matches:",
    (current["matched_entity_ids"].fillna("") == "").sum()
)

# Check duplicate S1 rows
print(
    "Duplicate S1 rows:",
    current["source1_entity_id"].duplicated().sum()
)

# Check duplicate IDs inside rows
duplicate_id_rows = 0

for value in current["matched_entity_ids"].fillna(""):
    ids = [x for x in value.split(",") if x]
    if len(ids) != len(set(ids)):
        duplicate_id_rows += 1

print(
    "Rows with duplicate matched IDs:",
    duplicate_id_rows
)

Current rows: 1732544
S3 additions: 22

Saved: /content/matching_results_v3.tsv

Rows: 1732544
S1 with matches: 425865
S1 without matches: 1306679
Duplicate S1 rows: 0
Rows with duplicate matched IDs: 0


In [48]:
# ============================================
# CHECK V3 MATCHES AGAINST CANDIDATE SET
# ============================================

# Read only the S3 candidate rows
candidate_s3 = pd.read_csv(
    "/content/candidate_pairs.tsv",
    sep="\t",
    dtype=str,
    usecols=["source1_entity_id", "matched_entity_id", "source"]
)

candidate_s3 = candidate_s3[
    candidate_s3["source"] == "S3"
]

candidate_lookup = set(
    zip(
        candidate_s3["source1_entity_id"],
        candidate_s3["matched_entity_id"]
    )
)

missing = []

for _, row in s3_unambiguous_zero.iterrows():
    pair = (
        row["entity_id_s1"],
        row["entity_id_matched"]
    )

    if pair not in candidate_lookup:
        missing.append(pair)

print("New S3 pairs:", len(s3_unambiguous_zero))
print("Missing from candidate_pairs.tsv:", len(missing))

if missing:
    print("\nMISSING PAIRS:")
    for pair in missing:
        print(pair)
else:
    print("\nALL 22 NEW PAIRS ARE PRESENT IN candidate_pairs.tsv")

ValueError: Usecols do not match columns, columns expected but not found: ['matched_entity_id', 'source1_entity_id']

In [49]:
# ============================================
# CHECK CANDIDATE FILE COLUMNS
# ============================================

candidate_header = pd.read_csv(
    "/content/candidate_pairs.tsv",
    sep="\t",
    dtype=str,
    nrows=3
)

print("Columns:")
print(candidate_header.columns.tolist())

print("\nFirst 3 rows:")
print(candidate_header.to_string(index=False))

Columns:
['entity_id_s1', 'entity_id_matched', 'source']

First 3 rows:
entity_id_s1 entity_id_matched source
S1-100000556      S2-109665397     S2
S1-100000556       S2-11431119     S2
S1-100000556      S2-119912657     S2


In [50]:
# ============================================
# CHECK V3 MATCHES AGAINST CANDIDATE SET
# ============================================

candidate_s3 = pd.read_csv(
    "/content/candidate_pairs.tsv",
    sep="\t",
    dtype=str,
    usecols=[
        "entity_id_s1",
        "entity_id_matched",
        "source"
    ]
)

# Keep only S3 candidates
candidate_s3 = candidate_s3[
    candidate_s3["source"] == "S3"
]

candidate_lookup = set(
    zip(
        candidate_s3["entity_id_s1"],
        candidate_s3["entity_id_matched"]
    )
)

missing = []

for _, row in s3_unambiguous_zero.iterrows():

    pair = (
        row["entity_id_s1"],
        row["entity_id_matched"]
    )

    if pair not in candidate_lookup:
        missing.append(pair)

print("New S3 pairs:", len(s3_unambiguous_zero))
print("Missing from candidate_pairs.tsv:", len(missing))

if missing:
    print("\nMISSING PAIRS:")
    for pair in missing:
        print(pair)
else:
    print("\nALL 22 NEW PAIRS ARE PRESENT IN candidate_pairs.tsv")

New S3 pairs: 22
Missing from candidate_pairs.tsv: 0

ALL 22 NEW PAIRS ARE PRESENT IN candidate_pairs.tsv


In [51]:
from google.colab import files

files.download("/content/matching_results_v3.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [52]:
# ============================================
# S3 ZERO-MATCH TOKEN SIMILARITY
# ============================================

from rapidfuzz.fuzz import token_set_ratio

s3_zero_scored["name_token_ratio"] = [
    token_set_ratio(str(a), str(b))
    for a, b in zip(
        s3_zero_scored["s1_business_name"],
        s3_zero_scored["s3_business_name"]
    )
]

thresholds = [
    (95, 90),
    (95, 93),
    (95, 95),
    (97, 90),
    (97, 93),
    (98, 90),
    (98, 93),
    (99, 90),
]

print("S3 zero-match candidates:", len(s3_zero_scored))
print()

for name_t, addr_t in thresholds:

    x = s3_zero_scored[
        (s3_zero_scored["name_token_ratio"] >= name_t) &
        (s3_zero_scored["address_ratio"] >= addr_t)
    ]

    print(
        f"token_name >= {name_t}, "
        f"address >= {addr_t}: "
        f"{len(x):,} pairs | "
        f"{x['entity_id_s1'].nunique():,} S1"
    )

S3 zero-match candidates: 892384

token_name >= 95, address >= 90: 23,433 pairs | 21,201 S1
token_name >= 95, address >= 93: 8,366 pairs | 7,798 S1
token_name >= 95, address >= 95: 2,797 pairs | 2,650 S1
token_name >= 97, address >= 90: 21,062 pairs | 19,308 S1
token_name >= 97, address >= 93: 7,799 pairs | 7,326 S1
token_name >= 98, address >= 90: 20,419 pairs | 18,773 S1
token_name >= 98, address >= 93: 7,644 pairs | 7,195 S1
token_name >= 99, address >= 90: 20,164 pairs | 18,551 S1


In [53]:
# ============================================
# STRICT S3 TOKEN-NAME CANDIDATES
# ============================================

thresholds = [
    (95, 97),
    (96, 97),
    (97, 97),
    (98, 97),
    (99, 97),
    (95, 98),
    (97, 98),
    (98, 98),
    (99, 98),
    (95, 99),
    (97, 99),
    (98, 99),
    (99, 99),
]

print("S3 zero-match strict token candidates\n")

for name_t, addr_t in thresholds:

    x = s3_zero_scored[
        (s3_zero_scored["name_token_ratio"] >= name_t) &
        (s3_zero_scored["address_ratio"] >= addr_t)
    ]

    print(
        f"token_name >= {name_t}, "
        f"address >= {addr_t}: "
        f"{len(x):>6} pairs | "
        f"{x['entity_id_s1'].nunique():>6} S1"
    )

S3 zero-match strict token candidates

token_name >= 95, address >= 97:    459 pairs |    431 S1
token_name >= 96, address >= 97:    450 pairs |    422 S1
token_name >= 97, address >= 97:    436 pairs |    409 S1
token_name >= 98, address >= 97:    432 pairs |    405 S1
token_name >= 99, address >= 97:    429 pairs |    402 S1
token_name >= 95, address >= 98:    178 pairs |    165 S1
token_name >= 97, address >= 98:    169 pairs |    157 S1
token_name >= 98, address >= 98:    169 pairs |    157 S1
token_name >= 99, address >= 98:    169 pairs |    157 S1
token_name >= 95, address >= 99:     82 pairs |     75 S1
token_name >= 97, address >= 99:     74 pairs |     68 S1
token_name >= 98, address >= 99:     74 pairs |     68 S1
token_name >= 99, address >= 99:     74 pairs |     68 S1


In [54]:
# ============================================
# INSPECT STRONGEST S3 TOKEN CANDIDATES
# ============================================

s3_token_99 = s3_zero_scored[
    (s3_zero_scored["name_token_ratio"] >= 99) &
    (s3_zero_scored["address_ratio"] >= 99)
].copy()

print("Candidates:", len(s3_token_99))
print("S1 entities:", s3_token_99["entity_id_s1"].nunique())

cols = [
    "entity_id_s1",
    "entity_id_matched",
    "s1_business_name",
    "s3_business_name",
    "s1_business_address",
    "s3_business_address",
    "s1_country",
    "s3_country",
    "name_ratio",
    "name_token_ratio",
    "address_ratio"
]

print(
    s3_token_99[
        cols
    ]
    .sort_values(
        ["name_token_ratio", "address_ratio", "name_ratio"],
        ascending=False
    )
    .to_string(index=False)
)

Candidates: 74
S1 entities: 68
entity_id_s1 entity_id_matched                            s1_business_name                             s3_business_name                                                                                                                                   s1_business_address                                                                                                                                      s3_business_address s1_country s3_country  name_ratio  name_token_ratio  address_ratio
S1-493093454      S3-766893578                      Pristine Match Pvt Ltd                           Pristine Match Pvt                    Tc 20/1470(9), Prasadam, Mera 23 Kunjalumood, Ramachavila, Karamana Po, Thiruvananthapuram, Thiruvananthapuram, Trivandrum, Kerala                      Tc 20/1470(9), Prasadam, Mera 23 Kunjalumood, Ramachavila, Karamana Po, Thiruvananthapuram, Thiruvananthapuram, Trivandrum, Keralam      India      India   90.000000             100.0     

In [55]:
# ============================================
# STRICT TOKEN + NORMAL NAME S3 CANDIDATES
# ============================================

s3_strict_token = s3_zero_scored[
    (s3_zero_scored["name_token_ratio"] >= 99) &
    (s3_zero_scored["address_ratio"] >= 99) &
    (s3_zero_scored["name_ratio"] >= 85)
].copy()

# Count candidates per S1
candidate_counts = (
    s3_strict_token
    .groupby("entity_id_s1")
    .size()
)

# Keep only unambiguous S1s
s3_strict_unambiguous = s3_strict_token[
    s3_strict_token["entity_id_s1"].map(candidate_counts) == 1
].copy()

print("Strict candidates:", len(s3_strict_token))
print(
    "S1 entities:",
    s3_strict_token["entity_id_s1"].nunique()
)

print("\nUnambiguous pairs:", len(s3_strict_unambiguous))
print(
    "Unambiguous S1s:",
    s3_strict_unambiguous["entity_id_s1"].nunique()
)

print("\nSelected:")
print(
    s3_strict_unambiguous[
        [
            "entity_id_s1",
            "entity_id_matched",
            "s1_business_name",
            "s3_business_name",
            "name_ratio",
            "name_token_ratio",
            "address_ratio"
        ]
    ]
    .sort_values(
        ["name_ratio", "address_ratio"],
        ascending=False
    )
    .to_string(index=False)
)

Strict candidates: 25
S1 entities: 23

Unambiguous pairs: 21
Unambiguous S1s: 21

Selected:
entity_id_s1 entity_id_matched                           s1_business_name                    s3_business_name  name_ratio  name_token_ratio  address_ratio
 S1-26243654      S3-500857538            DIG Development Private Limited The DIG Development Private Limited   93.939394             100.0      99.224806
S1-765793097      S3-935341377              Tech Infotech Private Limited  Shri Tech Infotech Private Limited   92.063492             100.0      99.408284
S1-106842122      S3-356556065                Supreme Consultants Pvt Ltd             Supreme Consultants Pvt   92.000000             100.0      99.310345
S1-943011619      S3-499324346     International Aerostar Lifestyle Group    International Aerostar Lifestyle   91.428571             100.0      99.319728
S1-617094307      S3-490396616                  Bravo Lifescience Pvt Ltd               Bravo Lifescience Pvt   91.304348            

In [56]:
# ============================================
# CHECK 21 STRICT S3 PAIRS IN CANDIDATE SET
# ============================================

candidate_s3 = pd.read_csv(
    "/content/candidate_pairs.tsv",
    sep="\t",
    dtype=str,
    usecols=[
        "entity_id_s1",
        "entity_id_matched",
        "source"
    ]
)

candidate_s3 = candidate_s3[
    candidate_s3["source"] == "S3"
]

candidate_lookup = set(
    zip(
        candidate_s3["entity_id_s1"],
        candidate_s3["entity_id_matched"]
    )
)

missing = []

for _, row in s3_strict_unambiguous.iterrows():
    pair = (
        row["entity_id_s1"],
        row["entity_id_matched"]
    )

    if pair not in candidate_lookup:
        missing.append(pair)

print("New pairs:", len(s3_strict_unambiguous))
print("Missing from candidate_pairs.tsv:", len(missing))

if missing:
    print("\nMISSING:")
    for x in missing:
        print(x)
else:
    print("\nALL 21 PAIRS ARE PRESENT")

New pairs: 21
Missing from candidate_pairs.tsv: 0

ALL 21 PAIRS ARE PRESENT


In [57]:
# ============================================
# CREATE V4 = V3 + 21 STRICT S3 MATCHES
# ============================================

# Load v3, which already contains the previous 22 additions
v4 = pd.read_csv(
    "/content/matching_results_v3.tsv",
    sep="\t",
    dtype=str
)

print("V3 rows:", len(v4))

added = 0

for _, row in s3_strict_unambiguous.iterrows():

    s1_id = row["entity_id_s1"]
    s3_id = row["entity_id_matched"]

    idx = v4.index[
        v4["source1_entity_id"] == s1_id
    ]

    if len(idx) != 1:
        print("WARNING:", s1_id, "rows =", len(idx))
        continue

    i = idx[0]

    existing = v4.at[i, "matched_entity_ids"]

    if pd.isna(existing) or existing == "":
        v4.at[i, "matched_entity_ids"] = s3_id
        added += 1

    else:
        existing_ids = set(existing.split(","))

        if s3_id not in existing_ids:
            v4.at[i, "matched_entity_ids"] = (
                existing + "," + s3_id
            )
            added += 1

# Save as a new file
output_v4 = "/content/matching_results_v4.tsv"

v4.to_csv(
    output_v4,
    sep="\t",
    index=False
)

print("\nAdded:", added)
print("Saved:", output_v4)

# --------------------------------------------
# Validation
# --------------------------------------------

print("\nRows:", len(v4))

print(
    "S1 with matches:",
    (v4["matched_entity_ids"].fillna("") != "").sum()
)

print(
    "S1 without matches:",
    (v4["matched_entity_ids"].fillna("") == "").sum()
)

print(
    "Duplicate S1 rows:",
    v4["source1_entity_id"].duplicated().sum()
)

duplicate_id_rows = 0

for value in v4["matched_entity_ids"].fillna(""):
    ids = [x for x in value.split(",") if x]

    if len(ids) != len(set(ids)):
        duplicate_id_rows += 1

print(
    "Rows with duplicate matched IDs:",
    duplicate_id_rows
)

V3 rows: 1732544

Added: 21
Saved: /content/matching_results_v4.tsv

Rows: 1732544
S1 with matches: 425886
S1 without matches: 1306658
Duplicate S1 rows: 0
Rows with duplicate matched IDs: 0


In [58]:
from google.colab import files

files.download("/content/matching_results_v4.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
# ============================================
# CHECK AVAILABLE TEST S1 INDEXES
# ============================================

print("S1 compact-name index:", len(s1_compact_name_index))
print("S1 exact-name index:", len(s1_exact_name))
print("S1 exact-address index:", len(s1_exact_address))

NameError: name 's1_compact_name_index' is not defined

In [60]:
# ============================================
# REBUILD COMPACT-NAME INDEX FOR ZERO-MATCH S1s
# ============================================

import re
import pandas as pd

DOMAIN_STOPWORDS = {
    "limited","private","llc","inc","ltd","pvt","llp",
    "incorporated","corp","corporation","company","co","plc"
}

def compact_business_key(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # Remove accents
    import unicodedata
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    # Remove common domain extensions
    text = re.sub(
        r"\.(com|net|org|co|in|biz|info|io)$",
        "",
        text
    )

    # Keep alphanumeric tokens
    text = re.sub(r"[^a-z0-9]+", " ", text)

    tokens = [
        x for x in text.split()
        if x not in DOMAIN_STOPWORDS
    ]

    return "".join(tokens)


# --------------------------------------------
# Build only for currently ZERO-MATCH S1s
# --------------------------------------------

zero_s1_df = test_s1[
    test_s1["entity_id"].isin(zero_match_s1)
][
    ["entity_id", "business_name", "country"]
].copy()

zero_s1_df["compact_key"] = (
    zero_s1_df["business_name"]
    .map(compact_business_key)
)

# Remove empty keys
zero_s1_df = zero_s1_df[
    zero_s1_df["compact_key"] != ""
].copy()

compact_zero_index = (
    zero_s1_df
    .groupby("compact_key")["entity_id"]
    .agg(list)
    .to_dict()
)

print("Zero-match S1 records:", len(zero_s1_df))
print("Unique compact keys:", len(compact_zero_index))

# Show a few examples
print("\nExamples:")
print(
    zero_s1_df[
        ["entity_id", "business_name", "compact_key"]
    ].head(10).to_string(index=False)
)

Zero-match S1 records: 1306701
Unique compact keys: 855733

Examples:
   entity_id            business_name       compact_key
S1-714132312          Zephay Labs Inc        zephaylabs
S1-106407869     Vision Partners Corp    visionpartners
S1-156285671            << Team Ecole         teamecole
S1-689823050      Red Perfect Trading redperfecttrading
S1-921369899            ZNB Club SARL       znbclubsarl
S1-481669221        Nandlal Kisan LLP      nandlalkisan
S1-280204013 Om Constructions Pvt Ltd   omconstructions
S1-742053041            Roongta Sangh      roongtasangh
S1-913506265      Thermal & Fils SASU   thermalfilssasu
S1-680637447   Siliguri Media Pvt Ltd     siligurimedia


In [61]:
# ============================================
# SCAN S2 FOR COMPACT-NAME + COUNTRY MATCHES
# ============================================

from collections import defaultdict

# Target keys from ZERO-MATCH S1s
target_keys = set(
    zip(
        zero_s1_df["compact_key"],
        zero_s1_df["country"].fillna("")
    )
)

print("Target S1 compact+country keys:", len(target_keys))

s2_compact_matches = []

chunksize = 500_000

for chunk in pd.read_csv(
    "/content/test_source2.tsv",
    sep="\t",
    dtype=str,
    usecols=[
        "entity_id",
        "business_name",
        "country"
    ],
    chunksize=chunksize
):

    chunk["compact_key"] = (
        chunk["business_name"]
        .map(compact_business_key)
    )

    chunk["_country"] = chunk["country"].fillna("")

    # Only retain keys that occur among zero-match S1s
    mask = [
        (k, c) in target_keys
        for k, c in zip(
            chunk["compact_key"],
            chunk["_country"]
        )
    ]

    matched = chunk.loc[
        mask,
        [
            "entity_id",
            "business_name",
            "country",
            "compact_key"
        ]
    ].copy()

    if len(matched):
        s2_compact_matches.append(matched)

    print(
        "Processed:",
        len(s2_compact_matches),
        "chunks | matches:",
        sum(len(x) for x in s2_compact_matches)
    )

s2_compact_matches = pd.concat(
    s2_compact_matches,
    ignore_index=True
)

print("\n========== S2 COMPACT RESULTS ==========")
print("Rows:", len(s2_compact_matches))
print(
    "Unique compact keys:",
    s2_compact_matches["compact_key"].nunique()
)

Target S1 compact+country keys: 857329
Processed: 1 chunks | matches: 153282
Processed: 2 chunks | matches: 306522
Processed: 3 chunks | matches: 460134
Processed: 4 chunks | matches: 613259
Processed: 5 chunks | matches: 766554
Processed: 6 chunks | matches: 920341
Processed: 7 chunks | matches: 1074178
Processed: 8 chunks | matches: 1227637
Processed: 9 chunks | matches: 1381589
Processed: 10 chunks | matches: 1500200

========== S2 COMPACT RESULTS ==========
Rows: 1500200
Unique compact keys: 579057


In [62]:
# ============================================
# ANALYZE COMPACT-KEY AMBIGUITY
# ============================================

# Number of zero-match S1 records per compact+country key
s1_key_counts = (
    zero_s1_df
    .groupby(["compact_key", "country"])
    .size()
    .rename("s1_count")
    .reset_index()
)

# Number of S2 records per compact+country key
s2_key_counts = (
    s2_compact_matches
    .groupby(["compact_key", "country"])
    .size()
    .rename("s2_count")
    .reset_index()
)

key_counts = s1_key_counts.merge(
    s2_key_counts,
    on=["compact_key", "country"],
    how="inner"
)

print("Keys shared by S1 and S2:", len(key_counts))

print("\n========== KEY AMBIGUITY ==========")

print(
    "S1 key count = 1:",
    (key_counts["s1_count"] == 1).sum()
)

print(
    "S2 key count = 1:",
    (key_counts["s2_count"] == 1).sum()
)

print(
    "Both S1 and S2 count = 1:",
    (
        (key_counts["s1_count"] == 1) &
        (key_counts["s2_count"] == 1)
    ).sum()
)

print("\nS1 multiplicity:")
print(key_counts["s1_count"].value_counts().sort_index().head(15))

print("\nS2 multiplicity:")
print(key_counts["s2_count"].value_counts().sort_index().head(15))

Keys shared by S1 and S2: 580295

========== KEY AMBIGUITY ==========
S1 key count = 1: 488392
S2 key count = 1: 288759
Both S1 and S2 count = 1: 274944

S1 multiplicity:
s1_count
1     488392
2      53256
3      16334
4       6243
5       3071
6       1728
7       1100
8        759
9        531
10       387
11       367
12       279
13       275
14       268
15       226
Name: count, dtype: int64

S2 multiplicity:
s2_count
1     288759
2     154948
3      65320
4      26790
5      12949
6       7320
7       4448
8       2873
9       1878
10      1383
11      1013
12       828
13       646
14       587
15       508
Name: count, dtype: int64


In [63]:
# ============================================
# BUILD 1-TO-1 S2 COMPACT-NAME PAIRS
# ============================================

# Keep only keys where:
#   exactly 1 zero-match S1
#   exactly 1 S2 record

one_to_one_keys = key_counts[
    (key_counts["s1_count"] == 1) &
    (key_counts["s2_count"] == 1)
][
    ["compact_key", "country"]
].copy()

print("1-to-1 compact keys:", len(one_to_one_keys))

# --------------------------------------------
# Get the S1 record for each key
# --------------------------------------------

s1_one = zero_s1_df.merge(
    one_to_one_keys,
    on=["compact_key", "country"],
    how="inner"
)

# --------------------------------------------
# Get the S2 record for each key
# --------------------------------------------

s2_one = s2_compact_matches.merge(
    one_to_one_keys,
    on=["compact_key", "country"],
    how="inner"
)

print("S1 records:", len(s1_one))
print("S2 records:", len(s2_one))

# --------------------------------------------
# Merge S1 and S2 on compact key + country
# --------------------------------------------

s2_compact_1to1 = s1_one.merge(
    s2_one,
    on=["compact_key", "country"],
    suffixes=("_s1", "_s2")
)

print("\nFinal 1-to-1 pairs:", len(s2_compact_1to1))

print(
    s2_compact_1to1[
        [
            "entity_id",
            "entity_id_s2",
            "business_name_s1",
            "business_name_s2",
            "country"
        ]
    ].head(10).to_string(index=False)
)

1-to-1 compact keys: 274944
S1 records: 274944
S2 records: 274944

Final 1-to-1 pairs: 274944


KeyError: "['entity_id'] not in index"

In [64]:
print(s2_compact_1to1.columns.tolist())

['entity_id_s1', 'business_name_s1', 'country', 'compact_key', 'entity_id_s2', 'business_name_s2']


In [65]:
# ============================================
# SCORE S2 COMPACT 1-TO-1 PAIRS
# ============================================

from rapidfuzz.fuzz import ratio

# Load only S2 address information
s2_address = pd.read_csv(
    "/content/test_source2.tsv",
    sep="\t",
    dtype=str,
    usecols=[
        "entity_id",
        "business_address"
    ]
)

s2_address = s2_address.rename(columns={
    "entity_id": "entity_id_s2",
    "business_address": "s2_business_address"
})

# Add S1 address from test_s1
s1_address = test_s1[
    [
        "entity_id",
        "business_address"
    ]
].copy()

s1_address = s1_address.rename(columns={
    "entity_id": "entity_id_s1",
    "business_address": "s1_business_address"
})

# --------------------------------------------
# Join addresses
# --------------------------------------------

s2_compact_scored = s2_compact_1to1.merge(
    s1_address,
    on="entity_id_s1",
    how="left"
)

s2_compact_scored = s2_compact_scored.merge(
    s2_address,
    on="entity_id_s2",
    how="left"
)

print("Joined shape:", s2_compact_scored.shape)

# --------------------------------------------
# Calculate similarities
# --------------------------------------------

s2_compact_scored["name_ratio"] = [
    ratio(str(a), str(b))
    for a, b in zip(
        s2_compact_scored["business_name_s1"],
        s2_compact_scored["business_name_s2"]
    )
]

s2_compact_scored["address_ratio"] = [
    ratio(str(a), str(b))
    for a, b in zip(
        s2_compact_scored["s1_business_address"],
        s2_compact_scored["s2_business_address"]
    )
]

print("\nScoring complete.")

print(
    s2_compact_scored[
        ["name_ratio", "address_ratio"]
    ].describe()
)

Joined shape: (274944, 8)

Scoring complete.
          name_ratio  address_ratio
count  274944.000000  274944.000000
mean       74.408979      37.275547
std        27.552475      12.651465
min         0.000000       0.000000
25%        60.465116      29.411765
50%        87.179487      37.522563
75%        94.736842      45.528455
max       100.000000      96.875000


In [66]:
# ============================================
# S2 COMPACT 1-TO-1 THRESHOLD ANALYSIS
# ============================================

thresholds = [
    (95, 50),
    (95, 60),
    (95, 70),
    (95, 80),
    (95, 90),
    (95, 95),
    (97, 50),
    (97, 60),
    (97, 70),
    (97, 80),
    (97, 90),
    (98, 80),
    (98, 90),
    (99, 80),
    (99, 90),
    (100, 80),
    (100, 90),
]

print("S2 compact 1-to-1 candidates:", len(s2_compact_scored))
print()

for name_t, addr_t in thresholds:

    x = s2_compact_scored[
        (s2_compact_scored["name_ratio"] >= name_t) &
        (s2_compact_scored["address_ratio"] >= addr_t)
    ]

    print(
        f"name >= {name_t:>3}, "
        f"address >= {addr_t:>2}: "
        f"{len(x):>6} pairs"
    )

S2 compact 1-to-1 candidates: 274944

name >=  95, address >= 50:   7915 pairs
name >=  95, address >= 60:   1829 pairs
name >=  95, address >= 70:    351 pairs
name >=  95, address >= 80:     22 pairs
name >=  95, address >= 90:      0 pairs
name >=  95, address >= 95:      0 pairs
name >=  97, address >= 50:   5213 pairs
name >=  97, address >= 60:   1164 pairs
name >=  97, address >= 70:    218 pairs
name >=  97, address >= 80:     15 pairs
name >=  97, address >= 90:      0 pairs
name >=  98, address >= 80:     11 pairs
name >=  98, address >= 90:      0 pairs
name >=  99, address >= 80:      8 pairs
name >=  99, address >= 90:      0 pairs
name >= 100, address >= 80:      8 pairs
name >= 100, address >= 90:      0 pairs


In [67]:
# ============================================
# INSPECT S2 COMPACT HIGH-SIMILARITY TAIL
# ============================================

s2_compact_high = s2_compact_scored[
    (s2_compact_scored["name_ratio"] >= 95) &
    (s2_compact_scored["address_ratio"] >= 80)
].copy()

print("Candidates:", len(s2_compact_high))

cols = [
    "entity_id_s1",
    "entity_id_s2",
    "business_name_s1",
    "business_name_s2",
    "s1_business_address",
    "s2_business_address",
    "country",
    "name_ratio",
    "address_ratio"
]

print(
    s2_compact_high[
        cols
    ]
    .sort_values(
        ["name_ratio", "address_ratio"],
        ascending=False
    )
    .to_string(index=False)
)

Candidates: 22
entity_id_s1 entity_id_s2                         business_name_s1                         business_name_s2                                                                                    s1_business_address                                                                   s2_business_address country  name_ratio  address_ratio
S1-886703513 S2-598342969                Lions Rendez Parents SASU                Lions Rendez Parents SASU                                         133 RUE P BROSSOLETTE, 59100 ROUBAIX, Roubaix, Hauts-de-France                                        133 RUE P BOSSOLETTE, ROUBAIX, Hauts-de-France  France  100.000000      85.185185
S1-874929284  S2-10488643                           Bulles Culture                           Bulles Culture  19 ESPL1 DES ANTILLES, UNIVERSITE BORDEAUX MONTAIGNE MAISON DES ETUDIANTS, Pessac, Nouvelle-Aquitaine     19 ESPL1 DES ANTILLES, UNIVERSITE BORDEAUX MONTAIGNE MAISON DES ETUDIANTS, PESSAC  France  100.000000     

In [68]:
# ============================================
# NORMALIZED ADDRESS SIMILARITY
# ============================================

from rapidfuzz.fuzz import ratio, token_set_ratio

def normalize_address_for_match(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    import unicodedata
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    # Common address abbreviations
    replacements = {
        "road": "rd",
        "street": "st",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "place": "pl",
        "parkway": "pkwy",
        "highway": "hwy",
        "apartment": "apt",
        "floor": "fl",
        "number": "no",
    }

    text = re.sub(r"[^a-z0-9]+", " ", text)
    tokens = text.split()

    tokens = [
        replacements.get(t, t)
        for t in tokens
    ]

    return " ".join(tokens)


s2_compact_scored["norm_addr_s1"] = (
    s2_compact_scored["s1_business_address"]
    .map(normalize_address_for_match)
)

s2_compact_scored["norm_addr_s2"] = (
    s2_compact_scored["s2_business_address"]
    .map(normalize_address_for_match)
)

# Two address metrics
s2_compact_scored["norm_address_ratio"] = [
    ratio(a, b)
    for a, b in zip(
        s2_compact_scored["norm_addr_s1"],
        s2_compact_scored["norm_addr_s2"]
    )
]

s2_compact_scored["norm_address_token_ratio"] = [
    token_set_ratio(a, b)
    for a, b in zip(
        s2_compact_scored["norm_addr_s1"],
        s2_compact_scored["norm_addr_s2"]
    )
]

print(
    s2_compact_scored[
        [
            "address_ratio",
            "norm_address_ratio",
            "norm_address_token_ratio"
        ]
    ].describe()
)

       address_ratio  norm_address_ratio  norm_address_token_ratio
count  274944.000000       274944.000000             274944.000000
mean       37.275547           73.506795                 87.606180
std        12.651465           23.843498                 24.437408
min         0.000000            0.000000                  0.000000
25%        29.411765           64.197531                 89.552239
50%        37.522563           80.536913                 96.774194
75%        45.528455           90.322581                100.000000
max        96.875000          100.000000                100.000000


In [69]:
# ============================================
# S2 COMPACT + NORMALIZED ADDRESS THRESHOLDS
# ============================================

thresholds = [
    (95, 90),
    (95, 92),
    (95, 94),
    (95, 95),
    (95, 97),
    (97, 90),
    (97, 92),
    (97, 94),
    (97, 95),
    (98, 90),
    (98, 92),
    (98, 94),
    (99, 90),
    (99, 92),
    (99, 94),
    (100, 90),
    (100, 92),
    (100, 94),
]

for name_t, addr_t in thresholds:

    x = s2_compact_scored[
        (s2_compact_scored["name_ratio"] >= name_t) &
        (s2_compact_scored["norm_address_ratio"] >= addr_t)
    ]

    print(
        f"name >= {name_t:>3}, "
        f"normalized address >= {addr_t:>2}: "
        f"{len(x):>6} pairs, "
        f"{x['entity_id_s1'].nunique():>6} S1"
    )

name >=  95, normalized address >= 90:  11440 pairs,  11440 S1
name >=  95, normalized address >= 92:   7409 pairs,   7409 S1
name >=  95, normalized address >= 94:   3851 pairs,   3851 S1
name >=  95, normalized address >= 95:   2726 pairs,   2726 S1
name >=  95, normalized address >= 97:   1959 pairs,   1959 S1
name >=  97, normalized address >= 90:   7274 pairs,   7274 S1
name >=  97, normalized address >= 92:   4642 pairs,   4642 S1
name >=  97, normalized address >= 94:   2325 pairs,   2325 S1
name >=  97, normalized address >= 95:   1588 pairs,   1588 S1
name >=  98, normalized address >= 90:   5235 pairs,   5235 S1
name >=  98, normalized address >= 92:   3351 pairs,   3351 S1
name >=  98, normalized address >= 94:   1694 pairs,   1694 S1
name >=  99, normalized address >= 90:   3315 pairs,   3315 S1
name >=  99, normalized address >= 92:   2159 pairs,   2159 S1
name >=  99, normalized address >= 94:   1144 pairs,   1144 S1
name >= 100, normalized address >= 90:   3306 pairs,   

In [70]:
# ============================================
# INSPECT STRONG S2 COMPACT MATCHES
# ============================================

s2_compact_strong = s2_compact_scored[
    (s2_compact_scored["name_ratio"] >= 100) &
    (s2_compact_scored["norm_address_ratio"] >= 94)
].copy()

print("Strong candidates:", len(s2_compact_strong))

cols = [
    "entity_id_s1",
    "entity_id_s2",
    "business_name_s1",
    "business_name_s2",
    "s1_business_address",
    "s2_business_address",
    "country",
    "name_ratio",
    "address_ratio",
    "norm_address_ratio"
]

print(
    s2_compact_strong[cols]
    .sort_values(
        ["norm_address_ratio", "name_ratio"],
        ascending=[True, False]
    )
    .to_string(index=False)
)

Strong candidates: 1143
entity_id_s1 entity_id_s2                                    business_name_s1                                    business_name_s2                                                                                                                           s1_business_address                                                                                                                     s2_business_address country  name_ratio  address_ratio  norm_address_ratio
S1-382323425 S2-951998232                          Kool Paper Private Limited                          Kool Paper Private Limited                     Plot No. 91- B, Kh No. 1063, Ground Floor Blk- U, Prem Nagar Ph 2, Landmark Near N, Aag Mandir, Delhi, North Delhi, Delhi                            PLOT NO. 91- B, KH NO. 1063, GROUND FLOOR BLK- U, PREM NAGAR PH 2, LANDMARK NEAR N, AAG MANDIR, DELHI, Delhi   India       100.0      55.021834           94.000000
S1-149472596 S2-508832639                         

In [71]:
# ============================================
# CHECK ADDRESS NUMBER COMPATIBILITY
# ============================================

def extract_address_numbers(text):
    if pd.isna(text):
        return set()

    return set(re.findall(r"\d+", str(text)))


s2_compact_strong["nums_s1"] = (
    s2_compact_strong["s1_business_address"]
    .map(extract_address_numbers)
)

s2_compact_strong["nums_s2"] = (
    s2_compact_strong["s2_business_address"]
    .map(extract_address_numbers)
)

def numbers_compatible(a, b):
    # If neither address has numbers, don't reject it.
    if not a and not b:
        return True

    # If both have numbers, require at least one shared number.
    if a and b:
        return bool(a & b)

    # One has numbers and the other doesn't -> risky.
    return False


s2_compact_strong["numbers_compatible"] = [
    numbers_compatible(a, b)
    for a, b in zip(
        s2_compact_strong["nums_s1"],
        s2_compact_strong["nums_s2"]
    )
]

print(
    s2_compact_strong["numbers_compatible"]
    .value_counts()
)

print(
    "\nSafe candidates:",
    s2_compact_strong["numbers_compatible"].sum()
)

print(
    "Rejected:",
    (~s2_compact_strong["numbers_compatible"]).sum()
)

numbers_compatible
True     929
False    214
Name: count, dtype: int64

Safe candidates: 929
Rejected: 214


In [72]:
# ============================================
# INSPECT SAFE COMPACT MATCHES
# ============================================

safe_compact = s2_compact_strong[
    s2_compact_strong["numbers_compatible"]
].copy()

print("Safe candidates:", len(safe_compact))

cols = [
    "entity_id_s1",
    "entity_id_s2",
    "business_name_s1",
    "business_name_s2",
    "s1_business_address",
    "s2_business_address",
    "name_ratio",
    "address_ratio",
    "norm_address_ratio",
    "nums_s1",
    "nums_s2"
]

# Show the weakest 100 first
weakest = safe_compact.sort_values(
    ["norm_address_ratio", "address_ratio"],
    ascending=[True, True]
).head(100)

print(weakest[cols].to_string(index=False))

Safe candidates: 929
entity_id_s1 entity_id_s2                            business_name_s1                            business_name_s2                                                                                                                           s1_business_address                                                                                                                     s2_business_address  name_ratio  address_ratio  norm_address_ratio                      nums_s1                       nums_s2
S1-444129290 S2-528692304                      Sid (India) Properties                      Sid (India) Properties                                                                                    C/O- Balwant Singh, Vill-Basti Jalal, Chapra, Saran, Bihar                                                                                     C/O- BALWANT SINGH, VILL-BASTI JALAL, CHAPRA, Bihar       100.0      45.871560           94.000000                           {}              

In [73]:
# ============================================
# STRICTER ADDRESS NUMBER COMPATIBILITY
# ============================================

def first_address_number(text):
    if pd.isna(text):
        return None

    nums = re.findall(r"\d+", str(text))

    if not nums:
        return None

    # Normalize leading zeros: 00105 -> 105
    return str(int(nums[0]))


safe_compact["first_num_s1"] = (
    safe_compact["s1_business_address"]
    .map(first_address_number)
)

safe_compact["first_num_s2"] = (
    safe_compact["s2_business_address"]
    .map(first_address_number)
)

safe_compact["first_num_match"] = (
    (safe_compact["first_num_s1"] == safe_compact["first_num_s2"]) &
    safe_compact["first_num_s1"].notna()
)

print(
    safe_compact["first_num_match"].value_counts()
)

print(
    "\nFirst-number matches:",
    safe_compact["first_num_match"].sum()
)

print(
    "No first-number match:",
    (~safe_compact["first_num_match"]).sum()
)

first_num_match
True     859
False     70
Name: count, dtype: int64

First-number matches: 859
No first-number match: 70


In [74]:
# ============================================
# FINAL HIGH-CONFIDENCE S2 COMPACT POOL
# ============================================

compact_final = safe_compact[
    (safe_compact["first_num_match"]) &
    (safe_compact["norm_address_ratio"] >= 94) &
    (safe_compact["address_ratio"] >= 50)
].copy()

print("Final compact candidates:", len(compact_final))
print("Unique S1:", compact_final["entity_id_s1"].nunique())

print("\nAddress ratio distribution:")
print(compact_final["address_ratio"].describe())

print("\nNormalized address ratio:")
print(compact_final["norm_address_ratio"].describe())

Final compact candidates: 491
Unique S1: 491

Address ratio distribution:
count    491.000000
mean      57.226180
std        6.385608
min       50.000000
25%       52.500695
50%       55.357143
75%       60.380747
max       82.352941
Name: address_ratio, dtype: float64

Normalized address ratio:
count    491.000000
mean      97.458736
std        2.351164
min       94.000000
25%       94.858578
50%       97.777778
75%      100.000000
max      100.000000
Name: norm_address_ratio, dtype: float64


In [75]:
# ============================================
# CHECK OVERLAP WITH CURRENT MATCHES
# ============================================

# baseline is your current 0.228 matching_results dataframe

existing_s2_pairs = set()

for _, row in baseline.iterrows():
    s1 = row["source1_entity_id"]
    ids = str(row["matched_entity_ids"])

    if ids and ids != "nan":
        for x in ids.split(","):
            x = x.strip()
            if x.startswith("S2-"):
                existing_s2_pairs.add((s1, x))

compact_final["pair"] = list(
    zip(
        compact_final["entity_id_s1"],
        compact_final["entity_id_s2"]
    )
)

compact_final["already_existing"] = (
    compact_final["pair"].isin(existing_s2_pairs)
)

print(
    compact_final["already_existing"].value_counts()
)

print(
    "\nNew pairs:",
    (~compact_final["already_existing"]).sum()
)

already_existing
False    491
Name: count, dtype: int64

New pairs: 491


In [76]:
# ============================================
# ADD 491 HIGH-CONFIDENCE S2 PAIRS
# ============================================

new_rows = compact_final[
    ~compact_final["already_existing"]
][[
    "entity_id_s1",
    "entity_id_s2"
]].copy()

new_rows.columns = [
    "source1_entity_id",
    "matched_entity_id"
]

# Convert to lookup dictionary
new_lookup = {}

for _, row in new_rows.iterrows():
    s1 = row["source1_entity_id"]
    s2 = row["matched_entity_id"]
    new_lookup.setdefault(s1, []).append(s2)

# Start from current submission
improved_491 = baseline.copy()

def add_new_ids(row):
    s1 = row["source1_entity_id"]
    existing = str(row["matched_entity_ids"])

    additions = new_lookup.get(s1, [])

    if not additions:
        return existing

    existing_ids = set(
        x.strip()
        for x in existing.split(",")
        if x.strip()
    )

    for x in additions:
        existing_ids.add(x)

    return ",".join(sorted(existing_ids))

improved_491["matched_entity_ids"] = improved_491.apply(
    add_new_ids,
    axis=1
)

# Save
output_path = "/content/matching_results_compact491.tsv"

improved_491.to_csv(
    output_path,
    sep="\t",
    index=False
)

print("Saved:", output_path)
print("Rows:", len(improved_491))

# Basic checks
print(
    "Duplicate S1:",
    improved_491["source1_entity_id"].duplicated().sum()
)

print(
    "Matched S1:",
    (improved_491["matched_entity_ids"].astype(str).str.len() > 0).sum()
)

print(
    "No-match S1:",
    (improved_491["matched_entity_ids"].astype(str).str.len() == 0).sum()
)

print(
    "Pairs added:",
    len(new_rows)
)

Saved: /content/matching_results_compact491.tsv
Rows: 1732544
Duplicate S1: 0
Matched S1: 1732544
No-match S1: 0
Pairs added: 491


In [77]:
# ============================================
# VALIDATE NEW SUBMISSION
# ============================================

test_ids_s2 = set(test_s2["entity_id"].astype(str))
test_ids_s3 = set(test_s3["entity_id"].astype(str))

duplicate_s1 = improved_491["source1_entity_id"].duplicated().sum()

invalid_ids = []
duplicate_match_ids = 0
total_pairs = 0

for _, row in improved_491.iterrows():

    ids = str(row["matched_entity_ids"])

    if not ids or ids == "nan":
        continue

    parts = [x.strip() for x in ids.split(",") if x.strip()]

    if len(parts) != len(set(parts)):
        duplicate_match_ids += 1

    for x in parts:
        total_pairs += 1

        if x.startswith("S2-"):
            if x not in test_ids_s2:
                invalid_ids.append(x)

        elif x.startswith("S3-"):
            if x not in test_ids_s3:
                invalid_ids.append(x)

        else:
            invalid_ids.append(x)

print("Duplicate S1:", duplicate_s1)
print("Rows:", len(improved_491))
print("Total submitted pairs:", total_pairs)
print("Rows with duplicate matched IDs:", duplicate_match_ids)
print("Invalid IDs:", len(invalid_ids))

print(
    "\nVALID:",
    duplicate_s1 == 0
    and duplicate_match_ids == 0
    and len(invalid_ids) == 0
)

NameError: name 'test_s2' is not defined

In [78]:
# Check whether the new file accidentally converted empty matches to "nan"

print("NaN values:",
      improved_491["matched_entity_ids"].isna().sum())

print("Literal 'nan' values:",
      (improved_491["matched_entity_ids"].astype(str) == "nan").sum())

print("Empty strings:",
      (improved_491["matched_entity_ids"].fillna("").astype(str).str.strip() == "").sum())

print("\nSample values:")
print(improved_491["matched_entity_ids"].head(20).tolist())

NaN values: 0
Literal 'nan' values: 1306210
Empty strings: 0

Sample values:
['nan', 'nan', 'nan', 'nan', 'nan', 'nan', 'S2-528901108', 'nan', 'nan', 'nan', 'S2-262582939', 'nan', 'nan', 'S2-487943573', 'nan', 'nan', 'nan', 'nan', 'nan', 'nan']


In [79]:
# ============================================
# FIX "nan" VALUES AND CREATE CLEAN SUBMISSION
# ============================================

clean_491 = improved_491.copy()

# Convert literal "nan" back to empty matches
clean_491["matched_entity_ids"] = (
    clean_491["matched_entity_ids"]
    .replace("nan", "")
)

# Also handle actual NaN just in case
clean_491["matched_entity_ids"] = (
    clean_491["matched_entity_ids"]
    .fillna("")
    .astype(str)
    .str.strip()
)

output_path = "/content/matching_results_compact491_clean.tsv"

clean_491.to_csv(
    output_path,
    sep="\t",
    index=False
)

# Verify counts
has_match = clean_491["matched_entity_ids"].ne("")

total_pairs = clean_491["matched_entity_ids"].apply(
    lambda x: len([v for v in x.split(",") if v.strip()])
).sum()

print("Saved:", output_path)
print("Rows:", len(clean_491))
print("Matched S1:", has_match.sum())
print("No-match S1:", (~has_match).sum())
print("Total pairs:", total_pairs)
print("Duplicate S1:", clean_491["source1_entity_id"].duplicated().sum())
print("Literal nan:", (clean_491["matched_entity_ids"] == "nan").sum())
print("Actual empty:", (clean_491["matched_entity_ids"] == "").sum())

Saved: /content/matching_results_compact491_clean.tsv
Rows: 1732544
Matched S1: 426334
No-match S1: 1306210
Total pairs: 565451
Duplicate S1: 0
Literal nan: 0
Actual empty: 1306210


In [80]:
# ============================================
# COMPARE CLEAN FILE WITH ORIGINAL 0.228 FILE
# ============================================

original = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

new = clean_491.copy()

def pair_set(df):
    pairs = set()

    for _, row in df.iterrows():
        s1 = row["source1_entity_id"]
        ids = str(row["matched_entity_ids"])

        if ids == "nan" or not ids.strip():
            continue

        for s2 in ids.split(","):
            s2 = s2.strip()
            if s2:
                pairs.add((s1, s2))

    return pairs

old_pairs = pair_set(original)
new_pairs = pair_set(new)

added_pairs = new_pairs - old_pairs
removed_pairs = old_pairs - new_pairs

print("Original pairs:", len(old_pairs))
print("New pairs:", len(new_pairs))
print("Added pairs:", len(added_pairs))
print("Removed pairs:", len(removed_pairs))

print("\nExpected new pair count:")
print("Original + added =", len(old_pairs) + len(added_pairs))

print("\nS1 counts:")
print("Original matched S1:",
      sum(
          bool(str(x).strip()) and str(x) != "nan"
          for x in original["matched_entity_ids"]
      ))

print("New matched S1:",
      new["matched_entity_ids"].ne("").sum())

KeyboardInterrupt: 

In [81]:
# ============================================
# FAST COMPARISON WITH 0.228 SUBMISSION
# ============================================

original = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

new = clean_491.copy()

# Normalize empty/nan values
original["matched_entity_ids"] = (
    original["matched_entity_ids"]
    .fillna("")
    .replace("nan", "")
    .astype(str)
    .str.strip()
)

new["matched_entity_ids"] = (
    new["matched_entity_ids"]
    .fillna("")
    .replace("nan", "")
    .astype(str)
    .str.strip()
)

# Count pairs without creating giant Python sets
def count_pairs(df):
    return (
        df["matched_entity_ids"]
        .str.split(",")
        .explode()
        .str.strip()
        .loc[lambda x: x.ne("")]
        .shape[0]
    )

old_pair_count = count_pairs(original)
new_pair_count = count_pairs(new)

print("Original pairs:", old_pair_count)
print("New pairs:", new_pair_count)
print("Difference:", new_pair_count - old_pair_count)

print("\nOriginal matched S1:",
      original["matched_entity_ids"].ne("").sum())

print("New matched S1:",
      new["matched_entity_ids"].ne("").sum())

print("\nOriginal no-match S1:",
      original["matched_entity_ids"].eq("").sum())

print("New no-match S1:",
      new["matched_entity_ids"].eq("").sum())

Original pairs: 564469
New pairs: 565451
Difference: 982

Original matched S1: 425843
New matched S1: 426334

Original no-match S1: 1306701
New no-match S1: 1306210


In [82]:
# ============================================
# BUILD CLEAN FINAL FROM ORIGINAL 0.228 FILE
# ============================================

# Reload the original submission
original = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

# Properly convert missing values to empty strings FIRST
original["matched_entity_ids"] = (
    original["matched_entity_ids"]
    .fillna("")
    .replace("nan", "")
    .astype(str)
    .str.strip()
)

# Build lookup from the 491 high-confidence pairs
new_lookup = {}

for _, row in compact_final.iterrows():
    s1 = row["entity_id_s1"]
    s2 = row["entity_id_s2"]

    new_lookup.setdefault(s1, []).append(s2)

# Add exactly one new S2 ID to each of the 491 S1s
def add_pair_correctly(row):
    s1 = row["source1_entity_id"]
    existing = row["matched_entity_ids"]

    additions = new_lookup.get(s1, [])

    if not additions:
        return existing

    existing_ids = [
        x.strip()
        for x in existing.split(",")
        if x.strip()
    ]

    for x in additions:
        if x not in existing_ids:
            existing_ids.append(x)

    return ",".join(existing_ids)

final_491 = original.copy()

final_491["matched_entity_ids"] = final_491.apply(
    add_pair_correctly,
    axis=1
)

# Save
final_path = "/content/matching_results_compact491_FINAL.tsv"

final_491.to_csv(
    final_path,
    sep="\t",
    index=False
)

# ============================================
# VERIFY
# ============================================

has_match = final_491["matched_entity_ids"].ne("")

total_pairs = (
    final_491["matched_entity_ids"]
    .str.split(",")
    .explode()
    .str.strip()
    .loc[lambda x: x.ne("")]
    .shape[0]
)

print("Saved:", final_path)
print()
print("Rows:", len(final_491))
print("Matched S1:", has_match.sum())
print("No-match S1:", (~has_match).sum())
print("Total pairs:", total_pairs)
print("Duplicate S1:",
      final_491["source1_entity_id"].duplicated().sum())

print(
    "Literal nan:",
    (final_491["matched_entity_ids"] == "nan").sum()
)

Saved: /content/matching_results_compact491_FINAL.tsv

Rows: 1732544
Matched S1: 426334
No-match S1: 1306210
Total pairs: 564960
Duplicate S1: 0
Literal nan: 0


In [83]:
with open("/content/candidate_pairs.tsv", "r", encoding="utf-8") as f:
    print(f.readline().strip())

entity_id_s1	entity_id_matched	source


In [84]:
# ============================================
# CHECK WHETHER ALL 491 NEW S2 PAIRS
# EXIST IN candidate_pairs.tsv
# ============================================

# Create the 491 target pairs
target_pairs = set(
    zip(
        compact_final["entity_id_s1"].astype(str),
        compact_final["entity_id_s2"].astype(str)
    )
)

print("Pairs we need to find:", len(target_pairs))

found_pairs = set()

# Read the large candidate file in chunks
for chunk in pd.read_csv(
    "/content/candidate_pairs.tsv",
    sep="\t",
    dtype=str,
    chunksize=1_000_000
):
    # Only S2 candidates matter
    chunk = chunk[chunk["source"].astype(str) == "S2"]

    pairs = zip(
        chunk["entity_id_s1"],
        chunk["entity_id_matched"]
    )

    for pair in pairs:
        if pair in target_pairs:
            found_pairs.add(pair)

print("Found in candidate file:", len(found_pairs))
print("Missing from candidate file:", len(target_pairs - found_pairs))

missing_pairs = target_pairs - found_pairs

if missing_pairs:
    print("\nFirst missing pairs:")
    for pair in list(missing_pairs)[:20]:
        print(pair)
else:
    print("\nALL 491 PAIRS ARE IN candidate_pairs.tsv")

Pairs we need to find: 491
Found in candidate file: 491
Missing from candidate file: 0

ALL 491 PAIRS ARE IN candidate_pairs.tsv


In [85]:
from google.colab import files

files.download("/content/matching_results_compact491_FINAL.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [86]:
# ============================================
# INSPECT EXISTING S2 ADDRESS LOOKUP
# ============================================

import pandas as pd

s2_lookup = pd.read_csv(
    "/content/s2_address_lookup.tsv",
    sep="\t",
    dtype=str,
    nrows=5
)

print(s2_lookup.columns.tolist())
print()
print(s2_lookup.head())

['entity_id', 'business_name', 'business_address', 'country']

      entity_id         business_name  \
0   S2-89663826           Apex Summit   
1  S2-884102769          Fresh Truist   
2  S2-292567038  KERALA INDIA LIMITED   
3  S2-357438552     Dmb Skills L.L.P.   
4  S2-223826524              Rizabrix   

                                    business_address country  
0                   67 KENTUCKY ST, SALYERSVILLE, KY      US  
1               8264 FILLY COURT, ROANOKE COUNTY, VA      US  
2  P. NO. 482, KN 131 & 133 HANWANT A, BJS COLONY...   India  
3  C/O GRP SURYA PEB ENGG CONST. OFFCE NO 263 264...   India  
4  PUNE, GODREJ INFINITY-3A HAVELI, S.NO. ##9 TO ...   India  


In [87]:
# ============================================
# STEP 2 — GET ZERO-MATCH S1s FROM 0.228
# ============================================

baseline = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

baseline["matched_entity_ids"] = (
    baseline["matched_entity_ids"]
    .fillna("")
    .replace("nan", "")
    .astype(str)
    .str.strip()
)

zero_match_s1 = set(
    baseline.loc[
        baseline["matched_entity_ids"].eq(""),
        "source1_entity_id"
    ]
)

print("Total S1:", len(baseline))
print("Zero-match S1:", len(zero_match_s1))
print("Matched S1:", len(baseline) - len(zero_match_s1))

Total S1: 1732544
Zero-match S1: 1306701
Matched S1: 425843


In [88]:
# ============================================
# STEP 3 — ZERO-MATCH S1 × S2 ADDRESS CANDIDATES
# ============================================

s2_addr_candidates = pd.read_csv(
    "/content/test_s2_address_candidates.tsv",
    sep="\t",
    dtype=str
)

print("All S2 address candidates:", len(s2_addr_candidates))

zero_s2 = s2_addr_candidates[
    s2_addr_candidates["entity_id_s1"].isin(zero_match_s1)
].copy()

print("Zero-match S1 S2 candidates:", len(zero_s2))
print("Unique zero-match S1:", zero_s2["entity_id_s1"].nunique())
print("Unique S2 IDs:", zero_s2["entity_id_matched"].nunique())

print("\nColumns:")
print(zero_s2.columns.tolist())

All S2 address candidates: 1408537
Zero-match S1 S2 candidates: 995466
Unique zero-match S1: 264254
Unique S2 IDs: 598234

Columns:
['entity_id_s1', 'entity_id_matched', 'source']


In [1]:
# ============================================
# STEP 4 — ATTACH S1 + S2 RECORD DETAILS
# ============================================

# Load only the columns we need
s1_info = pd.read_csv(
    "/content/test_source1.tsv",
    sep="\t",
    dtype=str,
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

# Load the S2 address lookup
s2_info = pd.read_csv(
    "/content/s2_address_lookup.tsv",
    sep="\t",
    dtype=str
)

# Rename for merging
s1_info = s1_info.rename(columns={
    "entity_id": "entity_id_s1",
    "business_name": "s1_name",
    "business_address": "s1_address",
    "country": "s1_country"
})

s2_info = s2_info.rename(columns={
    "entity_id": "entity_id_matched",
    "business_name": "s2_name",
    "business_address": "s2_address",
    "country": "s2_country"
})

# Keep only S1s that actually occur in our candidate set
needed_s1 = zero_s2["entity_id_s1"].unique()
needed_s2 = zero_s2["entity_id_matched"].unique()

s1_small = s1_info[
    s1_info["entity_id_s1"].isin(needed_s1)
].copy()

s2_small = s2_info[
    s2_info["entity_id_matched"].isin(needed_s2)
].copy()

print("S1 records loaded:", len(s1_small))
print("S2 records loaded:", len(s2_small))

# Merge
zero_scored = zero_s2.merge(
    s1_small,
    on="entity_id_s1",
    how="left"
)

zero_scored = zero_scored.merge(
    s2_small,
    on="entity_id_matched",
    how="left"
)

print("Candidate rows:", len(zero_scored))
print("Missing S1 names:",
      zero_scored["s1_name"].isna().sum())
print("Missing S2 names:",
      zero_scored["s2_name"].isna().sum())

print("\nColumns:")
print(zero_scored.columns.tolist())

NameError: name 'pd' is not defined

In [2]:
# ============================================
# STEP 4 — ATTACH S1 + S2 RECORD DETAILS
# ============================================

import pandas as pd

# Load only the columns we need
s1_info = pd.read_csv(
    "/content/test_source1.tsv",
    sep="\t",
    dtype=str,
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

s2_info = pd.read_csv(
    "/content/s2_address_lookup.tsv",
    sep="\t",
    dtype=str
)

# Rename columns
s1_info = s1_info.rename(columns={
    "entity_id": "entity_id_s1",
    "business_name": "s1_name",
    "business_address": "s1_address",
    "country": "s1_country"
})

s2_info = s2_info.rename(columns={
    "entity_id": "entity_id_matched",
    "business_name": "s2_name",
    "business_address": "s2_address",
    "country": "s2_country"
})

# Get IDs actually needed
needed_s1 = zero_s2["entity_id_s1"].unique()
needed_s2 = zero_s2["entity_id_matched"].unique()

s1_small = s1_info[
    s1_info["entity_id_s1"].isin(needed_s1)
].copy()

s2_small = s2_info[
    s2_info["entity_id_matched"].isin(needed_s2)
].copy()

print("S1 records loaded:", len(s1_small))
print("S2 records loaded:", len(s2_small))

# Merge
zero_scored = zero_s2.merge(
    s1_small,
    on="entity_id_s1",
    how="left"
)

zero_scored = zero_scored.merge(
    s2_small,
    on="entity_id_matched",
    how="left"
)

print("Candidate rows:", len(zero_scored))
print("Missing S1 names:",
      zero_scored["s1_name"].isna().sum())
print("Missing S2 names:",
      zero_scored["s2_name"].isna().sum())

print("\nColumns:")
print(zero_scored.columns.tolist())

NameError: name 'zero_s2' is not defined

In [3]:
# ============================================
# RECOVER ZERO-MATCH S2 ADDRESS CANDIDATES
# ============================================

import pandas as pd

# Load baseline submission
baseline = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

baseline["matched_entity_ids"] = (
    baseline["matched_entity_ids"]
    .fillna("")
    .replace("nan", "")
    .astype(str)
    .str.strip()
)

# S1 entities with NO match in the 0.228 submission
zero_match_s1 = set(
    baseline.loc[
        baseline["matched_entity_ids"].eq(""),
        "source1_entity_id"
    ]
)

print("Zero-match S1:", len(zero_match_s1))

# Load existing S2 address candidates
s2_addr_candidates = pd.read_csv(
    "/content/test_s2_address_candidates.tsv",
    sep="\t",
    dtype=str
)

print("All S2 address candidates:", len(s2_addr_candidates))

# Keep only candidates belonging to zero-match S1s
zero_s2 = s2_addr_candidates[
    s2_addr_candidates["entity_id_s1"].isin(zero_match_s1)
].copy()

print("Zero-match S2 candidates:", len(zero_s2))
print("Unique S1:", zero_s2["entity_id_s1"].nunique())
print("Unique S2:", zero_s2["entity_id_matched"].nunique())

Zero-match S1: 1306701
All S2 address candidates: 1408537
Zero-match S2 candidates: 995466
Unique S1: 264254
Unique S2: 598234


In [4]:
# ============================================
# STREAMING RECOVERY — NO LARGE DATAFRAME
# ============================================

import pandas as pd
import re
import unicodedata
from rapidfuzz.fuzz import ratio

print("Environment ready")

Environment ready


In [5]:
# ============================================
# STEP 1 — STREAM ZERO-MATCH CANDIDATES
# ============================================

import pandas as pd
import os

# Reload baseline
baseline = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

baseline["matched_entity_ids"] = (
    baseline["matched_entity_ids"]
    .fillna("")
    .replace("nan", "")
    .astype(str)
    .str.strip()
)

zero_match_s1 = set(
    baseline.loc[
        baseline["matched_entity_ids"].eq(""),
        "source1_entity_id"
    ]
)

print("Zero-match S1:", len(zero_match_s1))

# Stream candidate file
candidate_path = "/content/test_s2_address_candidates.tsv"
filtered_path = "/content/zero_s2_candidates.tsv"

# Remove old file if it exists
if os.path.exists(filtered_path):
    os.remove(filtered_path)

total_kept = 0
first_write = True

for chunk in pd.read_csv(
    candidate_path,
    sep="\t",
    dtype=str,
    chunksize=200_000
):
    part = chunk[
        chunk["entity_id_s1"].isin(zero_match_s1)
    ]

    if len(part) > 0:
        part.to_csv(
            filtered_path,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False
        total_kept += len(part)

print("Filtered candidates:", total_kept)
print("Saved:", filtered_path)

Zero-match S1: 1306701
Filtered candidates: 995466
Saved: /content/zero_s2_candidates.tsv


In [6]:
# ============================================
# STEP 2 — EXTRACT ONLY NEEDED RECORDS
# ============================================

import pandas as pd
import os

# Get the unique IDs from the filtered candidate file
needed_s1 = set()
needed_s2 = set()

for chunk in pd.read_csv(
    "/content/zero_s2_candidates.tsv",
    sep="\t",
    dtype=str,
    chunksize=200_000
):
    needed_s1.update(chunk["entity_id_s1"])
    needed_s2.update(chunk["entity_id_matched"])

print("Needed S1:", len(needed_s1))
print("Needed S2:", len(needed_s2))

# --------------------------------------------
# Extract S1 records
# --------------------------------------------

s1_needed_path = "/content/zero_needed_s1.tsv"

if os.path.exists(s1_needed_path):
    os.remove(s1_needed_path)

s1_count = 0
first_write = True

for chunk in pd.read_csv(
    "/content/test_source1.tsv",
    sep="\t",
    dtype=str,
    chunksize=200_000
):
    part = chunk[
        chunk["entity_id"].isin(needed_s1)
    ]

    if len(part):
        part.to_csv(
            s1_needed_path,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )
        first_write = False
        s1_count += len(part)

print("S1 records extracted:", s1_count)

# --------------------------------------------
# Extract S2 records
# --------------------------------------------

s2_needed_path = "/content/zero_needed_s2.tsv"

if os.path.exists(s2_needed_path):
    os.remove(s2_needed_path)

s2_count = 0
first_write = True

for chunk in pd.read_csv(
    "/content/s2_address_lookup.tsv",
    sep="\t",
    dtype=str,
    chunksize=200_000
):
    part = chunk[
        chunk["entity_id"].isin(needed_s2)
    ]

    if len(part):
        part.to_csv(
            s2_needed_path,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )
        first_write = False
        s2_count += len(part)

print("S2 records extracted:", s2_count)

print("\nFiles created:")
print(s1_needed_path)
print(s2_needed_path)

Needed S1: 264254
Needed S2: 598234
S1 records extracted: 264254
S2 records extracted: 598234

Files created:
/content/zero_needed_s1.tsv
/content/zero_needed_s2.tsv


In [7]:
# ============================================
# STEP 3 — STREAMING FUZZY SCORING
# ============================================

import pandas as pd
import re
import unicodedata
from rapidfuzz.fuzz import ratio

# --------------------------------------------
# Normalization
# --------------------------------------------

def norm_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def norm_address(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    replacements = {
        "road": "rd",
        "street": "st",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "place": "pl",
        "parkway": "pkwy",
        "highway": "hwy",
        "apartment": "apt",
        "floor": "fl",
        "number": "no",
    }

    text = re.sub(r"[^a-z0-9]+", " ", text)
    tokens = text.split()
    tokens = [replacements.get(t, t) for t in tokens]

    return " ".join(tokens)


# --------------------------------------------
# Load only the extracted records
# --------------------------------------------

s1 = pd.read_csv(
    "/content/zero_needed_s1.tsv",
    sep="\t",
    dtype=str
)

s2 = pd.read_csv(
    "/content/zero_needed_s2.tsv",
    sep="\t",
    dtype=str
)

print("S1:", len(s1))
print("S2:", len(s2))

# --------------------------------------------
# Pre-normalize once
# --------------------------------------------

s1["name_norm"] = s1["business_name"].map(norm_text)
s1["addr_norm"] = s1["business_address"].map(norm_address)

s2["name_norm"] = s2["business_name"].map(norm_text)
s2["addr_norm"] = s2["business_address"].map(norm_address)

# Dictionaries for fast lookup
s1_map = s1.set_index("entity_id")[
    ["business_name", "business_address", "country",
     "name_norm", "addr_norm"]
].to_dict("index")

s2_map = s2.set_index("entity_id")[
    ["business_name", "business_address", "country",
     "name_norm", "addr_norm"]
].to_dict("index")

print("Normalization complete.")

# --------------------------------------------
# Stream candidates
# --------------------------------------------

candidate_path = "/content/zero_s2_candidates.tsv"

results = []

processed = 0

for chunk in pd.read_csv(
    candidate_path,
    sep="\t",
    dtype=str,
    chunksize=100_000
):

    for s1_id, s2_id in zip(
        chunk["entity_id_s1"],
        chunk["entity_id_matched"]
    ):

        a = s1_map.get(s1_id)
        b = s2_map.get(s2_id)

        if a is None or b is None:
            continue

        name_score = ratio(
            a["name_norm"],
            b["name_norm"]
        )

        if name_score < 90:
            continue

        addr_score = ratio(
            a["addr_norm"],
            b["addr_norm"]
        )

        if addr_score < 95:
            continue

        results.append({
            "entity_id_s1": s1_id,
            "entity_id_s2": s2_id,
            "name_ratio": name_score,
            "norm_address_ratio": addr_score,
            "s1_name": a["business_name"],
            "s2_name": b["business_name"],
            "s1_address": a["business_address"],
            "s2_address": b["business_address"],
            "country": a["country"]
        })

    processed += len(chunk)

    if processed % 500_000 == 0:
        print("Processed:", processed)

scored_candidates = pd.DataFrame(results)

print("\n============================================")
print("SCORING COMPLETE")
print("============================================")
print("Candidates processed:", processed)
print("High-confidence candidates:", len(scored_candidates))
print(
    "Unique S1:",
    scored_candidates["entity_id_s1"].nunique()
    if len(scored_candidates) else 0
)

S1: 264254
S2: 598234
Normalization complete.
Processed: 500000

SCORING COMPLETE
Candidates processed: 995466
High-confidence candidates: 9751
Unique S1: 8792


In [8]:
# ============================================
# STEP 4 — THRESHOLD + AMBIGUITY ANALYSIS
# ============================================

def analyze_threshold(name_thr, addr_thr):

    x = scored_candidates[
        (scored_candidates["name_ratio"] >= name_thr) &
        (scored_candidates["norm_address_ratio"] >= addr_thr)
    ].copy()

    counts = x.groupby("entity_id_s1").size()

    unique = x[
        x["entity_id_s1"].map(counts) == 1
    ]

    print(
        f"name >= {name_thr}, "
        f"address >= {addr_thr}"
    )
    print("  candidates:", len(x))
    print("  S1:", x["entity_id_s1"].nunique())
    print("  unique S1:", unique["entity_id_s1"].nunique())
    print("  ambiguous S1:", (counts > 1).sum())
    print()

for n, a in [
    (90, 95),
    (92, 95),
    (94, 95),
    (95, 95),
    (96, 95),
    (97, 95),
    (98, 95),
    (95, 96),
    (95, 97),
    (96, 97),
    (97, 97),
    (98, 97),
    (99, 97),
    (95, 98),
    (97, 98),
    (98, 98),
]:
    analyze_threshold(n, a)

name >= 90, address >= 95
  candidates: 9751
  S1: 8792
  unique S1: 7908
  ambiguous S1: 884

name >= 92, address >= 95
  candidates: 6117
  S1: 5504
  unique S1: 4932
  ambiguous S1: 572

name >= 94, address >= 95
  candidates: 2053
  S1: 1704
  unique S1: 1376
  ambiguous S1: 328

name >= 95, address >= 95
  candidates: 1401
  S1: 1100
  unique S1: 815
  ambiguous S1: 285

name >= 96, address >= 95
  candidates: 1280
  S1: 1040
  unique S1: 813
  ambiguous S1: 227

name >= 97, address >= 95
  candidates: 1156
  S1: 964
  unique S1: 782
  ambiguous S1: 182

name >= 98, address >= 95
  candidates: 1107
  S1: 931
  unique S1: 765
  ambiguous S1: 166

name >= 95, address >= 96
  candidates: 1262
  S1: 984
  unique S1: 721
  ambiguous S1: 263

name >= 95, address >= 97
  candidates: 992
  S1: 745
  unique S1: 512
  ambiguous S1: 233

name >= 96, address >= 97
  candidates: 909
  S1: 707
  unique S1: 517
  ambiguous S1: 190

name >= 97, address >= 97
  candidates: 825
  S1: 658
  unique S

In [9]:
# ============================================
# STEP 5 — INSPECT STRONG UNIQUE MATCHES
# ============================================

x = scored_candidates[
    (scored_candidates["name_ratio"] >= 98) &
    (scored_candidates["norm_address_ratio"] >= 98)
].copy()

# Candidate count per S1
counts = x.groupby("entity_id_s1").size()

# Keep only exactly one candidate per S1
strong_unique = x[
    x["entity_id_s1"].map(counts) == 1
].copy()

# Sort weakest first
strong_unique = strong_unique.sort_values(
    ["name_ratio", "norm_address_ratio"]
)

print("Strong unique candidates:", len(strong_unique))
print(
    "Name range:",
    strong_unique["name_ratio"].min(),
    "to",
    strong_unique["name_ratio"].max()
)
print(
    "Address range:",
    strong_unique["norm_address_ratio"].min(),
    "to",
    strong_unique["norm_address_ratio"].max()
)

print("\nWeakest 50:")
print(
    strong_unique[
        [
            "entity_id_s1",
            "entity_id_s2",
            "name_ratio",
            "norm_address_ratio",
            "s1_name",
            "s2_name",
            "s1_address",
            "s2_address"
        ]
    ].head(50).to_string(index=False)
)

Strong unique candidates: 435
Name range: 98.18181818181819 to 100.0
Address range: 98.0392156862745 to 100.0

Weakest 50:
entity_id_s1 entity_id_s2  name_ratio  norm_address_ratio                                      s1_name                                      s2_name                                 s1_address                             s2_address
S1-812796473 S2-742872634   98.181818          100.000000                 Cristy Gagnon Rothschild Inc                  CRISTY GGNON ROTHSCHILD-INC             470 Joye Street, Millcreek, UT             470 JOYE ST, MILLCREEK, UT
S1-515064429 S2-949457553   98.245614           98.181818                 Optimal Crystal Newtekone LP                Optimal Crystal Newtebkone LP           19118 Bickham Drive, Cypress, TX      19118 BICKHAM DRIVE, CYPPRESS, TX
S1-137279962 S2-854238614   98.245614           98.412698                Green Creative Solutions, Inc               GREEN CRSEATIVE SOLUTIONS, INC     3488 Whisnant Street, Iron Station,

In [10]:
# ============================================
# STEP 6 — FIRST HOUSE NUMBER CHECK
# ============================================

def first_address_number(text):
    if pd.isna(text):
        return None

    nums = re.findall(r"\d+", str(text))

    if not nums:
        return None

    return str(int(nums[0]))

strong_unique["first_num_s1"] = (
    strong_unique["s1_address"].map(first_address_number)
)

strong_unique["first_num_s2"] = (
    strong_unique["s2_address"].map(first_address_number)
)

strong_unique["first_num_match"] = (
    strong_unique["first_num_s1"].notna() &
    strong_unique["first_num_s2"].notna() &
    (
        strong_unique["first_num_s1"]
        == strong_unique["first_num_s2"]
    )
)

print("Total 98/98 unique:", len(strong_unique))
print()
print(strong_unique["first_num_match"].value_counts())
print()
print(
    "First-number matches:",
    strong_unique["first_num_match"].sum()
)
print(
    "First-number mismatches:",
    (~strong_unique["first_num_match"]).sum()
)

Total 98/98 unique: 435

first_num_match
True     434
False      1
Name: count, dtype: int64

First-number matches: 434
First-number mismatches: 1


In [11]:
print(
    strong_unique.loc[
        ~strong_unique["first_num_match"],
        [
            "entity_id_s1",
            "entity_id_s2",
            "name_ratio",
            "norm_address_ratio",
            "s1_name",
            "s2_name",
            "s1_address",
            "s2_address",
            "first_num_s1",
            "first_num_s2"
        ]
    ].to_string(index=False)
)

entity_id_s1 entity_id_s2  name_ratio  norm_address_ratio             s1_name              s2_name                      s1_address                  s2_address first_num_s1 first_num_s2
  S1-1277906 S2-585715788       100.0           98.039216 Garcia Manulife Inc Garcia Manulife Inc. 38970 360th Avenue, Fosston, MN 3897 360TH AVE, FOSSTON, MN        38970         3897


In [12]:
# ============================================
# CHECK 434 HIGH-CONFIDENCE PAIRS ARE NEW
# ============================================

strong_final = strong_unique[
    strong_unique["first_num_match"]
].copy()

print("High-confidence pairs:", len(strong_final))
print(
    "Unique S1:",
    strong_final["entity_id_s1"].nunique()
)

# Build existing pair lookup from the ORIGINAL 0.228 submission
original = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

original["matched_entity_ids"] = (
    original["matched_entity_ids"]
    .fillna("")
    .replace("nan", "")
    .astype(str)
    .str.strip()
)

existing_pairs = set()

for s1_id, ids in zip(
    original["source1_entity_id"],
    original["matched_entity_ids"]
):
    if not ids:
        continue

    for matched_id in ids.split(","):
        matched_id = matched_id.strip()
        if matched_id:
            existing_pairs.add((s1_id, matched_id))

strong_final["pair"] = list(
    zip(
        strong_final["entity_id_s1"],
        strong_final["entity_id_s2"]
    )
)

strong_final["already_existing"] = (
    strong_final["pair"].isin(existing_pairs)
)

print("\nAlready existing:")
print(strong_final["already_existing"].value_counts())

print(
    "\nNew pairs:",
    (~strong_final["already_existing"]).sum()
)

High-confidence pairs: 434
Unique S1: 434

Already existing:
already_existing
False    434
Name: count, dtype: int64

New pairs: 434


In [13]:
# ============================================
# CHECK 434 PAIRS AGAINST CANDIDATE FILE
# ============================================

target_pairs = set(
    zip(
        strong_final["entity_id_s1"].astype(str),
        strong_final["entity_id_s2"].astype(str)
    )
)

found_pairs = set()

for chunk in pd.read_csv(
    "/content/candidate_pairs.tsv",
    sep="\t",
    dtype=str,
    chunksize=1_000_000
):
    chunk = chunk[
        chunk["source"].astype(str) == "S2"
    ]

    for s1_id, s2_id in zip(
        chunk["entity_id_s1"],
        chunk["entity_id_matched"]
    ):
        pair = (s1_id, s2_id)

        if pair in target_pairs:
            found_pairs.add(pair)

print("Pairs to add:", len(target_pairs))
print("Found in candidate file:", len(found_pairs))
print("Missing:", len(target_pairs - found_pairs))

if target_pairs == found_pairs:
    print("\nALL 434 PAIRS ARE COVERED")
else:
    print("\nWARNING: SOME PAIRS ARE MISSING")
    print(list(target_pairs - found_pairs)[:20])

Pairs to add: 434
Found in candidate file: 434
Missing: 0

ALL 434 PAIRS ARE COVERED


In [14]:
# ============================================
# BUILD FINAL SUBMISSION: +434 HIGH-CONFIDENCE
# ============================================

# Reload clean 0.228 baseline
original = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

original["matched_entity_ids"] = (
    original["matched_entity_ids"]
    .fillna("")
    .replace("nan", "")
    .astype(str)
    .str.strip()
)

# Build lookup for exactly the 434 new pairs
new_lookup = {}

for _, row in strong_final.iterrows():
    s1 = row["entity_id_s1"]
    s2 = row["entity_id_s2"]

    new_lookup.setdefault(s1, []).append(s2)


def add_434(row):
    s1 = row["source1_entity_id"]
    existing = row["matched_entity_ids"]

    additions = new_lookup.get(s1, [])

    if not additions:
        return existing

    existing_ids = [
        x.strip()
        for x in existing.split(",")
        if x.strip()
    ]

    for x in additions:
        if x not in existing_ids:
            existing_ids.append(x)

    return ",".join(existing_ids)


final_434 = original.copy()

final_434["matched_entity_ids"] = final_434.apply(
    add_434,
    axis=1
)

# Save
final_path = "/content/matching_results_434_FINAL.tsv"

final_434.to_csv(
    final_path,
    sep="\t",
    index=False
)

# ============================================
# VERIFY
# ============================================

has_match = final_434["matched_entity_ids"].ne("")

total_pairs = (
    final_434["matched_entity_ids"]
    .str.split(",")
    .explode()
    .str.strip()
    .loc[lambda x: x.ne("")]
    .shape[0]
)

print("Saved:", final_path)
print()
print("Rows:", len(final_434))
print("Matched S1:", has_match.sum())
print("No-match S1:", (~has_match).sum())
print("Total pairs:", total_pairs)
print("Duplicate S1:",
      final_434["source1_entity_id"].duplicated().sum())
print("Literal nan:",
      (final_434["matched_entity_ids"] == "nan").sum())

Saved: /content/matching_results_434_FINAL.tsv

Rows: 1732544
Matched S1: 426277
No-match S1: 1306267
Total pairs: 564903
Duplicate S1: 0
Literal nan: 0


In [15]:
from google.colab import files

files.download("/content/matching_results_434_FINAL.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
# ============================================
# S3 FINAL EXPERIMENT — STEP 1
# STREAM ZERO-MATCH S3 CANDIDATES
# ============================================

import pandas as pd
import os

# Load current 0.228 baseline
baseline = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

baseline["matched_entity_ids"] = (
    baseline["matched_entity_ids"]
    .fillna("")
    .replace("nan", "")
    .astype(str)
    .str.strip()
)

zero_match_s1 = set(
    baseline.loc[
        baseline["matched_entity_ids"].eq(""),
        "source1_entity_id"
    ]
)

print("Zero-match S1:", len(zero_match_s1))

# Existing S3 address candidates
candidate_path = "/content/test_s3_address_candidates.tsv"
filtered_path = "/content/zero_s3_candidates.tsv"

if os.path.exists(filtered_path):
    os.remove(filtered_path)

total_kept = 0
first_write = True

for chunk in pd.read_csv(
    candidate_path,
    sep="\t",
    dtype=str,
    chunksize=200_000
):
    part = chunk[
        chunk["entity_id_s1"].isin(zero_match_s1)
    ]

    if len(part):
        part.to_csv(
            filtered_path,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )

        first_write = False
        total_kept += len(part)

print("Zero-match S3 candidates:", total_kept)
print("Saved:", filtered_path)

Zero-match S1: 1306701
Zero-match S3 candidates: 892384
Saved: /content/zero_s3_candidates.tsv


In [17]:
# ============================================
# S3 EXPERIMENT — STEP 2
# EXTRACT ONLY NEEDED RECORDS
# ============================================

import pandas as pd
import os

# Get required IDs without loading another giant source
needed_s1 = set()
needed_s3 = set()

for chunk in pd.read_csv(
    "/content/zero_s3_candidates.tsv",
    sep="\t",
    dtype=str,
    chunksize=200_000
):
    needed_s1.update(chunk["entity_id_s1"])
    needed_s3.update(chunk["entity_id_matched"])

print("Needed S1:", len(needed_s1))
print("Needed S3:", len(needed_s3))

# --------------------------------------------
# Extract S1
# --------------------------------------------

s1_path = "/content/zero_s3_needed_s1.tsv"

if os.path.exists(s1_path):
    os.remove(s1_path)

first_write = True
s1_count = 0

for chunk in pd.read_csv(
    "/content/test_source1.tsv",
    sep="\t",
    dtype=str,
    chunksize=200_000
):
    part = chunk[
        chunk["entity_id"].isin(needed_s1)
    ]

    if len(part):
        part.to_csv(
            s1_path,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )
        first_write = False
        s1_count += len(part)

print("S1 records extracted:", s1_count)

# --------------------------------------------
# Extract S3
# --------------------------------------------

s3_path = "/content/zero_s3_needed_s3.tsv"

if os.path.exists(s3_path):
    os.remove(s3_path)

first_write = True
s3_count = 0

for chunk in pd.read_csv(
    "/content/s3_address_lookup.tsv",
    sep="\t",
    dtype=str,
    chunksize=200_000
):
    part = chunk[
        chunk["entity_id"].isin(needed_s3)
    ]

    if len(part):
        part.to_csv(
            s3_path,
            sep="\t",
            index=False,
            mode="w" if first_write else "a",
            header=first_write
        )
        first_write = False
        s3_count += len(part)

print("S3 records extracted:", s3_count)

print("\nFiles:")
print(s1_path)
print(s3_path)

Needed S1: 262820
Needed S3: 555564
S1 records extracted: 262820
S3 records extracted: 555564

Files:
/content/zero_s3_needed_s1.tsv
/content/zero_s3_needed_s3.tsv


In [18]:
# ============================================
# S3 EXPERIMENT — STEP 3
# STREAMING FUZZY SCORING
# ============================================

import pandas as pd
import re
import unicodedata
from rapidfuzz.fuzz import ratio

def norm_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def norm_address(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        ch for ch in text
        if not unicodedata.combining(ch)
    )

    replacements = {
        "road": "rd",
        "street": "st",
        "avenue": "ave",
        "boulevard": "blvd",
        "drive": "dr",
        "lane": "ln",
        "place": "pl",
        "parkway": "pkwy",
        "highway": "hwy",
        "apartment": "apt",
        "floor": "fl",
        "number": "no",
    }

    text = re.sub(r"[^a-z0-9]+", " ", text)
    tokens = text.split()
    tokens = [replacements.get(t, t) for t in tokens]

    return " ".join(tokens)


# Load only the extracted records
s1 = pd.read_csv(
    "/content/zero_s3_needed_s1.tsv",
    sep="\t",
    dtype=str
)

s3 = pd.read_csv(
    "/content/zero_s3_needed_s3.tsv",
    sep="\t",
    dtype=str
)

print("S1:", len(s1))
print("S3:", len(s3))

# Normalize once
s1["name_norm"] = s1["business_name"].map(norm_text)
s1["addr_norm"] = s1["business_address"].map(norm_address)

s3["name_norm"] = s3["business_name"].map(norm_text)
s3["addr_norm"] = s3["business_address"].map(norm_address)

# Dictionaries
s1_map = s1.set_index("entity_id")[
    [
        "business_name",
        "business_address",
        "country",
        "name_norm",
        "addr_norm"
    ]
].to_dict("index")

s3_map = s3.set_index("entity_id")[
    [
        "business_name",
        "business_address",
        "country",
        "name_norm",
        "addr_norm"
    ]
].to_dict("index")

print("Normalization complete.")

# Score candidates in chunks
results = []
processed = 0

for chunk in pd.read_csv(
    "/content/zero_s3_candidates.tsv",
    sep="\t",
    dtype=str,
    chunksize=100_000
):

    for s1_id, s3_id in zip(
        chunk["entity_id_s1"],
        chunk["entity_id_matched"]
    ):

        a = s1_map.get(s1_id)
        b = s3_map.get(s3_id)

        if a is None or b is None:
            continue

        name_score = ratio(
            a["name_norm"],
            b["name_norm"]
        )

        # Initial strict screen
        if name_score < 90:
            continue

        addr_score = ratio(
            a["addr_norm"],
            b["addr_norm"]
        )

        if addr_score < 95:
            continue

        results.append({
            "entity_id_s1": s1_id,
            "entity_id_s3": s3_id,
            "name_ratio": name_score,
            "norm_address_ratio": addr_score,
            "s1_name": a["business_name"],
            "s3_name": b["business_name"],
            "s1_address": a["business_address"],
            "s3_address": b["business_address"],
            "country": a["country"]
        })

    processed += len(chunk)

    if processed % 500_000 == 0:
        print("Processed:", processed)

s3_scored = pd.DataFrame(results)

print("\n============================================")
print("S3 SCORING COMPLETE")
print("============================================")
print("Candidates processed:", processed)
print("High-confidence candidates:", len(s3_scored))

if len(s3_scored):
    print(
        "Unique S1:",
        s3_scored["entity_id_s1"].nunique()
    )

S1: 262820
S3: 555564
Normalization complete.
Processed: 500000

S3 SCORING COMPLETE
Candidates processed: 892384
High-confidence candidates: 4451
Unique S1: 4087


In [19]:
# ============================================
# S3 STEP 4 — STRICT UNIQUE MATCHES
# ============================================

x = s3_scored[
    (s3_scored["name_ratio"] >= 98) &
    (s3_scored["norm_address_ratio"] >= 98)
].copy()

counts = x.groupby("entity_id_s1").size()

s3_strong_unique = x[
    x["entity_id_s1"].map(counts) == 1
].copy()

print("98/98 candidates:", len(x))
print("S1:", x["entity_id_s1"].nunique())
print("Unique S1:", len(s3_strong_unique))
print("Ambiguous S1:", (counts > 1).sum())

print("\nScore ranges:")
print(
    "Name:",
    s3_strong_unique["name_ratio"].min(),
    "to",
    s3_strong_unique["name_ratio"].max()
)
print(
    "Address:",
    s3_strong_unique["norm_address_ratio"].min(),
    "to",
    s3_strong_unique["norm_address_ratio"].max()
)

print("\nWeakest 30:")
print(
    s3_strong_unique.sort_values(
        ["name_ratio", "norm_address_ratio"]
    )[
        [
            "entity_id_s1",
            "entity_id_s3",
            "name_ratio",
            "norm_address_ratio",
            "s1_name",
            "s3_name",
            "s1_address",
            "s3_address"
        ]
    ].head(30).to_string(index=False)
)

98/98 candidates: 0
S1: 0
Unique S1: 0
Ambiguous S1: 0

Score ranges:
Name: nan to nan
Address: nan to nan

Weakest 30:
Empty DataFrame
Columns: [entity_id_s1, entity_id_s3, name_ratio, norm_address_ratio, s1_name, s3_name, s1_address, s3_address]
Index: []


In [20]:
# ============================================
# S3 — FIND THE STRONGEST AVAILABLE TIERS
# ============================================

def analyze_s3(name_thr, addr_thr):
    x = s3_scored[
        (s3_scored["name_ratio"] >= name_thr) &
        (s3_scored["norm_address_ratio"] >= addr_thr)
    ].copy()

    if len(x) == 0:
        print(f"{name_thr}/{addr_thr}: 0")
        return

    counts = x.groupby("entity_id_s1").size()

    unique = x[
        x["entity_id_s1"].map(counts) == 1
    ]

    print(
        f"{name_thr}/{addr_thr}: "
        f"candidates={len(x)}, "
        f"S1={x['entity_id_s1'].nunique()}, "
        f"unique={len(unique)}, "
        f"ambiguous={(counts > 1).sum()}"
    )


for n, a in [
    (97, 97),
    (96, 97),
    (95, 97),
    (97, 96),
    (96, 96),
    (95, 96),
    (94, 97),
    (94, 96),
    (93, 97),
    (93, 96),
    (92, 97),
]:
    analyze_s3(n, a)

97/97: candidates=28, S1=22, unique=16, ambiguous=6
96/97: candidates=37, S1=29, unique=21, ambiguous=8
95/97: candidates=48, S1=34, unique=20, ambiguous=14
97/96: candidates=357, S1=323, unique=289, ambiguous=34
96/96: candidates=385, S1=346, unique=307, ambiguous=39
95/96: candidates=425, S1=370, unique=318, ambiguous=52
94/97: candidates=215, S1=199, unique=183, ambiguous=16
94/96: candidates=779, S1=706, unique=638, ambiguous=68
93/97: candidates=460, S1=440, unique=421, ambiguous=19
93/96: candidates=1303, S1=1202, unique=1109, ambiguous=93
92/97: candidates=739, S1=709, unique=681, ambiguous=28


In [21]:
# ============================================
# S3 — 97/96 UNIQUE + FIRST NUMBER CHECK
# ============================================

s3_x = s3_scored[
    (s3_scored["name_ratio"] >= 97) &
    (s3_scored["norm_address_ratio"] >= 96)
].copy()

counts = s3_x.groupby("entity_id_s1").size()

s3_unique = s3_x[
    s3_x["entity_id_s1"].map(counts) == 1
].copy()


def first_address_number(text):
    if pd.isna(text):
        return None

    nums = re.findall(r"\d+", str(text))

    if not nums:
        return None

    return str(int(nums[0]))


s3_unique["first_num_s1"] = (
    s3_unique["s1_address"].map(first_address_number)
)

s3_unique["first_num_s3"] = (
    s3_unique["s3_address"].map(first_address_number)
)

s3_unique["first_num_match"] = (
    s3_unique["first_num_s1"].notna() &
    s3_unique["first_num_s3"].notna() &
    (
        s3_unique["first_num_s1"]
        == s3_unique["first_num_s3"]
    )
)

print("97/96 unique:", len(s3_unique))
print()
print(s3_unique["first_num_match"].value_counts())
print()
print(
    "First-number matches:",
    s3_unique["first_num_match"].sum()
)
print(
    "First-number mismatches:",
    (~s3_unique["first_num_match"]).sum()
)

print("\nMismatches:")
print(
    s3_unique.loc[
        ~s3_unique["first_num_match"],
        [
            "entity_id_s1",
            "entity_id_s3",
            "name_ratio",
            "norm_address_ratio",
            "s1_name",
            "s3_name",
            "s1_address",
            "s3_address",
            "first_num_s1",
            "first_num_s3"
        ]
    ].to_string(index=False)
)

97/96 unique: 289

first_num_match
True    289
Name: count, dtype: int64

First-number matches: 289
First-number mismatches: 0

Mismatches:
Empty DataFrame
Columns: [entity_id_s1, entity_id_s3, name_ratio, norm_address_ratio, s1_name, s3_name, s1_address, s3_address, first_num_s1, first_num_s3]
Index: []


In [22]:
# ============================================
# S3 FINAL CHECKS — NEW + CANDIDATE COVERAGE
# ============================================

# Keep only the 289 high-confidence S3 pairs
s3_final = s3_unique[
    s3_unique["first_num_match"]
].copy()

print("S3 pairs:", len(s3_final))
print("Unique S1:", s3_final["entity_id_s1"].nunique())

# --------------------------------------------
# CHECK 1 — NEW RELATIVE TO 0.228 BASELINE
# --------------------------------------------

original = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

original["matched_entity_ids"] = (
    original["matched_entity_ids"]
    .fillna("")
    .replace("nan", "")
    .astype(str)
    .str.strip()
)

existing_pairs = set()

for s1_id, ids in zip(
    original["source1_entity_id"],
    original["matched_entity_ids"]
):
    if not ids:
        continue

    for matched_id in ids.split(","):
        matched_id = matched_id.strip()

        if matched_id:
            existing_pairs.add(
                (s1_id, matched_id)
            )

s3_final["pair"] = list(
    zip(
        s3_final["entity_id_s1"],
        s3_final["entity_id_s3"]
    )
)

s3_final["already_existing"] = (
    s3_final["pair"].isin(existing_pairs)
)

print("\nAlready existing:")
print(
    s3_final["already_existing"].value_counts()
)

new_s3 = s3_final[
    ~s3_final["already_existing"]
].copy()

print("\nNew S3 pairs:", len(new_s3))

# --------------------------------------------
# CHECK 2 — CANDIDATE COVERAGE
# --------------------------------------------

target_pairs = set(
    zip(
        new_s3["entity_id_s1"].astype(str),
        new_s3["entity_id_s3"].astype(str)
    )
)

found_pairs = set()

for chunk in pd.read_csv(
    "/content/candidate_pairs.tsv",
    sep="\t",
    dtype=str,
    chunksize=1_000_000
):
    chunk = chunk[
        chunk["source"].astype(str) == "S3"
    ]

    for s1_id, s3_id in zip(
        chunk["entity_id_s1"],
        chunk["entity_id_matched"]
    ):
        pair = (s1_id, s3_id)

        if pair in target_pairs:
            found_pairs.add(pair)

print("\nPairs to add:", len(target_pairs))
print("Found in candidate file:", len(found_pairs))
print("Missing:", len(target_pairs - found_pairs))

if target_pairs == found_pairs:
    print("\nALL S3 PAIRS ARE NEW AND COVERED")
else:
    print("\nWARNING: SOME S3 PAIRS FAILED COVERAGE")

S3 pairs: 289
Unique S1: 289

Already existing:
already_existing
False    289
Name: count, dtype: int64

New S3 pairs: 289

Pairs to add: 289
Found in candidate file: 289
Missing: 0

ALL S3 PAIRS ARE NEW AND COVERED


In [23]:
# ============================================
# BUILD FINAL: 0.228 BASELINE + 289 S3 PAIRS
# ============================================

# Reload clean 0.228 baseline
original = pd.read_csv(
    "/content/matching_results.tsv",
    sep="\t",
    dtype=str
)

original["matched_entity_ids"] = (
    original["matched_entity_ids"]
    .fillna("")
    .replace("nan", "")
    .astype(str)
    .str.strip()
)

# Build lookup for the 289 S3 pairs
new_lookup = {}

for _, row in new_s3.iterrows():
    s1 = row["entity_id_s1"]
    s3 = row["entity_id_s3"]

    new_lookup.setdefault(s1, []).append(s3)


def add_s3(row):
    s1 = row["source1_entity_id"]
    existing = row["matched_entity_ids"]

    additions = new_lookup.get(s1, [])

    if not additions:
        return existing

    existing_ids = [
        x.strip()
        for x in existing.split(",")
        if x.strip()
    ]

    for x in additions:
        if x not in existing_ids:
            existing_ids.append(x)

    return ",".join(existing_ids)


final_s3 = original.copy()

final_s3["matched_entity_ids"] = final_s3.apply(
    add_s3,
    axis=1
)

# Save
final_path = "/content/matching_results_S3_289_FINAL.tsv"

final_s3.to_csv(
    final_path,
    sep="\t",
    index=False
)

# ============================================
# VERIFY
# ============================================

has_match = final_s3["matched_entity_ids"].ne("")

total_pairs = (
    final_s3["matched_entity_ids"]
    .str.split(",")
    .explode()
    .str.strip()
    .loc[lambda x: x.ne("")]
    .shape[0]
)

print("Saved:", final_path)
print()
print("Rows:", len(final_s3))
print("Matched S1:", has_match.sum())
print("No-match S1:", (~has_match).sum())
print("Total pairs:", total_pairs)
print("Duplicate S1:",
      final_s3["source1_entity_id"].duplicated().sum())
print("Literal nan:",
      (final_s3["matched_entity_ids"] == "nan").sum())

Saved: /content/matching_results_S3_289_FINAL.tsv

Rows: 1732544
Matched S1: 426132
No-match S1: 1306412
Total pairs: 564758
Duplicate S1: 0
Literal nan: 0


In [24]:
from google.colab import files

files.download("/content/matching_results_S3_289_FINAL.tsv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>